# GNN Powerflow V2.8 — UQ Analysis

<!-- [STSI 231026]:new notebook (task 105) - UQ analysis skeleton + origin of the UQ common library -->

**Provenance (task 105, 2026-10-23).**
- Cells `1`–`30` are **verbatim copies** of `GNN_Powerflow_V2.7_Analysis.ipynb` cells `1`–`30` (same cell ids, outputs cleared). They are the library only (model, dataset with `bfs_path_matrix`, predict / evaluate, baselines, `check_hparam_results`); no analysis runs. Byte-parity is checked by `test_uq_notebook_parity.py`. Fix shared code in V2.7 Analysis, then re-copy.
- Cell `uq-split-helpers` holds function-level copies of the split helpers of the canonical `GNN_Powerflow_V2.7_Training.ipynb` (each preceded by a `# [COPY of GNN_Powerflow_V2.7_Training::<name>]` marker; checked by `test_uq_copy_parity.py`).
- Cell `uq-common` is the **origin** (ground truth) of the UQ common library: the P6 flat-row prediction contract, the θ path matrix, the streaming eval-mode forward pass, Gaussian / ensemble score helpers, the interval score, the latency timer and the split assertion, plus the [PROP-2] line-flow helpers. Trainer notebooks copy individual definitions with `# [COPY of GNN_Powerflow_V2.8_UQ_Analysis::<name>]`.

**Section plan** (the sections are added in tasks 109 (harness, CAL-2, run config) and 110 (sections (a)–(e))):
- **(a) Original** — the existing deterministic models, incl. the published `check_hparam_results(external_test_set=True)` view.
- **(b) UQ-A1** — two-stage mean + scale heads.
- **(c) UQ-B1** — deep ensemble.
- **(d) BLL C1/C2** — Bayesian last layer (post-hoc C1, variational C2).
- **(e) Cross-method** — coverage, sharpness, scores and latency side by side.

**Rules.**
- Only `unknown` coordinates are scored (PQ → Vmag, θ; PV → θ, Q; slack → P, Q). θ is in **rad** internally and in degrees in plots only.
- The cal split is used only by CAL-2.
- Line rows cover **lines only** (no transformers); loading = smax / s_limit_pu, NaN where unrated.

In [ ]:
# %%
import copy
import logging
import os
import pickle
import random
import time
from typing import Dict, List, Optional, Tuple
import json
from datetime import datetime
from dataclasses import dataclass, field, asdict

# Fix for numba >= 0.60: version_version attribute was removed; inject it before pandapower imports it
import sys
print("pandapower already imported:", "pandapower" in sys.modules)
try:
    import numba._version as _nv
    if not hasattr(_nv, 'version_version'):
        import numba as _n; _nv.version_version = _n.__version__
except ImportError:
    pass

%matplotlib inline
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import pypsa
import pypowsybl as pp
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch_geometric.data import Batch, Data, Dataset
from torch_geometric.nn import GATv2Conv, GCNConv, GraphConv, TransformerConv  # [STSI120526] Added TransformerConv import
from tqdm import tqdm

import urllib

import statsmodels.api as sm
from statsmodels.formula.api import ols


import numba, pandapower as pdp
from pandapower.converter import from_ppc
import pandapower.networks as pn

# Suppress verbose output from PyPSA and dependencies during data generation
for _log in ("pypsa", "pypsa.pf", "pypsa.components", "numexpr", "linopy", "fontTools", "fontTools.subset"):
    logging.getLogger(_log).setLevel(logging.WARNING)
logging.getLogger("numexpr").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

import re
from scipy import ndimage

In [ ]:
# ── Configurable paths ───────────────────────────────────────────────────────
# Set DATA_ROOT to wherever your data folders live.
# Use "." to keep everything relative to the repo (default).
# Example: DATA_ROOT = r"C:\Users\you\OneDrive\Data_PF_GNN"
import os as _os
DATA_ROOT             = r"C:\Users\STSI\OneDrive - USN\Data_PF_GNN"                                             # ← change this
GRID_MODEL_FILES      = _os.path.join(DATA_ROOT, "grid_model_files")    # static grid CSV/RAW files
TRAINING_NETWORKS_DIR = _os.path.join(DATA_ROOT, "training_networks_saved")  # dataset JSONs
TRAINING_RESULTS_DIR  = _os.path.join(DATA_ROOT, "training_results_saved")   # sweep result JSONs/CSVs


In [ ]:
#test and execution toggles
run_smoke_test=False # to run a smoke test on network generation
run_regression_test=False # to run a regression test on the GNN training loop (with very limited epochs and data)
run_powsybl_extract = False # extract data from powsybl for the first time, or re-extract after edits to the extraction code
run_cigre_extract = False # extract data from CIGRE for the first time, or re-extract after edits to the extraction code    
run_powsybl_extract = False
run_cigre_extract = False

In [ ]:
import itertools

# ── Style pools ──────────────────────────────────────────────────────────────
_LINE_STYLES  = ["-", "--", "-.", ":"]
_MARKERS      = ["", "o", "s", "^", "D", "v", "P", "X"]
_COLOR_CYCLE  = (plt.cm.tab20.colors[:20])  # [STSI 260526]:20 distinct colors instead of default 10


## Data Generation Functions
_Paste from V2.5.3 cells 20, 22, 23, 25, 26_

In [ ]:
#==================save and load genrated datasets========================


def load_dataset_list(index_path: str) -> list:
    """
    Reload a list of PyPSA networks saved by save_dataset_list.
    """
    import json

    with open(index_path, "r") as f:
        meta = json.load(f)

    save_dir = os.path.dirname(index_path) or "."
    networks: list[pypsa.Network] = []
    for nc_name in meta["files"]:
        nc_path = os.path.join(save_dir, nc_name)
        networks.append(pypsa.Network(nc_path))

    print(f"Loaded {len(networks)} networks from {index_path}")
    return networks



## Power System Helpers

## GNN Architecture

In [ ]:
from collections import deque  # [STSI 260526]:edge_delta BFS utilities

def precompute_bfs_order(edge_index_fwd, slack_idx, num_nodes):  # [STSI 260526]:BFS traversal for theta reconstruction
    """
    Compute BFS traversal from slack for θ reconstruction from Δθ.
    
    Returns:
        bfs_edge_idx:   [N-1] indices into forward edges — edges in BFS tree order
        bfs_signs:      [N-1] float tensor — +1 or -1
        bfs_node_order: [N-1] node indices in BFS visit order (excludes slack)
        bfs_parent:     [N-1] parent node index for each BFS child
    
    Convention: θ_child = θ_parent - sign * Δθ_edge
        sign = +1 when edge is parent→child (forward direction = from→to)
        sign = -1 when edge is child→parent (traverse against forward direction)
    """
    # constructing the adjecency matrix for BFS traversal, with edge indices and signs for forward/backward direction
    # use manual rather than graph extaction because we need to track edge indices and signs for both forward and backward traversal, which is not directly provided by PyG's utilities
    adj = [[] for _ in range(num_nodes)]
    n_edges = edge_index_fwd.size(1)
    for e in range(n_edges):
        u = edge_index_fwd[0, e].item()
        v = edge_index_fwd[1, e].item()
        adj[u].append((v, e, +1.0))
        adj[v].append((u, e, -1.0))
    
    visited = [False] * num_nodes
    visited[slack_idx] = True
    queue = deque([slack_idx])
    
    bfs_edge_idx = []
    bfs_signs = []
    bfs_node_order = []
    bfs_parent = []
    
    while queue:
        node = queue.popleft()
        for neighbor, edge_e, sign in adj[node]:
            if not visited[neighbor]:
                visited[neighbor] = True
                bfs_edge_idx.append(edge_e)
                bfs_signs.append(sign)
                bfs_node_order.append(neighbor)
                bfs_parent.append(node)
                queue.append(neighbor)
    
    assert len(bfs_node_order) == num_nodes - 1, (
        f"BFS did not reach all nodes: visited {len(bfs_node_order)+1}/{num_nodes}"
    )
    
    return (
        torch.tensor(bfs_edge_idx, dtype=torch.long),
        torch.tensor(bfs_signs, dtype=torch.float32),
        torch.tensor(bfs_node_order, dtype=torch.long),
        torch.tensor(bfs_parent, dtype=torch.long),
    )


def reconstruct_theta_from_delta(delta_theta_fwd, bfs_edge_idx, bfs_signs,  # [STSI 260526]:theta reconstruction from edge deltas
                                  bfs_node_order, bfs_parent, num_nodes):
    """
    Reconstruct absolute θ from edge Δθ predictions using pre-computed BFS order.
    Differentiable w.r.t. delta_theta_fwd (chain of additions).
    
    Args:
        delta_theta_fwd: [E_fwd] predicted Δθ per forward edge
        bfs_edge_idx:    [N-1] edge indices in BFS traversal order
        bfs_signs:       [N-1] signs (+1/-1)
        bfs_node_order:  [N-1] child node indices in BFS visit order
        bfs_parent:      [N-1] parent node index for each child
        num_nodes:       int, total nodes
    
    Returns:
        theta: [N] reconstructed absolute angles (θ_slack = 0)
    """
    theta = torch.zeros(num_nodes, device=delta_theta_fwd.device, dtype=delta_theta_fwd.dtype)
    
    for i in range(len(bfs_node_order)):
        child = bfs_node_order[i].item()
        parent_node = bfs_parent[i].item()
        e = bfs_edge_idx[i]
        s = bfs_signs[i]
        theta[child] = theta[parent_node] - s * delta_theta_fwd[e]
    
    return theta


def reconstruct_theta_from_path_matrix(delta_theta_fwd, bfs_path_matrix):  # [STSI 240726]:vectorized θ recon via path-incidence matrix
    """
    Reconstruct absolute θ from edge Δθ using precomputed path-incidence matrix.
    Replaces the sequential BFS loop with a single matmul: θ = P @ Δθ.
    P[i, e] encodes the sign of edge e on the spanning-tree path from slack to node i.
    θ_slack = 0 (slack row of P is all zeros).

    For meshed networks, the result is BFS-tree-dependent (spanning-tree choice matters).
    delta_theta_mae is the tree-independent primary angle metric for edge_delta models.

    Args:
        delta_theta_fwd:  [E_fwd] tensor of predicted Δθ per forward edge
        bfs_path_matrix:  [N, E_fwd] precomputed path-incidence matrix (from Data)
    Returns:
        theta: [N] reconstructed absolute angles (θ_slack = 0)
    """
    return bfs_path_matrix.to(delta_theta_fwd.device) @ delta_theta_fwd


# =============================================================================
# SECTION 5: GNN MODEL
# Must be defined here so pickle can reconstruct saved PowerFlowGNN models.
# =============================================================================



def compute_bfs_depth(edge_index_fwd, slack_idx, num_nodes):  # [STSI 270526]:task 029 — hop distance utility
    """
    Compute BFS hop distance from slack bus for each node.
    Returns np.ndarray [num_nodes] with hop distances (slack=0, unreachable=-1).
    """
    from collections import deque
    adj = [[] for _ in range(num_nodes)]
    E = edge_index_fwd.shape[1]
    for e in range(E):
        u, v = int(edge_index_fwd[0, e]), int(edge_index_fwd[1, e])
        adj[u].append(v)
        adj[v].append(u)
    
    depth = np.full(num_nodes, -1, dtype=int)
    depth[slack_idx] = 0
    queue = deque([slack_idx])
    while queue:
        node = queue.popleft()
        for nbr in adj[node]:
            if depth[nbr] == -1:
                depth[nbr] = depth[node] + 1
                queue.append(nbr)
    return depth



# =============================================================================
# SECTION 5: GNN MODEL
# =============================================================================

class PowerFlowGNN(nn.Module):
    """
    Graph Attention Network (GATv2) for power flow prediction.
    Predicts per-bus: [vmag, vang, P, Q]
    Predicts per-edge bilinear PTDF: hedges @ W @ hnodes.T
    """

    def __init__(
        self,
        node_features: int = 7,
        edge_features: int = 4,
        hidden_dim: int = 64, # Hiddem dimensions means the size of the feature vectors that are passed between layers in the GNN. A larger hidden_dim allows the model to capture more complex relationships, but also increases computational cost and risk of overfitting. 64 is a common choice for a balance between expressiveness and efficiency.
        num_layers: int = 3,# Number of GNN layers (graph convolutional layers). More layers allow the model to capture more complex interactions between nodes, but can also lead to over-smoothing where node features become too similar. 3 layers is a common choice for many graph tasks.
        heads: int = 4,# Number of attention heads in GAT layers. Multiple heads allow the model to attend to different aspects of the graph structure simultaneously. 4 heads is a common choice that provides a good balance between expressiveness and computational cost.
        dropout: float = 0.0,
        conv_type: str = "gatv2", # Placeholder for potential future extension to other convolution types
        head_mode:str = "standard", # or with new encoder
        # ── Conv block flags (independent toggles, replace conv_mode) ── [STSI 120526]:refactored conv_mode→independent flags
        use_residual: bool = False,
        norm_type: str = None,         # None | "layer" | "graph"
        activation: str = "leaky_relu",  # "leaky_relu" | "gelu" | "relu" | "elu" | "silu"
        use_dropout: bool = False,
        drop_rate: float = 0.1,
        # ── Deprecated shorthand ──
        conv_mode: str = None,         # if set, overrides individual flags
        # ── Edge Δθ prediction ── [STSI 250526]: added angle_mode to control whether we predict node angles (standard) or edge angle differences (edge_delta)
        angle_mode: str = "node",      # "node" (default), "edge_delta", or "both"  # [STSI 020626]:added "both" for dual-head
        # ── V_mag residual ── [STSI 280526]: predict ΔV from flat baseline instead of absolute V_mag
        vmag_mode: str = "absolute",   # "absolute" (default) or "residual"
        # ── Global graph context ── [STSI 290526]: whole-graph awareness via mean+max pooling
        use_global_pool: bool = False,
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.heads = heads
        self.dropout = dropout
        self.conv_type = conv_type
        self.head_mode = head_mode
        self.angle_mode = angle_mode  # [STSI 250526]:new angle_mode option
        self.vmag_mode = vmag_mode    # [STSI 280526]:residual V_mag prediction
        self.use_global_pool = use_global_pool  # [STSI 290526]
        # ── Deprecated conv_mode shorthand ── [STSI 120526]:backward compat mapping
        if conv_mode is not None:
            _map = {
                "old":            (False, None,    "leaky_relu", False),
                "residual":       (True,  None,    "gelu",       False),
                "res_norm":       (True,  "layer", "gelu",       False),
                "res_norm_drop":  (True,  "layer", "gelu",       True),
            }
            use_residual, norm_type, activation, use_dropout = _map[conv_mode]

        self.use_residual = use_residual
        self.norm_type = norm_type
        self.use_dropout = use_dropout
        self.drop_rate = drop_rate
        self.activation_name = activation
        # 1. Node embedding
        self.node_embedding = nn.Linear(node_features, hidden_dim)

        # 2. GAT convolution layers
        self.convs = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(self._make_conv(hidden_dim, hidden_dim, edge_features))

        # 2b. Optional normalization + dropout for conv block [STSI 120526]:independent flags
        if self.norm_type is not None:
            if self.norm_type == "layer":
                self.norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in range(num_layers)])
            elif self.norm_type == "graph":
                from torch_geometric.nn.norm import GraphNorm
                self.norms = nn.ModuleList([GraphNorm(hidden_dim) for _ in range(num_layers)])
            else:
                raise ValueError(f"Unknown norm_type: {self.norm_type!r}")
        if self.use_dropout:
            self.drop = nn.Dropout(self.drop_rate)


        # 2c. Optional global pooling projection [STSI 290526]: graph-level context → per-node
        if self.use_global_pool:
            self.global_proj = nn.Linear(2 * hidden_dim, hidden_dim)

        # Activation function lookup [STSI 120526]:configurable activation
        _act_map = {
            "leaky_relu": F.leaky_relu,
            "gelu": F.gelu,
            "relu": F.relu,
            "elu": F.elu,
            "silu": F.silu,
        }
        if activation not in _act_map:
            raise ValueError(f"Unknown activation: {activation!r}")
        self.activation_fn = _act_map[activation]

        # 3. Output layers for different predictions
        if head_mode=="standard":
            if angle_mode == "edge_delta":  # [STSI 250526]:no angle head for node prediction
                self.vmag_pred = nn.Linear(hidden_dim, 1)
                self.p_pred = nn.Linear(hidden_dim, 1)
                self.q_pred = nn.Linear(hidden_dim, 1)
            else:
                # one prediction head for each of the properties we want to predict
                self.vmag_pred = nn.Linear(hidden_dim, 1)   # Voltage magnitude
                self.vang_pred = nn.Linear(hidden_dim, 1)   # Voltage angle
                self.p_pred = nn.Linear(hidden_dim, 1)      # Active power
                self.q_pred = nn.Linear(hidden_dim, 1)      # Reactive power
        if head_mode=="with_encoder":
            if angle_mode == "edge_delta":  
                self.pq_head = nn.Linear(hidden_dim, 1)    # PQ: predict Vmag only
                self.pv_head = nn.Linear(hidden_dim, 1)    # PV: predict Q only
                self.slack_head = nn.Linear(hidden_dim, 2) # Slack: predict P, Q
            else:
                self.pq_head = nn.Linear(hidden_dim, 2)      # for PQ nodes
                self.pv_head = nn.Linear(hidden_dim, 2)      # For PV nodes
                self.slack_head = nn.Linear(hidden_dim, 2)   # For Slack nodes

        # STSI 26.02.11: Edge embedding MLP: (h_src || h_dst || edge_attr) -> hidden_dim
        self.edge_mlp = nn.Sequential(
            nn.Linear(2 * hidden_dim + edge_features, hidden_dim),
            nn.LeakyReLU(),
        )

        # [STSI 250526]:edge angle head for edge_delta/both mode
        if angle_mode in ("edge_delta", "both"):  # [STSI 020626]:extended to support dual-head "both" mode
            self.edge_angle_pred = nn.Linear(hidden_dim, 1)

        # Bilinear PTDF parameter: h_edge^T W h_bus
        self.ptdf_W = nn.Parameter(torch.randn(hidden_dim, hidden_dim) * 0.01)
        
        self._initialize_weights()

    def _make_conv(self, in_dim: int, out_dim: int, edge_features: int):
        if self.conv_type == "gatv2":
            return GATv2Conv(
                in_dim,
                out_dim // self.heads,
                heads=self.heads,
                edge_dim=edge_features,
                concat=True,
                dropout=0.0, #self.dropout, # Dropout is handled separately in conv_mode, so set to 0 here to avoid double dropout when conv_mode="res_norm_drop"
            )
        elif self.conv_type == "transformer":  # [STSI120526] New conv_type option: dot-product attention, edge features in K+V, gated residual (beta=True)
            return TransformerConv(
                in_dim,
                out_dim // self.heads,
                heads=self.heads,
                edge_dim=edge_features,
                concat=True,
                dropout=0.0,
                beta=not self.use_residual,  # [STSI 120526]:avoid double residual: external skip OR internal gated, not both
            )
        elif self.conv_type == "gcn":   # [STSI120526] DEPRECATED — no edge_attr support
            return GCNConv(in_dim, out_dim, add_self_loops=False)
        elif self.conv_type == "graphconv":   # [STSI120526] DEPRECATED — no edge_attr support
            return GraphConv(in_dim, out_dim)
        else:
            raise ValueError(f"Unknown conv_type: {self.conv_type}")

    def _apply_conv(self, conv, h, edge_index, edge_attr):
        """Apply a single conv layer, dispatching on conv type."""
        if isinstance(conv, (GATv2Conv, TransformerConv)):  # [STSI120526] Added TransformerConv to edge_attr dispatch
            return conv(h, edge_index, edge_attr)
        else:
            return conv(h, edge_index)

    def forward(self, data, return_embeddings=False):
        """
        Forward pass: embed -> GAT layers -> node heads + bilinear PTDF.

        Args:
            data: PyG Data/Batch object
            return_embeddings: if True, return (node_pred, h_nodes, h_edges)
                               for use in PTDF loss outside forward
        Returns:
            (node_pred [num_nodes, 4], ptdf_pred [num_edges, num_nodes])
            or (node_pred, h_nodes, h_edges) if return_embeddings=True
        """
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr

        # Initial embedding
        h = self.node_embedding(x)

        # Conv layers — unified path with independent flags [STSI 120526]:replaced 4-way if/elif
        for i, conv in enumerate(self.convs):
            h_in = h
            h = self._apply_conv(conv, h, edge_index, edge_attr)
            if self.norm_type is not None:
                h = self.norms[i](h)
            h = self.activation_fn(h)
            if self.use_dropout:
                h = self.drop(h)
            if self.use_residual:
                h = h + h_in


        h_nodes = h  # Final node features after all GAT layers

        # Global pooling context injection [STSI 290526]: broadcast graph-level mean+max into per-node repr
        if self.use_global_pool:
            batch_idx = data.batch if hasattr(data, 'batch') and data.batch is not None else torch.zeros(
                h_nodes.size(0), dtype=torch.long, device=h_nodes.device)
            h_mean = global_mean_pool(h_nodes, batch_idx)   # [num_graphs, hidden_dim]
            h_max = global_max_pool(h_nodes, batch_idx)     # [num_graphs, hidden_dim]
            h_global = self.global_proj(torch.cat([h_mean, h_max], dim=-1))  # [num_graphs, hidden_dim]
            h_nodes = h_nodes + h_global[batch_idx]  # additive residual injection


        # Node heads
        # Predictions are made using the output of the last graph convolution layer

        if self.head_mode=="standard":
            if self.angle_mode == "edge_delta":  # [STSI 250526]: node pred is [N,3]
                vmag_pred = self.vmag_pred(h_nodes)
                if self.vmag_mode == "residual":  # [STSI 280526]
                    vmag_pred = vmag_pred + 1.0
                p_pred = self.p_pred(h_nodes)
                q_pred = self.q_pred(h_nodes)
                node_pred = torch.cat([vmag_pred, p_pred, q_pred], dim=-1)  # [num_nodes, 3]
            else:
                vmag_pred = self.vmag_pred(h_nodes)
                if self.vmag_mode == "residual":  # [STSI 280526]
                    vmag_pred = vmag_pred + 1.0
                vang_pred = self.vang_pred(h_nodes)
                p_pred = self.p_pred(h_nodes)
                q_pred = self.q_pred(h_nodes)
                node_pred = torch.cat([vmag_pred, vang_pred, p_pred, q_pred], dim=-1)  # [num_nodes, 4]
        elif self.head_mode=="with_encoder":
            pq_mask = data.pq_mask.bool()
            pv_mask = data.pv_mask.bool()
            slack_mask = data.slack_mask.bool()
            if self.angle_mode == "edge_delta":  # [STSI 250526]: [N,3]=[Vmag,P,Q]
                node_pred = torch.zeros(h_nodes.size(0), 3, device=h_nodes.device)
                if pq_mask.any():
                    raw_vmag = self.pq_head(h_nodes[pq_mask]).squeeze(-1)  # Vmag
                    if self.vmag_mode == "residual":  # [STSI 280526]
                        node_pred[pq_mask, 0] = raw_vmag + 1.0
                    else:
                        node_pred[pq_mask, 0] = raw_vmag
                if pv_mask.any():
                    node_pred[pv_mask, 2] = self.pv_head(h_nodes[pv_mask]).squeeze(-1)  # Q
                if slack_mask.any():
                    slack_out = self.slack_head(h_nodes[slack_mask])
                    node_pred[slack_mask, 1] = slack_out[:, 0]  # P
                    node_pred[slack_mask, 2] = slack_out[:, 1]  # Q
            else:
                node_pred = torch.zeros(h_nodes.size(0), 4, device=h_nodes.device)
                if pq_mask.any():
                    pq_out = self.pq_head(h_nodes[pq_mask])  # [N_pq, 2] = [Vmag, Vang]
                    if self.vmag_mode == "residual":  # [STSI 280526]
                        pq_out = pq_out.clone()
                        pq_out[:, 0] = pq_out[:, 0] + 1.0
                    node_pred[pq_mask, 0:2] = pq_out
                if pv_mask.any():
                    pv_out=self.pv_head(h_nodes[pv_mask])  # Predict Vmag, Vang for PV nodes
                    node_pred[pv_mask,1]=pv_out[:,0]  # Vmag from PV head
                    node_pred[pv_mask,3]=pv_out[:,1]  # Q from PV head
                if slack_mask.any():
                    slack_out=self.slack_head(h_nodes[slack_mask])  # Predict Vmag, Vang for Slack nodes
                    node_pred[slack_mask, 2] = slack_out[:, 0]  # P from Slack head
                    node_pred[slack_mask, 3] = slack_out[:, 1]  # Q from Slack head

        # Edge embeddings
        row, col = data.edge_index
        h_src = h_nodes[row]   # [num_edges, hidden_dim]
        h_dst = h_nodes[col]   # [num_edges, hidden_dim]
        edge_input = torch.cat([h_src, h_dst, edge_attr], dim=-1)  # [num_edges, 2*hidden_dim + edge_features]
        h_edges = self.edge_mlp(edge_input)  # [num_edges, hidden_dim]

        # [STSI 250526]:edge angle prediction for edge_delta mode
        delta_theta_pred = None
        if self.angle_mode in ("edge_delta", "both"):  # [STSI 020626]:dual-head produces Δθ in both modes
            # All forward edges (lines + transformers) — even indices
            all_fwd = torch.zeros(h_edges.size(0), dtype=torch.bool, device=h_edges.device)
            all_fwd[::2] = True
            h_edges_fwd = h_edges[all_fwd]    # [E_all_fwd, hidden_dim]
            delta_theta_pred = self.edge_angle_pred(h_edges_fwd).squeeze(-1)  # [E_fwd]

        if return_embeddings:
            return node_pred, h_nodes, h_edges, delta_theta_pred

        # Bilinear PTDF: H_W = h_edges @ W,  ptdf_pred = H_W @ h_nodes.T
        H_W = h_edges @ self.ptdf_W                   # [num_edges, hidden_dim]
        ptdf_pred = H_W @ h_nodes.T                   # [num_edges, num_nodes]
        return node_pred, ptdf_pred, delta_theta_pred

    def _initialize_weights(self, gain=1.0):
        """
        STSI 24.09.25: Enhanced weight initialization for the entire network.
        """
        # 1. Initialize node embedding layer (Added 24.09.25)
        nn.init.xavier_uniform_(self.node_embedding.weight, gain=gain)
        nn.init.zeros_(self.node_embedding.bias)

        # 2. Initialize GAT convolution layers (Added 24.09.25)
        for conv in self.convs:
            if hasattr(conv, "lin_l") and conv.lin_l is not None:
                nn.init.xavier_uniform_(conv.lin_l.weight, gain=gain)
            if hasattr(conv, "lin_r") and conv.lin_r is not None:
                nn.init.xavier_uniform_(conv.lin_r.weight, gain=gain)
            if hasattr(conv, "lin_edge") and conv.lin_edge is not None:
                nn.init.xavier_uniform_(conv.lin_edge.weight, gain=gain)
            # GAT layers have multiple linear transformations

        # 3. Initialize output layers
        _vmag_bias = 0.0 if self.vmag_mode == "residual" else 1.0  # [STSI 280526]
        if self.head_mode == "standard":
            if self.angle_mode == "edge_delta":  # [STSI 250526]no angle head, only Vmag, P, Q
                prediction_layers = [
                    (self.vmag_pred, _vmag_bias),
                    (self.p_pred,    0.0),
                    (self.q_pred,    0.0),
                ]
            else:
                prediction_layers = [
                    (self.vmag_pred, _vmag_bias),
                    (self.vang_pred, 0.0),
                    (self.p_pred,    0.0),
                    (self.q_pred,    0.0),
                ]
            for layer, bias_init in prediction_layers:
                nn.init.xavier_uniform_(layer.weight, gain=gain)
                nn.init.constant_(layer.bias, bias_init)
            # Zero-init vmag_pred weights so output starts exactly at bias [STSI 280526]
            nn.init.zeros_(self.vmag_pred.weight)
        elif self.head_mode == "with_encoder":
            for layer in (self.pq_head, self.pv_head, self.slack_head):
                nn.init.xavier_uniform_(layer.weight, gain=gain)
                nn.init.zeros_(layer.bias)
            # Vmag outputs (col 0 of pq_head) should start near 1.0 pu
            if self.angle_mode != "edge_delta":  # [STSI 250526]:only for 2-output pq_head
                nn.init.constant_(self.pq_head.bias[0], _vmag_bias)
            else:
                nn.init.constant_(self.pq_head.bias[0], _vmag_bias)  # single Vmag output
            nn.init.zeros_(self.pq_head.weight[0])  # Zero-init Vmag row [STSI 280526]

        # [STSI 250526]:initialize edge angle head  # [STSI 020626]:Fix — "both" also needs zero init
        if self.angle_mode in ("edge_delta", "both"):  # [STSI 020626]:was == "edge_delta" only
            nn.init.zeros_(self.edge_angle_pred.weight)
            nn.init.zeros_(self.edge_angle_pred.bias)

        # [STSI 290526]: init global pooling projection
        if self.use_global_pool:
            nn.init.xavier_uniform_(self.global_proj.weight)


# =============================================================================
# SECTION 6: PHYSICS-INFORMED LOSS
# =============================================================================



def compute_power_flow_residual_from_pred(
    pred, x, Y_matrix, network, bus_masks=None,
    use_q_partial_mode: bool = False,
    w_P: float = 0.5,  # [STSI 210526]: weight on P residual (guides v_ang)
    w_Q: float = 0.5,  # [STSI 210526]: weight on Q residual (guides v_mag)
):
    """
    Compute AC power-flow residual.

    use_q_partial_mode=False (default/old behaviour):
        All bus types contribute both P and Q residuals.
    use_q_partial_mode=True:
        PV buses contribute only P residual (Q is a free variable for them).
    """
    Y_real, Y_imag = Y_matrix
    n_nodes = pred.size(0)
    device  = pred.device

    if bus_masks is None:
        slack_mask = (x[:, 0] == 1.0)
        pv_mask    = (x[:, 1] == 1.0)
        pq_mask    = (x[:, 2] == 1.0)
    else:
        slack_mask, pv_mask, pq_mask = bus_masks

    # ── Assemble injections ─────────────────────────────────────
    p_inj = torch.zeros(n_nodes, device=device)
    q_inj = torch.zeros(n_nodes, device=device)

    p_inj[pq_mask]    = x[pq_mask, 3]       # known P at PQ buses
    q_inj[pq_mask]    = x[pq_mask, 4]       # known Q at PQ buses
    p_inj[pv_mask]    = x[pv_mask, 3]       # known P at PV buses
    q_inj[pv_mask]    = pred[pv_mask, 3]    # predicted Q at PV buses
    p_inj[slack_mask] = pred[slack_mask, 2] # predicted P at slack
    q_inj[slack_mask] = pred[slack_mask, 3] # predicted Q at slack

    # ── Assemble voltages ────────────────────────────────────────
    v_mag = torch.zeros(n_nodes, device=device)
    v_ang = torch.zeros(n_nodes, device=device) #

    v_mag[pq_mask]    = pred[pq_mask, 0]    # predicted Vmag at PQ
    v_ang[pq_mask]    = pred[pq_mask, 1]    # predicted Vang at PQ
    v_mag[pv_mask]    = x[pv_mask, 5]       # known Vmag at PV
    v_ang[pv_mask]    = pred[pv_mask, 1]    # predicted Vang at PV
    v_mag[slack_mask] = x[slack_mask, 5]    # known Vmag at slack
    v_ang[slack_mask] = x[slack_mask, 6]    # known Vang (= 0) at slack

    # ── AC power calculation via Y-bus ──────────────────────────
    v_real = v_mag * torch.cos(v_ang)
    v_imag = v_mag * torch.sin(v_ang)

    I_real = torch.matmul(Y_real, v_real) - torch.matmul(Y_imag, v_imag)
    I_imag = torch.matmul(Y_real, v_imag) + torch.matmul(Y_imag, v_real)

    p_calc = v_real * I_real + v_imag * I_imag
    q_calc = v_imag * I_real - v_real * I_imag

    # ── Residuals ────────────────────────────────────────────────
    b_diag = torch.diag(Y_imag).abs().clamp(min=1e-6)  # [STSI 210526]: avoid division by zero for isolated buses
    # [STSI 270526]: divide by b_diag to normalize by typical power flow magnitudes at each bus, avoiding domination of high-injection buses in the loss
    p_residual = ((p_calc - p_inj) / b_diag) ** 2

    if use_q_partial_mode:
        # exclude PV buses from Q residual — they are Q-free by definition
        q_mask = pq_mask | slack_mask
        q_residual = torch.zeros(n_nodes, device=device)
        q_residual[q_mask] = ((q_calc[q_mask] - q_inj[q_mask]) / b_diag[q_mask]) ** 2
    else:
        #[STSI 270526]: Divide by b_diag for all buses in Q residual as well, for consistent normalization across P and Q and to prevent domination by high-injection buses.
        q_residual = ((q_calc - q_inj) / b_diag) ** 2 # include all bus types in the Q residual calculation

    p_res_mean = p_residual.mean().detach()  # [STSI 210526]: diagnostic, no grad
    q_res_mean = q_residual.mean().detach()  # [STSI 210526]: diagnostic, no grad
    physics_loss = torch.mean(w_P * p_residual + w_Q * q_residual)
    return physics_loss, p_res_mean, q_res_mean  # [STSI 210526]: 3-tuple return


@dataclass
class PhysicsConfig:
    """
    Toggles for every physics guidance term.
    Designed to be swept as a hyperparameter in run_hparam_sweep.
    """
    w_phys: float = 0.0                     # static weight on physics loss (used in "fixed" mode)

    # Dynamic loss weighting [STSI 200526]
    loss_weight_mode: str = "fixed"         # "fixed" | "adaptive"
    fraction_physics: float = 0.1           # target: physics = 10% of MSE (adaptive only)
    fraction_ptdf: float = 0.1              # target: PTDF = 10% of MSE (adaptive only)
    max_w_phys: float = 100.0               # cap for adaptive physics weight
    max_w_ptdf: float = 100.0               # cap for adaptive PTDF weight

    # Variant selection
    physics_mode: str = "rich"            # reserved — "rich" only; "simple" removed 2026-05-21  # [STSI 210526]

    # Shared options
    use_power_balance: bool = True          # AC residual on P,Q
    use_angle_ref_penalty: bool = False     # slack angle mean^2

    # P/Q residual weights [STSI 210526]
    w_P: float = 0.5    # weight on P residual in physics loss (guides v_ang)
    w_Q: float = 0.5    # weight on Q residual in physics loss (guides v_mag)

    # Rich-mode options (full PV/slack handling, partial Q)
    use_q_partial_mode: bool = False        # exclude PV buses from Q residual
    
    # PTDF supervision
    use_ptdf_loss: bool = False    # DC PTDF head on/off
    weight_ptdf: float = 0.0      # weight applied to the PTDF loss term
    ptdf_loss_mode: str = "matrix" # "matrix", "flows", "mixed", or "learned_edge" (STSI 26.04.07)
    ptdf_alpha: float = 0.5       # mix ratio for mode="mixed" (matrix share)
    ptdf_branch_mode: str = "lines"  # branches in PTDF: "lines" or "all"
    ptdf_changepoint: int = None     # [STSI 290526] epoch offset (from ptdf_activation_start) to switch matrix→flows in "stepwise" mode

    # ── Per-sub-loss flow fractions (replace w_flow hierarchy) ── [STSI 040626]:per-sub-loss flow fractions synced with Training
    fraction_dcf_local: float = 0.0      # adaptive target for DC-local flow loss (0=disabled)
    fraction_dcf_global: float = 0.0     # adaptive target for DC-global flow loss (0=disabled)
    fraction_acf_local: float = 0.0      # adaptive target for AC-local flow loss (0=disabled)
    fraction_acf_global: float = 0.0     # adaptive target for AC-global flow loss (0=disabled)
    max_w_flow: float = 100.0            # cap for all flow sub-loss adaptive weights
    flow_target: str = "pq"              # "p" (P only), "q" (Q only), or "pq" (both, AC only)

    def label(self) -> str:
        # [STSI 200526] Distinguishes adaptive vs fixed mode in run labels
        if self.loss_weight_mode == "adaptive":
            parts = [f"adap-fp{self.fraction_physics}"]
            if self.use_ptdf_loss:
                parts.append(f"ft{self.fraction_ptdf}")
        else:
            parts = [f"w-{self.w_phys}"]
            if self.use_ptdf_loss:
                ptdf_part = f"PTDF-{self.ptdf_loss_mode}-pt{self.weight_ptdf}-{self.ptdf_branch_mode}"
                if self.ptdf_loss_mode == "mixed":
                    ptdf_part += f"-a{self.ptdf_alpha}"
                if self.ptdf_loss_mode == "stepwise" and self.ptdf_changepoint is not None:
                    ptdf_part += f"-cp{self.ptdf_changepoint}"
                parts.append(ptdf_part)
        if self.w_P != 0.5 or self.w_Q != 0.5:  # [STSI 210526]: non-default weights
            parts.append(f"wP{self.w_P}wQ{self.w_Q}")
        # [STSI 040626]:show active flow fractions in label
        _fl_parts = []
        if self.fraction_dcf_local > 0: _fl_parts.append(f"dcL{self.fraction_dcf_local}")
        if self.fraction_dcf_global > 0: _fl_parts.append(f"dcG{self.fraction_dcf_global}")
        if self.fraction_acf_local > 0: _fl_parts.append(f"acL{self.fraction_acf_local}")
        if self.fraction_acf_global > 0: _fl_parts.append(f"acG{self.fraction_acf_global}")
        if _fl_parts:
            parts.append("flow-" + "-".join(_fl_parts))
        return "_".join(parts)


def physics_informed_loss_batch(
    pred, target, batch, networks, Y_cache,
    physics_cfg: PhysicsConfig,
    #ptdf_loss: Optional[torch.Tensor] = None, # STSI 260403:Removed as this is no longer computed in this function; it is now computed externally and passed in as an argument to the training loop, which combines it with the physics loss according to PhysicsConfig settings.
):
    """
    Modular physics-informed loss using PhysicsConfig flags.

    total = mse + w_phys * physics_terms

    STSI 26.03.25: MSE is no longer computed here — the training loop passes
    in masked MSE via _masked_mse_loss. This function returns physics loss only.
    The mse_loss return value is kept for API compatibility but is now the
    full (unmasked) MSE for logging purposes only — it is NOT used in total_loss.
    """
    # STSI 26.03.25: full MSE kept for logging/history only — not used in total_loss
    mse_loss = F.mse_loss(pred, target)

    if physics_cfg.w_phys == 0.0 and physics_cfg.loss_weight_mode != "adaptive":  # [STSI 200526] allow adaptive mode to compute physics even with w_phys=0
        zero = torch.tensor(0.0, device=pred.device)
        return mse_loss, mse_loss, zero, zero, 0.0, 0.0  # [STSI 210526]: extended to 6-tuple

    graph_ids  = batch.batch
    num_graphs = int(graph_ids.max().item()) + 1

    total_phys  = 0.0
    total_nodes = 0
    total_angle_ref = 0.0
    total_p_res = 0.0  # [STSI 210526]: per-component P residual accumulator
    total_q_res = 0.0  # [STSI 210526]: per-component Q residual accumulator

    for g in range(num_graphs):
        node_mask = (graph_ids == g) # mask for nodes in graph g
        if not node_mask.any():
            continue

        if batch.network_idx.dim() == 0:
            net_idx = int(batch.network_idx.item())
        else:
            net_idx = int(batch.network_idx[g].item())

        network  = networks[net_idx]
        Y_matrix = Y_cache[net_idx]

        x_g    = batch.x[node_mask]
        pred_g = pred[node_mask]

        # STSI 26.03.25: Fix: — assemble mixed known+predicted state vector
        # per bus type before passing to residual functions, restoring the
        # physically correct formulation from the early implementation.
        # Known values override predictions where the power flow solution
        # provides them as inputs:
        #   PQ  buses: Vmag, Vang predicted;  P, Q    known from x
        #   PV  buses: Vmag, P    known;       Vang, Q predicted
        #   Slack bus: Vmag, Vang known (=ref); P, Q  predicted
        slack_mask_g = batch.slack_mask[node_mask].bool()
        pv_mask_g    = batch.pv_mask[node_mask].bool()
        pq_mask_g    = batch.pq_mask[node_mask].bool()

        # Start from predictions for all variables
        v_mag_g = pred_g[:, 0].clone()
        v_ang_g = pred_g[:, 1].clone()
        p_g     = pred_g[:, 2].clone()
        q_g     = pred_g[:, 3].clone()

        # Override with known values where applicable
        # x layout: [is_slack, is_PV, is_PQ, P, Q, Vmag, Vang]
        v_mag_g[pv_mask_g]    = x_g[pv_mask_g,    5]   # PV:    Vmag known
        p_g[pv_mask_g]        = x_g[pv_mask_g,    3]   # PV:    P known
        p_g[pq_mask_g]        = x_g[pq_mask_g,    3]   # PQ:    P known
        q_g[pq_mask_g]        = x_g[pq_mask_g,    4]   # PQ:    Q known
        v_mag_g[slack_mask_g] = x_g[slack_mask_g, 5]   # Slack: Vmag known
        v_ang_g[slack_mask_g] = x_g[slack_mask_g, 6]   # Slack: Vang=0 known

        # Pack mixed state back into pred-shaped tensor for residual functions
        # Shape: [n_buses_g, 4] = [Vmag, Vang, P, Q]
        pred_g_mixed = torch.stack([v_mag_g, v_ang_g, p_g, q_g], dim=1)

        # [STSI 210526]: simple mode removed; rich mode is now unconditional
        bus_masks = (
            slack_mask_g,
            pv_mask_g,
            pq_mask_g,
        )
        residual, p_res_g, q_res_g = compute_power_flow_residual_from_pred(
            pred_g_mixed, x_g, Y_matrix, network,
            bus_masks=bus_masks,
            use_q_partial_mode=physics_cfg.use_q_partial_mode,
            w_P=physics_cfg.w_P,
            w_Q=physics_cfg.w_Q,
        )

        n_g = node_mask.sum().item()
        total_p_res += p_res_g.item() * n_g  # [STSI 210526]: weighted accumulation
        total_q_res += q_res_g.item() * n_g  # [STSI 210526]: weighted accumulation
        
            # angle-ref penalty (optional, both modes)
        # ── RESTORED FROM V2.1.2 ──────────────────────────────────────────
        # In V2.1.2 the angle-ref penalty was computed inside the residual
        # function, once per graph, so squaring always happened before any
        # cross-graph aggregation. When PhysicsConfig was introduced (Block 7)
        # this was lifted out to batch level — introducing a cancellation bug.
        # Fix: compute it here inside the per-graph loop, matching V2.1.2.
        angle_ref_g = torch.tensor(0.0, device=pred.device)
        if physics_cfg.use_angle_ref_penalty and slack_mask_g.any():
            angle_ref_g = pred[node_mask][slack_mask_g, 1].mean() ** 2        
        total_angle_ref += angle_ref_g * n_g          #  accumulate separatelyfor logging

        total_phys  += (residual+angle_ref_g) * n_g # include angle_ref in the physics loss for proper weighting and to avoid cross-graph cancellation
        total_nodes += n_g
    physics_loss = total_phys / total_nodes if total_nodes > 0 else torch.tensor( #[STSI110526] Before fix: works with RELU activations but can lead to scale issues with GELU. Fix: log1p of average residual for better scaling when using GELU, with zero-handling.   
    #physics_loss = torch.log1p(total_phys / total_nodes) if total_nodes > 0 else torch.tensor( #[STSI110526] Fix: log1p of average residual for better scaling when using GELU, with zero-handling
        0.0, device=pred.device
    )
    angle_ref_loss = total_angle_ref / total_nodes if total_nodes > 0 else torch.tensor(
        0.0, device=pred.device
        )


    # PTDF term (optional) # STSI 260403: Removed from here as it is now added separately during the training loop
    #ptdf_term = torch.tensor(0.0, device=pred.device)
    #if physics_cfg.use_ptdf_loss and ptdf_loss is not None:
    #    ptdf_term = ptdf_loss

    combined_physics = physics_loss #+ ptdf_term

    # STSI 26.03.25: total_loss no longer includes MSE — the training loop
    # adds masked MSE externally. Return only the physics component so the
    # caller can combine: loss = masked_mse + w_phys * physics
    p_res_mean = total_p_res / total_nodes if total_nodes > 0 else 0.0  # [STSI 210526]
    q_res_mean = total_q_res / total_nodes if total_nodes > 0 else 0.0  # [STSI 210526]
    return combined_physics, mse_loss, physics_loss, angle_ref_loss, p_res_mean, q_res_mean  # [STSI 210526]: 6-tuple

# get_effective_w_phys removed — obsolete with activation ramp system [STSI 040626]:obsolete with activation ramp system


In [ ]:
# =============================================================================
# SECTION 4: DATASET
# =============================================================================

import torch
from torch_geometric.data import Data, Dataset

class PowerFlowDataset(Dataset):
    """
    One data object per (network, snapshot) pair.

    The graph topology reflects the physical network topology:
    - buses → GNN nodes
    - lines + transformers → GNN edges (bidirectional)

    Each graph carries:
      x         : [n_buses, 7 or 8] node features (bus type flags, known P/Q/V/Vang, optional p_nom_share)
      y         : [n_buses, 4]       targets (vmag, vang, p, q)
      edge_index: [2, n_edges]       bidirectional
      edge_attr : [n_edges, 6]       edge features (r, x, b/2, tap, g_series, b_series)
      masks     : slack_mask, pv_mask, pq_mask
      y_line_p  : [n_fwd_lines]      forward-direction real power per supervised edge (PTDF supervision)
      ptdf_edge_row_idx : [n_edges]  PTDF row index for each edge (-1 for non-supervised/reverse)  # STSI 26.04.07
      network_idx: int
    """

    def __init__(
        self,
        networks: list,
        use_edge_features: bool = True,
        use_pnet_balance: bool = True,   # [STSI100526] kept for backfill compat (ignored internally)
        use_pnom_share: bool = False,     # [STSI100526] col7: p_nom_share (was col7=p_bal + col8=pnom_share before 100526)
        transform=None,
        pre_transform=None,
        ptdf_branch_mode: str = "lines",  # "lines" or "all" — controls PTDF supervision scope
        angle_mode: str = "node",          # [STSI 250526] "node" (default) or "edge_delta"
    ):
        super().__init__(root=None, transform=transform, pre_transform=pre_transform)
        self.networks          = networks
        self.use_edge_features = use_edge_features
        self.use_pnom_share    = use_pnom_share
        self.ptdf_branch_mode  = ptdf_branch_mode #STSI260402: pass ptdf_branch_mode to _create_graph_data for edge feature control
        self.angle_mode        = angle_mode  # [STSI 250526] "node" or "edge_delta"

        # Build flat index: list of (network_idx, timestep_idx)
        self._index = []
        for net_idx, net in enumerate(networks):
            for t_idx in range(len(net.snapshots)):
                self._index.append((net_idx, t_idx))

    def len(self):
        return len(self._index)

    def get(self, idx):
        net_idx, t_idx = self._index[idx]
        network = self.networks[net_idx]
        data = self._create_graph_data(network, t_idx, net_idx)
        return data

    def _create_graph_data(self, network: pypsa.Network, t_idx: int, net_idx: int) -> Data:
        buses     = list(network.buses.index)
        n_buses   = len(buses)
        bus_to_i  = {b: i for i, b in enumerate(buses)}
        snapshot  = network.snapshots[t_idx]

        # ── Bus type flags ────────────────────────────────────────────────────
        is_slack = torch.zeros(n_buses, dtype=torch.float)
        is_pv    = torch.zeros(n_buses, dtype=torch.float)
        is_pq    = torch.zeros(n_buses, dtype=torch.float)

        slack_mask = torch.zeros(n_buses, dtype=torch.bool)
        pv_mask    = torch.zeros(n_buses, dtype=torch.bool)
        pq_mask    = torch.zeros(n_buses, dtype=torch.bool)

        for gen_name, gen in network.generators.iterrows():
            bus_name = gen["bus"]
            if bus_name not in bus_to_i:
                continue
            i = bus_to_i[bus_name]
            ctrl = gen["control"]
            if ctrl == "Slack":
                is_slack[i] = 1.0
                slack_mask[i] = True
            elif ctrl == "PV":
                is_pv[i] = 1.0
                pv_mask[i] = True

        # Buses with no generator → PQ
        for i in range(n_buses):
            if not slack_mask[i] and not pv_mask[i]:
                is_pq[i]   = 1.0
                pq_mask[i] = True

        # ── Bus injections (P, Q) from PyPSA post-solve ───────────────────────
        # STSI260404: BUGFIX — use .loc[snapshot, buses] to force column order
        # to match network.buses.index.  Without this, buses_t columns may be
        # in lexicographic order (e.g. "Bus 10" before "Bus 2"), mis-aligning
        # P/Q/Vang with bus-type masks and the Y-matrix built from buses.index.
        p_bus = torch.tensor(
            network.buses_t.p.loc[snapshot, buses].values, dtype=torch.float
        )
        q_bus = torch.tensor(
            network.buses_t.q.loc[snapshot, buses].values, dtype=torch.float
        )

        # ── Voltages ─────────────────────────────────────────────────────────────
        v_mag = torch.tensor(
            network.buses_t.v_mag_pu.loc[snapshot, buses].values, dtype=torch.float
        )
        v_ang = torch.tensor(
            network.buses_t.v_ang.loc[snapshot, buses].values, dtype=torch.float
        )

        # ── Input features: GNN receives known values, zeros where unknown ────
        # Known before solve: bus type flags, P_inj for PQ+PV, Q_inj for PQ,
        #                     Vmag for PV+Slack, Vang for Slack (=0)
        #
        # x layout: [is_slack(0), is_PV(1), is_PQ(2), P(3), Q(4), Vmag(5), Vang(6)]
        #
        # NOTE: Slack P input x[:,3]=0 is a known issue — the model sees no
        # varying signal for slack P.  # [STSI100526] removed p_net_balance (was here)
        x_p    = p_bus.clone()
        x_q    = q_bus.clone()
        x_vmag = v_mag.clone()
        x_vang = v_ang.clone()

        # Mask unknowns to zero in input (prevents trivial leakage)
        x_p[slack_mask]    = 0.0   # Slack P unknown (to be predicted)
        x_q[slack_mask]    = 0.0   # Slack Q unknown
        x_q[pv_mask]       = 0.0   # PV Q unknown
        x_vmag[pq_mask]    = 0.0   # PQ Vmag unknown
        x_vang[pq_mask]    = 0.0   # PQ Vang unknown
        x_vang[pv_mask]    = 0.0   # PV Vang unknown

        x = torch.stack([is_slack, is_pv, is_pq, x_p, x_q, x_vmag, x_vang], dim=1)

        if self.use_pnom_share:  # [STSI100526] removed p_bal_col (redundant); GNN computes load sum internally
            # col7: static p_nom share per gen bus (p_nom_i / Σp_nom)
            # Tells the model each generator's relative capacity for slack distribution.
            p_nom_share_col = torch.zeros(n_buses, 1)
            p_noms = network.generators["p_nom"].values.astype(float)
            total_p_nom = float(p_noms.sum()) if p_noms.sum() > 0 else 1.0
            for gen_name, gen in network.generators.iterrows():
                b = gen["bus"]
                if b in bus_to_i:
                    p_nom_share_col[bus_to_i[b], 0] += gen["p_nom"] / total_p_nom
            x = torch.cat([x, p_nom_share_col], dim=1)

        # ── Target: full post-solve state for all buses ──────────────────────────
        y = torch.stack([v_mag, v_ang, p_bus, q_bus], dim=1)

        # ── Edge construction (lines + transformers) ──────────────────────────────
        edge_index_list = []
        edge_attr_list  = []
        forward_edge_mask = []   # True for first direction (not reverse duplicate)
        dc_flow_mask = []        # [STSI 020626]:True only for forward LINE edges (not transformers) — for DC flow loss
        fwd_line_idx_list  = []   # one entry per forward supervised edge
        ptdf_edge_row_idx_list = []  # STSI 26.04.07: PTDF row index per edge for learned_edge mode

        branches = []
        #STSI260402: control which branches are included in PTDF supervision via self.ptdf_branch_mode
        # lines always included
        for line_name, line in network.lines.iterrows():
            if line["bus0"] in bus_to_i and line["bus1"] in bus_to_i:
                branches.append(
                    (
                        line.name,  # name (not used in PTDF but useful for debugging)
                        line["bus0"], line["bus1"],
                        line["r"], line["x"],
                        line.get("b", 0.0), 1.0, "line"
                    )
                )

        # STSI260404: BUGFIX — transformers must ALWAYS be added as graph edges
        # for GNN message passing, regardless of ptdf_branch_mode.
        # ptdf_branch_mode only controls which branches are supervised by PTDF
        # loss (via forward_edge_mask / fwd_line_idx_list), NOT graph topology.
        # Previously gated behind `if self.ptdf_branch_mode == "all":` which
        # silently severed the graph when mode was "lines" (the default).
        for tr_name, tr in network.transformers.iterrows():
            if tr["bus0"] in bus_to_i and tr["bus1"] in bus_to_i:
                tap = tr.get("tap_ratio", 1.0)
                tap = tap if not pd.isna(tap) else 1.0
                branches.append(
                    (
                        tr.name,  # name (not used in PTDF but useful for debugging)
                        tr["bus0"], tr["bus1"],
                        tr["r"], tr["x"],
                        tr.get("b", 0.0), float(tap), "trafo"
                    )
                )


        for branch_idx, (name, b0, b1, r, x_val, b_val, tap, btype) in enumerate(branches):
            i0, i1 = bus_to_i[b0], bus_to_i[b1]
            z      = complex(r, x_val)
            y_ser  = (1.0 / z) if abs(z) > 1e-12 else 0.0
            g_ser, b_ser = y_ser.real, y_ser.imag

            if self.use_edge_features:
                attr = [r, x_val, b_val / 2.0, tap, g_ser, b_ser]
            else:
                attr = [1.0]

            # Forward edge
            edge_index_list.append([i0, i1])
            edge_attr_list.append(attr)

            # PTDF supervision: only lines (or all branches) get True in forward_edge_mask
            is_supervised = (btype == "line") if self.ptdf_branch_mode == "lines" else True
            forward_edge_mask.append(is_supervised)
            dc_flow_mask.append(btype == "line")  # [STSI 020626]:DC mask excludes transformers
            if is_supervised:
                fwd_line_idx_list.append(branch_idx)
                ptdf_edge_row_idx_list.append(branch_idx)  # STSI 26.04.07: supervised forward edge
            else:
                ptdf_edge_row_idx_list.append(-1)  # STSI 26.04.07: unsupervised forward edge

            # Reverse edge
            edge_index_list.append([i1, i0])
            edge_attr_list.append(attr)
            forward_edge_mask.append(False)
            dc_flow_mask.append(False)  # [STSI 020626]:reverse edge always False
            ptdf_edge_row_idx_list.append(-1)  # STSI 26.04.07: reverse edge

        if edge_index_list:
            edge_index = torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()
            edge_attr  = torch.tensor(edge_attr_list,  dtype=torch.float)
            fwd_mask   = torch.tensor(forward_edge_mask, dtype=torch.bool)
            dc_fwd_mask = torch.tensor(dc_flow_mask, dtype=torch.bool)  # [STSI 020626]:DC-only line mask
            line_idx   = torch.tensor(fwd_line_idx_list, dtype=torch.long)
            ptdf_edge_row_idx = torch.tensor(ptdf_edge_row_idx_list, dtype=torch.long)  # STSI 26.04.07
        else:
            edge_index = torch.zeros((2, 0), dtype=torch.long)
            edge_attr  = torch.zeros((0, len(attr) if branches else 1), dtype=torch.float)
            fwd_mask   = torch.zeros(0, dtype=torch.bool)
            dc_fwd_mask = torch.zeros(0, dtype=torch.bool)  # [STSI 020626]:DC-only line mask (empty)
            line_idx   = torch.zeros(0, dtype=torch.long)
            ptdf_edge_row_idx = torch.zeros(0, dtype=torch.long)  # STSI 26.04.07


        try:
            ptdf_matrix = compute_ptdf_matrix(network, ptdf_branch_mode=self.ptdf_branch_mode)  # [STSI 260526]:pass branch mode 
            y_ptdf = torch.tensor(ptdf_matrix, dtype=torch.float)
        except Exception:
            n_lines_fb = len(network.lines)
            n_branches_fb = n_lines_fb + len(network.transformers) if self.ptdf_branch_mode == "all" else n_lines_fb
            y_ptdf = torch.zeros((n_branches_fb, n_buses), dtype=torch.float)  # [STSI 260526]:use branch count in fallback 

        # STSI260402 Assertions — verify PTDF supervision scope and consistency
        n_lines = len(network.lines)
        if self.ptdf_branch_mode == "lines":
            assert int(fwd_mask.sum()) == n_lines, \
                f"forward_edge_mask true count {int(fwd_mask.sum())} != n_lines {n_lines}"
        else:  # ptdf_branch_mode == "all": lines + transformers
            n_trafos = len(network.transformers)
            assert int(fwd_mask.sum()) == n_lines + n_trafos, \
                f"forward_edge_mask true count {int(fwd_mask.sum())} != n_lines+n_trafos {n_lines + n_trafos}"
        expected_ptdf_rows = n_lines + len(network.transformers) if self.ptdf_branch_mode == "all" else n_lines  # [STSI 260526]: expected PTDF rows based on supervised branches
        assert y_ptdf.shape[0] == expected_ptdf_rows, \
            f"y_ptdf rows {y_ptdf.shape[0]} != expected {expected_ptdf_rows} (mode={self.ptdf_branch_mode})"

        # ── Line active power flows ───────────────────────────────────────────
        if not network.lines_t.p0.empty:
            y_line_p = torch.tensor(
                network.lines_t.p0.loc[snapshot].values, dtype=torch.float
            )
        else:
            y_line_p = torch.zeros(len(network.lines), dtype=torch.float)

        # [STSI 260526]: Append transformer flows when supervising all branches 
        if self.ptdf_branch_mode == "all" and len(network.transformers) > 0:
            if not network.transformers_t.p0.empty:
                y_trafo_p = torch.tensor(
                    network.transformers_t.p0.loc[snapshot].values, dtype=torch.float
                )
            else:
                y_trafo_p = torch.zeros(len(network.transformers), dtype=torch.float)
            y_line_p = torch.cat([y_line_p, y_trafo_p])

        # ── Q line flows (for AC per-edge flow loss) ──── [STSI 020626]
        if not network.lines_t.q0.empty:
            y_line_q = torch.tensor(
                network.lines_t.q0.loc[snapshot].values, dtype=torch.float
            )
        else:
            y_line_q = torch.zeros(len(network.lines), dtype=torch.float)

        # Append transformer Q flows when supervising all branches
        if self.ptdf_branch_mode == "all" and len(network.transformers) > 0:
            if not network.transformers_t.q0.empty:
                y_trafo_q = torch.tensor(
                    network.transformers_t.q0.loc[snapshot].values, dtype=torch.float
                )
            else:
                y_trafo_q = torch.zeros(len(network.transformers), dtype=torch.float)
            y_line_q = torch.cat([y_line_q, y_trafo_q])


        # [STSI 260526]: Precompute diagonal admittance for edge-local physics 
        # g_diag[i] = Σ_j g_series_ij ; b_diag[i] = Σ_j (b_series_ij + b_half_ij)
        if edge_index.size(1) > 0 and edge_attr.size(0) > 0:
            from torch_geometric.utils import scatter
            _src_nodes = edge_index[0]  # [2E] source node indices
            _g_ser_all = edge_attr[:, 4]  # [2E] series conductance
            _b_ser_all = edge_attr[:, 5]  # [2E] series susceptance
            _b_half_all = edge_attr[:, 2]  # [2E] line charging b/2 per end
            # STSI 220926 (task 065, Bug B fix): forward-edge series scaled by 1/tap^2 so the
            # edge-local diagonal matches compute_admittance_matrix's Y_ii tap handling. tap=1 ->
            # no-op (regression-safe).
            _tap_all = edge_attr[:, 3].clamp(min=1e-6)
            _g_ser_diag = _g_ser_all.clone()
            _b_ser_diag = _b_ser_all.clone()
            _g_ser_diag[fwd_mask] = _g_ser_diag[fwd_mask] / (_tap_all[fwd_mask] ** 2)
            _b_ser_diag[fwd_mask] = _b_ser_diag[fwd_mask] / (_tap_all[fwd_mask] ** 2)
            g_diag = scatter(_g_ser_diag, _src_nodes, dim=0, dim_size=n_buses, reduce='sum')
            b_diag = scatter(_b_ser_diag + _b_half_all, _src_nodes, dim=0, dim_size=n_buses, reduce='sum')
            # STSI 220926 (task 065, Bug A fix): bus shunt impedances (bus-to-ground) -- diagonal
            # only add so edge-local physics matches PyPSA at shunt buses.
            if len(network.shunt_impedances) > 0:
                _bidx = {b: i for i, b in enumerate(network.buses.index)}
                for _sh in network.shunt_impedances.index:
                    _sr = network.shunt_impedances.loc[_sh]
                    _i = _bidx.get(_sr['bus'])
                    if _i is not None:
                        g_diag[_i] += float(_sr.get('g', 0.0) or 0.0)
                        b_diag[_i] += float(_sr.get('b', 0.0) or 0.0)
        else:
            g_diag = torch.zeros(n_buses, dtype=torch.float)
            b_diag = torch.zeros(n_buses, dtype=torch.float)

        # ── Assemble Data object ──────────────────────────────────────────────
        data = Data(
            x          = x,
            y          = y,
            edge_index = edge_index,
            edge_attr  = edge_attr,
        )

        # Bus-type masks — node-level bool tensors (critical for collate + loss)
        data.slack_mask = slack_mask   # [n_nodes] bool
        data.pv_mask    = pv_mask      # [n_nodes] bool
        data.pq_mask    = pq_mask      # [n_nodes] bool

        # Variable-size targets (handled specially in collate_with_ptdf)
        data.y_ptdf      = y_ptdf
        data.y_line_p    = y_line_p
        data.y_line_q    = y_line_q  # [STSI 020626]: reactive flow target for AC per-edge loss
        data.forward_edge_mask = fwd_mask
        data.dc_flow_mask = dc_fwd_mask  # [STSI 020626]:lines-only mask for DC flow loss (excludes transformers)
        data.g_diag = g_diag  # [STSI 260526]: diagonal conductance for edge-local physics
        data.b_diag = b_diag  # [STSI 260526]: diagonal susceptance for edge-local physics
        data.ptdf_line_index = line_idx
        data.ptdf_edge_row_idx = ptdf_edge_row_idx  # STSI 26.04.07: per-edge PTDF row index for learned_edge mode

        # Graph-level metadata
        data.network_idx    = torch.tensor([net_idx], dtype=torch.long)

        # ── Δθ target and BFS metadata (for angle_mode="edge_delta" or "both") ── [STSI 250526]
        if self.angle_mode in ("edge_delta", "both"):  # [STSI 020626]:store Δθ targets for both modes
            # All forward edges (lines + transformers) for full BFS connectivity
            all_fwd_mask = torch.zeros(edge_index.size(1), dtype=torch.bool)
            all_fwd_mask[::2] = True  # even indices = forward edges
            edge_index_fwd = edge_index[:, all_fwd_mask]
            
            # Compute Δθ target: θ_from - θ_to for each forward edge
            theta_true = y[:, 1]  # absolute angles from power flow solution
            y_delta_theta = theta_true[edge_index_fwd[0]] - theta_true[edge_index_fwd[1]]
            data.y_delta_theta = y_delta_theta  # [E_fwd]
            
            # Precompute BFS order from slack
            slack_idx_val = int(slack_mask.nonzero(as_tuple=True)[0][0])
            bfs_edge_idx, bfs_signs, bfs_node_order, bfs_parent = precompute_bfs_order(
                edge_index_fwd, slack_idx_val, n_buses
            )

            # [STSI 240726]: Build dense path-incidence matrix P [N × E_fwd] for fast θ reconstruction
            # P[i, e] is non-zero for every edge e on the spanning-tree path from slack to node i.
            # P[child] = P[parent] (inherit full path) then P[child, edge] = -sign for the new edge.
            # θ = P @ delta_theta_fwd  (slack row is all zeros → θ_slack = 0)
            # Build in BFS order so parent rows are complete before child rows are computed.
            _e_fwd = edge_index_fwd.size(1)  # number of forward edges
            bfs_path_matrix = torch.zeros(n_buses, _e_fwd, dtype=torch.float32)
            for _bi in range(len(bfs_node_order)):  # BFS order guarantees parents precede children
                _child  = bfs_node_order[_bi].item()
                _parent = bfs_parent[_bi].item()
                _edge   = bfs_edge_idx[_bi].item()
                _sign   = bfs_signs[_bi].item()
                bfs_path_matrix[_child] = bfs_path_matrix[_parent].clone()  # inherit parent path
                bfs_path_matrix[_child, _edge] = -_sign  # add direct edge contribution
            # Note: for meshed networks, vang reconstructed via this BFS tree is tree-dependent
            # (spanning-tree choice matters). delta_theta_mae is tree-independent.
            data.bfs_path_matrix = bfs_path_matrix  # [N, E_fwd]
            data.bfs_edge_idx = bfs_edge_idx
            data.bfs_signs = bfs_signs
            data.bfs_node_order = bfs_node_order
            data.bfs_parent = bfs_parent

        return data

def collate_with_ptdf(batch):
    """
    Custom collate function for variable-size PTDF matrices and per-graph attributes.
    
    FIX 2026-03-25: Explicitly propagate slack_mask, pv_mask, pq_mask so that
    _masked_mse_loss and physics_informed_loss_batch receive valid mask tensors.
    Without this, all masked losses silently return 0.

    """
    # ── Variable-size lists: remove before PyG batching ──────────────────────
    y_ptdf_list   = [data.y_ptdf   for data in batch]
    y_line_p_list = [data.y_line_p for data in batch]
    y_line_q_list = [data.y_line_q for data in batch]  # [STSI 020626]
    ptdf_line_index_list = [data.ptdf_line_index for data in batch]


    # [STSI 250526]:also remove BFS metadata before PyG batching (variable-size per graph)
    bfs_keys = ['bfs_edge_idx', 'bfs_signs', 'bfs_node_order', 'bfs_parent', 'y_delta_theta', 'bfs_path_matrix']  # [STSI 240726]:add bfs_path_matrix for vectorized θ recon
    bfs_data = {k: [] for k in bfs_keys}
    for data in batch:
        for k in bfs_keys:
            if hasattr(data, k):
                bfs_data[k].append(getattr(data, k))
                delattr(data, k)

    for data in batch:
        del data.y_ptdf
        del data.y_line_p #STSI260402: added deletion of y_line_p to prevent PyG from trying to batch it and causing errors due to variable sizes
        del data.y_line_q  # [STSI 020626]
        del data.ptdf_line_index
    # ── PyG auto-batches all node-level tensors (x, y, edge_index, etc.) ─────
    # slack_mask / pv_mask / pq_mask are node-level bool tensors stored on each
    # Data object by PowerFlowDataset — Batch.from_data_list will concat them
    # along the node dimension automatically IF they are present.
    # ptdf_edge_row_idx is edge-level [n_edges] long — also auto-concatenated (STSI 26.04.07)
    batched = Batch.from_data_list(batch)

    # ── Restore variable-size list attributes ────────────────────────────────
    batched.y_ptdf_list   = y_ptdf_list
    batched.y_line_p_list = y_line_p_list
    batched.y_line_q_list = y_line_q_list  # [STSI 020626]
    batched.ptdf_line_index_list = ptdf_line_index_list

    # [STSI 250526]:restore BFS metadata as lists (for angle_mode="edge_delta")
    for k, v_list in bfs_data.items():
        if v_list:
            setattr(batched, k, v_list)

    # Restore on originals for re-use in dataset
    for i, (data, y_ptdf, y_line_p, y_line_q, ptdf_idx) in enumerate(zip(batch, y_ptdf_list, y_line_p_list, y_line_q_list, ptdf_line_index_list)):  # [STSI 020626]:added y_line_q
        data.y_ptdf   = y_ptdf
        data.y_line_p = y_line_p
        data.y_line_q = y_line_q  # [STSI 020626]
        data.ptdf_line_index = ptdf_idx
        # [STSI 250526]:restore BFS attrs on originals
        for k in bfs_keys:
            if bfs_data[k]:
                setattr(data, k, bfs_data[k][i])

    # ── Defensive check: abort loudly if masks are missing ───────────────────
    for mask_attr in ("slack_mask", "pv_mask", "pq_mask"):
        if not hasattr(batched, mask_attr):
            raise RuntimeError(
                f"collate_with_ptdf: batched object missing '{mask_attr}'. "
                f"Ensure PowerFlowDataset stores data.{mask_attr} as a node-level "
                f"bool tensor (shape [n_nodes]) on every Data object."
            )

    return batched

# training data validation
def validate_training_data(networks: list, name: str = "dataset", n_sample: int = 2) -> bool:
    """
    Validate that a list of PyPSA networks is compatible with PowerFlowDataset.
    Checks all fields accessed by _create_graph_data and compute_ptdf_matrix.

    Parameters
    ----------
    networks : list of pypsa.Network (post-PF)
    name     : label for print output
    n_sample : how many networks to inspect in detail

    Returns
    -------
    bool : True if all checks pass, False if any critical issue found
    """
    import numpy as np
    import torch

    issues   = []   # critical — will break the loader
    warnings = []   # non-critical — may degrade training

    print(f"\n{'='*60}")
    print(f"Validating '{name}': {len(networks)} networks")
    print(f"{'='*60}")

    if len(networks) == 0:
        print("CRITICAL: Empty network list")
        return False

    for net_idx, net in enumerate(networks[:n_sample]):
        tag = f"[net {net_idx}]"
        print(f"\n--- Network {net_idx} ---")

        # ── 1. Topology ──────────────────────────────────────────
        n_buses  = len(net.buses)
        n_lines  = len(net.lines)
        n_trafos = len(net.transformers)
        n_gens   = len(net.generators)
        n_loads  = len(net.loads)
        n_snaps  = len(net.snapshots)
        print(f"  Buses={n_buses}, Lines={n_lines}, Trafos={n_trafos}, "
              f"Gens={n_gens}, Loads={n_loads}, Snapshots={n_snaps}")

        if n_buses == 0:
            issues.append(f"{tag} No buses")
        if n_lines + n_trafos == 0:
            issues.append(f"{tag} No lines or transformers — edge_index will be empty")
        if n_snaps == 0:
            issues.append(f"{tag} No snapshots — dataset will be empty")
        if n_gens == 0:
            warnings.append(f"{tag} No generators — bus type will default to PQ everywhere")

        # ── 2. Required buses_t tables (read in _create_graph_data) ──
        print(f"\n  buses_t tables:")
        for tbl, attr in [("buses_t.p",       "p"),
                           ("buses_t.q",       "q"),
                           ("buses_t.v_mag_pu","v_mag_pu"),
                           ("buses_t.v_ang",   "v_ang")]:
            df = getattr(net.buses_t, attr)
            ok = not df.empty and df.shape == (n_snaps, n_buses)
            finite = np.isfinite(df.values).all() if ok else False
            status = "✅" if (ok and finite) else "❌"
            if not ok:
                issues.append(f"{tag} {tbl} missing or wrong shape: {df.shape} "
                              f"(expect {(n_snaps, n_buses)})")
            elif not finite:
                issues.append(f"{tag} {tbl} contains NaN/Inf")
            print(f"    {status} {tbl}: shape={df.shape}, finite={finite}")

        # ── 3. Voltage range sanity ───────────────────────────────
        if not net.buses_t.v_mag_pu.empty:
            v = net.buses_t.v_mag_pu.values.flatten()
            v_ok = np.all((v > 0.5) & (v < 1.5))
            status = "✅" if v_ok else "⚠️"
            print(f"    {status} v_mag_pu range: [{v.min():.4f}, {v.max():.4f}]")
            if not v_ok:
                warnings.append(f"{tag} v_mag_pu out of [0.5, 1.5]: "
                                f"min={v.min():.4f} max={v.max():.4f}")

        # ── 4. lines_t.p0 (used for y_line_p) ────────────────────
        print(f"\n  lines_t tables:")
        p0 = net.lines_t.p0
        ok_p0 = not p0.empty and p0.shape == (n_snaps, n_lines)
        finite_p0 = np.isfinite(p0.values).all() if ok_p0 else False
        status = "✅" if (ok_p0 and finite_p0) else "❌"
        if not ok_p0:
            issues.append(f"{tag} lines_t.p0 missing or wrong shape: {p0.shape} "
                         f"(expect {(n_snaps, n_lines)})")
        elif not finite_p0:
            issues.append(f"{tag} lines_t.p0 contains NaN/Inf")
        print(f"    {status} lines_t.p0: shape={p0.shape}, finite={finite_p0}")

        # ── 5. generators_t.p and q (used for node features) ─────
        print(f"\n  generators_t tables:")
        for attr in ["p", "q"]:
            df = getattr(net.generators_t, attr)
            ok = not df.empty
            finite = np.isfinite(df.values).all() if ok else False
            status = "✅" if (ok and finite) else "⚠️"
            if not ok:
                warnings.append(f"{tag} generators_t.{attr} empty")
            print(f"    {status} generators_t.{attr}: shape={df.shape}, finite={finite}")

        # ── 6. Generator control column ───────────────────────────
        print(f"\n  Generator control types:")
        if "control" not in net.generators.columns:
            issues.append(f"{tag} net.generators missing 'control' column")
        else:
            ctrl_counts = net.generators["control"].value_counts().to_dict()
            has_slack = "Slack" in ctrl_counts
            status = "✅" if has_slack else "❌"
            print(f"    {status} control types: {ctrl_counts}")
            if not has_slack:
                issues.append(f"{tag} No Slack generator — bus type encoding will be wrong")

        # ── 7. Line parameters (edge_attr: r, x, b, s_nom) ───────
        print(f"\n  Line parameters:")
        for col in ["r", "x", "b", "s_nom"]:
            present = col in net.lines.columns
            if present:
                vals = net.lines[col].values.astype(float)
                finite = np.isfinite(vals).all()
                status = "✅" if finite else "❌"
                print(f"    {status} lines.{col}: min={vals.min():.5f} "
                      f"max={vals.max():.5f}")
                if not finite:
                    issues.append(f"{tag} lines.{col} contains NaN/Inf")
                if col in ["r", "x"] and (vals <= 0).any():
                    warnings.append(f"{tag} lines.{col} has zero/negative values")
            else:
                issues.append(f"{tag} lines.{col} column missing")
                print(f"    ❌ lines.{col}: MISSING")

        # ── 8. Bus–line connectivity (bus0/bus1 in buses.index) ───
        missing_buses = set()
        for col in ["bus0", "bus1"]:
            missing = set(net.lines[col]) - set(net.buses.index)
            missing_buses |= missing
        if missing_buses:
            issues.append(f"{tag} Lines reference buses not in buses.index: "
                         f"{missing_buses}")
        else:
            print(f"\n    ✅ All line endpoints exist in buses.index")

        # ── 9. Simulate one graph build (catches index errors) ────
        print(f"\n  Graph construction smoke test (snapshot 0):")
        try:
            snap = net.snapshots[0]
            bus_to_idx = {b: i for i, b in enumerate(net.buses.index)}
            edges = []
            for _, line in net.lines.iterrows():
                i = bus_to_idx[line.bus0]
                j = bus_to_idx[line.bus1]
                edges += [[i, j], [j, i]]
            edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
            # STSI260404: BUGFIX — align buses_t columns to buses.index order
            x_check = np.column_stack([
                net.buses_t.v_mag_pu.loc[snap, list(net.buses.index)].values,
                net.buses_t.v_ang.loc[snap, list(net.buses.index)].values,
                net.buses_t.p.loc[snap, list(net.buses.index)].values,
                net.buses_t.q.loc[snap, list(net.buses.index)].values,
            ])
            assert np.isfinite(x_check).all(), "Non-finite node features"
            print(f"    ✅ edge_index: {edge_index.shape}, "
                  f"node features: {x_check.shape}, all finite")
        except Exception as e:
            issues.append(f"{tag} Graph construction failed: {e}")
            print(f"    ❌ Graph construction failed: {e}")

    # ── Summary ───────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"SUMMARY for '{name}'")
    if issues:
        print(f"\n❌ {len(issues)} CRITICAL issue(s):")
        for iss in issues:
            print(f"   • {iss}")
    if warnings:
        print(f"\n⚠️  {len(warnings)} warning(s):")
        for w in warnings:
            print(f"   • {w}")
    if not issues and not warnings:
        print("✅ All checks passed — dataset is compatible with PowerFlowDataset")
    elif not issues:
        print("✅ No critical issues — dataset should load correctly")

    print(f"{'='*60}\n")
    return len(issues) == 0




In [ ]:
# =============================================================================
# SECTION 7: LINE FLOW CALCULATION
# =============================================================================

def calculate_line_flows(network, vmag, vang, t_idx=None):
    """
    Calculate AC line flows from predicted voltages.

    Args:
        network: PyPSA network object
        vmag:    Voltage magnitudes [num_buses], numpy array, p.u.
        vang:    Voltage angles [num_buses], numpy array, radians
        t_idx:   Snapshot index (used for nominal values only)
    Returns:
        dict with keys p0, p1, q0, q1 - each a numpy array of length num_lines (p.u.)
    """
    num_lines = len(network.lines)
    p0 = np.zeros(num_lines)
    p1 = np.zeros(num_lines)
    q0 = np.zeros(num_lines)
    q1 = np.zeros(num_lines)

    all_buses = list(network.buses.index)
    # BUGFIX: use bus_to_idx dict instead of string parsing (topology-safe)
    bus_to_idx = {bus_name: idx for idx, bus_name in enumerate(all_buses)}

    for i, line in enumerate(network.lines.index):
        from_bus = network.lines.loc[line, "bus0"]
        to_bus = network.lines.loc[line, "bus1"]

        # BUGFIX: use dict lookup instead of int(bus.split('-')[1]) - 1
        from_idx = bus_to_idx[from_bus]
        to_idx = bus_to_idx[to_bus]

        r = network.lines.loc[line, "r"]
        x = network.lines.loc[line, "x"]
        b = network.lines.loc[line, "b"] if "b" in network.lines.columns else 0.0

        # Line impedance and admittance
        z = complex(r, x)
        y_series = 1.0 / z if abs(z) > 1e-12 else 0.0
        y_shunt = complex(0, b / 2.0)

        # Complex voltages at from/to buses
        V_from = vmag[from_idx] * np.exp(1j * vang[from_idx])
        V_to = vmag[to_idx] * np.exp(1j * vang[to_idx])

        # Line current from sending end
        I_from = (V_from - V_to) * y_series + V_from * y_shunt
        I_to = (V_to - V_from) * y_series + V_to * y_shunt

        S_from = V_from * np.conj(I_from)
        S_to = V_to * np.conj(I_to)

        s_nom = network.lines.loc[line, "s_nom"]
        # Convert from p.u. to MW/MVAr using system base
        #base_mva = getattr(network, "sn_mva", 100.0)

        p0[i] = S_from.real #* base_mva # removed the base_mva scaling to keep flows in p.u. for better numerical stability and consistency with the rest of the model's inputs/outputs
        q0[i] = S_from.imag #* base_mva
        p1[i] = S_to.real #* base_mva  # STSI 220926 (task 065, sign fix): was -S_to.real; PyPSA p1 convention is NOT negated
        q1[i] = S_to.imag #* base_mva  # STSI 220926 (task 065, sign fix): was -S_to.imag

    return {"p0": p0, "p1": p1, "q0": q0, "q1": q1}


def build_line_results_from_flows(network, flows, snapshot_label):
    """
    Helper to build a line_results-style DataFrame for one snapshot
    from a dict produced by calculate_line_flows.
    """
    idx = pd.Index([snapshot_label], name="snapshot")
    cols = pd.MultiIndex.from_product(
        [network.lines.index, ["P0 pu", "P1 pu", "Q0 pu", "Q1 pu"]]
    )
    df = pd.DataFrame(index=idx, columns=cols, dtype=float)
    for i, line in enumerate(network.lines.index):
        df.loc[snapshot_label, (line, "P0 pu")] = flows["p0"][i]
        df.loc[snapshot_label, (line, "P1 pu")] = flows["p1"][i]
        df.loc[snapshot_label, (line, "Q0 pu")] = flows["q0"][i]
        df.loc[snapshot_label, (line, "Q1 pu")] = flows["q1"][i]
    return df



def calculate_line_flows_from_delta_theta(  # [STSI 270526]: task 028 — direct Δθ line flows (copied from Training V2.6.2)
    delta_theta,     # [E_fwd] numpy — Δθ per forward edge (θ_from - θ_to)
    vmag,            # [N] numpy — voltage magnitudes
    edge_index_fwd,  # [2, E_fwd] numpy — forward edge source/target
    edge_attr_fwd,   # [E_fwd, 6] numpy — [r, x, b_half, tap, g_ser, b_ser]
) -> dict:
    """
    Compute line flows directly from Δθ without θ reconstruction.
    Uses π-model: P_from = V_from²×(g_s+g_sh) - V_from×V_to×(g_s×cos(Δθ) + b_s×sin(Δθ))
    Returns dict with keys: 'p0', 'q0', 'p1', 'q1' (from/to in per-unit).
    """
    from_bus = edge_index_fwd[0]  # [E_fwd]
    to_bus = edge_index_fwd[1]    # [E_fwd]
    v_from = vmag[from_bus]
    v_to = vmag[to_bus]

    r = edge_attr_fwd[:, 0]
    x_imp = edge_attr_fwd[:, 1]
    b_half = edge_attr_fwd[:, 2]
    tap = edge_attr_fwd[:, 3]

    # Series admittance
    z = r + 1j * x_imp
    y_s = 1.0 / z
    g_s = y_s.real
    b_s = y_s.imag

    # Handle tap ratio for transformers
    tap_nz = np.where(np.abs(tap) > 1e-12, tap, 1.0)
    # Tap-adjusted: effective from-end voltage = V_from / tap
    v_from_eff = v_from / tap_nz

    # Shunt susceptance per end
    b_sh = b_half

    cos_dt = np.cos(delta_theta)
    sin_dt = np.sin(delta_theta)

    # From-end power (π-model with tap on from side)
    p0 = v_from_eff**2 * g_s - v_from_eff * v_to * (g_s * cos_dt + b_s * sin_dt)
    q0 = -v_from_eff**2 * (b_s + b_sh) - v_from_eff * v_to * (g_s * sin_dt - b_s * cos_dt)

    # To-end power (Δθ reversed; PyPSA convention S1 = V1·conj(I1) = power into the branch at bus1)  # [STSI 191026]:task 096 F7 - outer negation removed
    p1 = v_to**2 * g_s - v_to * v_from_eff * (g_s * cos_dt - b_s * sin_dt)
    q1 = -v_to**2 * (b_s + b_sh) + v_to * v_from_eff * (g_s * sin_dt + b_s * cos_dt)

    return {'p0': p0, 'q0': q0, 'p1': p1, 'q1': q1}



In [ ]:
# =============================================================================
# SECTION 9: INFERENCE & PREDICTION
# =============================================================================

def _augment_edge_attr_learned(data, ptdf_params, max_buses, net_idx, ptdf_offset=0, device=None):
    """Augment data.edge_attr with learned PTDF rows for learned_edge models.

    Shared helper used by evaluate_model, compare_pf_conventional_vs_gnn,
    predict_network_results_with_masks, and evaluate_gnn_on_test_set.

    Args:
        data:          single PyG Data object (already moved to device)
        ptdf_params:   list[nn.Parameter] or None
        max_buses:     int, PTDF parameter width
        net_idx:       int, network index within the dataset
        ptdf_offset:   int, added to net_idx to get global ptdf_params index
        device:        torch device (inferred from data.edge_attr if None)

    Returns:
        data with edge_attr augmented in-place. No-op if ptdf_params is None.
    """
    if ptdf_params is None or max_buses <= 0:
        return data
    if device is None:
        device = data.edge_attr.device
    row_idx = data.ptdf_edge_row_idx  # [E] line index or -1
    ptdf_rows = torch.zeros(data.edge_attr.size(0), max_buses, device=device)
    valid = (row_idx >= 0) & (row_idx < ptdf_params[net_idx + ptdf_offset].size(0))
    if valid.any():
        ptdf_rows[valid] = ptdf_params[net_idx + ptdf_offset][row_idx[valid]].to(device)
    data.edge_attr = torch.cat([data.edge_attr, ptdf_rows], dim=-1)
    return data


def model_results(network, print_results=False, plot_results=False, compute_ptdf=False):
    """
    Run PyPSA power flow and store results in DataFrames.
    Print if print_results=True, plot if plot_results=True.

    DISTSLACK: pass distribute_slack=True to use PyPSA's distributed slack solver.
    """
    # DISTSLACK: use distribute_slack=True to properly handle distributed slack
    network.pf(use_seed=True, distribute_slack=False)  # DISTSLACK: change to True when using distributed slack

    snapshots = network.snapshots
    buses = network.buses.index
    lines = network.lines.index

    bus_results = pd.DataFrame(
        index=snapshots,
        columns=pd.MultiIndex.from_product([buses, ["P pu", "Q pu", "V pu", "Angle deg"]]),
        dtype=float,
    )
    line_results = pd.DataFrame(
        index=snapshots,
        columns=pd.MultiIndex.from_product([lines, ["P0 pu", "P1 pu", "Q0 pu", "Q1 pu"]]),
        dtype=float,
    )

    for t in snapshots:
        for bus in buses:
            bus_results.loc[t, (bus, "P pu")]    = network.buses_t.p.loc[t, bus]
            bus_results.loc[t, (bus, "Q pu")]  = network.buses_t.q.loc[t, bus]
            bus_results.loc[t, (bus, "V pu")]    = network.buses_t.v_mag_pu.loc[t, bus]
            bus_results.loc[t, (bus, "Angle deg")] = np.degrees(network.buses_t.v_ang.loc[t, bus])
        for line in lines:
            line_results.loc[t, (line, "P0 pu")]   = network.lines_t.p0.loc[t, line]
            line_results.loc[t, (line, "P1 pu")]   = network.lines_t.p1.loc[t, line]
            line_results.loc[t, (line, "Q0 pu")] = network.lines_t.q0.loc[t, line]
            line_results.loc[t, (line, "Q1 pu")] = network.lines_t.q1.loc[t, line]

    if print_results:
        print(bus_results)
        print(line_results)
    if plot_results:
        plot_network_results(network, bus_results)

    ptdf_df = None
    if compute_ptdf:
        ptdf_df = compute_ptdf_matrix(network)

    return bus_results, line_results, ptdf_df


def batched_gnn_forward(model, data_list):
    """Run GNN on a batch of graphs in a single forward pass."""
    exclude_keys = ['y_ptdf', 'ptdf_line_index', 'y_line_p',
                    'bfs_edge_idx', 'bfs_signs', 'bfs_node_order',
                    'bfs_parent', 'y_delta_theta']  # [STSI 260526]:edge_delta BFS keys
    batch = Batch.from_data_list(data_list, exclude_keys=exclude_keys)
    out = model(batch)
    if isinstance(out, tuple):
        out = out[0]
    results = []
    for g in range(len(data_list)):
        mask = (batch.batch == g)
        results.append(out[mask])
    return results


def predict_network_results_with_masks(
    model, network, use_edge_features=True, debug=False, use_pnom_share=None,
    # STSI 26.04.07: learned_edge PTDF support
    ptdf_params=None, max_buses=0, ptdf_offset=0,
    collect_delta_theta=False,  # [STSI 010626]:optionally collect edge-level delta_theta pred/true
):
    """
    Run GNN prediction on a network and return bus/line results with prediction masks.

    DISTSLACK: prediction masks now also flag distributed slack buses correctly.
    Distributed slack buses predict P and Q (same as single slack).
    STSI 26.04.07: Augments edge_attr when ptdf_params is provided (learned_edge).

    Args:
        model:   Trained PowerFlowGNN model
        network: PyPSA network object (must have solved power flow for feature extraction)
    Returns:
        bus_results, line_results, prediction_masks
    """
    _angle_mode = getattr(model, "angle_mode", "node")
    dataset = PowerFlowDataset(
        [network],
        use_edge_features=use_edge_features,
        use_pnom_share=(model.node_embedding.in_features >= 8) if use_pnom_share is None else use_pnom_share,  # [STSI100526] was >=9 (2-col); now >=8 (1-col: pnom_share only)
        angle_mode=_angle_mode,  # [STSI 260526]:pass angle_mode
        ptdf_branch_mode="all" if _angle_mode in ("edge_delta", "both") else "lines",  # [STSI 260526]:edge_delta needs all branches for BFS connectivity
    )
    snapshots = network.snapshots
    buses = list(network.buses.index)
    bus_to_idx = {b: i for i, b in enumerate(buses)}

    bus_results = pd.DataFrame(
        index=snapshots,
        columns=pd.MultiIndex.from_product([buses, ["P pu", "Q pu", "V pu", "Angle deg"]]),
        dtype=float,
    )
    line_results = pd.DataFrame(
        index=snapshots,
        columns=pd.MultiIndex.from_product(
            [network.lines.index, ["P0 pu", "P1 pu", "Q0 pu", "Q1 pu"]]
        ),
        dtype=float,
    )
    # Prediction masks: which properties were predicted (not taken from input)
    prediction_masks = {
        "P":     pd.DataFrame(index=snapshots, columns=buses, dtype=bool),
        "Q":     pd.DataFrame(index=snapshots, columns=buses, dtype=bool),
        "V":     pd.DataFrame(index=snapshots, columns=buses, dtype=bool),
        "Angle": pd.DataFrame(index=snapshots, columns=buses, dtype=bool),
    }

    model.eval()
    _dt_pred_rows, _dt_true_rows = [], []  # [STSI 010626]:delta_theta accumulators
    with torch.no_grad():
        for t_idx, t in enumerate(snapshots):
            data = dataset[t_idx]

            # STSI 26.04.07: augment edge_attr for learned_edge models
            _augment_edge_attr_learned(data, ptdf_params, max_buses, net_idx=0, ptdf_offset=ptdf_offset)

            # Fallback: zero-pad if model expects wider edge_attr (old runs without stored ptdf_params)
            if hasattr(model, 'convs') and model.convs and hasattr(model.convs[0], 'lin_edge') and model.convs[0].lin_edge is not None:
                _expected = model.convs[0].lin_edge.in_channels
                if data.edge_attr.size(1) < _expected:
                    data.edge_attr = torch.cat([data.edge_attr, torch.zeros(data.edge_attr.size(0), _expected - data.edge_attr.size(1))], dim=1)

            # [STSI 260526]:handle 3-tuple return and edge_delta theta reconstruction
            angle_mode = getattr(model, "angle_mode", "node")
            out = model(data)
            node_pred = out[0]
            delta_theta_pred = out[2] if len(out) > 2 else None
            if collect_delta_theta and delta_theta_pred is not None:  # [STSI 010626]:accumulate edge delta_theta
                _dt_pred_rows.append(delta_theta_pred.cpu().numpy())
                if hasattr(data, "y_delta_theta") and data.y_delta_theta is not None:
                    _dt_true_rows.append(data.y_delta_theta.cpu().numpy())

            # θ reconstruction for edge_delta mode [STSI 260526]
            if angle_mode == "edge_delta" and delta_theta_pred is not None:
                theta_recon = reconstruct_theta_from_delta(
                    delta_theta_pred,
                    data.bfs_edge_idx,
                    data.bfs_signs,
                    data.bfs_node_order,
                    data.bfs_parent,
                    data.x.size(0),
                )
                node_pred_full = torch.zeros(data.x.size(0), 4, device=node_pred.device)
                node_pred_full[:, 0] = node_pred[:, 0]  # Vmag
                node_pred_full[:, 1] = theta_recon       # reconstructed θ
                node_pred_full[:, 2] = node_pred[:, 1]  # P (col1 in 3-col)
                node_pred_full[:, 3] = node_pred[:, 2]  # Q (col2 in 3-col)
                node_pred = node_pred_full

            pred_vmag = node_pred[:, 0]
            pred_vang = node_pred[:, 1]
            pred_p    = node_pred[:, 2]
            pred_q    = node_pred[:, 3]

            for i, bus in enumerate(buses):
                bus_type = "PQ"  # default
                connected_gens = network.generators[network.generators.bus == bus]
                if len(connected_gens) > 0:
                    ctrl = connected_gens.iloc[0]["control"]
                    if ctrl in ("Slack", "PV"):
                        bus_type = ctrl

                if bus_type == "PQ":
                    # For PQ buses: predict vmag, vang
                    pred_vmag_i = pred_vmag[i]
                    pred_vang_i = pred_vang[i]
                    pred_p_i    = network.buses_t.p.loc[t, bus]   # Known
                    pred_q_i    = network.buses_t.q.loc[t, bus]   # Known
                    prediction_masks["P"].loc[t, bus]     = False
                    prediction_masks["Q"].loc[t, bus]     = False
                    prediction_masks["V"].loc[t, bus]     = True
                    prediction_masks["Angle"].loc[t, bus] = True

                elif bus_type == "PV":
                    # For PV buses: known vmag, predict vang, q
                    pred_vmag_i = network.buses_t.v_mag_pu.loc[t, bus]  # Known
                    pred_vang_i = pred_vang[i]
                    pred_p_i    = network.buses_t.p.loc[t, bus]         # Known
                    pred_q_i    = pred_q[i]
                    prediction_masks["P"].loc[t, bus]     = False
                    prediction_masks["Q"].loc[t, bus]     = True
                    prediction_masks["V"].loc[t, bus]     = False
                    prediction_masks["Angle"].loc[t, bus] = True

                else:
                    # Slack (single or distributed): known vmag, vang; predict p, q
                    # DISTSLACK: for distributed slack, vmag is known but angle
                    # reference is only soft-fixed; we still use the known angle
                    # from the PF solution as input and predict P, Q
                    pred_vmag_i = network.buses_t.v_mag_pu.loc[t, bus]  # Known
                    pred_vang_i = network.buses_t.v_ang.loc[t, bus]     # Known (reference)
                    pred_p_i    = pred_p[i]
                    pred_q_i    = pred_q[i]
                    prediction_masks["P"].loc[t, bus]     = True
                    prediction_masks["Q"].loc[t, bus]     = True
                    prediction_masks["V"].loc[t, bus]     = False
                    prediction_masks["Angle"].loc[t, bus] = False

                bus_results.loc[t, (bus, "P pu")]      = pred_p_i.item() if torch.is_tensor(pred_p_i) else pred_p_i
                bus_results.loc[t, (bus, "Q pu")]      = pred_q_i.item() if torch.is_tensor(pred_q_i) else pred_q_i
                bus_results.loc[t, (bus, "V pu")]      = pred_vmag_i.item() if torch.is_tensor(pred_vmag_i) else pred_vmag_i
                bus_results.loc[t, (bus, "Angle deg")] = np.degrees(
                    pred_vang_i.item() if torch.is_tensor(pred_vang_i) else pred_vang_i
                )

            if debug:
                print(f"Predicted angles should be small radians: {pred_vang[:3].numpy()}")
                print(f"Predicted angles in degrees: {np.degrees(pred_vang[:3].numpy())}")
                print(f"Angle range: {pred_vang.min():.6f} to {pred_vang.max():.6f} radians")

            # Calculate line flows from predicted voltages
            vmag_np = np.array([
                bus_results.loc[t, (bus, "V pu")] for bus in buses
            ], dtype=float)
            vang_np = np.radians(np.array([
                bus_results.loc[t, (bus, "Angle deg")] for bus in buses
            ], dtype=float))
            # [STSI 270526]:task 028 — direct Δθ for edge_delta models
            if angle_mode in ("edge_delta", "both") and delta_theta_pred is not None:
                _fwd_mask = np.zeros(data.edge_index.shape[1], dtype=bool)
                _fwd_mask[::2] = True
                _ei_fwd = data.edge_index[:, _fwd_mask].cpu().numpy()
                _ea_fwd = data.edge_attr[_fwd_mask].cpu().numpy()
                _dt_np = delta_theta_pred.cpu().numpy()
                pred_line_flows = calculate_line_flows_from_delta_theta(
                    delta_theta=_dt_np,
                    vmag=vmag_np,
                    edge_index_fwd=_ei_fwd,
                    edge_attr_fwd=_ea_fwd,
                )
            else:
                pred_line_flows = calculate_line_flows(network, vmag_np, vang_np, t_idx)

            for i, line in enumerate(network.lines.index):
                line_results.loc[t, (line, "P0 pu")]   = pred_line_flows["p0"][i]
                line_results.loc[t, (line, "P1 pu")]   = pred_line_flows["p1"][i]
                line_results.loc[t, (line, "Q0 pu")] = pred_line_flows["q0"][i]
                line_results.loc[t, (line, "Q1 pu")] = pred_line_flows["q1"][i]

    dt_dict = {}  # [STSI 010626]:return delta_theta as 4th element
    if _dt_pred_rows:
        dt_dict["dt_pred"] = np.concatenate(_dt_pred_rows)
        if _dt_true_rows:
            dt_dict["dt_true"] = np.concatenate(_dt_true_rows)
    return bus_results, line_results, prediction_masks, dt_dict

def evaluate_pypsa_timing(networks, verbose=True):
    """Measure PyPSA AC PF wall time by re-running pf() on a network copy.
    
    Runs pf() on all snapshots at once per network (same as normal usage),
    divides by snapshot count for per-snapshot average.
    Error = 0 by definition (PyPSA is the ground truth).
    """
    all_times_ms = []
    for network in networks:
        net_copy = network.copy()
        t0 = time.perf_counter()
        net_copy.pf(use_seed=True)
        t1 = time.perf_counter()
        ms_per_snap = (t1 - t0) * 1000.0 / max(1, len(network.snapshots))
        all_times_ms.append(ms_per_snap)

    avg_ms = float(np.mean(all_times_ms))
    metrics = {  # [STSI 010626]:add all zero-valued keys so PyPSA appears on all scatter subplots
        'vmag_mae': 0.0, 'vmag_rmse': 0.0,
        'vang_mae': 0.0, 'vang_rmse': 0.0,
        'p_mae': 0.0, 'p_rmse': 0.0,
        'q_mae': 0.0, 'q_rmse': 0.0,
        'line_flow_mae': 0.0, 'line_flow_rmse': 0.0,
        'line_flow_q0_mae': 0.0, 'line_flow_q0_rmse': 0.0,
        'delta_theta_mae': 0.0, 'delta_theta_rmse': 0.0,
        'inference_time_per_snapshot_ms': avg_ms,
        'n_snapshots': sum(len(n.snapshots) for n in networks),
    }
    if verbose:
        print(f"PyPSA AC timing: {avg_ms:.3f} ms/snap avg over {len(networks)} network(s)")
    return metrics

def evaluate_pypsa_timing_single(networks, verbose=True):
    """Measure PyPSA AC PF time per individual snapshot (no batching)."""
    times_ms = []
    for net in networks:
        for snap in net.snapshots:
            net_copy = net.copy()
            net_copy.set_snapshots([snap])
            t0 = time.perf_counter()
            net_copy.pf(use_seed=True)
            t1 = time.perf_counter()
            times_ms.append((t1 - t0) * 1000)
    
    avg_ms = float(np.mean(times_ms))
    metrics = {  # [STSI 010626]:add all zero-valued keys so PyPSA appears on all scatter subplots
        'vmag_mae': 0.0, 'vmag_rmse': 0.0,
        'vang_mae': 0.0, 'vang_rmse': 0.0,
        'p_mae': 0.0, 'p_rmse': 0.0,
        'q_mae': 0.0, 'q_rmse': 0.0,
        'line_flow_mae': 0.0, 'line_flow_rmse': 0.0,
        'line_flow_q0_mae': 0.0, 'line_flow_q0_rmse': 0.0,
        'delta_theta_mae': 0.0, 'delta_theta_rmse': 0.0,
        'inference_time_per_snapshot_ms': avg_ms,
    }
    if verbose:
        print(f"PyPSA AC (single): {avg_ms:.3f} ms/snap over {len(times_ms)} snapshots")
    return metrics

# STSI 26.04.07: added backfill_line_flow_metrics to compute line flow errors separately from bus errors, and to support learned_edge models with augmented edge_attr for PTDF rows.

def backfill_gnn_timing(
    runs: list,
    test_networks: list,
    use_edge_features: bool = True,
    use_pnet_balance: bool = True,
    n_warmup: int = 5,
    batch_size: int = 64,
    device=None,
):
    """
    Backfill per-snapshot timing into sweep runs with live model objects.
    
    Three timing levels:
      solve_time_ms:       single forward pass per snapshot
      batch_solve_time_ms: batched forward pass / N snapshots
      postproc_time_ms:    mask overrides + cpu transfer + line flow calc
      total_time_ms:       solve + postproc (single-snapshot end-to-end)
    """
    if device is None:
        device = torch.device('cpu')
    
    dataset = PowerFlowDataset(
        test_networks,
        use_edge_features=use_edge_features,
        use_pnet_balance=use_pnet_balance,
    )
    n_snaps = len(dataset)
    print(f"Dataset: {n_snaps} snapshots, {n_warmup} warmup, batch_size={batch_size}")
    
    for ri, run in enumerate(runs):
        model = run['model']
        model.to(device)
        model.eval()
        
        ptdf_params = run.get('_ptdf_params', None)
        max_buses   = run.get('_max_buses', 0)
        
        # ── Phase 1: Per-snapshot timing ──
        solve_times = []
        postproc_times = []
        
        with torch.no_grad():
            for i in range(n_snaps + n_warmup):
                idx = i % n_snaps
                data = dataset[idx].to(device)
                net_idx, _ = dataset._index[idx]
                _augment_edge_attr_learned(data, ptdf_params, max_buses, net_idx, device=device)
                
                # Level 1: pure forward pass
                t0 = time.perf_counter()
                out = model(data)
                _raw_out = out
                if isinstance(out, tuple):
                    out = _raw_out[0]
                _angle_mode = getattr(model, "angle_mode", "node")  # [STSI 270526]:task 028
                _delta_theta_pred = _raw_out[2] if isinstance(_raw_out, tuple) and len(_raw_out) > 2 else None
                t_solve = time.perf_counter() - t0
                
                # Level 2: post-processing
                t1 = time.perf_counter()
                x = data.x
                bus_type = x[:, 0]
                pv_mask    = (bus_type == 1)
                slack_mask = (bus_type == 2)
                
                vmag_pred = out[:, 0].clone()
                vang_pred = out[:, 1].clone()
                vmag_pred[pv_mask]    = x[pv_mask, 5]
                vmag_pred[slack_mask] = x[slack_mask, 5]
                vang_pred[slack_mask] = x[slack_mask, 6]
                
                vmag_np = vmag_pred.cpu().numpy()
                vang_np = vang_pred.cpu().numpy()
                # [STSI 270526]:task 028 — direct Δθ for edge_delta models (timing)
                if _angle_mode in ("edge_delta", "both") and _delta_theta_pred is not None:
                    _fwd_mask = np.zeros(data.edge_index.shape[1], dtype=bool)
                    _fwd_mask[::2] = True
                    _ei_fwd = data.edge_index[:, _fwd_mask].cpu().numpy()
                    _ea_fwd = data.edge_attr[_fwd_mask].cpu().numpy()
                    _dt_np = _delta_theta_pred.cpu().numpy()
                    calculate_line_flows_from_delta_theta(
                        delta_theta=_dt_np,
                        vmag=vmag_np,
                        edge_index_fwd=_ei_fwd,
                        edge_attr_fwd=_ea_fwd,
                    )
                else:
                    calculate_line_flows(test_networks[net_idx], vmag_np, vang_np)
                t_post = time.perf_counter() - t1
                
                if i >= n_warmup:
                    solve_times.append(t_solve)
                    postproc_times.append(t_post)
        
        # ── Phase 2: Batched timing ──
        all_data = []
        with torch.no_grad():
            for i in range(n_snaps):
                data = dataset[i].to(device)
                net_idx, _ = dataset._index[i]
                _augment_edge_attr_learned(data, ptdf_params, max_buses, net_idx, device=device)
                all_data.append(data)
            
            # Warmup
            for _ in range(3):
                _ = batched_gnn_forward(model, all_data[:batch_size])
            
            # Timed batched forward in chunks
            batch_times = []
            batch_counts = []
            for start in range(0, n_snaps, batch_size):
                chunk = all_data[start:start + batch_size]
                t0 = time.perf_counter()
                _ = batched_gnn_forward(model, chunk)
                batch_times.append(time.perf_counter() - t0)
                batch_counts.append(len(chunk))
        
        batch_ms_per_snap = sum(batch_times) * 1000 / sum(batch_counts)
        
        # ── Store results ──
        solve_ms    = np.mean(solve_times) * 1000
        postproc_ms = np.mean(postproc_times) * 1000
        total_ms    = solve_ms + postproc_ms
        
        run['test_metrics']['solve_time_ms']       = solve_ms
        run['test_metrics']['postproc_time_ms']     = postproc_ms
        run['test_metrics']['total_time_ms']        = total_ms
        run['test_metrics']['batch_solve_time_ms']  = batch_ms_per_snap
        
        tag = run.get('tag', run.get('physics_label', '?'))
        print(f"  Run {ri:2d}: {tag:30s} | single={solve_ms:.2f}ms  batch={batch_ms_per_snap:.3f}ms  post={postproc_ms:.2f}ms  total={total_ms:.2f}ms")
    
    print(f"\nBackfilled timing for {len(runs)} runs.")

def backfill_line_flow_metrics(
    runs: list,
    test_networks: list,
    use_edge_features: bool = True,
    use_pnet_balance: bool = True,
    device=None,
):
    """
    Backfill line_flow_mae and line_flow_q0_mae into runs that are missing them.
    Requires live model objects in run['model'] and solved test_networks.
    """
    if device is None:
        device = torch.device('cpu')

    dataset = PowerFlowDataset(
        test_networks,
        use_edge_features=use_edge_features,
        use_pnet_balance=use_pnet_balance,
    )
    n_snaps = len(dataset)

    for ri, run in enumerate(runs):
        tm = run.get("test_metrics", {})
        if "line_flow_mae" in tm and not np.isnan(tm["line_flow_mae"]):
            print(f"  Run {ri}: already has line_flow_mae={tm['line_flow_mae']:.6f}, skipping")
            continue

        model = run['model']
        model.to(device)
        model.eval()

        ptdf_params = run.get('_ptdf_params', None)
        max_buses   = run.get('_max_buses', 0)

        lflow_p_errors = []
        lflow_q_errors = []

        with torch.no_grad():
            for i in range(n_snaps):
                data = dataset[i].to(device)
                net_idx, t_idx = dataset._index[i]
                network = test_networks[net_idx]
                snapshot = network.snapshots[t_idx]

                _augment_edge_attr_learned(data, ptdf_params, max_buses, net_idx, device=device)

                out = model(data)
                _raw_out = out
                if isinstance(out, tuple):
                    out = _raw_out[0]
                _angle_mode = getattr(model, "angle_mode", "node")  # [STSI 270526]:task 028
                _delta_theta_pred = _raw_out[2] if isinstance(_raw_out, tuple) and len(_raw_out) > 2 else None

                x = data.x
                bus_type   = x[:, 0]
                pv_mask    = (bus_type == 1)
                slack_mask = (bus_type == 2)

                vmag_pred = out[:, 0].clone()
                vang_pred = out[:, 1].clone()
                vmag_pred[pv_mask]    = x[pv_mask, 5]
                vmag_pred[slack_mask] = x[slack_mask, 5]
                vang_pred[slack_mask] = x[slack_mask, 6]

                vmag_np = vmag_pred.cpu().numpy()
                vang_np = vang_pred.cpu().numpy()

                try:
                    # [STSI 270526]:task 028 — direct Δθ for edge_delta models
                    if _angle_mode in ("edge_delta", "both") and _delta_theta_pred is not None:
                        _fwd_mask = np.zeros(data.edge_index.shape[1], dtype=bool)
                        _fwd_mask[::2] = True
                        _ei_fwd = data.edge_index[:, _fwd_mask].cpu().numpy()
                        _ea_fwd = data.edge_attr[_fwd_mask].cpu().numpy()
                        _dt_np = _delta_theta_pred.cpu().numpy()
                        flows = calculate_line_flows_from_delta_theta(
                            delta_theta=_dt_np,
                            vmag=vmag_np,
                            edge_index_fwd=_ei_fwd,
                            edge_attr_fwd=_ea_fwd,
                        )
                    else:
                        flows = calculate_line_flows(network, vmag_np, vang_np)

                    n_lines = len(network.lines)
                    if not network.lines_t.p0.empty and snapshot in network.lines_t.p0.index:
                        true_p0 = network.lines_t.p0.loc[snapshot].values
                        pred_p0 = np.array(flows["p0"])[:n_lines]
                        lflow_p_errors.extend(np.abs(pred_p0 - true_p0))  # [STSI 140926]:element-wise, not mean-of-means

                    if not network.lines_t.q0.empty and snapshot in network.lines_t.q0.index:
                        true_q0 = network.lines_t.q0.loc[snapshot].values
                        pred_q0 = np.array(flows["q0"])[:n_lines]
                        lflow_q_errors.extend(np.abs(pred_q0 - true_q0))
                except Exception as e:
                    if not lflow_p_errors:
                        print(f"    [Run {ri}] line flow calc failed: {e}")

        tm["line_flow_mae"]     = float(np.mean(lflow_p_errors)) if lflow_p_errors else float("nan")
        tm["line_flow_rmse"]    = float(np.sqrt(np.mean(np.array(lflow_p_errors)**2))) if lflow_p_errors else float("nan")
        tm["line_flow_q0_mae"]  = float(np.mean(lflow_q_errors)) if lflow_q_errors else float("nan")
        tm["line_flow_q0_rmse"] = float(np.sqrt(np.mean(np.array(lflow_q_errors)**2))) if lflow_q_errors else float("nan")
        run["test_metrics"] = tm

        tag = run.get("tag", run.get("physics_label", "?"))
        print(f"  Run {ri}: {tag:30s} | p0_mae={tm['line_flow_mae']:.6f}  q0_mae={tm['line_flow_q0_mae']:.6f}  ({len(lflow_p_errors)}/{n_snaps} snaps)")

    print(f"\nBackfilled line flow metrics for {len(runs)} runs.")

def backfill_all(
    runs: list,
    test_networks: list,
    use_edge_features: bool = True,
    use_pnet_balance: bool = True,
    n_warmup: int = 5,
    batch_size: int = 64,
    device=None,
    force: bool = False,
):
    """
    Single-pass backfill of timing + line flow metrics into sweep runs.

    Stores in test_metrics:
      solve_time_ms, postproc_time_ms, total_time_ms, batch_solve_time_ms,
      line_flow_mae, line_flow_q0_mae
    
    Skips runs that already have all keys unless force=True.
    """
    ALL_KEYS = {"solve_time_ms", "postproc_time_ms", "total_time_ms",
                "batch_solve_time_ms", "line_flow_mae", "line_flow_q0_mae"}

    if device is None:
        device = torch.device('cpu')

    dataset = PowerFlowDataset(
        test_networks,
        use_edge_features=use_edge_features,
        use_pnet_balance=use_pnet_balance,
    )
    n_snaps = len(dataset)
    print(f"Dataset: {n_snaps} snapshots, {n_warmup} warmup, batch_size={batch_size}")

    for ri, run in enumerate(runs):
        tm = run.setdefault("test_metrics", {})

        # Skip if all keys already present (and not NaN for line flows)
        if not force and ALL_KEYS.issubset(tm.keys()):
            lf = tm.get("line_flow_mae", float("nan"))
            if not np.isnan(lf):
                print(f"  Run {ri}: already complete, skipping (use force=True to redo)")
                continue

        model = run['model']
        model.to(device)
        model.eval()

        ptdf_params = run.get('_ptdf_params', None)
        max_buses   = run.get('_max_buses', 0)

        solve_times, postproc_times = [], []
        lflow_p_errors, lflow_q_errors = [], []
        # STSI 26.04.07: if ptdf_params is None, we can't augment edge_attr with learned PTDF rows, which may cause errors or degraded performance for models that expect those features. We check for this and skip timing if it looks like a mismatch.
        # Guard: skip learned_edge runs whose PTDF params were lost on save/load
        if ptdf_params is None:
            first_conv = model.convs[0]
            if hasattr(first_conv, 'lin_edge') and first_conv.lin_edge is not None:
                le = first_conv.lin_edge
                expected_edge_dim = getattr(le, 'in_channels', None) or getattr(le, 'in_features', None)
                sample_edge_dim = dataset[0].edge_attr.shape[1]
                if expected_edge_dim is not None and expected_edge_dim > sample_edge_dim:
                    tag = run.get("tag", run.get("physics_label", "?"))
                    print(f"  Run {ri}: SKIPPED — learned_edge PTDF params not available "
                          f"(model expects {expected_edge_dim} edge feats, data has {sample_edge_dim}) | {tag}")
                    continue
        # ── Phase 1: Per-snapshot timing + line flows ──
        with torch.no_grad():
            for i in range(n_snaps + n_warmup):
                idx = i % n_snaps
                data = dataset[idx].to(device)
                net_idx, t_idx = dataset._index[idx]
                network  = test_networks[net_idx]
                snapshot = network.snapshots[t_idx]
                _augment_edge_attr_learned(data, ptdf_params, max_buses, net_idx, device=device)

                # — Solve (timed) —
                t0 = time.perf_counter()
                out = model(data)
                _raw_out = out
                if isinstance(out, tuple):
                    out = _raw_out[0]
                _angle_mode = getattr(model, "angle_mode", "node")  # [STSI 270526]:task 028
                _delta_theta_pred = _raw_out[2] if isinstance(_raw_out, tuple) and len(_raw_out) > 2 else None
                t_solve = time.perf_counter() - t0

                # — Post-processing (timed) —
                t1 = time.perf_counter()
                x = data.x
                bus_type   = x[:, 0]
                pv_mask    = (bus_type == 1)
                slack_mask = (bus_type == 2)

                vmag_pred = out[:, 0].clone()
                vang_pred = out[:, 1].clone()
                vmag_pred[pv_mask]    = x[pv_mask, 5]
                vmag_pred[slack_mask] = x[slack_mask, 5]
                vang_pred[slack_mask] = x[slack_mask, 6]

                vmag_np = vmag_pred.cpu().numpy()
                vang_np = vang_pred.cpu().numpy()

                # [STSI 270526]:task 028 — direct Δθ for edge_delta models
                if _angle_mode in ("edge_delta", "both") and _delta_theta_pred is not None:
                    _fwd_mask = np.zeros(data.edge_index.shape[1], dtype=bool)
                    _fwd_mask[::2] = True
                    _ei_fwd = data.edge_index[:, _fwd_mask].cpu().numpy()
                    _ea_fwd = data.edge_attr[_fwd_mask].cpu().numpy()
                    _dt_np = _delta_theta_pred.cpu().numpy()
                    flows = calculate_line_flows_from_delta_theta(
                        delta_theta=_dt_np,
                        vmag=vmag_np,
                        edge_index_fwd=_ei_fwd,
                        edge_attr_fwd=_ea_fwd,
                    )
                else:
                    flows = calculate_line_flows(network, vmag_np, vang_np)
                t_post = time.perf_counter() - t1

                # Only record after warmup
                if i >= n_warmup:
                    solve_times.append(t_solve)
                    postproc_times.append(t_post)

                    # Line flow errors (always, not just after warmup — but we gate on i>=n_warmup)
                    n_lines = len(network.lines)
                    try:
                        if not network.lines_t.p0.empty and snapshot in network.lines_t.p0.index:
                            true_p0 = network.lines_t.p0.loc[snapshot].values
                            pred_p0 = np.array(flows["p0"])[:n_lines]
                            lflow_p_errors.extend(np.abs(pred_p0 - true_p0))  # [STSI 140926]:element-wise, not mean-of-means

                        if not network.lines_t.q0.empty and snapshot in network.lines_t.q0.index:
                            true_q0 = network.lines_t.q0.loc[snapshot].values
                            pred_q0 = np.array(flows["q0"])[:n_lines]
                            lflow_q_errors.extend(np.abs(pred_q0 - true_q0))
                    except Exception as e:
                        if not lflow_p_errors:
                            print(f"    [Run {ri}] line flow error: {e}")

        # ── Phase 2: Batched timing ──
        all_data = []
        with torch.no_grad():
            for i in range(n_snaps):
                data = dataset[i].to(device)
                net_idx, _ = dataset._index[i]
                _augment_edge_attr_learned(data, ptdf_params, max_buses, net_idx, device=device)
                all_data.append(data)

            # Warmup
            for _ in range(3):
                _ = batched_gnn_forward(model, all_data[:batch_size])

            batch_times, batch_counts = [], []
            for start in range(0, n_snaps, batch_size):
                chunk = all_data[start:start + batch_size]
                t0 = time.perf_counter()
                _ = batched_gnn_forward(model, chunk)
                batch_times.append(time.perf_counter() - t0)
                batch_counts.append(len(chunk))

        batch_ms_per_snap = sum(batch_times) * 1000 / max(1, sum(batch_counts))

        # ── Store everything ──
        solve_ms    = float(np.mean(solve_times)) * 1000
        postproc_ms = float(np.mean(postproc_times)) * 1000

        tm["solve_time_ms"]       = solve_ms
        tm["postproc_time_ms"]    = postproc_ms
        tm["total_time_ms"]       = solve_ms + postproc_ms
        tm["batch_solve_time_ms"] = batch_ms_per_snap
        tm["line_flow_mae"]       = float(np.mean(lflow_p_errors)) if lflow_p_errors else float("nan")
        tm["line_flow_rmse"]      = float(np.sqrt(np.mean(np.array(lflow_p_errors)**2))) if lflow_p_errors else float("nan")  # [STSI 140926]:added rmse alongside mae
        tm["line_flow_q0_mae"]    = float(np.mean(lflow_q_errors)) if lflow_q_errors else float("nan")
        tm["line_flow_q0_rmse"]   = float(np.sqrt(np.mean(np.array(lflow_q_errors)**2))) if lflow_q_errors else float("nan")

        tag = run.get("tag", run.get("physics_label", "?"))
        print(
            f"  Run {ri}: {tag:30s} | "
            f"single={solve_ms:.2f}ms  batch={batch_ms_per_snap:.3f}ms  "
            f"post={postproc_ms:.2f}ms  total={solve_ms+postproc_ms:.2f}ms | "
            f"p0_mae={tm['line_flow_mae']:.6f}  q0_mae={tm['line_flow_q0_mae']:.6f}"
        )

    print(f"\nBackfilled timing + line flows for {len(runs)} runs.")



## Comparison & Solver Baselines

In [ ]:
def _compute_delta_theta_metrics(vang_pred_bus, dataset, net_idx, t_idx, index_map):  # [STSI 010626]:shared helper for delta_theta error computation across all baselines
    """Compute delta_theta error arrays from per-bus vang prediction.
    
    Computes ground-truth delta_theta on-the-fly from data.y[:,1] (true bus angles)
    and forward_edge_mask. Does NOT depend on data.y_delta_theta (which only exists
    when angle_mode="edge_delta"). This allows all baselines (DC, PyPower, FDLF,
    pandapower) to report delta_theta metrics regardless of dataset angle_mode.
    
    vang_pred_bus: numpy array [n_buses] of predicted voltage angles (radians).
    Returns numpy array of absolute errors per forward edge, or None if lookup fails.
    """
    ds_idx = index_map.get((net_idx, t_idx))
    if ds_idx is None:
        return None
    data = dataset[ds_idx]
    # [STSI 020626]:compute truth from bus angles (always available) instead of
    # y_delta_theta (only set when angle_mode="edge_delta")
    fwd_mask = data.forward_edge_mask.numpy()
    ei = data.edge_index[:, fwd_mask].numpy()
    theta_true = data.y[:, 1].numpy()
    dt_true = theta_true[ei[0]] - theta_true[ei[1]]
    dt_pred = vang_pred_bus[ei[0]] - vang_pred_bus[ei[1]]
    return np.abs(dt_pred - dt_true)

def evaluate_gnn_on_test_set(
    model,
    test_networks: list,
    use_edge_features: bool = True,
    use_pnom_share: bool = None,  # None = auto-detect from model.node_embedding.in_features
    device=None,
    # STSI 26.04.07: learned_edge PTDF support
    ptdf_params=None,
    max_buses: int = 0,
    ptdf_offset: int = 0,
    # [STSI260502]: batch_size for batched timing measurement
    batch_size: int = 64,
    # [STSI260502]: flag to enable dangling bus postprocessing
    postprocess_dangling: bool = True,
    verbose: bool = False,
) -> dict:
    """
    Evaluate GNN on test networks snapshot-by-snapshot.

    KEY FIX (2026-03-25): After inference, override known variables with ground
    truth inputs — otherwise scatter plots show model predictions for variables
    the model was never asked to predict (Vmag/Vang at slack/PV), making results
    appear much worse than they are.

    NOTE: vang_mae/vang_rmse returned here are in RADIANS. evaluate_models_on_networks
    reports 'vang_mae_deg' in DEGREES -- do not compare the two directly
    (~57x = 180/pi unit difference).  # [STSI 140926]:clarify unit convention

    Override rules (mirror of _masked_mse_loss):
        PV    → vmag_eval[pv]    = x[pv,    5]  (known Vmag)
        Slack → vmag_eval[slack] = x[slack,  5]  (known Vmag = v_set)
        Slack → vang_eval[slack] = x[slack,  6]  (known Vang = 0)

    STSI 26.04.07: When ptdf_params is not None (learned_edge mode), augments
    data.edge_attr with learned PTDF rows before model forward pass so the
    model receives the same edge feature width it was trained with.

    [STSI260502]: Added multi-level timing:
      solve_time_ms:       pure forward pass per snapshot
      postproc_time_ms:    mask overrides + cpu transfer + line flow calc
      total_time_ms:       solve + postproc (single-snapshot end-to-end)
      batch_solve_time_ms: batched forward pass amortized per snapshot
    Dataset construction and data.to(device) excluded from all timers.

    Returns dict with MAE/RMSE for Vmag, Vang, P, Q + line flow MAE + timing.
    """
    if device is None:
        device = next(model.parameters()).device

    model.eval()
    angle_mode = getattr(model, "angle_mode", "node")  # [STSI 260526]:auto-detect angle_mode
    theta_recon_times = []  # [STSI 260526]:timing accumulator for edge_delta
    dataset = PowerFlowDataset(
        test_networks,
        use_edge_features=use_edge_features,
        use_pnom_share=(model.node_embedding.in_features >= 8) if use_pnom_share is None else use_pnom_share,  # [STSI100526] was >=9 (2-col); now >=8 (1-col: pnom_share only)
        angle_mode=angle_mode,  # [STSI 260526]:pass angle_mode
        ptdf_branch_mode="all" if angle_mode in ("edge_delta", "both") else "lines",  # [STSI 260526]:edge_delta needs all branches for BFS connectivity
    )

    all_vmag_err, all_vang_err, all_p_err, all_q_err = [], [], [], []  # [STSI 010626]:element-wise error lists for true RMSE
    vmag_true_all, vang_true_all, p_true_all, q_true_all = [], [], [], []

    all_lflow_err = []  # [STSI 010626]:element-wise line flow errors
    all_lflow_q0_err = []
    all_true_flows, all_pred_flows = [], []
    all_true_q0_flows, all_pred_q0_flows = [], []
    all_delta_theta_errors = []  # [STSI 270526]:task 028 — collect per-edge Δθ errors
    line_flow_errors_node_theta = []  # [STSI 030626]:dual-head comparison — node-θ line flow errors
    theta_consistency_errors = []  # [STSI 030626]:dual-head comparison — node-θ vs BFS-reconstructed-θ
    n_snapshots = 0

    # [STSI260502]: per-snapshot timing lists (solve and postproc measured separately)
    solve_times = []
    postproc_times = []
    # [STSI260502]: collect prepared data for batched timing later
    all_data_for_batch = []

    with torch.no_grad():
        for i in range(len(dataset)):
            data = dataset[i].to(device)

            # STSI 26.04.07: augment edge_attr for learned_edge models
            net_idx, _ = dataset._index[i]
            _augment_edge_attr_learned(data, ptdf_params, max_buses, net_idx, ptdf_offset, device)

            # [STSI260502]: save a copy for batched timing (before forward modifies anything)
            all_data_for_batch.append(data.clone())

            # [STSI260502]: Level 1 — time pure forward pass only
            t_solve_start = time.perf_counter()
            out = model(data, return_embeddings=True)
            node_pred = out[0]
            delta_theta_pred = out[3] if len(out) > 3 else None  # [STSI 260526]:4th elem in embeddings mode
            if device.type == 'cuda':
                torch.cuda.synchronize()
            t_solve = time.perf_counter() - t_solve_start
            # [STSI 240726]: start postproc timer BEFORE θ reconstruction so
            # total_time_ms includes θ recon (was leaking into theta_recon_times only)
            t_post_start = time.perf_counter()

            # θ reconstruction for edge_delta mode [STSI 260526]
            if angle_mode == "edge_delta" and delta_theta_pred is not None:
                t_recon_start = time.perf_counter()
                theta_recon = reconstruct_theta_from_path_matrix(  # [STSI 240726]:replaced BFS loop with matmul via path-incidence matrix
                    delta_theta_pred,
                    data.bfs_path_matrix,
                )
                theta_recon_times.append((time.perf_counter() - t_recon_start) * 1000)
                # Assemble [N, 4] from [N, 3] + reconstructed θ
                node_pred_full = torch.zeros(data.x.size(0), 4, device=device)
                node_pred_full[:, 0] = node_pred[:, 0]  # Vmag
                node_pred_full[:, 1] = theta_recon       # reconstructed θ
                node_pred_full[:, 2] = node_pred[:, 1]  # P
                node_pred_full[:, 3] = node_pred[:, 2]  # Q
                node_pred = node_pred_full

            # ── Extract masks ─────────────────────────────────────────────────
            slack_mask = data.slack_mask.bool()
            pv_mask    = data.pv_mask.bool()
            pq_mask    = data.pq_mask.bool()
            
            # ── Dangling bus post-processing ──────────────────────────────────
            if postprocess_dangling:
                node_pred = postprocess_dangling_buses(node_pred, data)
            
            # ── Raw predictions ───────────────────────────────────────────────
            vmag_eval = node_pred[:, 0].clone()
            vang_eval = node_pred[:, 1].clone()
            p_eval    = node_pred[:, 2].clone()
            q_eval    = node_pred[:, 3].clone()

            # ── Override known variables with ground-truth inputs ─────────────
            # PV: Vmag is known (set by voltage regulator)
            if pv_mask.any():
                vmag_eval[pv_mask] = data.x[pv_mask, 5]

            # Slack: both Vmag and Vang=0 are known
            if slack_mask.any():
                vmag_eval[slack_mask] = data.x[slack_mask, 5]
                vang_eval[slack_mask] = data.x[slack_mask, 6]  # should be ~0

            # PQ: P and Q are known inputs
            if pq_mask.any():
                p_eval[pq_mask] = data.x[pq_mask, 3]
                q_eval[pq_mask] = data.x[pq_mask, 4]

            # PV: P is known input
            if pv_mask.any():
                p_eval[pv_mask] = data.x[pv_mask, 3]

            # ── Ground truth ──────────────────────────────────────────────────
            vmag_true = data.y[:, 0]
            vang_true = data.y[:, 1]
            p_true    = data.y[:, 2]
            q_true    = data.y[:, 3]

            all_vmag_err.extend((vmag_eval - vmag_true).abs().cpu().numpy())  # [STSI 010626]:element-wise errors for correct RMSE
            all_vang_err.extend((vang_eval - vang_true).abs().cpu().numpy())
            all_p_err.extend((p_eval - p_true).abs().cpu().numpy())
            all_q_err.extend((q_eval - q_true).abs().cpu().numpy())
            vmag_true_all.append(vmag_true.detach())
            vang_true_all.append(vang_true.detach())
            p_true_all.append(p_true.detach())
            q_true_all.append(q_true.detach())
            n_snapshots += 1

            # ── Step 4 (partial): line flow MAE ──────────────────────────────
            if test_networks is not None and hasattr(dataset, '_index') and i < len(dataset._index):
                try:
                    net_idx, t_idx = dataset._index[i]
                    network  = test_networks[net_idx]
                    snapshot = network.snapshots[t_idx]
                    if not network.lines_t.p0.empty:
                        true_flows = network.lines_t.p0.loc[snapshot].values
                        vmag_np = vmag_eval.cpu().numpy()
                        vang_np = vang_eval.cpu().numpy()
                        # [STSI 270526]:task 028 — direct Δθ line flows for edge_delta models
                        if angle_mode in ("edge_delta", "both") and delta_theta_pred is not None:
                            _data_cpu = data.cpu()
                            _all_fwd_mask_np = np.zeros(_data_cpu.edge_index.shape[1], dtype=bool)
                            _all_fwd_mask_np[::2] = True
                            _ei_fwd_np = _data_cpu.edge_index[:, _all_fwd_mask_np].numpy()
                            _ea_fwd_np = _data_cpu.edge_attr[_all_fwd_mask_np].numpy()
                            _dt_pred_np = delta_theta_pred.cpu().numpy()
                            flows = calculate_line_flows_from_delta_theta(
                                delta_theta=_dt_pred_np,
                                vmag=vmag_np,
                                edge_index_fwd=_ei_fwd_np,
                                edge_attr_fwd=_ea_fwd_np,
                            )
                        else:
                            # [STSI 240726]:replaced pandas .loc loop with vectorized Δθ calc (avoids ~6ms/snap overhead)
                            _fwd_mask_np = np.zeros(data.edge_index.shape[1], dtype=bool)
                            _fwd_mask_np[::2] = True
                            _ei_fwd_np = data.edge_index[:, _fwd_mask_np].cpu().numpy()
                            _ea_fwd_np = data.edge_attr[_fwd_mask_np].cpu().numpy()
                            _dt_np = vang_np[_ei_fwd_np[0]] - vang_np[_ei_fwd_np[1]]
                            flows = calculate_line_flows_from_delta_theta(
                                delta_theta=_dt_np,
                                vmag=vmag_np,
                                edge_index_fwd=_ei_fwd_np,
                                edge_attr_fwd=_ea_fwd_np,
                            )
                        n_lines = len(network.lines)
                        pred_flows = np.array(flows["p0"])[:n_lines]  # lines only (excl transformers)
                        all_lflow_err.extend(np.abs(pred_flows - true_flows))  # [STSI 010626]:element-wise
                        all_true_flows.append(true_flows)
                        all_pred_flows.append(pred_flows)
                    if not network.lines_t.q0.empty:
                        true_q0 = network.lines_t.q0.loc[snapshot].values
                        pred_q0 = np.array(flows["q0"])[:n_lines]  # lines only
                        all_lflow_q0_err.extend(np.abs(pred_q0 - true_q0))  # [STSI 010626]:element-wise
                        all_true_q0_flows.append(true_q0)
                        all_pred_q0_flows.append(pred_q0)
                except Exception as _lfe:
                    if not all_lflow_err:  # only warn once
                        print(f"[line_flow_mae] skipped: {_lfe}")

            # [STSI 270526]:task 028 — collect per-edge Δθ errors for edge_delta models
            if angle_mode in ("edge_delta", "both") and hasattr(data, "y_delta_theta"):
                _dt_true = data.y_delta_theta.cpu().numpy()
                _dt_pred = (delta_theta_pred.cpu().numpy()
                            if delta_theta_pred is not None
                            else np.zeros_like(_dt_true))
                all_delta_theta_errors.append(np.abs(_dt_pred - _dt_true))

            # ── Dual-head θ comparison (angle_mode=="both" only) ──────────── [STSI 030626]
            if angle_mode == "both" and delta_theta_pred is not None:
                # 1. Line flows from NODE head θ (using θ_i − θ_j internally)
                if test_networks is not None and hasattr(dataset, '_index') and i < len(dataset._index):
                    try:
                        _net_idx_dh, _t_idx_dh = dataset._index[i]
                        _network_dh = test_networks[_net_idx_dh]
                        _snapshot_dh = _network_dh.snapshots[_t_idx_dh]
                        if not _network_dh.lines_t.p0.empty:
                            _true_flows_dh = _network_dh.lines_t.p0.loc[_snapshot_dh].values
                            _vmag_np_dh = vmag_eval.cpu().numpy()
                            _vang_np_dh = vang_eval.cpu().numpy()  # node-head θ
                            _flows_node = calculate_line_flows(_network_dh, _vmag_np_dh, _vang_np_dh)
                            _n_lines_dh = len(_network_dh.lines)
                            _pred_flows_node = np.array(_flows_node["p0"])[:_n_lines_dh]
                            line_flow_errors_node_theta.extend(np.abs(_pred_flows_node - _true_flows_dh))
                    except Exception:
                        pass

                # 2. θ consistency: node head θ vs BFS-reconstructed θ from edge Δθ
                _theta_recon_dh = reconstruct_theta_from_path_matrix(  # [STSI 240726]:use vectorized matmul recon
                    delta_theta_pred,
                    data.bfs_path_matrix,
                )
                _node_theta = vang_eval.cpu().numpy()
                _recon_theta = _theta_recon_dh.cpu().numpy()
                _slack_idx_dh = int(data.slack_mask.bool().nonzero(as_tuple=True)[0][0])
                _non_slack = np.ones(len(_node_theta), dtype=bool)
                _non_slack[_slack_idx_dh] = False
                theta_consistency_errors.extend(np.abs(_node_theta[_non_slack] - _recon_theta[_non_slack]))

            # [STSI260502]: end post-processing timer (includes masks + line flows)
            t_post = time.perf_counter() - t_post_start
            solve_times.append(t_solve)
            postproc_times.append(t_post)

    # [STSI260502]: Phase 2 — batched timing (forward + PARALLEL postproc)
    # [STSI 240726]: postproc now timed on the batched output (one unit per chunk),
    # amortized per snapshot. batch_solve_time_ms = forward + parallel postproc.
    n_batch_warmup = 3
    _EXCLUDE_BATCH = ['y_ptdf', 'ptdf_line_index', 'y_line_p',
                      'bfs_edge_idx', 'bfs_signs','bfs_path_matrix', 'bfs_node_order',
                      'bfs_parent', 'y_delta_theta']  # variable-size keys excluded from PyG batch
    with torch.no_grad():
        if all_data_for_batch:
            # Warmup batched forward (optionally run one postproc pass too, to
            # warm numpy / .cpu() paths so the first timed chunk isn't cold)
            for _ in range(n_batch_warmup):
                _wb = Batch.from_data_list(all_data_for_batch[:batch_size], exclude_keys=_EXCLUDE_BATCH)
                _ = model(_wb)

            batch_fwd_times, batch_pp_times, batch_counts = [], [], []
            for start in range(0, len(all_data_for_batch), batch_size):
                chunk = all_data_for_batch[start:start + batch_size]
                bp_batch = Batch.from_data_list(chunk, exclude_keys=_EXCLUDE_BATCH)

                # ── Level 1: batched forward (timed) ──────────────
                t_fwd = time.perf_counter()
                bp_out = model(bp_batch)
                if device.type == 'cuda':
                    torch.cuda.synchronize()
                batch_fwd_times.append(time.perf_counter() - t_fwd)

                bp_node = bp_out[0] if isinstance(bp_out, tuple) else bp_out
                bp_dt   = bp_out[2] if isinstance(bp_out, tuple) and len(bp_out) > 2 else None

                # ── Level 2: batched PARALLEL postproc (timed) ────────
                t_pp = time.perf_counter()

                # (a) θ reconstruction — edge_delta only, per-graph matmul via path-incidence matrix  # [STSI 240726]:replaced BFS loop with per-graph matmul
                if angle_mode == "edge_delta" and bp_dt is not None:
                    vang_stacked = torch.zeros(bp_node.size(0), device=device)
                    node_off, dt_off = 0, 0
                    for g_data in chunk:
                        n_g = g_data.x.size(0)
                        e_fwd_g = g_data.edge_index.size(1) // 2  # [::2] forward edges
                        theta_g = reconstruct_theta_from_path_matrix(
                            bp_dt[dt_off:dt_off + e_fwd_g],
                            g_data.bfs_path_matrix,  # [n_g, e_fwd_g] — one matmul, no scalar loop
                        )
                        vang_stacked[node_off:node_off + n_g] = theta_g
                        node_off += n_g
                        dt_off   += e_fwd_g
                    # bp_node is [N,3]=(Vmag,P,Q) for edge_delta → assemble [N,4]
                    bp_eval = torch.zeros(bp_node.size(0), 4, device=device)
                    bp_eval[:, 0] = bp_node[:, 0]
                    bp_eval[:, 1] = vang_stacked
                    bp_eval[:, 2] = bp_node[:, 1]
                    bp_eval[:, 3] = bp_node[:, 2]
                else:
                    bp_eval = bp_node  # [N,4] for node / both

                # (b) dangling-bus fix (works on batched graph — no cross-graph edges)
                if postprocess_dangling:
                    bp_eval = postprocess_dangling_buses(bp_eval, bp_batch)

                # (c) vectorized mask overrides on the stacked node tensor
                vmag_b = bp_eval[:, 0].clone()
                vang_b = bp_eval[:, 1].clone()
                _sm = bp_batch.slack_mask.bool()
                _pv = bp_batch.pv_mask.bool()
                if _pv.any():
                    vmag_b[_pv] = bp_batch.x[_pv, 5]
                if _sm.any():
                    vmag_b[_sm] = bp_batch.x[_sm, 5]
                    vang_b[_sm] = bp_batch.x[_sm, 6]

                # (d) PARALLEL line flows: one vectorized call over the whole chunk
                #     (batched edge_index already carries GLOBAL node offsets)
                _fwd = torch.zeros(bp_batch.edge_index.size(1), dtype=torch.bool, device=device)
                _fwd[::2] = True
                ei_fwd  = bp_batch.edge_index[:, _fwd].cpu().numpy()
                ea_fwd  = bp_batch.edge_attr[_fwd].cpu().numpy()
                vmag_np = vmag_b.cpu().numpy()
                if angle_mode in ("edge_delta", "both") and bp_dt is not None:
                    dt_np = bp_dt.cpu().numpy()             # aligned with edge_index[:, ::2]
                else:
                    vang_np = vang_b.cpu().numpy()
                    dt_np = vang_np[ei_fwd[0]] - vang_np[ei_fwd[1]]
                _ = calculate_line_flows_from_delta_theta(
                    delta_theta=dt_np, vmag=vmag_np,
                    edge_index_fwd=ei_fwd, edge_attr_fwd=ea_fwd,
                )

                if device.type == 'cuda':
                    torch.cuda.synchronize()
                batch_pp_times.append(time.perf_counter() - t_pp)
                batch_counts.append(len(chunk))

            _cnt = max(1, sum(batch_counts))
            batch_inference_only_ms = sum(batch_fwd_times) * 1000 / _cnt
            batch_postproc_ms       = sum(batch_pp_times)  * 1000 / _cnt
        else:
            batch_inference_only_ms = float('nan')
            batch_postproc_ms       = float('nan')

    metrics = {  # [STSI 010626]:element-wise MAE and RMSE (flat list pattern)
        "vmag_mae":      float(np.mean(all_vmag_err)),
        "vang_mae":      float(np.mean(all_vang_err)),
        "p_mae":         float(np.mean(all_p_err)),
        "q_mae":         float(np.mean(all_q_err)),
        "vmag_rmse":     float(np.sqrt(np.mean(np.array(all_vmag_err)**2))),
        "vang_rmse":     float(np.sqrt(np.mean(np.array(all_vang_err)**2))),
        "vmag_true_std": float(torch.cat(vmag_true_all).std().item()),
        "vang_true_std": float(torch.cat(vang_true_all).std().item()),
        "p_true_std":    float(torch.cat(p_true_all).std().item()),
        "q_true_std":    float(torch.cat(q_true_all).std().item()),
    }
    if all_lflow_err:  # [STSI 010626]:element-wise RMSE
        metrics["line_flow_mae"] = float(np.mean(all_lflow_err))
        metrics["line_flow_rmse"] = float(np.sqrt(np.mean(np.array(all_lflow_err)**2)))
    if all_lflow_q0_err:
        metrics["line_flow_q0_mae"] = float(np.mean(all_lflow_q0_err))
        metrics["line_flow_q0_rmse"] = float(np.sqrt(np.mean(np.array(all_lflow_q0_err)**2)))
    metrics["p_rmse"] = float(np.sqrt(np.mean(np.array(all_p_err)**2)))
    metrics["q_rmse"] = float(np.sqrt(np.mean(np.array(all_q_err)**2)))

    if all_true_flows:
        raw = {
            "p0_true": np.concatenate(all_true_flows),
            "p0_pred": np.concatenate(all_pred_flows),
        }
        if all_true_q0_flows:
            raw["q0_true"] = np.concatenate(all_true_q0_flows)
            raw["q0_pred"] = np.concatenate(all_pred_q0_flows)
        metrics["line_flow_raw"] = raw
    if verbose:     
        print("----------------\n Evaluation Metrics (known variables overridden) ---------------")
        for k, v in metrics.items():
            if isinstance(v, (int, float)):
                print(f"  {k:20s}: {v:.6f}")

    # [STSI260502]: monolithic timing kept for backward compat
    inference_time_s = sum(solve_times) + sum(postproc_times)
    metrics["inference_time_s"] = inference_time_s
    metrics["inference_time_per_snapshot_ms"] = (
        1000.0 * inference_time_s / max(1, n_snapshots)
    )
    
    # [STSI260502]: new multi-level timing metrics
    solve_ms    = float(np.mean(solve_times)) * 1000
    postproc_ms = float(np.mean(postproc_times)) * 1000
    metrics["solve_time_ms"]       = solve_ms
    metrics["postproc_time_ms"]    = postproc_ms
    metrics["total_time_ms"]       = solve_ms + postproc_ms
    # [STSI 240726]: batch_solve_time_ms is now the PARALLEL total (batched forward
    # + batched postproc), overwriting the 067 conservative (forward + serial) total.
    metrics["batch_inference_only_time_ms"] = batch_inference_only_ms
    metrics["batch_postproc_time_ms"]       = batch_postproc_ms  # diagnostic; NOT in REQUIRED_METRICS
    metrics["batch_solve_time_ms"]          = batch_inference_only_ms + batch_postproc_ms
    if theta_recon_times:  # [STSI 260526]:edge_delta timing metric
        metrics["theta_reconstruction_time_ms"] = float(np.mean(theta_recon_times))
    if all_delta_theta_errors:  # [STSI 270526]:task 028 — Δθ accuracy metrics
        _all_dt_err = np.concatenate(all_delta_theta_errors)
        metrics["delta_theta_mae"] = float(np.mean(_all_dt_err))
        metrics["delta_theta_rmse"] = float(np.sqrt(np.mean(_all_dt_err**2)))
    # [STSI 030626]:dual-head θ consistency metrics (angle_mode=="both" only)
    if angle_mode == "both":
        if all_lflow_err:
            metrics["line_flow_rmse_delta_theta"] = metrics.get("line_flow_rmse", float("nan"))
            metrics["line_flow_mae_delta_theta"] = metrics.get("line_flow_mae", float("nan"))
        if line_flow_errors_node_theta:
            _arr_nt = np.array(line_flow_errors_node_theta)
            metrics["line_flow_rmse_node_theta"] = float(np.sqrt(np.mean(_arr_nt**2)))
            metrics["line_flow_mae_node_theta"] = float(np.mean(_arr_nt))
        if theta_consistency_errors:
            _arr_tc = np.array(theta_consistency_errors)
            metrics["theta_consistency_mae"] = float(np.mean(_arr_tc))
            metrics["theta_consistency_rmse"] = float(np.sqrt(np.mean(_arr_tc**2)))
    if verbose:
        print(f"\n  --- Timing breakdown (per snapshot) ---")
        print(f"  {'solve_time_ms':20s}: {solve_ms:.3f}")
        print(f"  {'postproc_time_ms':20s}: {postproc_ms:.3f}")
        print(f"  {'total_time_ms':20s}: {solve_ms + postproc_ms:.3f}")
        print(f"  {'batch_fwd_only_ms':20s}: {batch_inference_only_ms:.3f}")
        print(f"  {'batch_postproc_ms':20s}: {batch_postproc_ms:.3f}")
        print(f"  {'batch_solve_time_ms':20s}: {(batch_inference_only_ms + batch_postproc_ms):.3f}")

    return metrics

def postprocess_dangling_buses(node_pred, data, p_thresh=1e-4):
    """
    Fix predictions for dangling buses (degree-1 PQ buses with P=Q=0).
    Physics: no current flows → Vmag/Vang must equal neighbor's values.
    Modifies vmag (col 0) and vang (col 1) only; leaves P, Q untouched.
    """
    pred = node_pred.clone()
    x = data.x          # [N,7]: [is_slack, is_pv, is_pq, P, Q, Vmag, Vang]
    ei = data.edge_index  # [2, E]
    N = x.shape[0]

    # Degree (edges appear in both directions in PyG undirected graphs)
    deg = torch.zeros(N, dtype=torch.long, device=x.device) # check for degree-1 buses
    deg.scatter_add_(0, ei[0], torch.ones(ei.shape[1], dtype=torch.long, device=x.device)) # count outgoing edges

    dangling = (deg == 1) & (x[:, 2] > 0.5) & (x[:, 3].abs() < p_thresh) & (x[:, 4].abs() < p_thresh)   # degree-1 PQ buses with P≈0 and Q≈0
    if not dangling.any(): # early exit if no dangling buses
        return pred 

    for i in dangling.nonzero(as_tuple=True)[0].tolist(): # for each dangling bus
        nbrs = ei[1][ei[0] == i]   # check which bus it's connected to (should be exactly one neighbor in an undirected graph)
        nbrs = torch.cat([nbrs, ei[0][ei[1] == i]])  # check both directions to be safe
        nbrs = nbrs[nbrs != i] # exclude self-loop if present
        if len(nbrs) == 0: # 
            continue
        j = nbrs[0].item()  # neighbor index (should be only one)
        pred[i, 0] = pred[j, 0]  # vmag ← neighbour # STSI 26.05.0: this is a simple heuristic fix for dangling buses — we copy the neighbor's Vmag and Vang predictions, since a dangling bus with no load should have the same voltage as its single neighbor. This is a post-processing step that modifies the model's raw predictions to enforce a known physical constraint, and should improve accuracy for these edge cases.
        pred[i, 1] = pred[j, 1]  # vang ← neighbour
    return pred

def evaluate_dc_baseline(
    test_networks,
    use_edge_features: bool = True,
    use_pnet_balance: bool = True,
    verbose: bool = True,
) -> dict:
    """
    Run PyPSA linear (DC) power flow on all test-set snapshots and compute
    the same MAE metrics as evaluate_gnn_on_test_set.

    DC capabilities:
        vmag:          flat 1.0 p.u. (DC assumption) — error at PQ buses
        vang:          solved by lpf()               — error at PQ+PV buses
        p injections:  lossless balance              — error at Slack bus
        q injections:  naive zero guess (DC has no Q model) — error at PV+Slack buses
        line_flow_mae: p0 from lpf() vs AC p0        — valid comparison
        line_flow_q0:  naive zero guess (DC has no Q model) — same treatment as q injections

    Both Q-related metrics are a zero-guess, not a real DC capability — plotting
    code (plot_timing_vs_accuracy) marks DC's Q-metric points with a distinct
    marker to avoid implying DC actually solves for reactive power.

    NOTE: vang_mae/vang_rmse returned here are in RADIANS. evaluate_models_on_networks
    reports 'vang_mae_deg' in DEGREES -- do not compare the two directly
    (~57x = 180/pi unit difference).  # [STSI 140926]:clarify unit convention

    Returns a dict compatible with evaluate_gnn_on_test_set output.
    """
    dataset = PowerFlowDataset(
        test_networks,
        use_edge_features=use_edge_features,
        use_pnet_balance=use_pnet_balance,
    )
    index_map = {(ni, ti): k for k, (ni, ti) in enumerate(dataset._index)}

    all_vmag_err, all_vang_err, all_p_err, all_q_err = [], [], [], []  # [STSI 010626]:element-wise errors for correct RMSE
    all_lflow_err = []
    all_lflow_q0_err = []  # naive Q=0 guess, mirrors node-level q_dc=0 treatment
    all_dt_errors = []
    n_snapshots = 0
    t_solve = 0.0

    for net_idx, network in enumerate(test_networks):
        if verbose and net_idx % 20 == 0:
            print(f"  [DC] network {net_idx + 1}/{len(test_networks)}")

        net_copy = network.copy()
        t0 = time.time()
        try:
            net_copy.lpf()
        except Exception as e:
            if verbose:
                print(f"  [DC] lpf() failed on network {net_idx}: {e}")
            continue
        t_solve += time.time() - t0

        buses = network.buses.index

        for t_idx, snap in enumerate(network.snapshots):
            ds_idx = index_map.get((net_idx, t_idx))
            if ds_idx is None:
                continue

            data       = dataset[ds_idx]
            slack_mask = data.slack_mask.bool().numpy()
            pv_mask    = data.pv_mask.bool().numpy()
            pq_mask    = data.pq_mask.bool().numpy()
            x_np       = data.x.numpy()
            y_np       = data.y.numpy()

            vmag_true = y_np[:, 0]
            vang_true = y_np[:, 1]
            p_true    = y_np[:, 2]
            q_true    = y_np[:, 3]

            # DC predictions --------------------------------------------------
            vmag_dc = np.ones(len(buses))                                      # flat 1.0 p.u.
            vang_dc = net_copy.buses_t.v_ang.loc[snap].reindex(buses).values   # from DC solve
            p_dc    = x_np[:, 3].copy()                                        # given P inputs
            q_dc    = np.zeros(len(buses))                                     # DC: Q=0

            # DC slack P: lossless power balance
            if slack_mask.any():
                p_dc[slack_mask] = -p_true[~slack_mask].sum() / slack_mask.sum()

            # Apply same known-variable overrides as evaluate_gnn_on_test_set -
            vmag_dc[pv_mask]    = x_np[pv_mask,    5]   # PV: Vmag given
            vmag_dc[slack_mask] = x_np[slack_mask, 5]   # Slack: Vmag given
            vang_dc[slack_mask] = x_np[slack_mask, 6]   # Slack: Vang=0 given
            p_dc[pq_mask]       = x_np[pq_mask,    3]   # PQ: P given
            q_dc[pq_mask]       = x_np[pq_mask,    4]   # PQ: Q given
            p_dc[pv_mask]       = x_np[pv_mask,    3]   # PV: P given

            all_vmag_err.extend(np.abs(vmag_dc - vmag_true))  # [STSI 010626]:element-wise errors
            all_vang_err.extend(np.abs(vang_dc - vang_true))
            all_p_err.extend(np.abs(p_dc - p_true))
            all_q_err.extend(np.abs(q_dc - q_true))
            n_snapshots += 1

            # DC line flows (p0 only — q0 is zero in DC) ----------------------
            if not network.lines_t.p0.empty and snap in network.lines_t.p0.index \
                    and snap in net_copy.lines_t.p0.index:
                true_p0 = network.lines_t.p0.loc[snap].values
                dc_p0   = net_copy.lines_t.p0.loc[snap].values
                # sn_mva is cosmetic metadata only (flat-pu convention) — no scaling needed
                all_lflow_err.extend(np.abs(dc_p0 - true_p0))  # [STSI 140926]:removed stale sn_mva division, matches unscaled true_p0

            # DC reactive line flow: naive zero guess (DC has no Q model, mirrors node-level q_dc=0)
            if not network.lines_t.q0.empty and snap in network.lines_t.q0.index:
                true_q0 = network.lines_t.q0.loc[snap].values
                all_lflow_q0_err.extend(np.abs(true_q0))  # guess = 0

            # DC delta_theta from bus angles  # [STSI 010626]:add delta_theta metrics to DC baseline
            dt_errs = _compute_delta_theta_metrics(vang_dc, dataset, net_idx, t_idx, index_map)
            if dt_errs is not None:
                all_dt_errors.extend(dt_errs)  # [STSI 010626]:element-wise for correct RMSE

    metrics = {  # [STSI 010626]:element-wise MAE and RMSE (flat list pattern)
        "vmag_mae":    float(np.mean(all_vmag_err)) if all_vmag_err else float("nan"),
        "vmag_rmse":   float(np.sqrt(np.mean(np.array(all_vmag_err)**2))) if all_vmag_err else float("nan"),
        "vang_mae":    float(np.mean(all_vang_err)) if all_vang_err else float("nan"),
        "vang_rmse":   float(np.sqrt(np.mean(np.array(all_vang_err)**2))) if all_vang_err else float("nan"),
        "p_mae":       float(np.mean(all_p_err))    if all_p_err    else float("nan"),
        "p_rmse":      float(np.sqrt(np.mean(np.array(all_p_err)**2))) if all_p_err else float("nan"),
        "q_mae":       float(np.mean(all_q_err))    if all_q_err    else float("nan"),
        "q_rmse":      float(np.sqrt(np.mean(np.array(all_q_err)**2))) if all_q_err else float("nan"),
        "line_flow_mae":     float(np.mean(all_lflow_err)) if all_lflow_err else float("nan"),
        "line_flow_rmse":    float(np.sqrt(np.mean(np.array(all_lflow_err)**2))) if all_lflow_err else float("nan"),
        "line_flow_q0_mae":  float(np.mean(all_lflow_q0_err)) if all_lflow_q0_err else float("nan"),   # naive Q=0 guess
        "line_flow_q0_rmse": float(np.sqrt(np.mean(np.array(all_lflow_q0_err)**2))) if all_lflow_q0_err else float("nan"),
        "delta_theta_mae":   float(np.mean(all_dt_errors)) if all_dt_errors else float("nan"),
        "delta_theta_rmse":  float(np.sqrt(np.mean(np.array(all_dt_errors)**2))) if all_dt_errors else float("nan"),
        "inference_time_s":               t_solve,
        "inference_time_per_snapshot_ms": 1000.0 * t_solve / max(1, n_snapshots),
        "n_snapshots":  n_snapshots,
    }

    if verbose:
        print(
            f"\n[DC baseline] {n_snapshots} snapshots — "
            f"pure solve time {t_solve:.2f}s "
            f"({metrics['inference_time_per_snapshot_ms']:.3f} ms/snap)"
        )
        for k in ["vmag_mae", "vang_mae", "p_mae", "q_mae", "line_flow_mae"]:
            print(f"  {k:25s}: {metrics[k]:.5f}")

    return metrics


In [ ]:
#=============================================================================
# SECTION 10b: COMPARISON WITH OTHER SOLVERS
#=============================================================================

#import numpy as np
#import time
from pypower.api import runpf, ppoption
from pypower.idx_bus import (BUS_I, BUS_TYPE, PD, QD, GS, BS,
                              BUS_AREA, VM, VA, BASE_KV, ZONE, VMAX, VMIN)
from pypower.idx_gen import (GEN_BUS, PG, QG, QMAX, QMIN, VG,
                              MBASE, GEN_STATUS, PMAX, PMIN)
from pypower.idx_brch import (F_BUS, T_BUS, BR_R, BR_X, BR_B, RATE_A,
                               RATE_B, RATE_C, TAP, SHIFT, BR_STATUS, PF, QF, PT, QT)


def pypsa_to_ppc(network, snapshot, base_mva=1.0):
    """Convert a PyPSA network at a single snapshot to a PyPower ppc dict.
    
    Assumes network values are already in p.u. (v_nom=1.0, line lengths=0).
    All outputs remain in p.u. with the given base_mva.
    
    Returns:
        ppc, bus_names, bus_idx, line_names, trafo_names
    """
    bus_names = network.buses.index.tolist()
    bus_idx = {name: i + 1 for i, name in enumerate(bus_names)}  # 1-indexed
    n = len(bus_names)

    # Bus type sets from generators
    slack_buses = set(network.generators[network.generators['control'] == 'Slack']['bus'])
    pv_buses = set(network.generators[network.generators['control'] == 'PV']['bus'])

    # Aggregate loads per bus for this snapshot
    lp = network.loads_t.p_set.loc[snapshot]
    lq = network.loads_t.q_set.loc[snapshot]
    bus_p, bus_q = {}, {}
    for lname, ldata in network.loads.iterrows():
        b = ldata['bus']
        bus_p[b] = bus_p.get(b, 0.0) + lp.get(lname, 0.0)
        bus_q[b] = bus_q.get(b, 0.0) + lq.get(lname, 0.0)

    # --- Bus matrix (13 cols) ---
    bus_mat = np.zeros((n, 13))
    for i, bname in enumerate(bus_names):
        btype = 3 if bname in slack_buses else (2 if bname in pv_buses else 1)
        bus_mat[i, BUS_I]    = i + 1
        bus_mat[i, BUS_TYPE] = btype
        bus_mat[i, PD]       = bus_p.get(bname, 0.0)
        bus_mat[i, QD]       = bus_q.get(bname, 0.0)
        bus_mat[i, VM]       = 1.0
        bus_mat[i, VA]       = 0.0
        bus_mat[i, BASE_KV]  = network.buses.loc[bname, 'v_nom']
        bus_mat[i, VMAX]     = 1.1
        bus_mat[i, VMIN]     = 0.9

    # Get solved generator outputs for this snapshot
    gen_p_solved, gen_q_solved = {}, {}
    if len(network.generators_t.p) > 0:
        row_gp = network.generators_t.p.loc[snapshot]
        for gname in network.generators.index:
            if gname in row_gp.index:
                gen_p_solved[gname] = float(row_gp[gname])
    if hasattr(network.generators_t, 'q') and len(network.generators_t.q) > 0:
        row_gq = network.generators_t.q.loc[snapshot]
        for gname in network.generators.index:
            if gname in row_gq.index:
                gen_q_solved[gname] = float(row_gq[gname])

    # --- Gen matrix (10 cols) ---
    gen_rows = []
    for gname, gdata in network.generators.iterrows():
        b = gdata['bus']
        vg = float(network.buses_t.v_mag_pu.loc[snapshot, b])
        row = np.zeros(10)
        row[GEN_BUS]    = bus_idx[b]
        row[VG]         = vg
        row[MBASE]      = base_mva
        row[GEN_STATUS] = 1
        row[QMAX] = 9999.0;  row[QMIN] = -9999.0
        row[PMAX] = 9999.0;  row[PMIN] = -9999.0
        if gdata['control'] != 'Slack':
            row[PG] = gen_p_solved.get(gname, 0.0)  # PV: fixed P, pypower solves Q
        if gdata['control'] == 'PQ':
            row[QG] = gen_q_solved.get(gname, 0.0)  # PQ: both fixed
        gen_rows.append(row)

    gen_mat = np.array(gen_rows)

    # --- Branch matrix — lines ---
    branch_rows = []
    line_names = network.lines.index.tolist()
    for lname, line in network.lines.iterrows():
        row = np.zeros(13)
        row[F_BUS]     = bus_idx[line['bus0']]
        row[T_BUS]     = bus_idx[line['bus1']]
        row[BR_R]      = line['r']
        row[BR_X]      = line['x']
        row[BR_B]      = line['b']
        row[RATE_A]    = line['s_nom']
        row[BR_STATUS] = 1
        row[TAP]       = 0.0   # 0 → treated as 1.0 by pypower
        branch_rows.append(row)

    # --- Branch matrix — transformers (use raw r/x, not r_pu/x_pu) ---
    trafo_names = network.transformers.index.tolist()
    for tname, tr in network.transformers.iterrows():
        row = np.zeros(13)
        row[F_BUS]     = bus_idx[tr['bus0']]
        row[T_BUS]     = bus_idx[tr['bus1']]
        row[BR_R]      = tr['r']
        row[BR_X]      = tr['x']
        row[BR_B]      = tr.get('b', 0.0)
        row[RATE_A]    = tr['s_nom']
        row[TAP]       = tr['tap_ratio'] if tr['tap_ratio'] != 0 else 1.0
        row[SHIFT]     = tr['phase_shift']
        row[BR_STATUS] = 1
        branch_rows.append(row)

    branch_mat = np.array(branch_rows)

    ppc = {'version': '2', 'baseMVA': base_mva,
           'bus': bus_mat, 'gen': gen_mat, 'branch': branch_mat}

    return ppc, bus_names, bus_idx, line_names, trafo_names


def evaluate_pypower_baseline(networks, snapshots=None, base_mva=1.0, verbose=True,
                              test_networks=None, use_edge_features=True, use_pnet_balance=True):  # [STSI 010626]:add test_networks for delta_theta
    """Run PyPower AC PF on all networks/snapshots, compare to PyPSA ground truth.
    
    All comparisons are in p.u.  Returns dict compatible with scatter plot functions:
      vmag_mae, vang_mae, p_mae, q_mae, line_flow_mae, line_flow_q0_mae,
      inference_time_per_snapshot_ms, n_snapshots, n_converged, n_failed
    """
    ppopt = ppoption(VERBOSE=0, OUT_ALL=0)

    all_vmag_err, all_vang_err = [], []
    all_p_err, all_q_err = [], []
    all_line_p_err, all_line_q_err = [], []
    all_times = []
    n_converged, n_failed = 0, 0

    # Optional: build dataset for delta_theta computation  # [STSI 010626]:delta_theta requires edge topology from dataset
    # Guard: skip if custom snapshots passed (index alignment would be wrong)
    _dataset, _index_map = None, None
    if test_networks is not None and snapshots is None:
        _dataset = PowerFlowDataset(test_networks, use_edge_features=use_edge_features,
                                    use_pnet_balance=use_pnet_balance)
        _index_map = {(ni, ti): k for k, (ni, ti) in enumerate(_dataset._index)}
    all_dt_errors = []

    for net_idx, network in enumerate(networks):
        snaps = snapshots if snapshots is not None else network.snapshots
        bus_names = network.buses.index.tolist()
        line_names = network.lines.index.tolist()
        n_buses = len(bus_names)
        n_lines = len(line_names)

        # Check which time-series exist in ground truth
        has_bus_p = hasattr(network.buses_t, 'p') and len(network.buses_t.p) > 0
        has_bus_q = hasattr(network.buses_t, 'q') and len(network.buses_t.q) > 0
        has_line_p = hasattr(network.lines_t, 'p0') and len(network.lines_t.p0) > 0
        has_line_q = hasattr(network.lines_t, 'q0') and len(network.lines_t.q0) > 0

        for t_idx, snap in enumerate(snaps):
            # --- Convert & ----
            ppc, _, bus_idx, _, _ = pypsa_to_ppc(network, snap, base_mva)
            # --- Solve (timed) ---
            t0 = time.perf_counter()
            result, success = runpf(ppc, ppopt)
            t1 = time.perf_counter()
            all_times.append((t1 - t0) * 1000.0)

            if not success:
                n_failed += 1
                continue
            n_converged += 1

            # Sort result buses by index
            res_bus = result['bus'][np.argsort(result['bus'][:, BUS_I].astype(int))]

            # --- Voltage comparison (p.u. / radians) ---
            pypsa_vmag = network.buses_t.v_mag_pu.loc[snap].reindex(bus_names).values
            pypsa_vang = network.buses_t.v_ang.loc[snap].reindex(bus_names).values
            pp_vmag = res_bus[:, VM]
            pp_vang = np.deg2rad(res_bus[:, VA])

            all_vmag_err.extend(np.abs(pypsa_vmag - pp_vmag))
            all_vang_err.extend(np.abs(pypsa_vang - pp_vang))

            # Delta_theta from bus angles (if dataset available)
            if _dataset is not None:
                dt_errs = _compute_delta_theta_metrics(pp_vang, _dataset, net_idx, t_idx, _index_map)
                if dt_errs is not None:
                    all_dt_errors.extend(dt_errs)  # [STSI 010626]:element-wise for correct RMSE

            # --- Bus P/Q injection (p.u.) ---
            if has_bus_p or has_bus_q:
                pp_gen = result['gen']
                pp_bus_pgen = np.zeros(n_buses)
                pp_bus_qgen = np.zeros(n_buses)
                for g in range(pp_gen.shape[0]):
                    bidx = int(pp_gen[g, GEN_BUS]) - 1
                    pp_bus_pgen[bidx] += pp_gen[g, PG]
                    pp_bus_qgen[bidx] += pp_gen[g, QG]
                if has_bus_p:
                    pypsa_p = network.buses_t.p.loc[snap].reindex(bus_names).values
                    pp_p_inj = pp_bus_pgen - res_bus[:, PD]
                    all_p_err.extend(np.abs(pypsa_p - pp_p_inj))
                if has_bus_q:
                    pypsa_q = network.buses_t.q.loc[snap].reindex(bus_names).values
                    pp_q_inj = pp_bus_qgen - res_bus[:, QD]
                    all_q_err.extend(np.abs(pypsa_q - pp_q_inj))

            # --- Line flows (p.u.) — lines only, first n_lines branches ---
            if has_line_p:
                pypsa_lp = network.lines_t.p0.loc[snap].reindex(line_names).values
                all_line_p_err.extend(np.abs(pypsa_lp - result['branch'][:n_lines, PF]))
            if has_line_q:
                pypsa_lq = network.lines_t.q0.loc[snap].reindex(line_names).values
                all_line_q_err.extend(np.abs(pypsa_lq - result['branch'][:n_lines, QF]))

    total = n_converged + n_failed
    metrics = {
        # RMSE computed over per-element errors (true element-wise RMSE)  # [STSI 010626]:full metric coverage for scatter
        'vmag_mae':       float(np.mean(all_vmag_err))   if all_vmag_err   else float('nan'),
        'vmag_rmse':      float(np.sqrt(np.mean(np.array(all_vmag_err)**2))) if all_vmag_err else float('nan'),
        'vang_mae':       float(np.mean(all_vang_err))   if all_vang_err   else float('nan'),
        'vang_rmse':      float(np.sqrt(np.mean(np.array(all_vang_err)**2))) if all_vang_err else float('nan'),
        'p_mae':          float(np.mean(all_p_err))      if all_p_err      else float('nan'),
        'p_rmse':         float(np.sqrt(np.mean(np.array(all_p_err)**2))) if all_p_err else float('nan'),
        'q_mae':          float(np.mean(all_q_err))      if all_q_err      else float('nan'),
        'q_rmse':         float(np.sqrt(np.mean(np.array(all_q_err)**2))) if all_q_err else float('nan'),
        'line_flow_mae':  float(np.mean(all_line_p_err)) if all_line_p_err else float('nan'),
        'line_flow_rmse': float(np.sqrt(np.mean(np.array(all_line_p_err)**2))) if all_line_p_err else float('nan'),
        'line_flow_q0_mae':  float(np.mean(all_line_q_err)) if all_line_q_err else float('nan'),
        'line_flow_q0_rmse': float(np.sqrt(np.mean(np.array(all_line_q_err)**2))) if all_line_q_err else float('nan'),
        'delta_theta_mae':   float(np.mean(all_dt_errors)) if all_dt_errors else float('nan'),
        'delta_theta_rmse':  float(np.sqrt(np.mean(np.array(all_dt_errors)**2))) if all_dt_errors else float('nan'),
        'inference_time_per_snapshot_ms': float(np.mean(all_times)) if all_times else float('nan'),
        'n_snapshots': total,
        'n_converged': n_converged,
        'n_failed': n_failed,
    }

    if verbose:
        print(f"PyPower AC baseline: {n_converged}/{total} converged, "
              f"{metrics['inference_time_per_snapshot_ms']:.3f} ms/snap")
        for k in ['vmag_mae','vang_mae','p_mae','q_mae','line_flow_mae','line_flow_q0_mae']:
            print(f"  {k}: {metrics[k]:.8f} p.u.")
    return metrics

def evaluate_fdlf_baseline(networks, snapshots=None, base_mva=1.0, verbose=True,
                              test_networks=None, use_edge_features=True, use_pnet_balance=True):  # [STSI 010626]:add test_networks for delta_theta
    """Run PyPower Fast Decoupled (XB) PF on all networks/snapshots.
    Identical to evaluate_pypower_baseline but uses PF_ALG=2 (FDLF-XB).
    """
    ppopt = ppoption(VERBOSE=0, OUT_ALL=0, PF_ALG=2)

    all_vmag_err, all_vang_err = [], []
    all_p_err, all_q_err = [], []
    all_line_p_err, all_line_q_err = [], []
    all_times = []
    n_converged, n_failed = 0, 0

    # Optional: build dataset for delta_theta computation  # [STSI 010626]:delta_theta requires edge topology from dataset
    # Guard: skip if custom snapshots passed (index alignment would be wrong)
    _dataset, _index_map = None, None
    if test_networks is not None and snapshots is None:
        _dataset = PowerFlowDataset(test_networks, use_edge_features=use_edge_features,
                                    use_pnet_balance=use_pnet_balance)
        _index_map = {(ni, ti): k for k, (ni, ti) in enumerate(_dataset._index)}
    all_dt_errors = []

    for net_idx, network in enumerate(networks):
        snaps = snapshots if snapshots is not None else network.snapshots
        bus_names = network.buses.index.tolist()
        line_names = network.lines.index.tolist()
        n_buses = len(bus_names)
        n_lines = len(line_names)

        has_bus_p = hasattr(network.buses_t, 'p') and len(network.buses_t.p) > 0
        has_bus_q = hasattr(network.buses_t, 'q') and len(network.buses_t.q) > 0
        has_line_p = hasattr(network.lines_t, 'p0') and len(network.lines_t.p0) > 0
        has_line_q = hasattr(network.lines_t, 'q0') and len(network.lines_t.q0) > 0

        for t_idx, snap in enumerate(snaps):
            ppc, _, bus_idx, _, _ = pypsa_to_ppc(network, snap, base_mva)

            t0 = time.perf_counter()
            result, success = runpf(ppc, ppopt)
            t1 = time.perf_counter()
            all_times.append((t1 - t0) * 1000.0)

            if not success:
                n_failed += 1
                continue
            n_converged += 1

            res_bus = result['bus'][np.argsort(result['bus'][:, BUS_I].astype(int))]

            pypsa_vmag = network.buses_t.v_mag_pu.loc[snap].reindex(bus_names).values
            pypsa_vang = network.buses_t.v_ang.loc[snap].reindex(bus_names).values
            pp_vmag = res_bus[:, VM]
            pp_vang = np.deg2rad(res_bus[:, VA])

            all_vmag_err.extend(np.abs(pypsa_vmag - pp_vmag))
            all_vang_err.extend(np.abs(pypsa_vang - pp_vang))

            # Delta_theta from bus angles (if dataset available)
            if _dataset is not None:
                dt_errs = _compute_delta_theta_metrics(pp_vang, _dataset, net_idx, t_idx, _index_map)
                if dt_errs is not None:
                    all_dt_errors.extend(dt_errs)  # [STSI 010626]:element-wise for correct RMSE

            if has_bus_p or has_bus_q:
                pp_gen = result['gen']
                pp_bus_pgen = np.zeros(n_buses)
                pp_bus_qgen = np.zeros(n_buses)
                for g in range(pp_gen.shape[0]):
                    bidx = int(pp_gen[g, GEN_BUS]) - 1
                    pp_bus_pgen[bidx] += pp_gen[g, PG]
                    pp_bus_qgen[bidx] += pp_gen[g, QG]
                if has_bus_p:
                    pypsa_p = network.buses_t.p.loc[snap].reindex(bus_names).values
                    pp_p_inj = pp_bus_pgen - res_bus[:, PD]
                    all_p_err.extend(np.abs(pypsa_p - pp_p_inj))
                if has_bus_q:
                    pypsa_q = network.buses_t.q.loc[snap].reindex(bus_names).values
                    pp_q_inj = pp_bus_qgen - res_bus[:, QD]
                    all_q_err.extend(np.abs(pypsa_q - pp_q_inj))

            if has_line_p:
                pypsa_lp = network.lines_t.p0.loc[snap].reindex(line_names).values
                all_line_p_err.extend(np.abs(pypsa_lp - result['branch'][:n_lines, PF]))
            if has_line_q:
                pypsa_lq = network.lines_t.q0.loc[snap].reindex(line_names).values
                all_line_q_err.extend(np.abs(pypsa_lq - result['branch'][:n_lines, QF]))

    total = n_converged + n_failed
    metrics = {
        # RMSE computed over per-element errors (true element-wise RMSE)  # [STSI 010626]:full metric coverage for scatter
        'vmag_mae':       float(np.mean(all_vmag_err))   if all_vmag_err   else float('nan'),
        'vmag_rmse':      float(np.sqrt(np.mean(np.array(all_vmag_err)**2))) if all_vmag_err else float('nan'),
        'vang_mae':       float(np.mean(all_vang_err))   if all_vang_err   else float('nan'),
        'vang_rmse':      float(np.sqrt(np.mean(np.array(all_vang_err)**2))) if all_vang_err else float('nan'),
        'p_mae':          float(np.mean(all_p_err))      if all_p_err      else float('nan'),
        'p_rmse':         float(np.sqrt(np.mean(np.array(all_p_err)**2))) if all_p_err else float('nan'),
        'q_mae':          float(np.mean(all_q_err))      if all_q_err      else float('nan'),
        'q_rmse':         float(np.sqrt(np.mean(np.array(all_q_err)**2))) if all_q_err else float('nan'),
        'line_flow_mae':  float(np.mean(all_line_p_err)) if all_line_p_err else float('nan'),
        'line_flow_rmse': float(np.sqrt(np.mean(np.array(all_line_p_err)**2))) if all_line_p_err else float('nan'),
        'line_flow_q0_mae':  float(np.mean(all_line_q_err)) if all_line_q_err else float('nan'),
        'line_flow_q0_rmse': float(np.sqrt(np.mean(np.array(all_line_q_err)**2))) if all_line_q_err else float('nan'),
        'delta_theta_mae':   float(np.mean(all_dt_errors)) if all_dt_errors else float('nan'),
        'delta_theta_rmse':  float(np.sqrt(np.mean(np.array(all_dt_errors)**2))) if all_dt_errors else float('nan'),
        'inference_time_per_snapshot_ms': float(np.mean(all_times)) if all_times else float('nan'),
        'n_snapshots': total,
        'n_converged': n_converged,
        'n_failed': n_failed,
    }

    if verbose:
        print(f"FDLF (XB) baseline: {n_converged}/{total} converged, "
              f"{metrics['inference_time_per_snapshot_ms']:.3f} ms/snap")
        for k in ['vmag_mae','vang_mae','p_mae','q_mae','line_flow_mae','line_flow_q0_mae']:
            print(f"  {k}: {metrics[k]:.8f} p.u.")
    return metrics

def evaluate_pandapower_baseline(networks, snapshots=None, base_mva=1.0, verbose=True,
                              test_networks=None, use_edge_features=True, use_pnet_balance=True):  # [STSI 010626]:add test_networks for delta_theta
    """Run pandapower AC NR PF on all networks/snapshots.
    Path: pypsa_to_ppc → from_ppc (untimed) → runpp (timed).
    With baseMVA=1.0, pandapower MW/MVAr values equal p.u. numerically.
    """
    import pandapower as pdp
    from pandapower.converter import from_ppc
    import warnings
    warnings.filterwarnings("ignore", category=FutureWarning)

    all_vmag_err, all_vang_err = [], []
    all_p_err, all_q_err = [], []
    all_line_p_err, all_line_q_err = [], []
    all_times = []
    n_converged, n_failed = 0, 0

    # Optional: build dataset for delta_theta computation  # [STSI 010626]:delta_theta requires edge topology from dataset
    # Guard: skip if custom snapshots passed (index alignment would be wrong)
    _dataset, _index_map = None, None
    if test_networks is not None and snapshots is None:
        _dataset = PowerFlowDataset(test_networks, use_edge_features=use_edge_features,
                                    use_pnet_balance=use_pnet_balance)
        _index_map = {(ni, ti): k for k, (ni, ti) in enumerate(_dataset._index)}
    all_dt_errors = []

    for net_idx, network in enumerate(networks):
        snaps = snapshots if snapshots is not None else network.snapshots
        bus_names = network.buses.index.tolist()
        line_names = network.lines.index.tolist()
        n_buses = len(bus_names)
        n_lines = len(line_names)

        has_bus_p = hasattr(network.buses_t, 'p') and len(network.buses_t.p) > 0
        has_bus_q = hasattr(network.buses_t, 'q') and len(network.buses_t.q) > 0
        has_line_p = hasattr(network.lines_t, 'p0') and len(network.lines_t.p0) > 0
        has_line_q = hasattr(network.lines_t, 'q0') and len(network.lines_t.q0) > 0

        for t_idx, snap in enumerate(snaps):
            # Conversion (untimed — same overhead as any solver needing its own format)
            ppc, _, _, _, _ = pypsa_to_ppc(network, snap, base_mva)
            try:
                pp_net = from_ppc(ppc, f_hz=50.0)
            except Exception as e:
                n_failed += 1
                if verbose and n_failed <= 3:
                    print(f"  [pandapower] from_ppc failed: {e}")
                continue

            # Solve (timed)
            t0 = time.perf_counter()
            try:
                pdp.runpp(pp_net, algorithm='nr', init='flat', numba=True)
            except Exception as e:
                n_failed += 1
                if verbose and n_failed <= 3:
                    print(f"  [pandapower] runpp failed: {e}")
                continue
            t1 = time.perf_counter()
            all_times.append((t1 - t0) * 1000.0)

            if not pp_net.converged:
                n_failed += 1
                continue
            n_converged += 1

            # Results — baseMVA=1 so MW == p.u. numerically
            pp_vmag = pp_net.res_bus.vm_pu.values[:n_buses]
            pp_vang = np.deg2rad(pp_net.res_bus.va_degree.values[:n_buses])

            pypsa_vmag = network.buses_t.v_mag_pu.loc[snap].reindex(bus_names).values
            pypsa_vang = network.buses_t.v_ang.loc[snap].reindex(bus_names).values

            all_vmag_err.extend(np.abs(pypsa_vmag - pp_vmag))
            all_vang_err.extend(np.abs(pypsa_vang - pp_vang))

            # Delta_theta from bus angles (if dataset available)
            if _dataset is not None:
                dt_errs = _compute_delta_theta_metrics(pp_vang, _dataset, net_idx, t_idx, _index_map)
                if dt_errs is not None:
                    all_dt_errors.extend(dt_errs)  # [STSI 010626]:element-wise for correct RMSE

            # Bus P/Q: res_bus.p_mw is net injection (gen-load) in MW (= p.u. @ base=1)
            if has_bus_p:
                pypsa_p = network.buses_t.p.loc[snap].reindex(bus_names).values
                pp_p = -pp_net.res_bus.p_mw.values[:n_buses]
                all_p_err.extend(np.abs(pypsa_p - pp_p))
            if has_bus_q:
                pypsa_q = network.buses_t.q.loc[snap].reindex(bus_names).values
                pp_q = -pp_net.res_bus.q_mvar.values[:n_buses]
                all_q_err.extend(np.abs(pypsa_q - pp_q))

            # Line flows
            n_res_lines = len(pp_net.res_line)
            if has_line_p and n_res_lines >= n_lines:
                pypsa_lp = network.lines_t.p0.loc[snap].reindex(line_names).values
                pp_lp = pp_net.res_line.p_from_mw.values[:n_lines]
                all_line_p_err.extend(np.abs(pypsa_lp - pp_lp))
            if has_line_q and n_res_lines >= n_lines:
                pypsa_lq = network.lines_t.q0.loc[snap].reindex(line_names).values
                pp_lq = pp_net.res_line.q_from_mvar.values[:n_lines]
                all_line_q_err.extend(np.abs(pypsa_lq - pp_lq))

    total = n_converged + n_failed
    metrics = {
        # RMSE computed over per-element errors (true element-wise RMSE)  # [STSI 010626]:full metric coverage for scatter
        'vmag_mae':       float(np.mean(all_vmag_err))   if all_vmag_err   else float('nan'),
        'vmag_rmse':      float(np.sqrt(np.mean(np.array(all_vmag_err)**2))) if all_vmag_err else float('nan'),
        'vang_mae':       float(np.mean(all_vang_err))   if all_vang_err   else float('nan'),
        'vang_rmse':      float(np.sqrt(np.mean(np.array(all_vang_err)**2))) if all_vang_err else float('nan'),
        'p_mae':          float(np.mean(all_p_err))      if all_p_err      else float('nan'),
        'p_rmse':         float(np.sqrt(np.mean(np.array(all_p_err)**2))) if all_p_err else float('nan'),
        'q_mae':          float(np.mean(all_q_err))      if all_q_err      else float('nan'),
        'q_rmse':         float(np.sqrt(np.mean(np.array(all_q_err)**2))) if all_q_err else float('nan'),
        'line_flow_mae':  float(np.mean(all_line_p_err)) if all_line_p_err else float('nan'),
        'line_flow_rmse': float(np.sqrt(np.mean(np.array(all_line_p_err)**2))) if all_line_p_err else float('nan'),
        'line_flow_q0_mae':  float(np.mean(all_line_q_err)) if all_line_q_err else float('nan'),
        'line_flow_q0_rmse': float(np.sqrt(np.mean(np.array(all_line_q_err)**2))) if all_line_q_err else float('nan'),
        'delta_theta_mae':   float(np.mean(all_dt_errors)) if all_dt_errors else float('nan'),
        'delta_theta_rmse':  float(np.sqrt(np.mean(np.array(all_dt_errors)**2))) if all_dt_errors else float('nan'),
        'inference_time_per_snapshot_ms': float(np.mean(all_times)) if all_times else float('nan'),
        'n_snapshots': total,
        'n_converged': n_converged,
        'n_failed': n_failed,
    }

    if verbose:
        print(f"pandapower NR baseline: {n_converged}/{total} converged, "
              f"{metrics['inference_time_per_snapshot_ms']:.3f} ms/snap")
        for k in ['vmag_mae','vang_mae','p_mae','q_mae','line_flow_mae','line_flow_q0_mae']:
            print(f"  {k}: {metrics[k]:.8f} p.u.")
    return metrics

# wrapper to run all the baselines and build ref_single and ref_batch dicts for scatter plots
def build_ref_baselines(networks, n_sample=None, ref_single=None, ref_batch=None, test_networks=None):  # [STSI 140926]:n_sample=None -> full test split; test_networks override
    """
    Build (or incrementally fill) ref_single and ref_batch baseline timing dicts.

    Parameters
    ----------
    networks  : full list of networks (train+val+test); test split reconstructed internally.
                Ignored if test_networks is provided.
    n_sample  : number of test networks to time. Default None = use the full test
                population (same population evaluate_gnn_on_test_set is run on for
                the GNN runs these baselines will be plotted against). Pass a smaller
                int to trade accuracy of the comparison for faster iteration — a
                warning is printed whenever the resolved sample is smaller than the
                full test population.
    ref_single : existing dict to fill; keys already present are skipped
    ref_batch  : existing dict to fill; keys already present are skipped
    test_networks : optional pre-selected test-network list to use directly, bypassing
                internal reconstruct_test_split(networks). Use this when `networks` is
                already a curated test set (e.g. a single-topology exploration list) —
                passing such a list as `networks` would otherwise be re-split 70/15/15,
                collapsing the effective sample size unexpectedly.

    Returns
    -------
    (ref_single, ref_batch)
    """
    if ref_single is None:
        ref_single = {}
    if ref_batch is None:
        ref_batch = {}

    if test_networks is not None:
        test_nets = list(test_networks)
    else:
        _, _, test_nets = reconstruct_test_split(networks)

    sample = test_nets if n_sample is None else test_nets[:n_sample]
    if len(sample) < len(test_nets):
        print(
            f"WARNING: baseline timing uses {len(sample)}/{len(test_nets)} test networks "
            f"— GNN runs plotted alongside these baselines are typically evaluated on the "
            f"full {len(test_nets)}-network test set. Metrics are not directly comparable "
            f"unless n_sample matches (or is None)."
        )
    print(f"Using {len(sample)} test networks for baseline timing.")

    # single-snapshot baselines
    if "PyPower AC" not in ref_single:
        print("Running PyPower AC...")
        ref_single["PyPower AC"] = evaluate_pypower_baseline(sample, test_networks=sample)  # [STSI 010626]:pass test_networks for delta_theta
    if "PyPSA single" not in ref_single:
        print("Running PyPSA single...")
        ref_single["PyPSA single"] = evaluate_pypsa_timing_single(sample)
    if "FDLF" not in ref_single:
        print("Running FDLF...")
        ref_single["FDLF"] = evaluate_fdlf_baseline(sample, test_networks=sample)
    if "pandapower" not in ref_single:
        print("Running pandapower...")
        ref_single["pandapower"] = evaluate_pandapower_baseline(sample, test_networks=sample)

    # batch baselines
    if "PyPSA batch" not in ref_batch:
        print("Running PyPSA batch...")
        ref_batch["PyPSA batch"] = evaluate_pypsa_timing(sample)
    if "DC lpf" not in ref_batch:
        print("Running DC lpf...")
        ref_batch["DC lpf"] = evaluate_dc_baseline(sample)

    print("\nref_single timings (ms/snap):")
    for k, v in ref_single.items():
        print(f"  {k}: {v['inference_time_per_snapshot_ms']:.2f}")
    print("ref_batch timings (ms/snap):")
    for k, v in ref_batch.items():
        print(f"  {k}: {v['inference_time_per_snapshot_ms']:.2f}")

    return ref_single, ref_batch


## Hyperparameter Sweep Helpers

In [ ]:
# Hyperparameter helpers

def _get_run_styles(runs):
    # [STSI100526] Removed physics_mode/n_node_features (dead); added conv_mode/drop_rate
    varying = [
        k for k in ("physics_label", "weight_ptdf",
                    "head_mode", "use_residual", "norm_type",  # [STSI 120526]:independent flags
                    "activation", "use_dropout", "drop_rate",
                    "batch_size", "lr", "conv_type", "y_matrix_source", "angle_mode",
                    "fraction_dcf_local", "fraction_dcf_global",  # [STSI 040626]:activation ramp keys
                    "fraction_acf_local", "fraction_acf_global",
                    "activation_ramp_epochs",
                    "fraction_kvl", "enable_balance_dc", "kvl_activation_start")  # [STSI 021026]
        if len({str(x.get(k)) for x in runs}) > 1
    ]
    if not varying:
        varying = ["physics_label"]

    dim_values = {}
    for k in varying:
        seen = {}
        for r in runs:
            v = str(r.get(k, ""))
            if v not in seen:
                seen[v] = len(seen)
        dim_values[k] = seen

    # [STSI 260526]:use per-run ordinal color when first dim has fewer unique values than runs
    n_colors_from_dim = len(dim_values[varying[0]]) if varying else 1
    use_ordinal_color = (n_colors_from_dim < len(runs))

    styles = {}
    for run_i, r in enumerate(runs):
        rid = id(r)
        if use_ordinal_color:
            c_idx = run_i  # unique color per run
        else:
            c_idx = dim_values[varying[0]][str(r.get(varying[0], ""))] if len(varying) >= 1 else 0
        ls_idx = dim_values[varying[1]][str(r.get(varying[1], ""))] if len(varying) >= 2 else 0
        mk_idx = dim_values[varying[2]][str(r.get(varying[2], ""))] if len(varying) >= 3 else 0

        styles[rid] = (
            _COLOR_CYCLE[c_idx  % len(_COLOR_CYCLE)],
            _LINE_STYLES[ls_idx % len(_LINE_STYLES)],
            _MARKERS[mk_idx     % len(_MARKERS)],
        )
    return styles

def _activation_scale(epoch: int, start, stop, ramp_epochs: int) -> float:  # [STSI 090626]:verbatim from Training
    """Linear ramp-up from start, optional ramp-down to stop. Returns 0.0-1.0."""
    if start is None or epoch < start:
        return 0.0
    if ramp_epochs > 0 and epoch < start + ramp_epochs:
        scale = (epoch - start + 1) / ramp_epochs
    else:
        scale = 1.0
    if stop is not None and stop > 0:
        if epoch >= stop:
            return 0.0
        if ramp_epochs > 0 and epoch > stop - ramp_epochs:
            scale *= (stop - epoch) / ramp_epochs
    return min(max(scale, 0.0), 1.0)

def _add_shared_legend(fig, label_entries, styles, ncol=None, bottom_frac=0.10,
                       extra_handles=None, legend_key="legend_label",
                       base_font_size=10, side="bottom", right_frac=0.80,
                       label_wrap=None):  # [STSI 040926]:side="right" → single-col RHS legend; label_wrap wraps long labels
    """Add a single shared legend using Line2D handles.

    extra_handles : list of Artist, optional — appended after run handles
    legend_key    : str — which label_entries key to use
    side          : "bottom" (default) or "right" (single column, right of plots)
    right_frac    : fraction of figure width reserved for axes when side="right"
    label_wrap    : int, optional — wrap legend text at this many characters (None/0 = off)
    """
    import textwrap
    from matplotlib.lines import Line2D
    run_handles = []
    for entry in label_entries:
        run = entry["run"]
        color, _, mk = styles[id(run)]
        lbl = entry[legend_key]
        if label_wrap:
            lbl = textwrap.fill(lbl, label_wrap)
        run_handles.append(
            Line2D([0], [0], color=color, linestyle="-",
                   marker=mk if mk else None, markersize=5,
                   label=lbl)
        )
    all_handles = run_handles + (extra_handles or [])  # [STSI 100626]:shared legend via _add_shared_legend
    if not all_handles:
        plt.tight_layout(rect=[0, 0, 1, 0.95])
        return
    if side == "right":  # [STSI 040926]:single-column legend on the right-hand side
        fig.legend(handles=all_handles, loc="center left",
                   bbox_to_anchor=(0.80, 0.5), ncol=1,
                   fontsize=base_font_size * 0.85, frameon=True)
        plt.tight_layout(rect=[0, 0, right_frac, 0.95])
    else:
        fig.legend(handles=all_handles, loc="lower center",
                   bbox_to_anchor=(0.5, -0.02),
                   ncol=ncol or max(1, min(4, len(all_handles))),
                   fontsize=base_font_size * 0.85, frameon=True)
        plt.tight_layout(rect=[0, bottom_frac, 1, 0.95])

def _add_shared_legend_from_dict(fig, labels, extra_handles=None,  # [STSI 100626]:dict-keyed multi-model shared legend
                                  ncol=None, bottom_frac=0.10, base_font_size=10):
    """Add a shared legend below figure for dict-keyed multi-model plots.

    Builds Line2D handles with tab10 colors matching the positional color
    assignment used by dict-based plot functions (scatter/plot order = dict order).

    Parameters
    ----------
    fig          : matplotlib Figure
    labels       : list of str — model/solver names (dict keys in plot order)
    extra_handles: list of Artist, optional — prepended (e.g. 'True (PyPSA)' handle)
    ncol         : int, optional — legend columns (auto-sized if None)
    bottom_frac  : float — fraction of figure height reserved below plot for legend
    """
    from matplotlib.lines import Line2D
    colors = plt.cm.tab10.colors
    handles = list(extra_handles or [])
    for i, lbl in enumerate(labels):
        handles.append(
            Line2D([0], [0], color=colors[i % 10], linestyle="-",
                   marker="o", markersize=5, label=lbl)
        )
    if handles:
        fig.legend(handles=handles, loc="lower center",
                   bbox_to_anchor=(0.5, -0.02),
                   ncol=ncol or max(1, min(4, len(handles))),
                   fontsize=base_font_size * 0.85, frameon=True)
        plt.tight_layout(rect=[0, bottom_frac, 1, 0.95])
    else:
        plt.tight_layout(rect=[0, 0, 1, 0.95])

def _frac_notation(short, frac, start, stop):  # [STSI 040926]:hparam-table style "frac(start/stop)" notation
    """Format a sub-loss fraction + activation window, e.g. 'dcL0.2(20/65)'; '-' = unset/never-stops."""
    frac_str = f"{frac:g}" if isinstance(frac, (int, float)) else str(frac)
    start_str = str(start) if start is not None else "-"
    stop_str = str(stop) if stop is not None else "-"
    return f"{short}{frac_str}({start_str}/{stop_str})"


def build_legend_label(r, runs):
    # [STSI100526] Removed physics_mode/n_node_features (dead); added conv_mode/drop_rate
    varying = [
        k for k in [
            "physics_label",
            "weight_ptdf",
            "ptdf_loss_mode",
            "ptdf_branch_mode",
            "ptdf_alpha",
            "ptdf_changepoint",
            "head_mode",
            "use_residual",
            "norm_type",
            "activation",
            "use_dropout",
            "drop_rate",
            "batch_size",
            "lr",
            "conv_type",
            "y_matrix_source",
            "physics_activation_start",  # [STSI 040626]:replaces warmup_epochs_*
            "ptdf_activation_start",
            "hidden_dim",
            "num_layers",
            "angle_mode",  # [STSI 260526]:edge-delta awareness
            "vmag_mode",
            "fraction_dcf_local", "fraction_dcf_global",  # [STSI 040626]:activation ramp keys
            "fraction_acf_local", "fraction_acf_global",
            "activation_ramp_epochs",
            "fraction_kvl", "kvl_activation_start",  # [STSI 021026]
        ]
        if len({str(x.get(k)) for x in runs}) > 1
    ]

    if not varying:
        varying = ["physics_label"]

    parts = []

    # [STSI100526] Removed physics_mode label (always-on)
    # Skip physics_label when other varying keys follow: it is a summary tag
    # already spelled out by the detailed parts below (frac/start/stop etc.).
    if "physics_label" in varying and len(varying) == 1:
        parts.append(r.get("physics_label", ""))

    if "weight_ptdf" in varying:
        parts.append(f"ptdf={r.get('weight_ptdf')}")
    if "ptdf_loss_mode" in varying:
        mode_str = r.get('ptdf_loss_mode')
        cp = r.get('ptdf_changepoint')
        if mode_str == 'stepwise' and cp is not None:
            mode_str = f'stepwise(cp={cp})'
        parts.append(f'mode={mode_str}')
    if "ptdf_changepoint" in varying and "ptdf_loss_mode" not in varying:
        parts.append(f"cp={r.get('ptdf_changepoint')}")
    if "ptdf_branch_mode" in varying:
        parts.append(f"br={r.get('ptdf_branch_mode')}")
    if "ptdf_alpha" in varying and r.get("ptdf_loss_mode") == "mixed":
        parts.append(f"a={r.get('ptdf_alpha')}")

    if "y_matrix_source" in varying:
        parts.append(f"y={r.get('y_matrix_source')}")
    if "physics_activation_start" in varying:  # [STSI 040926]:hparam-table notation frac(start/stop)
        parts.append(_frac_notation("phys", r.get("fraction_physics"),
                                    r.get("physics_activation_start"), r.get("physics_activation_stop")))
    if "ptdf_activation_start" in varying:
        parts.append(_frac_notation("ptdf", r.get("fraction_ptdf"),
                                    r.get("ptdf_activation_start"), r.get("ptdf_activation_stop")))
    if "hidden_dim" in varying:
        parts.append(f"h={r.get('hidden_dim')}")
    if "num_layers" in varying:
        parts.append(f"L={r.get('num_layers')}")
    if "batch_size" in varying:
        parts.append(f"bs={r.get('batch_size')}")
    if "lr" in varying:
        parts.append(f"lr={r.get('lr'):.0e}")
    if "conv_type" in varying:
        parts.append(r.get("conv_type", ""))
    # [STSI100526] Synced with Training: head_mode/conv_mode/drop_rate (removed n_node_features)
    if "head_mode" in varying:
        parts.append(f"hm={r.get('head_mode', 'standard')}")
    # Compact conv-block notation: only show non-default components [STSI 140526]:unconditional build + base fallback
    conv_parts = []
    if r.get("use_residual", False):
        conv_parts.append("res")
    nt = r.get("norm_type", None)
    if nt:
        norm_short = {"layer": "layN", "batch": "batN", "graph": "grpN"}.get(nt, nt)
        conv_parts.append(norm_short)
    act = r.get("activation", "leaky_relu")
    if act != "leaky_relu":
        conv_parts.append(act)
    if r.get("use_dropout", False):
        conv_parts.append(f"do{r.get('drop_rate', 0.1)}")
    # Only add if any conv-block key actually varies
    conv_keys = {"use_residual", "norm_type", "activation", "use_dropout", "drop_rate"}
    if conv_keys & set(varying):
        if conv_parts:
            parts.append("+".join(conv_parts))
        else:
            parts.append("base")
    if "angle_mode" in varying:  # [STSI 260526]:edge-delta label
        am = r.get("angle_mode", "node")
        parts.append("ed" if am == "edge_delta" else "n")
    if "vmag_mode" in varying:
        vm = r.get("vmag_mode", "none")
        parts.append(f"vm{vm[0]}" if vm != "none" else "vm0")
    # Per-sub-loss fractions [STSI 040926]:hparam-table notation frac(start/stop)
    _frac_keys = [
        ("fraction_dcf_local", "dcL", "dcf_local_activation_start", "dcf_local_activation_stop"),
        ("fraction_dcf_global", "dcG", "dcf_global_activation_start", "dcf_global_activation_stop"),
        ("fraction_acf_local", "acL", "acf_local_activation_start", "acf_local_activation_stop"),
        ("fraction_acf_global", "acG", "acf_global_activation_start", "acf_global_activation_stop"),
        ("fraction_kvl", "kvl", "kvl_activation_start", "kvl_activation_stop"),  # [STSI 021026]
    ]
    for fk, short, start_k, stop_k in _frac_keys:
        if fk in varying:
            parts.append(_frac_notation(short, r.get(fk), r.get(start_k), r.get(stop_k)))
    if "activation_ramp_epochs" in varying:
        parts.append(f"ramp={r.get('activation_ramp_epochs', 5)}")

    return " | ".join(parts)
    


RUN_LABEL_COMPACT_THRESHOLD = 4
RUN_LABEL_INDEX_THRESHOLD = 8


def build_compact_legend_label(r, runs):
    full_label = build_legend_label(r, runs)
    full_parts = [part for part in full_label.split(" | ") if part]
    known_prefixes = (
        "ptdf=", "mode=", "br=", "a=", "y=", "wu=",
        "h=", "L=", "bs=", "lr=", "hm=", "dr=",  # [STSI 120526]:removed cm=, added conv block tokens
        "res", "layN", "graN", "gelu", "relu", "elu", "silu", "do",
    )
    parts = []

    if full_parts and not full_parts[0].startswith(known_prefixes):
        parts.append(full_parts[0])

    for part in full_parts:
        if part.startswith("ptdf="):
            parts.append(part.replace("ptdf=", "pt"))
        elif part.startswith("mode="):
            parts.append(part.replace("mode=", "pm="))
        elif part.startswith(("br=", "a=", "y=", "wu=", "h=", "L=", "bs=", "lr=", "hm=", "dr=",
                              "res", "layN", "graN", "gelu", "relu", "elu", "silu", "do")):
            parts.append(part)

    if not parts:
        return full_label

    deduped = []
    seen = set()
    for part in parts:
        if part and part not in seen:
            deduped.append(part)
            seen.add(part)
    return " | ".join(deduped)


def resolve_run_label_policy(
    runs,
    label_mode="auto",
    compact_threshold=RUN_LABEL_COMPACT_THRESHOLD,
    index_threshold=RUN_LABEL_INDEX_THRESHOLD,
    index_prefix="R",
):
    if label_mode == "auto":
        if len(runs) >= index_threshold:
            active_mode = "index"
        elif len(runs) >= compact_threshold:
            active_mode = "compact"
        else:
            active_mode = "full"
    else:
        active_mode = label_mode

    if active_mode == "compact":
        compact_candidates = [build_compact_legend_label(r, runs) for r in runs]
        if len(set(compact_candidates)) < len(compact_candidates):
            active_mode = "full"

    entries = []
    for i, r in enumerate(runs, start=1):
        full_label = build_legend_label(r, runs)
        compact_label = build_compact_legend_label(r, runs)
        short_label = f"{index_prefix}{r.get('_orig_index', i)}"  # preserve original position label when set (e.g. by get_top_runs)
        plot_label = {
            "full": full_label,
            "compact": compact_label,
            "index": short_label,
        }[active_mode]
        mapping_label = f"{short_label}: {full_label}"
        legend_label = mapping_label if active_mode == "index" else plot_label
        entries.append({
            "run": r,
            "full_label": full_label,
            "compact_label": compact_label,
            "short_label": short_label,
            "plot_label": plot_label,
            "legend_label": legend_label,
            "mapping_label": mapping_label,
        })
    return active_mode, entries


def _index_label_entries(label_entries):
    return {id(entry["run"]): entry for entry in label_entries}


def _resolve_annotation_mode(annotation_mode, label_mode):
    if annotation_mode != "auto":
        return annotation_mode
    return {
        "full": "all",
        "compact": "first-per-run",
        "index": "none",
    }[label_mode]

def plot_all_runs_training_curves(
    runs,
    title_prefix="All runs",
    marker_every=10,
    label_mode="auto",
    compact_threshold=RUN_LABEL_COMPACT_THRESHOLD,
    index_threshold=RUN_LABEL_INDEX_THRESHOLD,
    base_font_size=10,
):
    """
    Plot train/val loss for all runs.
    - Color    separates the first varying hyperparameter (e.g. physics_label).
    - Linestyle separates the second varying hyperparameter (e.g. weight_ptdf).
    - Marker   separates the third varying hyperparameter (e.g. batch_size / lr).
    - marker_every: place a marker every N epochs (keeps plots readable).
    """
    styles = _get_run_styles(runs)
    _, label_entries = resolve_run_label_policy(
        runs,
        label_mode=label_mode,
        compact_threshold=compact_threshold,
        index_threshold=index_threshold,
    )
    label_entry_by_run = _index_label_entries(label_entries)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 10))

    for r in runs:
        label = label_entry_by_run[id(r)]["legend_label"]
        h = r["history"]
        color, ls, mk = styles[id(r)]
        n = len(h["train_total"])
        markevery = max(1, n // marker_every) if mk else None

        ax1.plot(
            h["train_total"], label=label, alpha=0.85,
            color=color, linestyle=ls, marker=mk,
            markevery=markevery, markersize=5,
        )
        ax2.plot(
            h["val_total"], label=label, alpha=0.85,
            color=color, linestyle=ls, marker=mk,
            markevery=markevery, markersize=5,
        )

    for ax, title, ylabel in [
        (ax1, f"{title_prefix} — Training Loss", "Training Loss"),
        (ax2, f"{title_prefix} — Validation Loss", "Validation Loss"),
    ]:
        ax.set_xlabel("Epoch", fontsize=base_font_size * 1.0)
        ax.set_ylabel(ylabel, fontsize=base_font_size * 1.0)
        ax.set_title(title, fontsize=base_font_size * 1.1)
        ax.tick_params(axis="both", labelsize=base_font_size * 0.85)
        ax.grid(True, alpha=0.3)

    _add_shared_legend(fig, label_entries, styles, base_font_size=base_font_size)
    plt.show()

def plot_loss_components_chart(
    runs,
    title_prefix="Loss Components",
    marker_every=10,
    label_mode="auto",
    compact_threshold=RUN_LABEL_COMPACT_THRESHOLD,
    index_threshold=RUN_LABEL_INDEX_THRESHOLD,
    base_font_size=10,
):
    """
    Plot train/val loss for each component (MSE, Physics, Angle-ref, PTDF)
    in a 2×2 grid of subplots, with each subplot showing train (solid) and
    val (dashed) curves for all runs.
    """
    styles = _get_run_styles(runs)
    _, label_entries = resolve_run_label_policy(
        runs,
        label_mode=label_mode,
        compact_threshold=compact_threshold,
        index_threshold=index_threshold,
    )
    label_entry_by_run = _index_label_entries(label_entries)

    components = [
        ("mse",       "MSE Loss"),
        ("phys",      "Physics Loss"),
        ("angle_ref", "Angle-Ref Loss"),
        ("ptdf",      "PTDF Loss"),
        ("p_res",     "P Residual"),    # [STSI 220526] new — train_p_res history key
        ("q_res",     "Q Residual"),    # [STSI 220526] new — train_q_res history key
        ("delta_theta_mse", "Δθ Edge MSE"),  # [STSI 260526]:edge-delta angle loss component
        ("flow_loss", "Flow Loss"),  # [STSI 030626]:flow loss component panel
    ]

    fig, axes = plt.subplots(2, 4, figsize=(48, 14))
    fig.suptitle(f"{title_prefix} (solid=train, dashed=val)", fontsize=base_font_size * 1.3)
    axes = axes.flatten()

    for _ax in axes[len(components):]:
        _ax.set_visible(False)  # [STSI 260526]:hide unused subplot slots

    for ax, (key, comp_title) in zip(axes, components):
        for r in runs:
            label = label_entry_by_run[id(r)]["legend_label"]
            h = r["history"]
            train_key = f"train_{key}"
            val_key = f"val_{key}"

            if train_key not in h or not h[train_key]:
                continue

            color, ls, mk = styles[id(r)]
            n = len(h[train_key])
            markevery = max(1, n // marker_every) if mk else None

            ax.plot(
                h[train_key], label=f"{label} (train)", alpha=0.85,
                color=color, linestyle="-", marker=mk,
                markevery=markevery, markersize=5,
            )
            ax.plot(
                h.get(val_key, []), label=f"{label} (val)", alpha=0.85,  # [STSI 220526]: guard old JSONs without val_p_res/val_q_res
                color=color, linestyle="--", marker=mk,
                markevery=markevery, markersize=5,
            )

        ax.set_xlabel("Epoch", fontsize=base_font_size * 1.0)
        ax.set_ylabel("Loss", fontsize=base_font_size * 1.0)
        ax.set_title(f"{title_prefix} — {comp_title}", fontsize=base_font_size * 1.1)
        ax.tick_params(axis="both", labelsize=base_font_size * 0.85)
        # clip to max = 95 percentile for y axis if max is more than 5 and disregard the first 5 epochs as these are usually unstable. keep min value at 0. this prevents outliers from dominating the plot while still showing the overall trend.
        ax.set_ylim(top=np.percentile(
            [max(h.get(f"train_{key}", [0])[5:]), max(h.get(f"val_{key}", [0])[5:])], 95)  if max(h.get(f"train_{key}", [0]) + h.get(f"val_{key}", [0])) > 5 else None, bottom=0)
        ax.grid(True, alpha=0.3)

    _add_shared_legend(fig, label_entries, styles, base_font_size=base_font_size)
    plt.show()

def plot_effective_weights_chart(  # [STSI 200526]:new function for dynamic loss weight visualization
    runs,
    title_prefix="Effective Weights",
    marker_every=10,
    max_w_phys=None,
    max_w_ptdf=None,
    max_w_flow=None,  # [STSI 030626]:flow weight cap reference line
    max_w_kvl=None,   # [STSI 021026]:KVL weight cap reference line
    label_mode="auto",
    compact_threshold=RUN_LABEL_COMPACT_THRESHOLD,
    index_threshold=RUN_LABEL_INDEX_THRESHOLD,
    base_font_size=10,
):
    """
    Plot effective w_phys and w_ptdf over epochs for all runs.
    1x2 subplot: left = w_phys, right = w_ptdf.
    Horizontal dashed line at max_w cap.
    Gracefully skips runs without weight history (pre-adaptive runs).
    """
    styles = _get_run_styles(runs)
    _, label_entries = resolve_run_label_policy(
        runs,
        label_mode=label_mode,
        compact_threshold=compact_threshold,
        index_threshold=index_threshold,
    )
    label_entry_by_run = _index_label_entries(label_entries)

    # [STSI 021026]:expanded to 2x4 to add the KVL effective-weight panel
    fig, axes = plt.subplots(2, 4, figsize=(40, 14))
    fig.suptitle(f"{title_prefix} — Effective Loss Weights", fontsize=base_font_size * 1.3)

    _eff_panels = [
        ("train_eff_w_phys", axes[0, 0], "w_phys"),
        ("train_eff_w_ptdf", axes[0, 1], "w_ptdf"),
        ("train_eff_w_dcf_local", axes[0, 2], "w_dcf_local"),
        ("train_eff_w_dcf_global", axes[0, 3], "w_dcf_global"),
        ("train_eff_w_acf_local", axes[1, 0], "w_acf_local"),
        ("train_eff_w_acf_global", axes[1, 1], "w_acf_global"),
        ("train_eff_w_kvl", axes[1, 2], "w_kvl"),
    ]
    axes[1, 3].axis("off")  # 8th slot unused -- 7 loss terms today

    has_data = False
    for r in runs:
        label = label_entry_by_run[id(r)]["legend_label"]
        h = r["history"]
        color, ls, mk = styles[id(r)]
        for key, ax, _ in _eff_panels:
            if key in h and h[key]:
                has_data = True
                n = len(h[key])
                markevery = max(1, n // marker_every) if mk else None
                ax.plot(h[key], label=label, alpha=0.85,
                        color=color, linestyle=ls, marker=mk,
                        markevery=markevery, markersize=5)

    if not has_data:
        plt.close(fig)
        return

    # Draw cap reference lines
    if max_w_phys is not None:
        axes[0, 0].axhline(max_w_phys, color="red", linestyle="--", alpha=0.4,
                    label=f"max_w_phys={max_w_phys}")
    if max_w_ptdf is not None:
        axes[0, 1].axhline(max_w_ptdf, color="red", linestyle="--", alpha=0.4,
                    label=f"max_w_ptdf={max_w_ptdf}")
    if max_w_flow is not None:
        for _, ax, title in _eff_panels[2:6]:  # the 4 flow panels only (dcf/acf local/global)
            ax.axhline(max_w_flow, color="red", linestyle="--", alpha=0.4,
                        label=f"max_w_flow={max_w_flow}")
    if max_w_kvl is not None:
        _, kvl_ax, _ = _eff_panels[6]
        kvl_ax.axhline(max_w_kvl, color="red", linestyle="--", alpha=0.4,
                        label=f"max_w_kvl={max_w_kvl}")

    for _, ax, title in _eff_panels:
        ax.set_xlabel("Epoch", fontsize=base_font_size * 1.0)
        ax.set_ylabel("Effective Weight", fontsize=base_font_size * 1.0)
        ax.set_title(f"{title_prefix} — {title}", fontsize=base_font_size * 1.1)
        ax.tick_params(axis="both", labelsize=base_font_size * 0.85)
        ax.grid(True, alpha=0.3)

    _add_shared_legend(fig, label_entries, styles, base_font_size=base_font_size)
    plt.show()

def plot_effective_fractions_chart(  # [STSI 090626]:effective fractions visualization
    runs,
    title_prefix="Effective Fractions",
    marker_every=10,
    label_mode="auto",
    compact_threshold=RUN_LABEL_COMPACT_THRESHOLD,
    index_threshold=RUN_LABEL_INDEX_THRESHOLD,
    base_font_size=10,
):
    """Plot fraction * _activation_scale(epoch) for each sub-loss. 2x3 layout.
    Skips runs without activation_ramp_epochs in run_info (pre-ramp runs)."""
    styles = _get_run_styles(runs)
    _, label_entries = resolve_run_label_policy(
        runs, label_mode=label_mode,
        compact_threshold=compact_threshold,
        index_threshold=index_threshold,
    )
    label_entry_by_run = _index_label_entries(label_entries)

    _frac_panels = [
        ("fraction_physics",    "physics_activation_start",    "physics_activation_stop",    "Physics"),
        ("fraction_ptdf",       "ptdf_activation_start",       "ptdf_activation_stop",       "PTDF"),
        ("fraction_dcf_local",  "dcf_local_activation_start",  "dcf_local_activation_stop",  "DC Flow Local"),
        ("fraction_dcf_global", "dcf_global_activation_start", "dcf_global_activation_stop", "DC Flow Global"),
        ("fraction_acf_local",  "acf_local_activation_start",  "acf_local_activation_stop",  "AC Flow Local"),
        ("fraction_acf_global", "acf_global_activation_start", "acf_global_activation_stop", "AC Flow Global"),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(30, 14))
    fig.suptitle(f"{title_prefix} — Effective Fractions (fraction \u00d7 activation ramp)", fontsize=base_font_size * 1.3)

    has_data = False
    for r in runs:
        ramp = r.get("activation_ramp_epochs")
        if ramp is None:
            continue
        label = label_entry_by_run[id(r)]["legend_label"]
        color, ls, mk = styles[id(r)]
        n_epochs = len(r.get("history", {}).get("train_total", []))
        if n_epochs == 0:
            continue

        for (frac_key, start_key, stop_key, panel_title), ax in zip(_frac_panels, axes.flat):
            frac = r.get(frac_key, 0.0) or 0.0
            start = r.get(start_key)
            stop = r.get(stop_key)
            eff = [frac * _activation_scale(ep, start, stop, ramp) for ep in range(n_epochs)]
            if any(v > 0 for v in eff):
                has_data = True
            markevery = max(1, n_epochs // marker_every) if mk else None
            ax.plot(eff, label=label, alpha=0.85,
                    color=color, linestyle=ls, marker=mk,
                    markevery=markevery, markersize=5)

    if not has_data:
        plt.close(fig)
        return

    for (_, _, _, panel_title), ax in zip(_frac_panels, axes.flat):
        ax.set_xlabel("Epoch", fontsize=base_font_size * 1.0)
        ax.set_ylabel("Effective Fraction", fontsize=base_font_size * 1.0)
        ax.set_title(f"{title_prefix} — {panel_title}", fontsize=base_font_size * 1.1)
        ax.tick_params(axis="both", labelsize=base_font_size * 0.85)
        ax.grid(True, alpha=0.3)

    _add_shared_legend(fig, label_entries, styles, base_font_size=base_font_size)
    plt.show()


def plot_flow_loss_breakdown_chart(  # [STSI 030626]:dedicated flow loss breakdown diagnostic
    runs,
    title_prefix="Flow Loss Breakdown",
    marker_every=10,
    label_mode="auto",
    compact_threshold=RUN_LABEL_COMPACT_THRESHOLD,
    index_threshold=RUN_LABEL_INDEX_THRESHOLD,
    figsize=(32, 8),
    base_font_size=10,
):
    """Plot flow loss breakdown: total, effective weight, and sub-components if available.

    Gracefully handles runs without flow loss history (backward-compat).
    """
    # Check if ANY run has flow loss history
    has_flow = any("train_flow_loss" in r.get("history", {}) for r in runs)
    if not has_flow:
        return None

    styles = _get_run_styles(runs)
    _, label_entries = resolve_run_label_policy(
        runs,
        label_mode=label_mode,
        compact_threshold=compact_threshold,
        index_threshold=index_threshold,
    )
    label_entry_by_run = _index_label_entries(label_entries)

    # Determine available sub-components [STSI 040626]:added per-sub-loss eff weight panels
    sample_h = next((r["history"] for r in runs if "train_flow_loss" in r.get("history", {})), {})
    has_sub = "train_flow_dc_local" in sample_h
    has_eff_sub = "train_eff_w_dcf_local" in sample_h

    n_panels = 2 + (2 if has_sub else 0) + (2 if has_eff_sub else 0)
    fig, axes = plt.subplots(1, n_panels, figsize=(figsize[0], figsize[1]))
    fig.suptitle(f"{title_prefix}", fontsize=base_font_size * 1.3)
    if not hasattr(axes, '__len__'):
        axes = [axes]
    else:
        axes = list(axes)
    while len(axes) < 6:
        axes.append(None)

    # Panel 1: Total flow loss (train + val)
    ax = axes[0]
    for r in runs:
        label = label_entry_by_run[id(r)]["legend_label"]
        h = r.get("history", {})
        color, ls, mk = styles[id(r)]
        if "train_flow_loss" in h and h["train_flow_loss"]:
            n = len(h["train_flow_loss"])
            markevery = max(1, n // marker_every) if mk else None
            ax.plot(h["train_flow_loss"], label=f"{label} (train)",
                    color=color, linestyle=ls, marker=mk,
                    markevery=markevery, markersize=5)
        if "val_flow_loss" in h and h["val_flow_loss"]:
            ax.plot(h["val_flow_loss"], label=f"{label} (val)",
                    color=color, linestyle="--", alpha=0.7)
    ax.set_title("Total Flow Loss", fontsize=base_font_size * 1.1)
    ax.set_xlabel("Epoch", fontsize=base_font_size * 1.0)
    ax.set_ylabel("Loss", fontsize=base_font_size * 1.0)
    ax.tick_params(axis="both", labelsize=base_font_size * 0.85)
    ax.grid(True, alpha=0.3)

    # Panel 2: Effective flow weight
    ax = axes[1]
    for r in runs:
        label = label_entry_by_run[id(r)]["legend_label"]
        h = r.get("history", {})
        color, ls, mk = styles[id(r)]
        if "train_eff_w_flow" in h and h["train_eff_w_flow"]:
            n = len(h["train_eff_w_flow"])
            markevery = max(1, n // marker_every) if mk else None
            ax.plot(h["train_eff_w_flow"], label=label,
                    color=color, linestyle=ls, marker=mk,
                    markevery=markevery, markersize=5)
    ax.set_title("Effective Flow Weight", fontsize=base_font_size * 1.1)
    ax.set_xlabel("Epoch", fontsize=base_font_size * 1.0)
    ax.set_ylabel("Weight", fontsize=base_font_size * 1.0)
    ax.tick_params(axis="both", labelsize=base_font_size * 0.85)
    ax.grid(True, alpha=0.3)

    if has_sub:
        # Panel 3: DC sub-components (local vs global)
        ax = axes[2]
        for r in runs:
            label = label_entry_by_run[id(r)]["legend_label"]
            h = r.get("history", {})
            color, ls, mk = styles[id(r)]
            if "train_flow_dc_local" in h and h["train_flow_dc_local"]:
                ax.plot(h["train_flow_dc_local"], label=f"{label} DC-local",
                        color=color, linestyle="-")
            if "train_flow_dc_global" in h and h["train_flow_dc_global"]:
                ax.plot(h["train_flow_dc_global"], label=f"{label} DC-global",
                        color=color, linestyle="--")
        ax.set_title("DC Flow Sub-Components", fontsize=base_font_size * 1.1)
        ax.set_xlabel("Epoch", fontsize=base_font_size * 1.0)
        ax.set_ylabel("MSE", fontsize=base_font_size * 1.0)
        ax.tick_params(axis="both", labelsize=base_font_size * 0.85)
        ax.grid(True, alpha=0.3)

        # Panel 4: AC sub-components (local vs global)
        ax = axes[3]
        for r in runs:
            label = label_entry_by_run[id(r)]["legend_label"]
            h = r.get("history", {})
            color, ls, mk = styles[id(r)]
            if "train_flow_ac_local" in h and h["train_flow_ac_local"]:
                ax.plot(h["train_flow_ac_local"], label=f"{label} AC-local",
                        color=color, linestyle="-")
            if "train_flow_ac_global" in h and h["train_flow_ac_global"]:
                ax.plot(h["train_flow_ac_global"], label=f"{label} AC-global",
                        color=color, linestyle="--")
        ax.set_title("AC Flow Sub-Components", fontsize=base_font_size * 1.1)
        ax.set_xlabel("Epoch", fontsize=base_font_size * 1.0)
        ax.set_ylabel("MSE", fontsize=base_font_size * 1.0)
        ax.tick_params(axis="both", labelsize=base_font_size * 0.85)
        ax.grid(True, alpha=0.3)

    if has_eff_sub:  # [STSI 040626]:per-sub-loss effective weight panels
        _panel_idx = 4 if has_sub else 2
        # Panel: DC sub-loss effective weights
        ax = axes[_panel_idx]
        for r in runs:
            label = label_entry_by_run[id(r)]["legend_label"]
            h = r.get("history", {})
            color, ls, mk = styles[id(r)]
            if "train_eff_w_dcf_local" in h and h["train_eff_w_dcf_local"]:
                ax.plot(h["train_eff_w_dcf_local"], label=f"{label} local",
                        color=color, linestyle="-")
            if "train_eff_w_dcf_global" in h and h["train_eff_w_dcf_global"]:
                ax.plot(h["train_eff_w_dcf_global"], label=f"{label} global",
                        color=color, linestyle="--")
        ax.set_title("DC Flow — Effective Weights", fontsize=base_font_size * 1.1)
        ax.set_xlabel("Epoch", fontsize=base_font_size * 1.0)
        ax.set_ylabel("Weight", fontsize=base_font_size * 1.0)
        ax.tick_params(axis="both", labelsize=base_font_size * 0.85)
        ax.grid(True, alpha=0.3)

        # Panel: AC sub-loss effective weights
        ax = axes[_panel_idx + 1]
        for r in runs:
            label = label_entry_by_run[id(r)]["legend_label"]
            h = r.get("history", {})
            color, ls, mk = styles[id(r)]
            if "train_eff_w_acf_local" in h and h["train_eff_w_acf_local"]:
                ax.plot(h["train_eff_w_acf_local"], label=f"{label} local",
                        color=color, linestyle="-")
            if "train_eff_w_acf_global" in h and h["train_eff_w_acf_global"]:
                ax.plot(h["train_eff_w_acf_global"], label=f"{label} global",
                        color=color, linestyle="--")
        ax.set_title("AC Flow — Effective Weights", fontsize=base_font_size * 1.1)
        ax.set_xlabel("Epoch", fontsize=base_font_size * 1.0)
        ax.set_ylabel("Weight", fontsize=base_font_size * 1.0)
        ax.tick_params(axis="both", labelsize=base_font_size * 0.85)
        ax.grid(True, alpha=0.3)

    _add_shared_legend(fig, label_entries, styles, base_font_size=base_font_size)
    plt.show()
    return fig

def plot_theta_consistency_chart(runs, title_prefix="Dual-Head θ Consistency",  # [STSI 030626]:dual-head comparison chart
                                metrics_key="test_metrics", base_font_size=10):
    """Bar chart comparing line_flow_rmse from node-head θ vs edge-head Δθ.

    Only renders for runs where both metrics exist (angle_mode='both').
    Panel 1: Line flow RMSE comparison (edge Δθ vs node θ).
    Panel 2: θ consistency (MAE + RMSE between node head and BFS-reconstructed θ).
    """
    _, label_entries = resolve_run_label_policy(runs)
    label_entry_by_run = _index_label_entries(label_entries)

    # Filter to runs that have the dual-head metric
    valid = [(i, r) for i, r in enumerate(runs)
             if "line_flow_rmse_delta_theta" in r.get(metrics_key, {})]
    if not valid:
        return None

    labels = [label_entry_by_run[id(r)]["legend_label"] for _, r in valid]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Panel 1: Line flow RMSE comparison
    ax = axes[0]
    x = np.arange(len(valid))
    width = 0.35
    rmse_dtheta = [r[metrics_key].get("line_flow_rmse_delta_theta", 0) for _, r in valid]
    rmse_node = [r[metrics_key].get("line_flow_rmse_node_theta", 0) for _, r in valid]
    ax.bar(x - width/2, rmse_dtheta, width, label="Edge Δθ head", color="steelblue")
    ax.bar(x + width/2, rmse_node, width, label="Node θ head", color="darkorange")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=base_font_size * 0.8)
    ax.set_ylabel("Line Flow RMSE [pu]", fontsize=base_font_size * 1.0)
    ax.set_title(f"{title_prefix} — Line Flow RMSE: Edge Δθ vs Node θ", fontsize=base_font_size * 1.1)
    ax.legend(fontsize=base_font_size * 0.85)

    # Panel 2: θ consistency (MAE + RMSE)
    ax = axes[1]
    cons_mae = [r[metrics_key].get("theta_consistency_mae", 0) for _, r in valid]
    cons_rmse = [r[metrics_key].get("theta_consistency_rmse", 0) for _, r in valid]
    ax.bar(x - width/2, cons_mae, width, label="θ consistency MAE", color="teal")
    ax.bar(x + width/2, cons_rmse, width, label="θ consistency RMSE", color="salmon")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=base_font_size * 0.8)
    ax.set_ylabel("Angle [rad]", fontsize=base_font_size * 1.0)
    ax.set_title(f"{title_prefix} — Dual-Head θ Consistency", fontsize=base_font_size * 1.1)
    ax.legend(fontsize=base_font_size * 0.85)

    plt.tight_layout()
    plt.show()
    return fig


def plot_all_runs_accuracy_metrics(
    runs,
    title_prefix="All runs",
    label_mode="auto",
    compact_threshold=RUN_LABEL_COMPACT_THRESHOLD,
    index_threshold=RUN_LABEL_INDEX_THRESHOLD,
    metrics_key: str = "test_metrics",  # [STSI 300526]:configurable key for external test set
    base_font_size=10,
):
    """
    Bar chart comparison of test accuracy metrics across runs.
    Reads from r["test_metrics"]["overall_metrics"] for mean/std per metric.
    Metrics: vmag_mae, vang_rmse, p_mae, q_mae.
    """
    metrics = [
        ("vmag_mae", "Vmag MAE  [p.u.]"),
        ("vang_rmse", "Vang RMSE [rad]"),
        ("p_mae", "P MAE     [p.u.]"),
        ("q_mae", "Q MAE     [p.u.]"),
        ("line_flow_mae", "Line Flow MAE [p.u.]"),
        ("line_flow_q0_mae", "Line Flow Q0 MAE [p.u.]"),
        ("delta_theta_mae", "Δθ MAE [rad]"),  # [STSI 270526]:task 028
        ("delta_theta_rmse", "Δθ RMSE [rad]"),  # [STSI 270526]:task 028
    ]
    x = list(range(len(runs)))
    active_label_mode, label_entries = resolve_run_label_policy(
        runs,
        label_mode=label_mode,
        compact_threshold=compact_threshold,
        index_threshold=index_threshold,
    )
    styles = _get_run_styles(runs)
    colors = [styles[id(r)][0] for r in runs]
    tick_labels = [entry["plot_label"] for entry in label_entries]
    tick_rotation = 0 if active_label_mode == "index" else 30
    tick_ha = "center" if active_label_mode == "index" else "right"
    tick_fontsize = base_font_size * (0.9 if active_label_mode == "index" else 0.8)

    fig, axes = plt.subplots(1, len(metrics), figsize=(max(16, 2.5 * len(metrics) + len(runs)), 6))
    fig.suptitle(f"{title_prefix} — Test Accuracy Metrics", fontsize=base_font_size * 1.3)
    for ax, (metric_key, metric_label) in zip(axes, metrics):
        means, stds = [], []
        for r in runs:
            tm = r.get(metrics_key, {})
            val = tm.get(metric_key, float("nan"))
            if isinstance(val, dict):
                mean = float(val.get("mean", float("nan")))
                std = float(val.get("std", 0.0))
            else:
                mean = float(val)
                std = 0.0
            means.append(mean)
            stds.append(std)

        bars = ax.bar(x, means, yerr=stds, capsize=5, alpha=0.8, color=colors)
        ax.set_xticks(x)
        ax.set_xticklabels(tick_labels, rotation=tick_rotation, ha=tick_ha, fontsize=tick_fontsize)
        ax.set_ylabel(metric_label, fontsize=base_font_size * 1.0)
        ax.set_title(metric_label, fontsize=base_font_size * 1.1)
        ax.grid(True, axis="y", alpha=0.3)

        valid_means = [m for m in means if m == m]
        top = max(valid_means) if valid_means else 1.0
        for bar, mean in zip(bars, means):
            if mean == mean:
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + top * 0.02,
                    f"{mean:.4f}",
                    ha="center", va="bottom", fontsize=base_font_size * 0.75,
                )

    if active_label_mode == "index":
        legend_lines = [
            plt.Line2D([0], [0], color=colors[i], linewidth=6, alpha=0.8,
                       label=label_entries[i]["mapping_label"])
            for i in range(len(runs))
        ]
        fig.legend(
            handles=legend_lines,
            loc="lower center",
            bbox_to_anchor=(0.5, -0.02),
            ncol=max(1, min(3, len(runs))),
            fontsize=base_font_size * 0.85,
            frameon=True,
        )
        plt.tight_layout(rect=[0, 0.12 + 0.03 * (len(runs) // 3), 1, 1])
    else:
        plt.tight_layout()
    plt.show()

def print_sweep_summary(runs_list, top_n=5, metrics_key="test_metrics"):  # [STSI 300526]:metrics_key param for external test set
    """
    Print a ranked summary table of hyperparameter sweep results.
    Shows top_n configurations by test loss.
    """
    sorted_runs = sorted(runs_list, key=lambda r: r.get(metrics_key, {}).get("test_total", float("inf")))  # [STSI 300526]:guard against missing key
    # [STSI100526] Added head_mode/conv_mode columns to summary table
    print(f"{'Rank':<5}{'w_phys':<9}{'w_ptdf':<9}{'w_P':<7}{'w_Q':<7}{'kvl':<6}{'bDC':<5}{'kvlAct':<9}{'am':<5}{'head':<14}{'res':<6}{'norm':<8}{'act':<12}{'do':<5}"  # [STSI 220526] w_P/w_Q; [STSI 280926] kvl/bDC; [STSI 021026] kvlAct
          f"{'bs':<6}{'lr':<11}"
          f"{'Train':<11}{'Val':<11}{'Test':<11}{'Time(s)':<9}")
    print("-" * 154)
    for i, run in enumerate(sorted_runs[:top_n], 1):
        tm = run.get(metrics_key, {})
        kvl_act_str = f"{run.get('kvl_activation_start', 0)}-{run.get('kvl_activation_stop') or 'end'}"  # [STSI 021026]
        print(
            f"{i:<5}{run['weight_physics']:<9}{run['weight_ptdf']:<9}"
            f"{run.get('w_P', 0.5):<7.2f}{run.get('w_Q', 0.5):<7.2f}"  # [STSI 220526]
            f"{run.get('fraction_kvl', 0.0):<6.2f}{str(run.get('enable_balance_dc', False)):<5}"  # [STSI 280926]
            f"{kvl_act_str:<9}"  # [STSI 021026]
            f"{'ed' if run.get('angle_mode', 'node') == 'edge_delta' else 'n':<5}"  # [STSI 260526]
            f"{run.get('head_mode','standard'):<14}"
            f"{str(run.get('use_residual', False)):<6}{str(run.get('norm_type', None)):<8}{run.get('activation', 'leaky_relu'):<12}{str(run.get('use_dropout', False)):<5}"  # [STSI 120526]
            f"{run['batch_size']:<6}{run['lr']:<11.2e}"
            f"{run['final_train_loss']:<11.6f}{run['final_val_loss']:<11.6f}"
            f"{tm['test_total']:<11.6f}{run['training_time']:<9.1f}"
        )

def _sanitise_filename(s: str) -> str:
    """Replace characters illegal on Windows filesystems with safe alternatives."""
    for ch in r'\/:*?"<>|=':
        s = s.replace(ch, "-")
    return s


class pdf_output:  # [STSI 250726]:context manager — save each plt.show() as a separate PDF in pdf_dir
    """Context manager: save every plt.show() figure as a separate PDF in pdf_dir.

    No imports required. Works as a with-statement or via __enter__/__exit__.

    Args:
        pdf_dir : directory to write PDFs into (created if absent)
        prefix  : filename prefix (e.g. sanitised title_prefix or sweep name)

    Example — standalone:
        with pdf_output(os.path.join(TRAINING_RESULTS_DIR, "plots"), prefix="R5bn"):
            plot_all_runs_training_curves(runs, title_prefix="R5bn")
            plot_all_runs_accuracy_metrics(runs, title_prefix="R5bn")

    Example — via orchestrator:
        check_hparam_results(runs, ..., save_to_pdf=True)
    """
    def __init__(self, pdf_dir: str, prefix: str = ""):
        self._pdf_dir = pdf_dir
        self._safe_prefix = _sanitise_filename(prefix)
        self._counter = [0]
        self._orig_show = None
        self._orig_fonttypes = None

    def __enter__(self):
        os.makedirs(self._pdf_dir, exist_ok=True)
        self._orig_show = plt.show
        self._counter[0] = 0
        # [STSI 070926]:embed real (selectable/editable) text in PDF instead of vector-outlined glyphs
        self._orig_fonttypes = (plt.rcParams["pdf.fonttype"], plt.rcParams["ps.fonttype"])
        plt.rcParams["pdf.fonttype"] = 42
        plt.rcParams["ps.fonttype"] = 42
        _dir = self._pdf_dir
        _sfx = self._safe_prefix
        _cnt = self._counter
        _orig = self._orig_show

        def _saving_show(*args, **kwargs):
            _fig = plt.gcf()
            if _fig.get_axes():
                _cnt[0] += 1
                _fname = (f"{_sfx}_{_cnt[0]:02d}.pdf"
                          if _sfx else f"plot_{_cnt[0]:02d}.pdf")
                _fig.savefig(os.path.join(_dir, _fname), bbox_inches="tight")
                print(f"  [PDF] Saved: {_fname}")
            _orig(*args, **kwargs)

        plt.show = _saving_show
        return self

    def __exit__(self, *_):
        if self._orig_show is not None:
            plt.show = self._orig_show
        if self._orig_fonttypes is not None:
            plt.rcParams["pdf.fonttype"], plt.rcParams["ps.fonttype"] = self._orig_fonttypes


def load_sweep_results(pkl_path: str) -> list:
    """
    Load a full sweep result list from a pickle file.
    Tags each run with _source_pkl for save_backfilled traceability.
    """
    with open(pkl_path, "rb") as f:
        runs = pickle.load(f)
    for r in runs:
        r["_source_pkl"] = pkl_path
    print(f"Loaded {len(runs)} runs from {pkl_path}")
    return runs


def load_sweep_metrics(csv_path: str) -> pd.DataFrame:
    """
    Load just the metrics CSV (no models) for fast analysis.
    """
    df = pd.read_csv(csv_path)
    print(f"Loaded metrics for {len(df)} runs from {csv_path}")
    return df

def reconstruct_test_split(networks, seed=42, train_frac=0.70, val_frac=0.15):
    """
    Reproduce the train/val/test split that train_power_flow_gnn uses internally.
    Returns (train_networks, val_networks, test_networks).
    Use this only as a fallback for old runs that don't store test_networks.
    """
    import random as _random
    net_copy = list(networks)
    _random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    _random.shuffle(net_copy)
    n = len(net_copy)
    n_train = int(train_frac * n)
    n_val = int((train_frac + val_frac) * n) - n_train
    train_nets = net_copy[:n_train]
    val_nets = net_copy[n_train : n_train + n_val]
    test_nets = net_copy[n_train + n_val :]
    return train_nets, val_nets, test_nets

def filter_runs(runs, criteria=None):
    """
    Filter runs such that for each column in criteria, the run value matches
    one of the allowed values for that column.

    - criteria is a dict: {col_name: [allowed_val1, allowed_val2, ...], ...}
    - columns combine with AND
    - values per column combine with OR
    """
    if criteria is None:
        criteria = {'weight_physics': ['0.1']}

    # convert to sets for efficiency
    crit_sets = {col: set(vals) for col, vals in criteria.items()}

    relevant_runs = []
    for r in runs:
        if all(r.get(col) in valset for col, valset in crit_sets.items()):
            relevant_runs.append(r)

    return relevant_runs

def check_hparam_results(runs,
                         networks=None,
                         training_networks=None,
                         title_prefix:str="Runs Analysis", 
                         factors=["weight_physics","weight_ptdf","head_mode","use_residual","norm_type","activation","use_dropout"],  # [STSI 140526]:added use_dropout
                         interactions=[("weight_physics","use_residual"),("weight_ptdf","norm_type")],  # [STSI 140526]:aligned interactions
                         test_network=None,
                         # ── Analysis toggles ─────────────────────  # [STSI 190526]:per-analysis toggles
                         plot_training_curves: bool = True,
                         plot_loss_components: bool = True,
                         plot_effective_weights: bool = True,  # [STSI 200526]:toggle for effective weight plot
                         plot_effective_fractions: bool = True,  # [STSI 090626]:toggle for effective fractions chart
                         plot_flow_loss_breakdown: bool = True,  # [STSI 030626]:toggle for flow loss breakdown chart
                         plot_theta_consistency: bool = True,  # [STSI 030626]:toggle for dual-head θ consistency chart
                         plot_accuracy_metrics: bool = True,
                         plot_inference_time: bool = True,
                         plot_timing_accuracy: bool = True,
                         timing_y_metrics: list = None,  # [STSI 260526]:pass-through for plot_timing_vs_accuracy y_metrics
                         timing_y_metrics_row2: list = None,  # [STSI 010626]:pass-through for plot_timing_vs_accuracy y_metrics_row2
                         run_factorial_analysis: bool = True,
                         composite_weights: dict = None,  # [STSI 150626]:pass-through for weighted composite response
                         run_multi_net_eval: bool = True,
                         physics_comparison_results: bool = True,
                         scatter_hist_layout: str = "by_run",  # [STSI 100926]:pass-through layout ("by_run"|"by_metric") for scatter/histogram plots
                         scatter_hist_max_cols: int = 3,  # [STSI 100926]:pass-through max_cols for by_metric grid wrapping
                         operating_ranges: dict = None,  # [STSI 100926]:pass-through metric-name-keyed axis-clip range for scatter/histogram/error-vs-range plots
                         # ──────────────────────────────────────────
                         ref_baselines: dict = None,
                         ref_baselines_batch: dict = None,   # [STSI260502] batch baseline group
                         include_dc_in_network_plot: bool = False,  # [STSI 080926]:add DC (lpf) to the single-network comparison plots
                         include_dc_in_test_eval: bool = False,  # [STSI 090926]:add DC (lpf) to full test-set scatter/histogram/error-vs-range plots
                         postprocess_dangling: bool = True,  # [STSI070526] dangling bus vmag/vang fix
                         use_pnom_share: Optional[bool] = None,  # None=auto from model.in_features; True/False forces mode
                         save_backfilled: bool = False,  # [STSI 270526]:persist backfilled metrics to {orig}_backfilled.pkl
                         external_test_set: bool = False,  # [STSI 300526]:force re-eval on networks=, store in test_metrics_external
                         force_external_reeval: bool = True,  # [STSI 310526]:if False, skip re-eval when test_metrics_external already exists
                         max_plot_buses: int = 10,  # [STSI 200526]:None=plot all; int=sample representative buses if network exceeds threshold
                         save_to_pdf: bool = False,  # [STSI 250726]:save each plot as separate PDF in TRAINING_RESULTS_DIR/plots/
                         base_font_size: float = 10,
                         heatmap_corr_method: str = "pearson",  # [STSI 030926]:'pearson' or 'spearman'
                         heatmap_fdr_alpha: float = 0.05,
                         verbose: bool = True,
                         ):
    # ── Determine test networks for evaluation ───────────────────────────────
    eval_networks = networks  # explicit override, if given
    if eval_networks is None:
        # Try to use test_networks stored during training
        stored = runs[0].get("test_networks") if runs else None
        if stored is not None:
            eval_networks = stored
            print(f"[check_hparam_results] Using stored test split ({len(eval_networks)} networks)")
        elif training_networks is not None:
            # Reconstruct from the full network list + seed
            seed = runs[0].get("seed", 42) if runs else 42
            _, _, eval_networks = reconstruct_test_split(training_networks, seed=seed)
            print(f"[check_hparam_results] Reconstructed test split ({len(eval_networks)} networks)")

    _, label_entries = resolve_run_label_policy(runs)
    label_entry_by_run = _index_label_entries(label_entries)

    # ── Re-evaluate any runs missing line_flow_mae / line_flow_q0_mae ────────
    REQUIRED_METRICS = {"vmag_mae", "vang_rmse", "p_mae", "q_mae",
                        "line_flow_mae", "line_flow_q0_mae",
                        "vmag_true_std", "vang_true_std", "p_true_std", "q_true_std",
                        "inference_time_per_snapshot_ms",
                        # [STSI260502]: multi-level timing
                        "solve_time_ms", "postproc_time_ms", "total_time_ms",
                        "batch_solve_time_ms"}

    if eval_networks is not None:
        if external_test_set:  # [STSI 300526]:always evaluate on provided networks, store in test_metrics_external
            for r in runs:
                if not force_external_reeval and r.get("test_metrics_external") is not None:
                    continue
                _lbl = label_entry_by_run[id(r)]["legend_label"]
                print(f"[check_hparam_results] Evaluating '{_lbl}' on external test set "
                      f"({len(eval_networks)} networks)")
                hparams_r     = r.get("hparams") or {}
                ptdf_params_r = r.get("_ptdf_params") or hparams_r.get("ptdf_params", None)
                max_buses_r   = r.get("_max_buses") or hparams_r.get("max_buses", 0)
                ptdf_offset_r = r.get("_ptdf_offset") or hparams_r.get("ptdf_offset", 0)
                try:
                    new_metrics = evaluate_gnn_on_test_set(
                        r["model"],
                        eval_networks,
                        use_edge_features=True,
                        use_pnom_share=use_pnom_share,
                        ptdf_params=ptdf_params_r,
                        max_buses=max_buses_r,
                        ptdf_offset=ptdf_offset_r,
                        postprocess_dangling=postprocess_dangling,
                        verbose=verbose
                    )
                    r["test_metrics_external"] = new_metrics
                except RuntimeError as e:
                    print(f"  [WARNING] External evaluation skipped for '{_lbl}': {e}")
        else:
            # ── Backfill: only re-evaluate runs missing required metrics ──
            for r in runs:
                tm = r.get("test_metrics") or {}
                missing = REQUIRED_METRICS - set(tm.keys())
                if missing:
                    _lbl = label_entry_by_run[id(r)]["legend_label"]
                    print(f"[check_hparam_results] Re-evaluating '{_lbl}' "
                        f"\u2014 missing metrics: {missing}")
                    hparams_r     = r.get("hparams") or {}
                    ptdf_params_r = r.get("_ptdf_params") or hparams_r.get("ptdf_params", None)
                    max_buses_r   = r.get("_max_buses") or hparams_r.get("max_buses", 0)
                    ptdf_offset_r = r.get("_ptdf_offset") or hparams_r.get("ptdf_offset", 0)
                    try:
                        new_metrics = evaluate_gnn_on_test_set(
                            r["model"],
                            eval_networks,
                            use_edge_features=True,
                            use_pnom_share=use_pnom_share,
                            ptdf_params=ptdf_params_r,
                            max_buses=max_buses_r,
                            ptdf_offset=ptdf_offset_r,
                            postprocess_dangling=postprocess_dangling,
                            verbose=verbose
                        )
                        r["test_metrics"] = new_metrics
                    except RuntimeError as e:
                        print(f"  [WARNING] Re-evaluation skipped for '{_lbl}': {e}")

    # ── Metrics key for downstream reads ──────────────────────────────────────  # [STSI 300526]:dual-storage key routing
    if external_test_set and eval_networks is not None:
        metrics_key = "test_metrics_external"
    else:
        metrics_key = "test_metrics"

    # [STSI 270526]:persist backfilled metrics to avoid re-evaluation on next load
    if save_backfilled and any(r.get("_source_pkl") for r in runs):
        # Group runs by source pkl
        from collections import defaultdict
        by_source = defaultdict(list)
        for r in runs:
            sp = r.get("_source_pkl")
            if sp:
                by_source[sp].append(r)
        for src_pkl, src_runs in by_source.items():
            base, ext = os.path.splitext(src_pkl)
            out_path = f"{base}_backfilled{ext}"
            with open(out_path, "wb") as f:
                pickle.dump(src_runs, f)
            print(f"[check_hparam_results] Saved backfilled metrics -> {out_path}")

    # ── PDF capture setup ─────────────────────────────────────────────────────
    if save_to_pdf:
        _pdf_obj = pdf_output(os.path.join(TRAINING_RESULTS_DIR, "plots"),
                              prefix=_sanitise_filename(title_prefix))
        _pdf_obj.__enter__()

    # ── Training curves & accuracy metrics ───────────────────────────────────────

    if plot_training_curves:  # [STSI 190526]:toggle guard
        plot_all_runs_training_curves(runs, title_prefix=title_prefix, base_font_size=base_font_size)
    if plot_loss_components:  # [STSI 190526]:toggle guard
        plot_loss_components_chart(runs, title_prefix=title_prefix, base_font_size=base_font_size)
    if plot_effective_weights:  # [STSI 200526]:dispatch effective weight chart
        plot_effective_weights_chart(runs, title_prefix=title_prefix, base_font_size=base_font_size)
    if plot_effective_fractions:  # [STSI 090626]:dispatch effective fractions chart
        plot_effective_fractions_chart(runs, title_prefix=title_prefix, base_font_size=base_font_size)
    if plot_flow_loss_breakdown and any("train_flow_loss" in r.get("history", {}) for r in runs):  # [STSI 030626]:flow loss breakdown dispatch
        plot_flow_loss_breakdown_chart(runs, title_prefix=title_prefix, base_font_size=base_font_size)
    if plot_theta_consistency:  # [STSI 030626]:dispatch dual-head θ consistency chart
        plot_theta_consistency_chart(runs, title_prefix=title_prefix, metrics_key=metrics_key, base_font_size=base_font_size)
    if plot_accuracy_metrics:  # [STSI 190526]:toggle guard
        plot_all_runs_accuracy_metrics(runs, title_prefix=title_prefix, metrics_key=metrics_key, base_font_size=base_font_size)  # [STSI 300526]:pass metrics_key for external test set
    if plot_inference_time:  # [STSI 190526]:toggle guard
        plot_inference_time_bar(runs, title_prefix=title_prefix, metrics_key=metrics_key)  # [STSI 300526]:pass metrics_key for external test set
    if plot_timing_accuracy:  # [STSI 190526]:toggle guard
        _all_baselines = {**(ref_baselines or {}), **(ref_baselines_batch or {})}
        _default_row1 = ["vmag_mae", "delta_theta_mae", "p_mae", "q_mae", "line_flow_mae", "line_flow_q0_mae"]  # [STSI 010626]:6-col x 2-row scatter grid
        _default_row2 = ["vmag_rmse", "delta_theta_rmse", "p_rmse", "q_rmse", "line_flow_rmse", "line_flow_q0_rmse"]
        plot_timing_vs_accuracy(runs, x_metric="infer_ms_per_snap",
                                y_metrics=timing_y_metrics or _default_row1,
                                y_metrics_row2=timing_y_metrics_row2 or _default_row2,
                                title_prefix=title_prefix,
                                ref_baselines=_all_baselines or None,
                                timing_key="both",
                                x_axis_log=True,
                                metrics_key=metrics_key,
                                base_font_size=base_font_size)

    # ── Factorial analysis ────────────────────────────────────────────────────────
    # Auto-include flow fraction factors when they vary [STSI 040626]:auto-include varying flow fractions
    for _fk in ("fraction_dcf_local", "fraction_dcf_global", "fraction_acf_local", "fraction_acf_global"):
        if len(set(r.get(_fk, 0.0) for r in runs)) > 1 and _fk not in factors:
            factors = list(factors) + [_fk]
    results, coefs = None, None
    if run_factorial_analysis:
        results, coefs = factorial_analysis(
            runs=runs,
            factors=factors,
            interactions=interactions,
            response_metrics=["vmag_mae", "vang_mae", "delta_theta_mae",
                              "p_mae", "q_mae", "line_flow_mae"],
            response_metrics_row2=["vmag_rmse", "vang_rmse", "delta_theta_rmse",
                                   None, None, "line_flow_q0_mae"],
            metrics_key=metrics_key,
            composite_weights=composite_weights,  # [STSI 150626]
            base_font_size=base_font_size,
        )
    
        if results:
            print(results["p_mae"].summary())

    # ── Factor-metric correlation heatmap (independent of OLS) ──────────────────
    if run_factorial_analysis:
        plot_factor_metric_heatmap(
            runs,
            response_metrics=["vmag_mae", "vang_rmse", "p_mae", "q_mae",
                              "line_flow_mae", "line_flow_q0_mae", "delta_theta_mae"],
            factors=factors + ["n_buses", "dataset_size"],
            metrics_key=metrics_key,
            composite_weights=composite_weights,  # [STSI 150626]
            base_font_size=base_font_size,
            corr_method=heatmap_corr_method,
            fdr_alpha=heatmap_fdr_alpha,
        )

    # ── Per-network visual comparison (qualitative) ─────────────────────────────
    if test_network is not None:
        # --- Bus sampling for readable plots [STSI 200526]:sample representative buses when network is large ---
        num_buses = len(test_network.buses)
        if max_plot_buses and num_buses > max_plot_buses:
            slack_buses = test_network.buses[test_network.buses.control == "Slack"].index.tolist()
            pv_buses = test_network.buses[test_network.buses.control == "PV"].index.tolist()
            pq_buses = test_network.buses[test_network.buses.control == "PQ"].index.tolist()
            sampled = slack_buses[:1] + pv_buses[:min(2, len(pv_buses))]
            remaining = max_plot_buses - len(sampled)
            if remaining > 0:
                pool = [b for b in pq_buses + pv_buses[min(2, len(pv_buses)):] if b not in sampled]
                sampled += list(np.random.choice(pool, size=min(remaining, len(pool)), replace=False))
            plot_buses = sorted(sampled, key=lambda b: list(test_network.buses.index).index(b))
            print(f"  Network has {num_buses} buses; plotting representative sample of {len(plot_buses)}.")
        else:
            plot_buses = None

        plot_single_network_comparisons(
            runs,
            test_network=test_network,
            title_prefix=title_prefix,
            use_pnom_share=use_pnom_share,
            plot_buses=plot_buses,
            base_font_size=base_font_size,
            include_dc=include_dc_in_network_plot,
        )


    # Build models dict using shared label policy
    models_dict = {}
    model_summary = {}
    for r in runs:
        label = label_entry_by_run[id(r)]["legend_label"]
        ptdf_params = r.get("_ptdf_params")
        max_buses   = r.get("_max_buses", 0)
        if ptdf_params is not None and max_buses > 0:
            models_dict[label] = {
                "model": r["model"],
                "ptdf_params": ptdf_params,
                "max_buses": max_buses,
            }
        else:
            models_dict[label] = r["model"]
        _tm = r.get("test_metrics") or {}
        model_summary[label] = {
            "test_metrics": {k: v for k, v in _tm.items() if k != "line_flow_raw"},
        }
    print("Model dict build")

    all_preds = {}  # init so return always works even when run_multi_net_eval=False
    multi_net_metrics = {}  # [STSI 190526]:init before conditional so return always works
    if eval_networks is not None and run_multi_net_eval:  # [STSI 190526]:toggle guard
        # ── Single evaluation pass: collect raw predictions ──────────────  # [STSI 290526]:eliminated redundant Pass 2 (task 033)
        all_preds = _collect_test_set_predictions(
            models_dict, eval_networks, use_pnom_share=use_pnom_share,
            include_dc=include_dc_in_test_eval,
        )

        # ── Derive aggregate MAE from raw predictions (replaces evaluate_models_on_networks call) ──
        for lbl, data in all_preds.items():
            if lbl not in models_dict:  # e.g. DC baseline — not part of models_dict-based bar mapping
                continue
            pred, true = data["pred"], data["true"]
            if len(pred) == 0:
                continue
            multi_net_metrics[lbl] = {
                "vmag_mae":     float(np.mean(np.abs(pred[:, 0] - true[:, 0]))),
                "vang_mae_deg": float(np.mean(np.abs(pred[:, 1] - true[:, 1]))),
                "p_mae":        float(np.mean(np.abs(pred[:, 2] - true[:, 2]))),
                "q_mae":        float(np.mean(np.abs(pred[:, 3] - true[:, 3]))),
            }
        print("multi_net_metrics done!")
        for mnm_lbl, mnm_vals in multi_net_metrics.items():
            if mnm_lbl in model_summary:
                model_summary[mnm_lbl]["multi_net_metrics"] = mnm_vals

        # ── Bar chart of per-model MAE ──────────────────────────────────
        plot_model_metrics_across_networks(
            models_dict,
            eval_networks,
            title_prefix=title_prefix,
            use_pnom_share=use_pnom_share,
            metrics=multi_net_metrics,
            base_font_size=base_font_size,
        )

        # ── Full test-set scatter & error histograms ────────────────────
        if physics_comparison_results:  # [STSI 190526]:toggle guard
            # ── Δθ distribution diagnostic ──  # [STSI 020626]:diagnostic print for delta-theta
            for lbl, data in all_preds.items():
                if "dt_true" in data and len(data["dt_true"]) > 0:
                    dt_deg = data["dt_true"] * (180.0 / np.pi)
                    n_outlier = int((np.abs(dt_deg) > 20).sum())
                    print(f"  Δθ diag [{lbl}]: range=[{dt_deg.min():.1f}°,{dt_deg.max():.1f}°] "
                          f"mean={dt_deg.mean():.2f}° std={dt_deg.std():.2f}° "
                          f"|Δθ|>20°: {n_outlier}/{len(dt_deg)} ({100*n_outlier/len(dt_deg):.1f}%)")
            try:
                plot_full_test_scatter(all_preds, title_prefix=title_prefix, base_font_size=base_font_size,
                                        layout=scatter_hist_layout, operating_ranges=operating_ranges,
                                        max_cols=scatter_hist_max_cols)  # [STSI 100926]:wire layout/operating_ranges/max_cols pass-through
                plot_error_histograms(all_preds, title_prefix=title_prefix, base_font_size=base_font_size,
                                       layout=scatter_hist_layout, operating_ranges=operating_ranges,
                                       max_cols=scatter_hist_max_cols)  # [STSI 100926]:wire layout/operating_ranges/max_cols pass-through
                plot_errors_vs_operating_range(all_preds, n_bins=40, title_prefix=title_prefix, base_font_size=base_font_size,
                                                operating_ranges=operating_ranges)  # [STSI 100926]:wire operating_ranges pass-through
                # ── Collect error stats per model ───────────────────────────
                col_names = ["V_mag (pu)", "V_ang (deg)", "P (MW)", "Q (MVAr)"]
                for es_lbl, es_data in all_preds.items():
                    error_stats = {}
                    for col_idx, cname in enumerate(col_names):
                        err = es_data["pred"][:, col_idx] - es_data["true"][:, col_idx]
                        ci_lo, ci_hi = np.percentile(err, [2.5, 97.5])
                        error_stats[cname] = {
                            "mae": float(np.mean(np.abs(err))),
                            "mean": float(np.mean(err)),
                            "std": float(np.std(err)),
                            "ci_lo": float(ci_lo),
                            "ci_hi": float(ci_hi),
                        }
                    if es_lbl in model_summary:
                        model_summary[es_lbl]["error_stats"] = error_stats
            except Exception as exc:
                print(f"[WARNING] scatter/histogram plots failed: {exc}")

    # ── Restore plt.show ──────────────────────────────────────────────────────
    if save_to_pdf:
        _pdf_obj.__exit__(None, None, None)

    # Expose already-computed predictions/metrics so callers can reuse them
    # (e.g. plot_full_test_scatter, plot_error_histograms) without recomputation.
    return {
        "all_preds": all_preds,
        "model_summary": model_summary,
        "models_dict": models_dict,
        "multi_net_metrics": multi_net_metrics,
        "metrics_key": metrics_key,
    }


In [ ]:
## =============================================================================
# SECTION 11b: Factorial analysis
# =============================================================================
_FACTOR_LABELS = {
    "n_buses":        "System size (n buses)",
    "bus_count_range": "Bus count range",
    "dataset_size":   "Dataset size (scenarios)",
    "weight_physics": "Physics loss weight",
    "weight_ptdf":    "PTDF loss weight",
    "num_layers":     "GNN layers",
    "w_phys":         "Physics loss weight",
    "ptdf_loss_mode": "PTDF loss mode",
    "physics_mode":   "Physics mode",  # legacy — always 'rich' with V2.6.1+
    "hidden_dim":     "Hidden dim",
    "batch_size":     "Batch size",
    "lr":             "Learning rate",
    "conv_mode":      "Conv block mode (legacy)",  # [STSI 140526]:new entries for conv-block flags
    "use_residual":   "Residual connections",
    "norm_type":      "Normalization type",
    "activation":     "Activation function",
    "use_dropout":    "Dropout enabled",
    "drop_rate":      "Dropout rate",
    "head_mode":      "Output head mode",
    "angle_mode":     "Angle mode",  # [STSI 260526]:edge-delta factorial support
    "conv_type":      "Conv layer type",
    "w_P":            "P residual weight",    # [STSI 220526]
    "w_Q":            "Q residual weight",    # [STSI 220526]
    "y_matrix_source": "Y-matrix source",
    "fraction_dcf_local":       "DC-local flow fraction",  # [STSI 040626]:activation ramp / per-sub-loss fractions
    "fraction_dcf_global":      "DC-global flow fraction",
    "fraction_acf_local":       "AC-local flow fraction",
    "fraction_acf_global":      "AC-global flow fraction",
    "activation_ramp_epochs":   "Activation ramp epochs",
    "fraction_physics":         "Physics loss fraction",
    "fraction_ptdf":            "PTDF loss fraction",
    "vmag_mode":                "V-mag mode",
    "ptdf_changepoint":         "PTDF changepoint",
    "physics_activation_start": "Physics activation start",
    "ptdf_activation_start":    "PTDF activation start",
    "physics_activation_stop":  "Physics activation stop",
    "ptdf_activation_stop":     "PTDF activation stop",
    "physics_activation_duration": "Physics activation duration",
    "ptdf_activation_duration":     "PTDF activation duration",
    "dcf_local_activation_start": "DC-local activation start",
    "dcf_global_activation_start": "DC-global activation start",
    "acf_local_activation_start": "AC-local activation start",
    "acf_global_activation_start": "AC-global activation start",
    "dcf_local_activation_stop":  "DC-local activation stop",
    "dcf_global_activation_stop": "DC-global activation stop",
    "acf_local_activation_stop":  "AC-local activation stop",
    "acf_global_activation_stop": "AC-global activation stop",
    "dcf_local_activation_duration":  "DC-local activation duration",
    "dcf_global_activation_duration": "DC-global activation duration",
    "acf_local_activation_duration":  "AC-local activation duration",
    "acf_global_activation_duration": "AC-global activation duration",
    "fraction_kvl":              "KVL loop-closure fraction",  # [STSI 021026]:task-081 fields, never added here
    "max_w_kvl":                 "KVL weight cap",
    "enable_balance_dc":         "DC balance loss enabled",
    "balance_dc_clamp_factor":   "DC balance clamp factor",
    "kvl_activation_start":      "KVL activation start",
    "kvl_activation_stop":       "KVL activation stop",
    "kvl_activation_duration":      "KVL activation duration",
    "physics_label":  "Physics label",
    "vmag_mae":         "Vmag MAE [p.u.]",
    "vmag_rmse":        "Vmag RMSE [p.u.]",
    "vang_mae":         "Vang MAE [p.u.]",
    "vang_rmse":        "Vang RMSE [p.u.]",
    "p_mae":            "P MAE [p.u.]",
    "p_rmse":            "P RMSE [p.u.]",
    "q_mae":            "Q MAE [p.u.]",
    "q_rmse":           "Q RMSE [p.u.]",
    "line_flow_mae":    "Line flow MAE [p.u.]",
    "line_flow_rmse":    "Line flow RMSE [p.u.]",
    "line_flow_q0_mae": "Line flow Q0 MAE [p.u.]",
    "line_flow_q0_rmse": "Line flow Q0 RMSE [p.u.]",
    "delta_theta_mae":  "\u0394\u03b8 MAE [rad]",
    "delta_theta_rmse": "\u0394\u03b8 RMSE [rad]",
    "vmag_mae_norm":    "Vmag MAE / σ(true)",
    "vang_mae_norm":    "Vang MAE / σ(true)",
    "p_mae_norm":       "P MAE / σ(true)",
    "p_rmse_norm":       "P RMSE / σ(true)",
    "q_mae_norm":       "Q MAE / σ(true)",
    "q_rmse_norm":       "Q RMSE / σ(true)",
    "line_flow_mae_norm":    "Line flow MAE / σ(true)",
    "line_flow_rmse_norm":    "Line flow RMSE / σ(true)",
    "line_flow_q0_mae_norm": "Line flow Q0 MAE / σ(true)",
    "line_flow_q0_rmse_norm": "Line flow Q0 RMSE / σ(true)",
    "composite":        "Composite (weighted)",  # [STSI 100626]:composite weighted response metric
}

# [STSI 100626]:gate→child dependency chains for conditional NaN masking in factorial/heatmap
_DEFAULT_DEPENDENT_SETS = {
    "D1_physics": {
        "gate": "fraction_physics",
        "condition": lambda v: v is not None and v > 0,
        "children": ["physics_activation_start", "physics_activation_stop",
                      "w_P", "w_Q", "weight_physics"],
    },
    "D2_ptdf": {
        "gate": "fraction_ptdf",
        "condition": lambda v: v is not None and v > 0,
        "children": ["ptdf_activation_start", "ptdf_activation_stop",
                      "weight_ptdf", "ptdf_loss_mode", "ptdf_alpha", "ptdf_branch_mode"],
    },
    "D3_ptdf_stepwise": {
        "gate": "ptdf_loss_mode",
        "condition": lambda v: v == "stepwise",
        "children": ["ptdf_changepoint"],
    },
    "D4_dcf_local": {
        "gate": "fraction_dcf_local",
        "condition": lambda v: v is not None and v > 0,
        "children": ["dcf_local_activation_start", "dcf_local_activation_stop"],
    },
    "D5_dcf_global": {
        "gate": "fraction_dcf_global",
        "condition": lambda v: v is not None and v > 0,
        "children": ["dcf_global_activation_start", "dcf_global_activation_stop"],
    },
    "D6_acf_local": {
        "gate": "fraction_acf_local",
        "condition": lambda v: v is not None and v > 0,
        "children": ["acf_local_activation_start", "acf_local_activation_stop"],
    },
    "D7_acf_global": {
        "gate": "fraction_acf_global",
        "condition": lambda v: v is not None and v > 0,
        "children": ["acf_global_activation_start", "acf_global_activation_stop"],
    },
    "D8_dropout": {
        "gate": "use_dropout",
        "condition": lambda v: bool(v),
        "children": ["drop_rate"],
    },
    "D9_kvl": {  # [STSI 021026]
        "gate": "fraction_kvl",
        "condition": lambda v: v is not None and v > 0,
        "children": ["kvl_activation_start", "kvl_activation_stop", "max_w_kvl"],
    },
    "D10_balance_dc": {  # [STSI 021026]:independent gate from fraction_kvl
        "gate": "enable_balance_dc",
        "condition": lambda v: bool(v),
        "children": ["balance_dc_clamp_factor"],
    },
}

_SYS_TO_BUSES = {
    "ieee9": 9, "ieee14": 14, "cigre14": 14,
    "ieee30": 30, "ieee57": 57, "ieee39": 39, "ieee118": 118,
}

import warnings
warnings.filterwarnings("ignore", message=".*kurtosistest.*", category=UserWarning)
warnings.filterwarnings("ignore", message=".*ConstantInputWarning.*")
warnings.filterwarnings("ignore", category=RuntimeWarning, message=".*input array is constant.*")
from scipy.stats import ConstantInputWarning
warnings.filterwarnings("ignore", category=ConstantInputWarning)

def create_comparison_dataframe(runs_list):
    """Create a comprehensive comparison DataFrame from sweep results."""
    rows = []
    for r in runs_list:
        tm = r["test_metrics"]
        rows.append({  # [STSI 140526]:added conv-block columns
            "tag":           r["tag"],
            "angle_mode":    r.get("angle_mode", "node"),  # [STSI 260526]:edge-delta column
            "w_phys":        r["weight_physics"],
            "w_ptdf":        r["weight_ptdf"],
            "head_mode":     r.get("head_mode", "standard"),
            "use_residual":  r.get("use_residual", False),
            "norm_type":     r.get("norm_type", None),
            "activation":    r.get("activation", "leaky_relu"),
            "use_dropout":   r.get("use_dropout", False),
            "drop_rate":     r.get("drop_rate", 0.0),
            "physics_label": r.get("physics_label", ""),
            "w_P":           r.get("w_P", 0.5),     # [STSI 220526] default 0.5 for old runs
            "w_Q":           r.get("w_Q", 0.5),     # [STSI 220526]
            "bs":            r["batch_size"],
            "lr":            r["lr"],
            "train_loss":    r["final_train_loss"],
            "val_loss":      r["final_val_loss"],
            "test_total":    tm["test_total"],
            "test_mse":      tm["test_mse"],
            "test_physics":  tm["test_physics"],
            "test_ptdf":     tm["test_ptdf"],
            "training_time": r["training_time"],
            "delta_theta_mae": tm.get("delta_theta_mae", float("nan")),  # [STSI 270526]:task 028
            "delta_theta_rmse": tm.get("delta_theta_rmse", float("nan")),  # [STSI 270526]:task 028
            "fraction_dcf_local": r.get("fraction_dcf_local", 0.0),  # [STSI 040626]:activation ramp / per-sub-loss fractions
            "fraction_dcf_global": r.get("fraction_dcf_global", 0.0),
            "fraction_acf_local": r.get("fraction_acf_local", 0.0),
            "fraction_acf_global": r.get("fraction_acf_global", 0.0),
            "fraction_kvl": r.get("fraction_kvl", 0.0),              # [STSI 280926]:run-df gap fix, task 081
            "max_w_kvl": r.get("max_w_kvl", 100.0),
            "enable_balance_dc": r.get("enable_balance_dc", False),
            "balance_dc_clamp_factor": r.get("balance_dc_clamp_factor", 3.0),
            "activation_ramp_epochs": r.get("activation_ramp_epochs", 5),
            "physics_activation_start": r.get("physics_activation_start", 0),
            "physics_activation_stop": r.get("physics_activation_stop"),   # [STSI 021026]:missing *_stop sibling
            "ptdf_activation_start": r.get("ptdf_activation_start", 0),
            "ptdf_activation_stop": r.get("ptdf_activation_stop"),
            "dcf_local_activation_start": r.get("dcf_local_activation_start"),   # [STSI 021026]:sibling gap bundled w/ kvl fix
            "dcf_local_activation_stop": r.get("dcf_local_activation_stop"),
            "dcf_global_activation_start": r.get("dcf_global_activation_start"),
            "dcf_global_activation_stop": r.get("dcf_global_activation_stop"),
            "acf_local_activation_start": r.get("acf_local_activation_start"),
            "acf_local_activation_stop": r.get("acf_local_activation_stop"),
            "acf_global_activation_start": r.get("acf_global_activation_start"),
            "acf_global_activation_stop": r.get("acf_global_activation_stop"),
            "kvl_activation_start": r.get("kvl_activation_start", 0),   # [STSI 021026]:literal historical default
            "kvl_activation_stop": r.get("kvl_activation_stop"),
        })
    return pd.DataFrame(rows)



def _get_factor_value(r, f):
    if f == "n_buses":
        v = r.get("n_buses")
        if v is None:
            bs = r.get("base_system") or (r.get("hparams") or {}).get("base_system", "")
            v = _SYS_TO_BUSES.get(str(bs).lower(), float("nan"))
        return float(v) if v is not None else float("nan")
    elif f == "dataset_size":
        v = r.get("dataset_size")
        if v is None:
            v = r.get("num_scenarios") or (r.get("hparams") or {}).get("num_scenarios")
        return float(v) if v is not None else float("nan")
    elif f == "max_buses":
        return float(r.get("_max_buses", 0))
    elif f.endswith("_activation_start"):  # [STSI 021026]:absent/None start => always-on from epoch 0, NOT "never"
        v = r.get(f)
        return int(v) if v is not None else 0
    elif f.endswith("_activation_stop"):  # None = runs to end of run, encoded as that run's epoch count (NaN if no history)
        v = r.get(f)
        if v is not None:
            return int(v)
        n_epochs = len(r.get("history", {}).get("train_total", []))
        return n_epochs if n_epochs > 0 else float("nan")
    elif f == "activation_ramp_epochs":  # [STSI 040626]:activation ramp / per-sub-loss fractions
        v = r.get(f)
        return int(v) if v is not None else 0
    else:
        return r.get(f)



def _apply_dependent_masking(df, dependent_sets, all_columns, verbose=True):  # [STSI 100626]:NaN-mask child factors when gate is inactive
    """NaN-mask child columns where their gate factor is inactive.

    Returns conditioned_children: dict mapping child col name → number of active rows.
    """
    conditioned_children = {}
    for dep_name, dep in dependent_sets.items():
        gate_col = dep["gate"]
        if gate_col not in all_columns:
            continue
        if gate_col not in df.columns:
            continue
        gate_inactive = ~df[gate_col].apply(dep["condition"])
        n_inactive = gate_inactive.sum()
        if n_inactive == 0:
            continue  # gate always active — no masking needed
        n_total = len(df)
        n_active = n_total - n_inactive
        for child in dep["children"]:
            if child not in df.columns:
                continue
            df.loc[gate_inactive, child] = np.nan
            conditioned_children[child] = n_active
            if verbose:
                print(f"  \u26a0 '{child}' depends on '{gate_col}' \u2014 "
                      f"active in {n_active}/{n_total} runs. "
                      f"Masked {n_inactive} inactive rows to NaN.")
    return conditioned_children

def _full_label(term):
    parts = term.split(":")
    translated = []
    for p in parts:
        if "[" in p:
            base, suffix = p.split("[", 1)
            translated.append(_FACTOR_LABELS.get(base, base) + " [" + suffix)
        else:
            translated.append(_FACTOR_LABELS.get(p, p))
    return " × ".join(translated)


def factorial_analysis(runs, response_metrics=None, factors=None, interactions=None, metrics_key="test_metrics", response_metrics_row2=None,
                       dependent_sets=None, composite_weights=None, base_font_size=10,
                       max_terms=None, term_label_wrap=40, plot_response_surface=True,
                       max_cols=None, col_width=5.0, term_height=0.3, bar_height=0.7,
                       wspace=0.35, hspace=None):  # [STSI 100626]:condition child factors on gate activity; composite weighted response

    if response_metrics is None:
        response_metrics = ["vmag_mae", "vang_mae", "delta_theta_mae", "p_mae", "q_mae", "line_flow_mae"]
    if response_metrics_row2 is None and response_metrics_row2 != []:
        response_metrics_row2 = ["vmag_rmse", "vang_rmse", "delta_theta_rmse", "line_flow_mae", "line_flow_q0_mae"]
    all_response_metrics = list(dict.fromkeys(m for m in (response_metrics + (response_metrics_row2 or [])) if m is not None))
    # Reserve composite slot in row 1 when weights provided  # [STSI 100626]:composite weighted response metric
    if composite_weights and "composite" not in response_metrics:
        response_metrics = list(response_metrics) + ["composite"]
    if factors is None:
        factors = ["weight_physics", "q_residual_mode", "weight_ptdf", "ptdf_loss_mode"]

    records = []
    for r in runs:
        tm = r.get(metrics_key, {})
        rec = {}
        for m in all_response_metrics:
            rec[m] = float(r.get(m, tm.get(m, float("nan"))))
        for f in factors:
            rec[f] = _get_factor_value(r, f)
        records.append(rec)
    df_raw = pd.DataFrame(records)

    # ── Gate/child confound caution ──  # [STSI 040926]:masking removed from OLS path (listwise dropna deleted whole runs). Masking kept only in the heatmap; warn here instead.
    if dependent_sets is None:
        dependent_sets = _DEFAULT_DEPENDENT_SETS
    _dep_info = {}  # children kept in the design (not NaN-masked); confound flagged below
    if dependent_sets:
        for _dep in dependent_sets.values():
            _gate = _dep["gate"]
            if _gate not in df_raw.columns:
                continue
            _n_off = int((~df_raw[_gate].apply(_dep["condition"])).sum())
            _present = [c for c in _dep["children"] if c in df_raw.columns]
            if _n_off and _present:
                print(f"  \u26a0 gate '{_gate}' OFF in {_n_off}/{len(df_raw)} runs; "
                      f"children {_present} confounded with it (rows kept, not deleted).")


    df = df_raw.copy()
    encoding    = {}
    factor_type = {}

    for f in factors:
        levels = sorted(df[f].dropna().unique(), key=lambda v: (str(type(v)), v))
        n = len(levels)
        is_numeric = all(isinstance(v, (int, float, np.integer, np.floating)) for v in levels)

        if f == "physics_mode":
            if n != 2:
                print(f"  WARNING: 'physics_mode' has {n} levels → expected 2; skipping.")
                continue
            lo, hi = sorted(levels)
            encoding[f] = {lo: -1, hi: +1}
            df[f] = df[f].map(encoding[f])
            factor_type[f] = "binary"
            print(f"  {f:30s} [binary]   : {str(lo):>10} → -1,  {str(hi):>10} → +1")
            continue

        if n == 2:
            
            lo, hi = levels[0], levels[1]
            encoding[f] = {lo: -1, hi: +1}
            df[f] = df[f].map(encoding[f])
            factor_type[f] = "binary"
            print(f"  {f:30s} [binary]   : {str(lo):>10} → -1,  {str(hi):>10} → +1")
        elif n >= 3 and is_numeric:
            factor_type[f] = "numeric"
            print(f"  {f:30s} [numeric]  : {levels} → as-is (linear trend)")
        elif n >= 3 and not is_numeric:
            df[f] = df[f].astype(str)
            factor_type[f] = "categorical"
            print(f"  {f:30s} [categoric]: {levels} → treatment coded, ref={levels[0]}")
        else:
            print(f"  WARNING: '{f}' has {n} levels and unknown type — skipping.")
            continue

    print()
    valid_factors = [f for f in factors if f in factor_type]
    if not valid_factors:
        print("[factorial_analysis] No varying factors — skipping OLS regression.")
        return {}, {}
    df = df.dropna(subset=valid_factors)

    # ── STEP 2b: DOE [-1,+1] coding for factors + range-norm responses ─────  # [STSI 190526]:DOE coding replaces z-score
    coding_info = {}  # store center/half_range for back-transform

    # Numeric factors with ≥3 levels: rescale to [-1, +1]
    for f in valid_factors:
        if factor_type[f] == "numeric":
            lo, hi = df[f].min(), df[f].max()
            center = (lo + hi) / 2
            half_range = (hi - lo) / 2
            if half_range > 0:
                df[f] = (df[f] - center) / half_range
                coding_info[f] = {"center": center, "half_range": half_range}
            else:
                coding_info[f] = {"center": center, "half_range": 1.0}
        elif factor_type[f] == "binary":
            # Already ±1 from encoding above — record original levels for contour labels
            levels = sorted(df_raw[f].dropna().unique(), key=lambda v: (str(type(v)), v))
            lo_val, hi_val = levels[0], levels[1]
            _both_numeric = isinstance(lo_val, (int, float)) and isinstance(hi_val, (int, float))  # [STSI 280926]:guard mixed str/int levels (e.g. "never" vs epoch int, from _get_factor_value's None->"never")
            coding_info[f] = {"center": (lo_val + hi_val) / 2 if _both_numeric else 0,
                              "half_range": (hi_val - lo_val) / 2 if _both_numeric else 1}

    # Responses: range normalization (y - mean) / (max - min)
    resp_info = {}
    for m in all_response_metrics:
        y_mean = df[m].mean()
        y_range = df[m].max() - df[m].min()
        if y_range > 0:
            df[m] = (df[m] - y_mean) / y_range
            resp_info[m] = {"mean": y_mean, "range": y_range}
        else:
            resp_info[m] = {"mean": y_mean, "range": 1.0}

    # ── Composite weighted response ──────────────────────────────  # [STSI 100626]:composite weighted response metric
    if composite_weights:
        active_cw = {m: w for m, w in composite_weights.items()
                     if w > 0 and m in df.columns}
        if active_cw:
            w_total = sum(active_cw.values())
            norm_w = {m: w / w_total for m, w in active_cw.items()}
            df["composite"] = sum(norm_w[m] * df[m] for m in norm_w)
            resp_info["composite"] = {"mean": 0.0, "range": 1.0}  # already normalized
            if "composite" not in all_response_metrics:
                all_response_metrics.append("composite")
            print(f"  Composite metric: weighted sum of {list(norm_w.keys())}")
            print(f"    Normalized weights: {dict((m, round(w, 3)) for m, w in norm_w.items())}")

    if interactions is None:
        active_interactions = [
            (a, b)
            for i, a in enumerate(valid_factors)
            for b in valid_factors[i+1:]
        ]
    else:
        active_interactions = [
            tup for tup in interactions
            if all(f in valid_factors for f in tup)
        ]

    print("Active interactions:")
    for tup in active_interactions:
        print(f"  {' : '.join(tup)}")
    print()

    rename  = {f: f"F{i}" for i, f in enumerate(valid_factors)}
    reverse = {v: k for k, v in rename.items()}
    df_model = df[valid_factors + all_response_metrics].rename(columns=rename)

    def _formula_term(f):
        fn = rename[f]
        return f"C({fn})" if factor_type[f] == "categorical" else fn

    mains  = " + ".join(_formula_term(f) for f in valid_factors)
    twoway = " + ".join(
        ":".join(_formula_term(f) for f in tup)
        for tup in active_interactions
    )
    formula_rhs = mains + (" + " + twoway if twoway else "")

    def _n_params(f):
        levels = sorted(df[f].dropna().unique(), key=lambda v: (str(type(v)), v))
        return len(levels) - 1 if factor_type[f] == "categorical" else 1

    n_main_params = sum(_n_params(f) for f in valid_factors)
    from functools import reduce
    import operator
    n_interaction_params = sum(
        reduce(operator.mul, (_n_params(f) for f in tup), 1) for tup in active_interactions
    )
    n_params = 1 + n_main_params + n_interaction_params
    df_error = len(df_model) - n_params
    print(f"Runs: {len(df_model)}   Parameters: {n_params}   Error df: {df_error}")
    if df_error <= 0:  # rank-deficient: coefficients and p-values are not identifiable
        print(f"  !! WARNING: error df = {df_error} <= 0. The design matrix is saturated or "
              f"rank-deficient, so coefficients, p-values and R^2 below are NOT trustworthy. "
              f"Reduce `factors` or pass an explicit `interactions` list.")
    elif df_error < n_params:
        print(f"  ! CAUTION: error df ({df_error}) < n_params ({n_params}) - estimates are "
              f"weakly identified.")
    print(f"Formula RHS: {formula_rhs}")
    print()

    results   = {}
    all_coefs = {}

    def _restore(term):
        parts = term.split(":")
        restored = []
        for p in parts:
            p_clean = p.replace("C(", "").replace(")", "")
            base = p_clean.split("[")[0]
            suffix = ("[" + p_clean.split("[")[1]) if "[" in p_clean else ""
            restored.append(reverse.get(base, base) + suffix)
        return ":".join(restored)

    for metric in all_response_metrics:
        formula = f"{metric} ~ {formula_rhs}"
        model   = ols(formula, data=df_model).fit()
        results[metric] = model

        coef = model.params.copy()
        pval = model.pvalues.copy()
        coef.index = [_restore(t) for t in coef.index]
        pval.index = coef.index

        # Back-transform to practical effect in original response units  # [STSI 190526]:DOE practical effect
        y_range = resp_info.get(metric, {"range": 1.0})["range"]
        effect_original = coef * y_range        # coef × y_range = half-effect in original units
        full_effect_pct = coef * 2 * 100        # full lo→hi effect as % of response range

        all_coefs[metric] = pd.DataFrame({
            "coef (DOE)":         coef,
            "full effect (%)":    full_effect_pct,
            "effect (original)":  effect_original * 2,  # full lo→hi in original units
            "p-value":            pval,
            "significant":        pval.apply(lambda p: "***" if p < 0.001
                                              else ("**" if p < 0.01
                                              else ("*"  if p < 0.05 else ""))),
        })
        print("=" * 60)
        print(f"Response: {metric}   R²={model.rsquared:.4f}   Adj-R²={model.rsquared_adj:.4f}   " # rsquared adjusted in OLS model means it is adjusted by 
              f"y_range={y_range:.6f}")  # [STSI 190526]:DOE header
        print("=" * 60)
        print(all_coefs[metric].to_string())
        print()

    # ── Coefficient plot (2-row grid: MAE top, RMSE bottom) ────────────────────
    # Coding is spelled out only for non-numeric levels; numeric factors map
    # low -> -1 / high -> +1 proportionally, which needs no annotation.
    _coding_suffix, _cat_ref = {}, {}
    for _f in valid_factors:
        if factor_type[_f] == "binary":
            _lo = next(k for k, v in encoding[_f].items() if v == -1)
            _hi = next(k for k, v in encoding[_f].items() if v == +1)
            if not all(isinstance(_v, (bool, np.bool_, int, float, np.integer, np.floating))
                       for _v in (_lo, _hi)):
                _coding_suffix[_f] = f"[{_hi}=+1 vs {_lo}=-1]"
        elif factor_type[_f] == "categorical":
            _cat_ref[_f] = str(sorted(df_raw[_f].dropna().unique(),
                                      key=lambda v: (str(type(v)), v))[0])

    def _coded_label(term):
        parts = []
        for p in term.split(":"):
            base, br, lvl = p.partition("[")
            lbl = _FACTOR_LABELS.get(base, base)
            if br:  # treatment-coded categorical level, e.g. "ptdf_loss_mode[T.stepwise]"
                lvl = lvl.rstrip("]")
                if lvl.startswith("T."):
                    lvl = lvl[2:]
                parts.append(f"{lbl} [{lvl}=1 vs {_cat_ref.get(base, 'ref')}=0]")
            elif base in _coding_suffix:
                parts.append(f"{lbl} {_coding_suffix[base]}")
            else:
                parts.append(lbl)
        return " \u00d7 ".join(parts)

    has_row2 = bool(response_metrics_row2)

    # Each block (MAE / RMSE) wraps into one or more grid rows of at most max_cols panels.
    def _chunk(seq, size):
        seq = [m for m in seq]
        if not size or size < 1 or size >= len(seq):
            return [seq]
        return [seq[i:i + size] for i in range(0, len(seq), size)]

    _blocks = [("MAE" if has_row2 else "", _chunk(response_metrics, max_cols))]
    if has_row2:
        _blocks.append(("RMSE", _chunk(response_metrics_row2, max_cols)))
    _grid = [(lbl, ch, ci) for lbl, chunks in _blocks for ci, ch in enumerate(chunks)]
    n_rows = len(_grid)
    n_cols = max(len(ch) for _, ch, _ in _grid)

    # One shared term subset across all subplots so the hidden y-axes stay aligned.
    _plot_terms = None
    if max_terms:
        _rank = {}
        for _mod in results.values():
            _c = _mod.params.drop("Intercept").abs()
            _p = _mod.pvalues.drop("Intercept")
            for _t in _c.index:
                _rt = _restore(_t)
                _prev = _rank.get(_rt, (0, 0.0))
                _rank[_rt] = (max(_prev[0], int(_p[_t] < 0.05)), max(_prev[1], float(_c[_t])))
        if len(_rank) > max_terms:
            _kept = sorted(_rank, key=lambda t: _rank[t], reverse=True)[:max_terms]
            _plot_terms = set(_kept)
            print(f"[factorial_analysis] Showing top {max_terms} of {len(_rank)} terms "
                  f"(significant first, then |coef|).")

    _n_terms_plot = (len(_plot_terms) if _plot_terms is not None
                     else max((len(m.params) - 1 for m in results.values()), default=1))
    _row_h = max(4.5, term_height * _n_terms_plot)
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(col_width * n_cols, _row_h * n_rows),
                             squeeze=False)

    def _plot_row(row_axes, metrics_list, row_idx):
        for col_idx, (ax, metric) in enumerate(zip(row_axes, metrics_list)):
            if metric not in results:
                ax.set_visible(False)
                continue
            coef = results[metric].params.drop("Intercept")
            pval = results[metric].pvalues.drop("Intercept")
            coef.index = [_restore(t) for t in coef.index]
            pval.index = coef.index
            if _plot_terms is not None:
                _keep = [t for t in coef.index if t in _plot_terms]
                coef, pval = coef[_keep], pval[_keep]
            labels = [_coded_label(lbl) for lbl in coef.index]
            # Annotate conditioned children with effective N  # [STSI 100626]:show active sample size on bar labels for masked factors
            if _dep_info:
                labels = [
                    f"{lbl} (N={_dep_info[raw_name]})" if raw_name in _dep_info else lbl
                    for lbl, raw_name in zip(labels, coef.index)
                ]
            if term_label_wrap:
                import textwrap
                labels = [textwrap.fill(l, term_label_wrap) for l in labels]
            colors = ["tomato" if p < 0.05 else "steelblue" for p in pval]
            ax.barh(labels, coef.values, color=colors, alpha=0.8, height=bar_height)
            if col_idx > 0:
                ax.tick_params(labelleft=False)
            ax.tick_params(axis="both", labelsize=base_font_size * 0.85)
            ax.axvline(0, color="black", linewidth=0.8)
            ax.set_title(
                f"{_FACTOR_LABELS.get(metric, metric)}\n"
                f"R²={results[metric].rsquared:.3f}  adj-R²={results[metric].rsquared_adj:.3f}", # adjusted R² is calculated by 
                fontsize=base_font_size * 1.1,
            )
            if row_idx == n_rows - 1:
                ax.set_xlabel("DOE coefficient", fontsize=base_font_size * 0.8)
            ax.grid(True, axis="x", alpha=0.3)
            max_abs = np.nanmax(np.abs(coef.values)) if len(coef) > 0 else 1.0
            for i, (val, p) in enumerate(zip(coef.values, pval)):
                stars = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else ""))
                if stars:
                    ax.text(val + np.sign(val) * max_abs * -0.1, # moving the stars inside the bar
                            i, stars, va="center", ha="left" if val >= 0 else "right",
                            fontsize=base_font_size * 0.7, color="black")

    for _r, (_lbl, _chunk_metrics, _ci) in enumerate(_grid):
        _padded = (list(_chunk_metrics) + [None] * n_cols)[:n_cols]
        for _c, _metric in enumerate(_padded):
            if _metric is None or _metric not in results:
                axes[_r, _c].set_visible(False)
        _plot_row(axes[_r], [m if m else "" for m in _padded], _r)

    from matplotlib.patches import Patch
    fig.legend(
        handles=[
            Patch(facecolor="tomato",    label="p < 0.05 (significant)"),
            Patch(facecolor="steelblue", label="p ≥ 0.05 (not significant)"),
        ],
        loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.03),
        fontsize=base_font_size * 0.85,
    )
    fig.text(0.5, -0.06, "Significance (raw p-value):   *** p<0.001    ** p<0.01    * p<0.05",
             ha="center", va="top", fontsize=base_font_size * 0.8,
             bbox=dict(facecolor="white", edgecolor="black", boxstyle="round,pad=0.3"))
    for _r, (_lbl, _chunk_metrics, _ci) in enumerate(_grid):
        if _lbl and _ci == 0:  # label only the first grid row of each block
            axes[_r, 0].annotate(
                _lbl, xy=(-0.3, 0.5),
                xycoords="axes fraction", fontsize=base_font_size * 1.4, fontweight="bold",
                ha="center", va="center", rotation=90,
            )
    plt.suptitle("Factorial DOE analysis \u2014 coded [-1,+1] coefficients", y=1.01, fontsize=base_font_size * 1.3)
    plt.tight_layout()
    # applied after tight_layout so it is not overwritten
    if wspace is not None or hspace is not None:
        fig.subplots_adjust(**{k: v for k, v in (("wspace", wspace), ("hspace", hspace))
                               if v is not None})
    plt.show()

    # ── Contour plots: predicted response surface for factor pairs ─────────  # [STSI 190526]:contour plots
    numeric_binary_factors = [f for f in valid_factors if factor_type[f] in ("binary", "numeric")]
    if plot_response_surface and len(numeric_binary_factors) >= 2:
        from itertools import combinations
        pairs = [(a, b) for tup in active_interactions if len(tup) == 2 for a, b in [tup] if a in numeric_binary_factors and b in numeric_binary_factors]
        for metric in response_metrics:
            n_pairs = len(pairs)
            ncols = min(3, n_pairs)
            if ncols == 0:
                continue
            nrows = (n_pairs + ncols - 1) // ncols
            fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), squeeze=False)
            fig.suptitle(f"DOE response surface — {_FACTOR_LABELS.get(metric, metric)}", y=1.02, fontsize=base_font_size * 1.3)

            for idx, (fa, fb) in enumerate(pairs):
                ax = axes[idx // ncols][idx % ncols]
                # Meshgrid in coded space
                grid = np.linspace(-1, 1, 25)
                Xa, Xb = np.meshgrid(grid, grid)
                # Build prediction DataFrame: numeric/binary at center (0), categorical at ref level  # [STSI 200526]:fix categorical predict crash
                pred_data = {}
                for f in valid_factors:
                    if factor_type[f] == "categorical":
                        cat_levels = sorted(df_raw[f].dropna().unique(), key=lambda v: (str(type(v)), v))
                        pred_data[rename[f]] = str(cat_levels[0])
                    else:
                        pred_data[rename[f]] = 0.0
                pred_df = pd.DataFrame(pred_data, index=range(625))
                pred_df[rename[fa]] = Xa.ravel()
                pred_df[rename[fb]] = Xb.ravel()
                Z = results[metric].predict(pred_df).values.reshape(25, 25)
                # Convert Z back to original units for display
                y_info = resp_info.get(metric, {"mean": 0, "range": 1})
                Z_orig = Z * y_info["range"] + y_info["mean"]

                cf = ax.contourf(Xa, Xb, Z_orig, levels=15, cmap="RdYlGn_r", alpha=0.85)
                plt.colorbar(cf, ax=ax, pad=0.02)

                # Overlay actual data points
                ax.scatter(df_model[rename[fa]], df_model[rename[fb]],
                           c="black", s=18, alpha=0.6, zorder=5, edgecolors="white", linewidths=0.3)

                # Axis labels with real-value ticks
                info_a = coding_info.get(fa, {"center": 0, "half_range": 1})
                info_b = coding_info.get(fb, {"center": 0, "half_range": 1})
                real_ticks = [-1, 0, 1]
                ax.set_xticks(real_ticks)
                ax.set_xticklabels([f"{info_a['center'] + t * info_a['half_range']:.3g}" for t in real_ticks], fontsize=base_font_size * 0.85)
                ax.set_yticks(real_ticks)
                ax.set_yticklabels([f"{info_b['center'] + t * info_b['half_range']:.3g}" for t in real_ticks], fontsize=base_font_size * 0.85)
                ax.set_xlabel(_FACTOR_LABELS.get(fa, fa), fontsize=base_font_size * 1.0)
                ax.set_ylabel(_FACTOR_LABELS.get(fb, fb), fontsize=base_font_size * 1.0)

            # Hide unused axes
            for idx in range(n_pairs, nrows * ncols):
                axes[idx // ncols][idx % ncols].set_visible(False)
            plt.tight_layout()
            plt.show()

    return results, all_coefs


def plot_factor_metric_heatmap(runs, response_metrics=None, factors=None, metrics_key="test_metrics",
                               dependent_sets=None, composite_weights=None, base_font_size=10,
                               corr_method="pearson", fdr_alpha=0.05,
                               ylabel_wrap=28):  # [STSI 030926]:categorical encoding + spearman + BH-FDR; ylabel_wrap chars (0/None = off)  # [STSI 100626]:condition child factors on gate activity; composite weighted response

    if response_metrics is None:
        response_metrics = ["vmag_mae", "vang_rmse", "p_mae", "q_mae",
                            "line_flow_mae", "line_flow_q0_mae"]
    if factors is None:
        factors = ["n_buses", "dataset_size", "weight_physics",
                   "weight_ptdf", "num_layers", "w_phys"]

    _system_to_buses = {
        "ieee9": 9, "ieee14": 14, "ieee30": 30, "ieee57": 57,
        "ieee118": 118, "ieee39": 39, "cigre14": 14,
    }

    def _n_buses(r):
        bs = r.get("base_system") or (r.get("hparams") or {}).get("base_system", "")
        return _system_to_buses.get(str(bs).lower(), float("nan"))

    def _dataset_size(r):
        v = r.get("num_scenarios") or (r.get("hparams") or {}).get("num_scenarios")
        return float(v) if v is not None else float("nan")

    factor_accessors = {
        "n_buses":        _n_buses,
        "dataset_size":   _dataset_size,
        "weight_physics": lambda r: r.get("weight_physics"),
        "weight_ptdf":    lambda r: r.get("weight_ptdf"),
        "num_layers":     lambda r: r.get("num_layers") or (r.get("hparams") or {}).get("num_layers"),
        "w_phys":         lambda r: r.get("w_phys") or r.get("weight_physics"),
    }

    available_factors = []
    seen = set()
    for f in factors:
        if f not in seen:
            available_factors.append(f)
            seen.add(f)

    records = []
    for r in runs:
        row = {}
        for f in available_factors:
            # raw value kept here; encoding happens after gate masking so string gates still compare
            row[f] = factor_accessors[f](r) if f in factor_accessors else _get_factor_value(r, f)
        tm = r.get(metrics_key) or {}
        for m in response_metrics:
            v = tm.get(m, float("nan"))
            if isinstance(v, dict):
                v = v.get("mean", float("nan"))
            row[m] = float(v) if v is not None else float("nan")
        records.append(row)

    df = pd.DataFrame(records)

    # ── Dependent-factor NaN masking ──────────────────────────────  # [STSI 100626]:mask child factors when gate is OFF to avoid correlation dilution
    if dependent_sets is None:
        dependent_sets = _DEFAULT_DEPENDENT_SETS
    if dependent_sets:
        _apply_dependent_masking(
            df, dependent_sets,
            all_columns=list(df.columns), verbose=True,
        )


    # ── Composite weighted response ──────────────────────────────  # [STSI 100626]:composite weighted response metric
    if composite_weights:
        active_cw = {m: w for m, w in composite_weights.items()
                     if w > 0 and m in df.columns}
        if active_cw:
            w_total = sum(active_cw.values())
            norm_w = {m: w / w_total for m, w in active_cw.items()}
            # Range-normalize each metric before combining so scale
            # differences don't dominate the composite  # [STSI 100626]:fix raw-vs-normalized
            composite = pd.Series(0.0, index=df.index)
            for m in norm_w:
                col = df[m]
                r = col.max() - col.min()
                normed = (col - col.mean()) / r if r > 0 else col * 0.0
                composite += norm_w[m] * normed
            df["composite"] = composite
            if "composite" not in response_metrics:
                response_metrics = list(response_metrics) + ["composite"]

    # ── Factor encoding ──────────────────────────────────────────  # [STSI 030926]:numeric as-is; 2-level -> +/-1; >=3-level -> one indicator per level
    def _is_missing(v):
        return v is None or (isinstance(v, float) and np.isnan(v))

    def _is_numeric(v):
        return isinstance(v, (bool, np.bool_, int, float, np.integer, np.floating))

    encoded, col_labels = {}, {}
    for f in available_factors:
        col = df[f]
        present = [v for v in col if not _is_missing(v)]
        if not present:
            continue
        base_label = _FACTOR_LABELS.get(f, f)
        if all(_is_numeric(v) for v in present):  # bools already coerce to 1/0
            encoded[f] = col.map(lambda v: np.nan if _is_missing(v) else float(v))
            col_labels[f] = base_label
            continue
        levels = sorted({str(v) for v in present})
        if len(levels) < 2:
            continue
        if len(levels) == 2:
            lo, hi = levels  # alphabetical order decides the sign, so spell it out in the label
            encoded[f] = col.map(
                lambda v: np.nan if _is_missing(v) else (1.0 if str(v) == hi else -1.0))
            col_labels[f] = f"{base_label} [{hi}=+1 vs {lo}=-1]"
        else:
            for lvl in levels:  # point-biserial: this level vs all others
                name = f"{f}[{lvl}]"
                encoded[name] = col.map(
                    lambda v, _l=lvl: np.nan if _is_missing(v) else (1.0 if str(v) == _l else 0.0))
                col_labels[name] = f"{base_label} [{lvl}=1 vs rest=0]"

    df_enc = pd.DataFrame(encoded, index=df.index)

    valid_factors = [
        c for c in df_enc.columns
        if len(df_enc[c].dropna()) >= 2 and df_enc[c].dropna().std() > 0
    ]
    valid_metrics = [m for m in response_metrics if len(df[m].dropna()) >= 2]

    if not valid_factors or not valid_metrics:
        print("[plot_factor_metric_heatmap] Not enough data to compute correlations.")
        return

    from scipy import stats as _stats
    try:
        _corr_fn = {"pearson": _stats.pearsonr, "spearman": _stats.spearmanr}[corr_method]
    except KeyError:
        raise ValueError(f"corr_method must be 'pearson' or 'spearman', got {corr_method!r}")
    corr_matrix = np.full((len(valid_factors), len(valid_metrics)), np.nan)
    pval_matrix = np.full_like(corr_matrix, np.nan)

    for i, f in enumerate(valid_factors):
        for j, m in enumerate(valid_metrics):
            x, y = df_enc[f], df[m]
            keep = x.notna() & y.notna()
            if keep.sum() >= 3 and x[keep].std() > 0 and y[keep].std() > 0:
                res = _corr_fn(x[keep], y[keep])
                corr_matrix[i, j] = res[0]
                pval_matrix[i, j] = res[1]

    # Filter out factors that are entirely NaN (constant within valid-metric rows)
    non_nan_rows = ~np.all(np.isnan(corr_matrix), axis=1)
    if not np.all(non_nan_rows):
        valid_factors = [f for f, keep in zip(valid_factors, non_nan_rows) if keep]
        corr_matrix = corr_matrix[non_nan_rows]
        pval_matrix = pval_matrix[non_nan_rows]

    if len(valid_factors) == 0:
        print("[plot_factor_metric_heatmap] All factors constant - nothing to plot.")
        return

    # ── Benjamini-Hochberg FDR over all computed cells ──────────  # [STSI 030926]:stars are FDR-adjusted, not raw p
    qval_matrix = np.full_like(pval_matrix, np.nan) # Initialize the matrix for BH-FDR adjusted q-values
    _finite = np.isfinite(pval_matrix) # Boolean mask of finite p-values to include in BH-FDR correction
    _n_tests = int(_finite.sum()) # Number of valid tests for BH-FDR correction where p-values are finite
    if _n_tests:
        _p = pval_matrix[_finite]   # Extract the finite p-values for BH-FDR correction
        _order = np.argsort(_p) # Get the indices that would sort the p-values in ascending order
        _q = _p[_order] * _n_tests / (np.arange(_n_tests) + 1) # Compute the BH-FDR adjusted q-values in the sorted order by 
        _q = np.minimum.accumulate(_q[::-1])[::-1].clip(0, 1) # Ensure monotonicity and clip to [0, 1]
        _ranked = np.empty(_n_tests) # Prepare an array to hold the q-values in the original order
        _ranked[_order] = _q # Place the sorted q-values back into the original order
        qval_matrix[_finite] = _ranked # Assign the BH-FDR adjusted q-values to the corresponding positions in the matrix
        print(f"[plot_factor_metric_heatmap] {corr_method} | {_n_tests} tests | "
              f"raw p<{fdr_alpha}: {int((_p < fdr_alpha).sum())} | "
              f"BH-FDR q<{fdr_alpha}: {int((_ranked < fdr_alpha).sum())}")

    _corr_label = {"pearson": "Pearson r", "spearman": "Spearman \u03c1"}[corr_method]

    fig, ax = plt.subplots(
        figsize=(max(6, 2.5 * len(valid_metrics)), max(3, 0.9 * len(valid_factors)))
    )
    im = ax.imshow(corr_matrix, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    plt.colorbar(im, ax=ax, label=_corr_label)
    ax.set_xticks(range(len(valid_metrics)))
    ax.set_xticklabels(
        [_FACTOR_LABELS.get(m, m) for m in valid_metrics],
        rotation=30, ha="right", fontsize=base_font_size * 0.9, # the font size for x-axis labels
    )
    ax.set_yticks(range(len(valid_factors)))
    import textwrap
    _ylabels = [col_labels.get(f, f) for f in valid_factors]
    if ylabel_wrap:
        _ylabels = [textwrap.fill(l, ylabel_wrap) for l in _ylabels]
    ax.set_yticklabels(_ylabels, fontsize=base_font_size * 0.9)
    ax.set_title(f"Factor–metric correlation heatmap ({_corr_label})", fontsize=base_font_size * 1.1)

    for i in range(len(valid_factors)):
        for j in range(len(valid_metrics)):
            r_val = corr_matrix[i, j]
            q_val = qval_matrix[i, j]
            if np.isnan(r_val):
                continue
            stars = ("***" if q_val < 0.001 else
                     "**"  if q_val < 0.01  else
                     "*"   if q_val < fdr_alpha else "")
            text_color = "white" if abs(r_val) > 0.6 else "black"
            ax.text(j, i, f"{r_val:.2f}{stars}",
                    ha="center", va="center", fontsize=min(base_font_size * 0.6,16), # the font size for the text inside heatmap cells is capped at 16 to avoid overcrowding
                    color=text_color)
            # include a framed text box with q-val star annotation explanation below the plot
    ax.text(0.5, -0.16, "Stars indicate BH-FDR q-value significance: *** q<0.001, ** q<0.01, * q<0.05",
            ha="center", va="center", fontsize=base_font_size * 0.8, transform=ax.transAxes,
            bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.25'))
    plt.tight_layout()
    plt.show()


In [ ]:
# =============================================================================
# SECTION 11c: CROSS-SYSTEM FACTORIAL ANALYSIS
# =============================================================================

_SYSTEM_TO_BUSES = {
    "ieee9": 9, "ieee14": 14, "cigre14": 14,
    "ieee30": 30, "ieee57": 57, "ieee39": 39, "ieee118": 118,
}


def multi_system_factorial_analysis(
    runs_by_system: dict,
    response_metrics=None,
    factors=None,
    interactions=None,
    normalization:str = "std",  # "std" to normalise MAE by true std, or "zscore", None for raw MAE
):
    """
    Factorial analysis across multiple train systems using each run's own
    same-system test metrics (no cross-testing).

    Injects 'n_buses' and 'dataset_size' into shallow-copied run dicts so
    factorial_analysis can treat them as ordinary numeric factors.

    Args:
        runs_by_system:  {"ieee9": runs_list, "cigre14": runs_list, ...}
        response_metrics, factors, interactions: passed to factorial_analysis.
            'n_buses', 'max_buses', and 'dataset_size' are always appended to factors.
    """
    if response_metrics is None:
        response_metrics = ["vmag_mae_norm", "vang_mae_norm", "p_mae_norm", "q_mae_norm"]

    if factors is None:
        factors = ["weight_physics", "weight_ptdf", "num_layers", "max_buses"]
    # Always include system-level factors, deduplicated
    extended_factors = factors + ["n_buses", "max_buses", "dataset_size"]
    seen = set()
    extended_factors = [f for f in extended_factors if not (f in seen or seen.add(f))]

    # ── Build augmented run list ──────────────────────────────────────────────
    combined_runs = []
    for sys_name, runs in runs_by_system.items():
        #n_buses = _SYSTEM_TO_BUSES.get(sys_name.lower(), float("nan"))
        for r in runs:
            r_aug = dict(r)   # shallow copy — don't mutate the original
            r_aug["_sys_name"] = sys_name   # tag for zscore grouping
            bs = (r.get("base_system") or "").lower()
            r_aug["n_buses"] = float(_SYSTEM_TO_BUSES.get(bs, float("nan")))
            #r_aug["n_buses"]      = n_buses
            r_aug["max_buses"]    = float(r.get("_max_buses", 0))
            r_aug["dataset_size"] = (
                r.get("num_scenarios")
                or r.get("dataset_size")
                or (r.get("hparams") or {}).get("num_scenarios", float("nan"))
            )
            combined_runs.append(r_aug)
    
    # ── Normalise MAE ─────────────────────────────────────────────────────────
    _MAE_PAIRS = [
        ("vmag_mae", "vmag_true_std"),
        ("vang_mae", "vang_true_std"),
        ("p_mae",    "p_true_std"),
        ("q_mae",    "q_true_std"),
    ]

    if normalization == "std":
        for r_aug in combined_runs:
            tm = r_aug.get("test_metrics", {})
            for mae_key, std_key in _MAE_PAIRS:
                mae = tm.get(mae_key, float("nan"))
                std = tm.get(std_key, float("nan"))
                r_aug[f"{mae_key}_norm"] = mae / std if std and not np.isnan(std) else float("nan")
        if any(m in response_metrics for m in ["vmag_mae_norm","vang_mae_norm","p_mae_norm","q_mae_norm"]):
            missing = sum(1 for r in combined_runs if np.isnan(r.get("vmag_mae_norm", float("nan"))))
            if missing:
                raise ValueError(
                    f"{missing}/{len(combined_runs)} runs are missing '*_true_std' in test_metrics. "
                    f"Re-run refresh_test_metrics_with_std(runs) first, or use normalization='zscore'."
                )

    elif normalization == "zscore":
        from collections import defaultdict
        for mae_key, _ in _MAE_PAIRS:
            groups = defaultdict(list)
            for r_aug in combined_runs:
                tm = r_aug.get("test_metrics", {})
                mae = tm.get(mae_key, float("nan"))
                if not np.isnan(mae):
                    groups[r_aug["_sys_name"]].append(mae)
            group_stats = {}
            for sys, vals in groups.items():
                if len(vals) < 2:
                    print(f"  WARNING: '{sys}' has only {len(vals)} run(s) for '{mae_key}' — zscore undefined.")
                    group_stats[sys] = (vals[0] if vals else float("nan"), float("nan"))
                else:
                    group_stats[sys] = (float(np.mean(vals)), float(np.std(vals, ddof=1)))
            for r_aug in combined_runs:
                tm = r_aug.get("test_metrics", {})
                mae = tm.get(mae_key, float("nan"))
                sys = r_aug["_sys_name"]
                mu, sigma = group_stats.get(sys, (float("nan"), float("nan")))
                if not np.isnan(mae) and not np.isnan(sigma) and sigma > 0:
                    r_aug[f"{mae_key}_norm"] = (mae - mu) / sigma
                else:
                    r_aug[f"{mae_key}_norm"] = float("nan")

    else:
        # normalization=None — copy raw MAE to *_norm keys for uniform downstream use
        for r_aug in combined_runs:
            tm = r_aug.get("test_metrics", {})
            for mae_key, _ in _MAE_PAIRS:
                r_aug[f"{mae_key}_norm"] = tm.get(mae_key, float("nan"))

    # Guard: Warn if any  factor has NaN values after augmentation
    for f in extended_factors:
        n_nan = sum(1 for r in combined_runs if np.isnan(r.get(f, float("nan"))))
        if n_nan:
            print(f"  WARNING: Factor '{f}' has {n_nan}/{len(combined_runs)} NaN values after augmentation.")
    print(f"multi_system_factorial_analysis: {len(combined_runs)} runs total "
          f"from {list(runs_by_system.keys())}")
    print(f"Factors: {extended_factors}")
    print(f"Responses: {response_metrics}\n")

    return factorial_analysis(
        runs=combined_runs,
        response_metrics=response_metrics,
        factors=extended_factors,
        interactions=interactions,
    )



def refresh_test_metrics_with_std(runs_by_system: dict, training_networks_by_system: dict, verbose=True):
    """
    Re-evaluates each run's model on its reconstructed test split to populate
    vmag_true_std, vang_true_std, p_true_std, q_true_std in test_metrics.

    Uses reconstruct_test_split with the seed stored in each run, matching
    exactly what train_power_flow_gnn used during training.

    Args:
        runs_by_system:              {"ieee9": runs_list, "cigre14": runs_list, ...}
        training_networks_by_system: {"ieee9": networks_ieee9_large, "cigre14": networks_cigre14_large, ...}
    """
    _STD_KEYS = ["vmag_true_std", "vang_true_std", "p_true_std", "q_true_std"]
    skipped, updated = 0, 0

    for sys_name, runs in runs_by_system.items():
        training_networks = training_networks_by_system.get(sys_name)
        if training_networks is None:
            print(f"  [{sys_name}] skip — no training_networks provided")
            skipped += len(runs)
            continue

        for i, r in enumerate(runs):
            model = r.get("model")
            if model is None:
                if verbose:
                    print(f"  [{sys_name}][{i}] skip — model not available")
                skipped += 1
                continue

            seed = r.get("seed", 42)
            _, _, test_nets = reconstruct_test_split(training_networks, seed=seed)

            if verbose:
                print(f"  [{sys_name}][{i}] seed={seed}, {len(test_nets)} test nets ...", end=" ", flush=True)

            hparams_r     = r.get("hparams") or {}
            ptdf_params_r = r.get("_ptdf_params") or hparams_r.get("ptdf_params", None)
            max_buses_r   = int(r.get("_max_buses") or hparams_r.get("max_buses", 0))
            ptdf_offset_r = int(r.get("_ptdf_offset") or hparams_r.get("ptdf_offset", 0))

            try:
                new_metrics = evaluate_gnn_on_test_set(
                    model=model,
                    test_networks=test_nets,
                    ptdf_params=ptdf_params_r,
                    max_buses=max_buses_r,
                    ptdf_offset=ptdf_offset_r,
                    postprocess_dangling=True,
                )
            except RuntimeError as e:
                print(f"[WARNING] skipped [{sys_name}][{i}]: {e}")
                skipped += 1
                continue

            for k in _STD_KEYS:
                r["test_metrics"][k] = new_metrics[k]

            if verbose:
                print(f"vmag_std={new_metrics.get('vmag_true_std', float('nan')):.4f}")
            updated += 1

    print(f"\nrefresh_test_metrics_with_std: {updated} updated, {skipped} skipped.")


## Visualisation Functions

In [ ]:
# =============================================================================
# SECTION 12: VISUALIZATION
# =============================================================================

def get_bus_type_label(network, bus):
    """Return a human-readable bus type label for plot titles."""
    connected_gens = network.generators[network.generators.bus == bus]
    if len(connected_gens) > 0:
        ctrl = connected_gens.iloc[0]["control"]
        if ctrl in ("Slack", "PV"):
            return ctrl
    return "PQ"


def plot_network_results(network, results_df=None):
    """
    Plot bus voltage and power results over time.
    If results_df is None, runs power flow first.
    """
    if results_df is None:
        network.pf(use_seed=True)
        buses = list(network.buses.index)
        snapshots = network.snapshots
        results_df = pd.DataFrame(
            index=snapshots,
            columns=pd.MultiIndex.from_product(
                [buses, ["P pu", "Q pu", "V pu", "Angle deg"]]
            ),
            dtype=float,
        )
        for t in snapshots:
            for bus in buses:
                results_df.loc[t, (bus, "P pu")]      = network.buses_t.p.loc[t, bus]
                results_df.loc[t, (bus, "Q pu")]    = network.buses_t.q.loc[t, bus]
                results_df.loc[t, (bus, "V pu")]      = network.buses_t.v_mag_pu.loc[t, bus]
                results_df.loc[t, (bus, "Angle deg")] = np.degrees(network.buses_t.v_ang.loc[t, bus])

    buses = results_df.columns.get_level_values(0).unique()
    n_buses = len(buses)
    fig, axs = plt.subplots(n_buses, 2, figsize=(14, 2.5 * n_buses), sharex=True)
    if n_buses == 1:
        axs = axs.reshape(1, -1)

    for i, bus in enumerate(buses):
        bus_label = get_bus_type_label(network, bus)
        axs[i, 0].plot(results_df.index, results_df[(bus, "V pu")],
                       marker="o", linewidth=2)
        axs[i, 0].set_ylabel("V (p.u.)", fontsize=11)
        axs[i, 0].set_title(f"Voltage - {bus} ({bus_label})", fontsize=11, fontweight="bold")
        axs[i, 0].grid(True, alpha=0.3)

        axs[i, 1].plot(results_df.index, results_df[(bus, "P pu")],
                       marker="o", linewidth=2, label="P pu")
        axs[i, 1].plot(results_df.index, results_df[(bus, "Q pu")],
                       marker="s", linewidth=2, label="Q pu")
        axs[i, 1].set_ylabel("Power", fontsize=11)
        axs[i, 1].set_title(f"Power - {bus} ({bus_label})", fontsize=11, fontweight="bold")
        axs[i, 1].legend(fontsize=9)
        axs[i, 1].grid(True, alpha=0.3)

    fig.text(0.5, 0.02, "Time Step", ha="center", fontsize=12)
    plt.tight_layout(rect=[0, 0.03, 1, 1])
    plt.show()


def plot_line_results(network, results_df=None):
    """
    Plot line flow results over time.
    If results_df is None, runs power flow first.
    """
    if results_df is None:
        network.pf(use_seed=True)
        lines = list(network.lines.index)
        snapshots = network.snapshots
        results_df = pd.DataFrame(
            index=snapshots,
            columns=pd.MultiIndex.from_product(
                [lines, ["P0 pu", "P1 pu", "Q0 pu", "Q1 pu"]]
            ),
            dtype=float,
        )
        for t in snapshots:
            for line in lines:
                results_df.loc[t, (line, "P0 pu")]   = network.lines_t.p0.loc[t, line]
                results_df.loc[t, (line, "P1 pu")]   = network.lines_t.p1.loc[t, line]
                results_df.loc[t, (line, "Q0 pu")] = network.lines_t.q0.loc[t, line]
                results_df.loc[t, (line, "Q1 pu")] = network.lines_t.q1.loc[t, line]

    lines = results_df.columns.get_level_values(0).unique()
    n_lines = len(lines)
    fig, axs = plt.subplots(n_lines, 2, figsize=(14, 2.5 * n_lines), sharex=True)
    if n_lines == 1:
        axs = axs.reshape(1, -1)

    for i, line in enumerate(lines):
        from_bus = network.lines.loc[line, "bus0"]
        to_bus   = network.lines.loc[line, "bus1"]
        from_label = get_bus_type_label(network, from_bus)
        to_label   = get_bus_type_label(network, to_bus)
        line_name  = f"{line} ({from_bus}→{to_bus})"
        s_nom = network.lines.loc[line, "s_nom"]

        # Active Power P - Left subplot
        axs[i, 0].plot(results_df.index, results_df[(line, "P0 pu")],
                       marker="o", linewidth=2, label=f"P from {from_bus}")
        axs[i, 0].plot(results_df.index, -results_df[(line, "P1 pu")],
                       marker="s", linewidth=2, label=f"P to {to_bus}")
        axs[i, 0].axhline(y=s_nom,  color="r", linestyle="--", alpha=0.7, label=f"Snom {s_nom:.2f} pu")
        axs[i, 0].axhline(y=-s_nom, color="r", linestyle="--", alpha=0.7)
        axs[i, 0].set_ylabel("P (pu)", fontsize=11)
        axs[i, 0].set_title(f"Active Power - {line_name}", fontsize=11, fontweight="bold")
        axs[i, 0].legend(fontsize=9)
        axs[i, 0].grid(True, alpha=0.3)

        # Reactive Power Q - Right subplot
        axs[i, 1].plot(results_df.index, results_df[(line, "Q0 pu")],
                       marker="o", linewidth=2, label=f"Q from {from_bus}")
        axs[i, 1].plot(results_df.index, -results_df[(line, "Q1 pu")],
                       marker="s", linewidth=2, label=f"Q to {to_bus}")
        axs[i, 1].set_ylabel("Q (pu)", fontsize=11)
        axs[i, 1].set_title(f"Reactive Power - {line_name}", fontsize=11, fontweight="bold")
        axs[i, 1].legend(fontsize=9)
        axs[i, 1].grid(True, alpha=0.3)

    fig.text(0.5, 0.02, "Time Step", ha="center", fontsize=12)
    plt.tight_layout(rect=[0, 0.03, 1, 1])
    plt.show()


def plot_transformer_results(network, results_df=None):
    """
    Plot transformer flow results separately.
    Similar to plot_line_results but for transformers.
    """
    if len(network.transformers) == 0:
        logger.warning("No transformers to plot")
        return

    # If results_df not provided, run power flow
    if results_df is None:
        network.pf(use_seed=True)
        trafos = list(network.transformers.index)
        snapshots = network.snapshots
        results_df = pd.DataFrame(
            index=snapshots,
            columns=pd.MultiIndex.from_product(
                [trafos, ["P0 pu", "P1 pu", "Q0 pu", "Q1 pu"]]
            ),
            dtype=float,
        )
        for t in snapshots:
            for trafo in trafos:
                results_df.loc[t, (trafo, "P0 pu")]   = network.transformers_t.p0.loc[t, trafo]
                results_df.loc[t, (trafo, "P1 pu")]   = network.transformers_t.p1.loc[t, trafo]
                results_df.loc[t, (trafo, "Q0 pu")] = network.transformers_t.q0.loc[t, trafo]
                results_df.loc[t, (trafo, "Q1 pu")] = network.transformers_t.q1.loc[t, trafo]

    n_trafos = len(network.transformers)
    fig, axs = plt.subplots(n_trafos, 2, figsize=(14, 2.5 * n_trafos), sharex=True)
    if n_trafos == 1:
        axs = axs.reshape(1, -1)

    for i, trafo in enumerate(network.transformers.index):
        from_bus = network.transformers.loc[trafo, "bus0"]
        to_bus   = network.transformers.loc[trafo, "bus1"]
        trafo_name = f"{trafo} ({from_bus}→{to_bus})"
        s_nom = network.transformers.loc[trafo, "s_nom"]

        # Active Power
        axs[i, 0].plot(results_df.index, results_df[(trafo, "P0 pu")],
                       marker="o", linewidth=2, label=f"P from {from_bus}")
        axs[i, 0].plot(results_df.index, -results_df[(trafo, "P1 pu")],
                       marker="s", linewidth=2, label=f"P to {to_bus}")
        axs[i, 0].axhline(y=s_nom,  color="r", linestyle="--", alpha=0.7, label=f"Snom {s_nom:.2f} pu")
        axs[i, 0].axhline(y=-s_nom, color="r", linestyle="--", alpha=0.7)
        axs[i, 0].set_ylabel("P (pu)", fontsize=11)
        axs[i, 0].set_title(f"Active Power - {trafo_name}", fontsize=11, fontweight="bold")
        axs[i, 0].legend(fontsize=9)
        axs[i, 0].grid(True, alpha=0.3)

        # Reactive Power
        axs[i, 1].plot(results_df.index, results_df[(trafo, "Q0 pu")],
                       marker="o", linewidth=2, label=f"Q from {from_bus}")
        axs[i, 1].plot(results_df.index, -results_df[(trafo, "Q1 pu")],
                       marker="s", linewidth=2, label=f"Q to {to_bus}")
        axs[i, 1].set_ylabel("Q (pu)", fontsize=11)
        axs[i, 1].set_title(f"Reactive Power - {trafo_name}", fontsize=11, fontweight="bold")
        axs[i, 1].legend(fontsize=9)
        axs[i, 1].grid(True, alpha=0.3)

    # Add common x-axis label
    fig.text(0.5, 0.02, "Time Step", ha="center", fontsize=12)
    plt.tight_layout(rect=[0, 0.03, 1, 1])
    plt.show()

def plot_test_metrics_barchart(df, metric="test_total", top_n=10, title=None):
    """
    Bar chart of a chosen test metric for the top_n configurations (sorted ascending).
    """
    df_sorted = df.sort_values(metric).head(top_n)
    labels = [
        f"w={row.w_phys}, pt={row.w_ptdf}, bs={row.bs}, lr={row.lr:.0e}"
        for _, row in df_sorted.iterrows()
    ]
    values = df_sorted[metric].values

    plt.figure(figsize=(12, 6))
    x = np.arange(len(values))
    plt.bar(x, values, alpha=0.8)
    plt.xticks(x, labels, rotation=45, ha="right")
    plt.ylabel(metric)
    plt.title(title or f"Top {top_n} configs by {metric}")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# =============================================================================
# SECTION 12b: VISUALIZATION multiruns 
# ============================================================================= 
def predict_network_results_dc(network):  # [STSI 080926]:DC (lpf) solution for single-network comparison plots
    """
    Run PyPSA linear (DC) power flow on one network and return bus/line results
    + prediction masks in the same format as predict_network_results_with_masks,
    so DC can be dropped into plot_single_network_comparisons as another "model".

    DC capabilities:
        vmag: flat 1.0 p.u. (PV/Slack keep known Vmag)
        vang: solved by lpf()
        P:    known inputs; Slack P via lossless power balance
        Q:    naive zero guess (DC has no reactive model)
        line P0/P1: taken directly from lpf() (lossless linear DC flow)
        line Q0/Q1: naive zero guess
    """
    net_copy = network.copy()
    net_copy.lpf()

    snapshots = network.snapshots
    buses = list(network.buses.index)
    lines = list(network.lines.index)

    bus_results = pd.DataFrame(
        index=snapshots,
        columns=pd.MultiIndex.from_product([buses, ["P pu", "Q pu", "V pu", "Angle deg"]]),
        dtype=float,
    )
    line_results = pd.DataFrame(
        index=snapshots,
        columns=pd.MultiIndex.from_product([lines, ["P0 pu", "P1 pu", "Q0 pu", "Q1 pu"]]),
        dtype=float,
    )
    prediction_masks = {
        "P":     pd.DataFrame(index=snapshots, columns=buses, dtype=bool),
        "Q":     pd.DataFrame(index=snapshots, columns=buses, dtype=bool),
        "V":     pd.DataFrame(index=snapshots, columns=buses, dtype=bool),
        "Angle": pd.DataFrame(index=snapshots, columns=buses, dtype=bool),
    }

    for t in snapshots:
        non_slack_p, slack_buses = {}, []
        for bus in buses:
            connected_gens = network.generators[network.generators.bus == bus]
            ctrl = connected_gens.iloc[0]["control"] if len(connected_gens) > 0 else "PQ"
            if ctrl == "Slack":
                slack_buses.append(bus)
            else:
                non_slack_p[bus] = network.buses_t.p.loc[t, bus]
        slack_p_total = -sum(non_slack_p.values())
        n_slack = max(len(slack_buses), 1)

        for bus in buses:
            connected_gens = network.generators[network.generators.bus == bus]
            bus_type = "PQ"
            if len(connected_gens) > 0:
                ctrl = connected_gens.iloc[0]["control"]
                if ctrl in ("Slack", "PV"):
                    bus_type = ctrl
            vang_dc = net_copy.buses_t.v_ang.loc[t, bus]

            if bus_type == "PQ":
                pred_vmag_i, pred_vang_i = 1.0, vang_dc
                pred_p_i = network.buses_t.p.loc[t, bus]
                pred_q_i = network.buses_t.q.loc[t, bus]
                prediction_masks["P"].loc[t, bus]     = False
                prediction_masks["Q"].loc[t, bus]     = False
                prediction_masks["V"].loc[t, bus]     = True
                prediction_masks["Angle"].loc[t, bus] = True
            elif bus_type == "PV":
                pred_vmag_i = network.buses_t.v_mag_pu.loc[t, bus]
                pred_vang_i = vang_dc
                pred_p_i = network.buses_t.p.loc[t, bus]
                pred_q_i = 0.0  # DC has no Q model
                prediction_masks["P"].loc[t, bus]     = False
                prediction_masks["Q"].loc[t, bus]     = True
                prediction_masks["V"].loc[t, bus]     = False
                prediction_masks["Angle"].loc[t, bus] = True
            else:  # Slack
                pred_vmag_i = network.buses_t.v_mag_pu.loc[t, bus]
                pred_vang_i = network.buses_t.v_ang.loc[t, bus]  # known = 0
                pred_p_i = slack_p_total / n_slack
                pred_q_i = 0.0  # DC has no Q model
                prediction_masks["P"].loc[t, bus]     = True
                prediction_masks["Q"].loc[t, bus]     = True
                prediction_masks["V"].loc[t, bus]     = False
                prediction_masks["Angle"].loc[t, bus] = False

            bus_results.loc[t, (bus, "P pu")]      = pred_p_i
            bus_results.loc[t, (bus, "Q pu")]      = pred_q_i
            bus_results.loc[t, (bus, "V pu")]      = pred_vmag_i
            bus_results.loc[t, (bus, "Angle deg")] = np.degrees(pred_vang_i)

        for line in lines:
            p0 = net_copy.lines_t.p0.loc[t, line] if not net_copy.lines_t.p0.empty else 0.0
            p1 = (net_copy.lines_t.p1.loc[t, line]
                  if hasattr(net_copy.lines_t, "p1") and not net_copy.lines_t.p1.empty
                  else -p0)
            line_results.loc[t, (line, "P0 pu")] = p0
            line_results.loc[t, (line, "P1 pu")] = p1
            line_results.loc[t, (line, "Q0 pu")] = 0.0  # naive Q=0 guess
            line_results.loc[t, (line, "Q1 pu")] = 0.0

    return bus_results, line_results, prediction_masks


def predict_network_results_with_masks_multi(models_dict, network, use_edge_features=True, use_pnom_share=None,
                                             include_dc=False):  # [STSI 080926]:optional DC (lpf) entry
    """
    Run GNN prediction for multiple models on one network.
    Returns dict: model_label -> (bus_results, line_results, prediction_masks)
    STSI260320: multi-model version for side-by-side comparison.
    STSI260407: models_dict values may be a bare model OR a dict with
    keys 'model', 'ptdf_params', 'max_buses' for learned_edge support.
    include_dc: if True, adds a "DC (lpf)" entry computed by predict_network_results_dc.
    """
    all_results = {}
    for label, entry in models_dict.items():
        if isinstance(entry, dict):
            model       = entry["model"]
            ptdf_params = entry.get("ptdf_params")
            max_buses   = entry.get("max_buses", 0)
        else:
            model       = entry
            ptdf_params = None
            max_buses   = 0
        bus_res, line_res, masks, _ = predict_network_results_with_masks(
            model, network, use_edge_features=use_edge_features,
            ptdf_params=ptdf_params, max_buses=max_buses,
            use_pnom_share=use_pnom_share,
        )
        all_results[label] = {"bus": bus_res, "line": line_res, "masks": masks}
    if include_dc:
        dc_bus, dc_line, dc_masks = predict_network_results_dc(network)
        all_results["DC (lpf)"] = {"bus": dc_bus, "line": dc_line, "masks": dc_masks}
    return all_results


def plot_comparison_results_with_masks(
    multi_results, network, true_bus_results=None, true_line_results=None,
    title_prefix="", snapshot_idx=0, plot_buses=None,  # [STSI 200526]:optional bus filter
    base_font_size=10,
):
    """
    Plot all models side-by-side against true PyPSA solution.
    multi_results: dict label -> {"bus": df, "line": df, "masks": df}
    true_bus_results: from model_results(network) — the PyPSA ground truth.
    STSI260320: overlays multiple models + truth on same axes.
    """
    buses = plot_buses if plot_buses else list(network.buses.index)  # [STSI 200526]:use sampled buses if provided
    props = [("V pu", "V"), ("Angle deg", "Angle"), ("P pu", "P"), ("Q pu", "Q")]
    n_models = len(multi_results)
    colors = plt.cm.tab10.colors

    # Use first model's masks for predicted/known coloring
    first_masks = list(multi_results.values())[0]["masks"]

    fig, axs = plt.subplots(
        len(buses), len(props),
        figsize=(4 * len(props), 2.5 * len(buses)),
        sharex=True
    )
    if len(buses) == 1:
        axs = axs.reshape(1, -1)

    snap = network.snapshots[snapshot_idx]

    for i, bus in enumerate(buses):
        bus_label = get_bus_type_label(network, bus)
        for j, (prop_name, mask_key) in enumerate(props):
            ax = axs[i, j]
            is_predicted = first_masks[mask_key][bus].any()

            # Plot true PyPSA solution
            if true_bus_results is not None and (bus, prop_name) in true_bus_results.columns:
                ax.plot(
                    true_bus_results.index,
                    true_bus_results[bus][prop_name],
                    color="black", linewidth=2.5, linestyle="-",
                    zorder=10
                )

            # Plot each model
            for k, (label, res) in enumerate(multi_results.items()):
                df = res["bus"]
                if (bus, prop_name) not in df.columns:
                    continue
                style = "-" if is_predicted else "--"
                alpha = 0.85
                ax.plot(
                    df.index, df[bus][prop_name],
                    color=colors[k % len(colors)],
                    linestyle=style, linewidth=1.5, alpha=alpha,
                    # [STSI 100626]:label removed, using shared legend
                )

            pred_str = "predicted" if is_predicted else "known"
            ax.set_title(f"{bus} ({bus_label}) — {prop_name} [{pred_str}]",
                         fontsize=base_font_size * 0.8, fontweight="bold" if is_predicted else "normal")
            ax.tick_params(axis="both", labelsize=base_font_size * 0.7)
            ax.grid(True, alpha=0.3)
            # [STSI 100626]:per-subplot legend removed

    fig.suptitle(f"{title_prefix} — Bus Results", fontsize=base_font_size * 1.2, y=1.01)
    fig.text(0.5, -0.01, "Time Step", ha="center", fontsize=base_font_size * 1.1)
    solver_names = list(multi_results.keys())
    from matplotlib.lines import Line2D

    truth_handle = Line2D([0], [0], color="black", linewidth=2, linestyle="-",
                          label="PyPSA (true)")
    _add_shared_legend_from_dict(fig, solver_names,
                                 extra_handles=[truth_handle],
                                 base_font_size=base_font_size)  # [STSI 100626]:dict-keyed multi-model shared legend
    plt.show()
    return fig

def plot_predictions_comparison_per_node(
    multi_results, network, true_bus_results=None,
    title_prefix="", predicted_only=True, base_font_size=10
):
    """
    Scatter: predicted vs true per node, one panel per output variable, one figure per model.
    STSI260320: replaces single-model version.
    """
    buses = list(network.buses.index)
    props = [("V pu", "V", "Vmag [p.u.]"),
             ("Angle deg", "Angle", "Vang [deg]"),
             ("P pu", "P", "P [pu]"),
             ("Q pu", "Q", "Q [pu]")]

    for label, res in multi_results.items():
        masks = res["masks"]
        df_model = res["bus"]

        fig, axs = plt.subplots(1, len(props), figsize=(5 * len(props), 5))

        for j, (prop_name, mask_key, axis_label) in enumerate(props):
            ax = axs[j]
            pred_vals, true_vals = [], []
            for bus in buses:
                if predicted_only and not masks[mask_key][bus].any():
                    continue
                if true_bus_results is None or (bus, prop_name) not in true_bus_results.columns:
                    continue
                if (bus, prop_name) not in df_model.columns:
                    continue
                pred_vals.extend(df_model[bus][prop_name].values)
                true_vals.extend(true_bus_results[bus][prop_name].values)

            if pred_vals:
                ax.scatter(true_vals, pred_vals,
                           alpha=0.5, s=15)#, label=label)

            # Identity line
            if true_bus_results is not None:
                all_true = []
                for bus in buses:
                    if (bus, prop_name) in true_bus_results.columns:
                        all_true.extend(true_bus_results[bus][prop_name].values)
                if all_true:
                    mn, mx = min(all_true), max(all_true)
                    ax.plot([mn, mx], [mn, mx], "k--", linewidth=1.5, label="Perfect")

            ax.set_xlabel(f"True {axis_label}", fontsize=base_font_size * 1.0)
            ax.set_ylabel(f"Predicted {axis_label}", fontsize=base_font_size * 1.0)
            ax.set_title(axis_label, fontsize=base_font_size * 1.1)
            ax.tick_params(axis="both", labelsize=base_font_size * 0.85)
            ax.legend(fontsize=base_font_size * 0.75)
            ax.grid(True, alpha=0.3)

        fig.suptitle(f"{title_prefix} — Predicted vs True (scatter) — {label}", fontsize=base_font_size * 1.2)
        plt.tight_layout()
    plt.show()


def plot_errors_comparison_per_node(
    multi_results, network, true_bus_results=None,
    title_prefix="", predicted_only=True, plot_buses=None,  # [STSI 200526]:optional bus filter
    base_font_size=10,
):
    """
    Per-bus error bars: MAE per bus per model, grouped by bus.
    STSI260320: shows which buses benefit/suffer from physics loss.
    """
    buses = plot_buses if plot_buses else list(network.buses.index)  # [STSI 200526]:use sampled buses if provided
    props = [("V pu", "V", "Vmag MAE [p.u.]"),
             ("Angle deg", "Angle", "Vang MAE [deg]"),
             ("P pu", "P", "P MAE [pu]"),
             ("Q pu", "Q", "Q MAE [pu]")]
    colors = plt.cm.tab10.colors
    labels = list(multi_results.keys())
    first_masks = list(multi_results.values())[0]["masks"]

    fig, axs = plt.subplots(1, len(props), figsize=(5 * len(props), 5))
    x = np.arange(len(buses))
    width = 0.8 / len(labels)

    for j, (prop_name, mask_key, axis_label) in enumerate(props):
        ax = axs[j]
        for k, (label, res) in enumerate(multi_results.items()):
            df_model = res["bus"]
            maes = []
            for bus in buses:
                if predicted_only and not first_masks[mask_key][bus].any():
                    maes.append(0.0)
                    continue
                if true_bus_results is None or (bus, prop_name) not in true_bus_results.columns:
                    maes.append(0.0)
                    continue
                if (bus, prop_name) not in df_model.columns:
                    maes.append(0.0)
                    continue
                err = np.mean(np.abs(
                    df_model[bus][prop_name].values -
                    true_bus_results[bus][prop_name].values
                ))
                maes.append(err)
            offset = (k - len(labels) / 2 + 0.5) * width
            ax.bar(x + offset, maes, width,
                   color=colors[k % len(colors)], alpha=0.8, label=label)

        bus_labels = [f"{b}\n({get_bus_type_label(network, b)})" for b in buses]
        ax.set_xticks(x)
        ax.set_xticklabels(bus_labels, fontsize=base_font_size * 0.8)
        ax.set_ylabel(axis_label, fontsize=base_font_size * 1.0)
        ax.set_title(axis_label, fontsize=base_font_size * 1.1)
        ax.tick_params(axis="y", labelsize=base_font_size * 0.85)

        ax.grid(True, axis="y", alpha=0.3)

    fig.suptitle(f"{title_prefix} — Per-bus MAE by model", fontsize=base_font_size * 1.2)
    # Shared legend from first subplot handles
    handles, leg_labels = axs[0].get_legend_handles_labels()
    fig.legend(
        handles=handles,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.02),
        ncol=max(1, min(4, len(handles))),
        fontsize=base_font_size * 0.85,
        frameon=True,
    )
    plt.tight_layout(rect=[0, 0.08, 1, 0.95])
    plt.show()

def plot_line_flow_timeseries_comparison(multi_results, network, true_line_results=None, title_prefix="", plot_lines=None, base_font_size=10):  # [STSI 200526]:optional line filter
    """One figure per line: P0 (left) and Q0 (right), all models vs PyPSA truth."""
    all_lines = list(network.lines.index)
    lines = [l for l in all_lines if l in plot_lines] if plot_lines else all_lines  # [STSI 200526]:filter to connected lines
    colors = plt.cm.tab10.colors
    for line in lines:
        from_bus = network.lines.loc[line, "bus0"]
        to_bus   = network.lines.loc[line, "bus1"]
        fig, (ax_p, ax_q) = plt.subplots(1, 2, figsize=(16, 5), sharex=True)
        fig.suptitle(f"{title_prefix} — Line: {line}  ({from_bus} → {to_bus})", fontsize=base_font_size * 1.2)
        if true_line_results is not None:
            if (line, "P0 pu") in true_line_results.columns:
                ax_p.plot(true_line_results.index, true_line_results[(line, "P0 pu")],
                          color="black", linewidth=2.5, zorder=10)
            if (line, "Q0 pu") in true_line_results.columns:
                ax_q.plot(true_line_results.index, true_line_results[(line, "Q0 pu")],
                          color="black", linewidth=2.5, zorder=10)
        for k, (label, res) in enumerate(multi_results.items()):
            line_df = res.get("line")
            if line_df is None:
                continue
            c = colors[k % len(colors)]
            if (line, "P0 pu") in line_df.columns:
                ax_p.plot(line_df.index, line_df[(line, "P0 pu")],
                          color=c, linewidth=1.5, alpha=0.85)  # [STSI 100626]:dict-keyed multi-model shared legend
            if (line, "Q0 pu") in line_df.columns:
                ax_q.plot(line_df.index, line_df[(line, "Q0 pu")],
                          color=c, linewidth=1.5, alpha=0.85)  # [STSI 100626]:dict-keyed multi-model shared legend
        ax_p.set_xlabel("Time Step", fontsize=base_font_size * 1.0); ax_p.set_ylabel("P0 [pu]", fontsize=base_font_size * 1.0)
        ax_p.set_title("Active Power P0", fontsize=base_font_size * 1.1); ax_p.grid(True, alpha=0.3)
        ax_q.set_xlabel("Time Step", fontsize=base_font_size * 1.0); ax_q.set_ylabel("Q0 [pu]", fontsize=base_font_size * 1.0)
        ax_q.set_title("Reactive Power Q0", fontsize=base_font_size * 1.1); ax_q.grid(True, alpha=0.3)
        for _ax in (ax_p, ax_q):
            _ax.tick_params(axis="both", labelsize=base_font_size * 0.85)
        solver_names = list(multi_results.keys())
        from matplotlib.lines import Line2D
        truth_handle = Line2D([0], [0], color="black", linewidth=2, linestyle="-",
                              marker="o", markersize=5, label="PyPSA (true)")
        _add_shared_legend_from_dict(fig, solver_names,
                                     extra_handles=[truth_handle],
                                     base_font_size=base_font_size)  # [STSI 100626]:dict-keyed multi-model shared legend
    plt.show()

def plot_single_network_comparisons(
    runs,
    test_network,
    title_prefix="",
    use_pnom_share=None,
    label_mode="auto",
    compact_threshold=RUN_LABEL_COMPACT_THRESHOLD,
    index_threshold=RUN_LABEL_INDEX_THRESHOLD,
    plot_buses=None,  # [STSI 200526]:optional list of buses to plot (None=all)
    base_font_size=10,
    include_dc=False,  # [STSI 080926]:add a "DC (lpf)" entry alongside GNN models
):
    """
    Multi-model overlay version.
    Build models dict: label -> model (or dict with ptdf info for learned_edge)
    """

    # Build true PyPSA results
    true_bus, true_line, _ = model_results(
        test_network,
        print_results=False,
        plot_results=False,
        compute_ptdf=False,
    )

    _, label_entries = resolve_run_label_policy(
        runs,
        label_mode=label_mode,
        compact_threshold=compact_threshold,
        index_threshold=index_threshold,
    )
    label_entry_by_run = _index_label_entries(label_entries)

    # Build models dict: label -> model (or dict with ptdf info for learned_edge)
    models_dict = {}
    for r in runs:
        label = label_entry_by_run[id(r)]["legend_label"]
        ptdf_params = r.get("_ptdf_params")
        max_buses = r.get("_max_buses", 0)
        if ptdf_params is not None and max_buses > 0:
            models_dict[label] = {
                "model": r["model"],
                "ptdf_params": ptdf_params,
                "max_buses": max_buses,
            }
        else:
            models_dict[label] = r["model"]

    # Get multi predictions
    multi_results = predict_network_results_with_masks_multi(
        models_dict, test_network, use_pnom_share=use_pnom_share, include_dc=include_dc
    )

    # Derive plot_lines from plot_buses [STSI 200526]:filter lines to those connected to sampled buses
    if plot_buses:
        bus_set = set(plot_buses)
        plot_lines = test_network.lines[
            test_network.lines.bus0.isin(bus_set) | test_network.lines.bus1.isin(bus_set)
        ].index.tolist()
    else:
        plot_lines = None

    # All three plots
    plot_comparison_results_with_masks(
        multi_results,
        test_network,
        true_bus_results=true_bus,
        true_line_results=true_line,
        title_prefix=title_prefix,
        plot_buses=plot_buses,
        base_font_size=base_font_size,
    )
    plot_line_flow_timeseries_comparison(
        multi_results,
        test_network,
        true_line_results=true_line,
        title_prefix=title_prefix,
        plot_lines=plot_lines,
        base_font_size=base_font_size,
    )

    plot_errors_comparison_per_node(
        multi_results,
        test_network,
        true_bus_results=true_bus,
        title_prefix=title_prefix,
        plot_buses=plot_buses,
        base_font_size=base_font_size,
    )

def evaluate_models_on_networks(models_dict, networks, use_edge_features=True, use_pnom_share=None):
    """
    Run each model on a list of networks and aggregate simple metrics.

    Returns: dict label -> dict with keys:
        'vmag_mae', 'vang_mae_deg', 'p_mae', 'q_mae'  (averaged over nets & buses)

    NOTE: vang_mae_deg is in DEGREES. evaluate_gnn_on_test_set / evaluate_dc_baseline /
    the solver baseline functions report 'vang_mae' in RADIANS -- do not compare the two
    directly (~57x = 180/pi unit difference).  # [STSI 140926]:clarify unit convention
    """
    results = {}

    for label, entry in models_dict.items():
        # Unpack model / ptdf info (entry can be bare model or dict)
        if isinstance(entry, dict):
            model       = entry["model"]
            ptdf_params = entry.get("ptdf_params")
            max_buses   = entry.get("max_buses", 0)
        else:
            model       = entry
            ptdf_params = None
            max_buses   = 0

        vmag_errs = []
        vang_errs = []
        p_errs    = []
        q_errs    = []

        n_nets = len(networks)
        prog_step = max(1, n_nets // 5)
        for i_net, net in enumerate(networks):
            true_bus, true_line, _ = model_results(
                net, print_results=False, plot_results=False, compute_ptdf=False
            )
            try:
                bus_res, _, masks, _ = predict_network_results_with_masks(
                    model, net, use_edge_features=use_edge_features,
                    ptdf_params=ptdf_params, max_buses=max_buses,
                    use_pnom_share=use_pnom_share,
                )
            except RuntimeError as exc:
                print(f"[WARNING] {label} failed on network: {exc}")
                continue

            buses = list(net.buses.index)
            for bus in buses:
                # Vmag
                if (bus, "V pu") in true_bus.columns and (bus, "V pu") in bus_res.columns:
                    vmag_errs.extend(
                        np.abs(
                            bus_res[bus]["V pu"].values -
                            true_bus[bus]["V pu"].values
                        )
                    )
                # Vang (deg)
                if (bus, "Angle deg") in true_bus.columns and (bus, "Angle deg") in bus_res.columns:
                    vang_errs.extend(
                        np.abs(
                            bus_res[bus]["Angle deg"].values -
                            true_bus[bus]["Angle deg"].values
                        )
                    )
                # P
                if (bus, "P pu") in true_bus.columns and (bus, "P pu") in bus_res.columns:
                    p_errs.extend(
                        np.abs(
                            bus_res[bus]["P pu"].values -
                            true_bus[bus]["P pu"].values
                        )
                    )
                # Q
                if (bus, "Q pu") in true_bus.columns and (bus, "Q pu") in bus_res.columns:
                    q_errs.extend(
                        np.abs(
                            bus_res[bus]["Q pu"].values -
                            true_bus[bus]["Q pu"].values
                        )
                    )
    
            if (i_net + 1) % prog_step == 0 or (i_net + 1) == n_nets:
                print(f"  [{label}] {i_net+1}/{n_nets} networks evaluated")
        results[label] = {
            "vmag_mae":     float(np.mean(vmag_errs)) if vmag_errs else np.nan,
            "vang_mae_deg": float(np.mean(vang_errs)) if vang_errs else np.nan,
            "p_mae":        float(np.mean(p_errs))    if p_errs    else np.nan,
            "q_mae":        float(np.mean(q_errs))    if q_errs    else np.nan,
        }

    return results

def plot_model_metrics_across_networks(models_dict, networks, title_prefix="Multi-net", use_pnom_share=None, metrics=None, base_font_size=10):
    if metrics is None:
        metrics = evaluate_models_on_networks(models_dict, networks, use_pnom_share=use_pnom_share)
    labels  = list(metrics.keys())
    keys    = ["vmag_mae", "vang_mae_deg", "p_mae", "q_mae"]
    pretty  = ["Vmag MAE [p.u.]", "Vang MAE [deg]", "P MAE [pu]", "Q MAE [pu]"]

    x = np.arange(len(labels))
    fig, axes = plt.subplots(1, len(keys), figsize=(12 + len(labels), 5))
    for ax, k, name in zip(axes, keys, pretty):
        vals = [metrics[l][k] for l in labels]
        ax.bar(x, vals, color=[f"C{i}" for i in x])
        ax.set_xticks(x)
        short_labels = [f"M{i+1}" for i in range(len(labels))]  # [STSI 100626]:dict-keyed multi-model shared legend
        ax.set_xticklabels(short_labels, rotation=0, ha="center", fontsize=base_font_size * 0.85)
        ax.set_ylabel(name, fontsize=base_font_size * 1.0)
        ax.set_title(name, fontsize=base_font_size * 1.1)
        ax.tick_params(axis="y", labelsize=base_font_size * 0.85)
        ax.grid(True, axis="y", alpha=0.3)

    fig.suptitle(f"{title_prefix} — averaged over {len(networks)} networks", fontsize=base_font_size * 1.3)
    # Add mapping legend: M1 → full label, M2 → full label  [STSI 100626]:mapping legend for dict-keyed model labels
    from matplotlib.patches import Patch
    model_names = list(models_dict.keys())
    mapping_handles = [
        Patch(facecolor="none", edgecolor="none",
              label=f"M{i+1}: {name}")
        for i, name in enumerate(model_names)
    ]
    fig.legend(handles=mapping_handles, loc="lower center",
               bbox_to_anchor=(0.5, -0.02),
               ncol=max(1, min(3, len(mapping_handles))),
               fontsize=base_font_size * 0.85, frameon=True)
    plt.tight_layout(rect=[0, 0.08, 1, 0.95])
    plt.show()

    

In [ ]:


def _collect_test_set_predictions(models_dict, networks, use_pnom_share=None, include_dc=False):  # [STSI 090926]:optional DC (lpf) baseline
    """
    Run every model on every network.
    Returns dict: label -> {'pred': [N,4], 'true': [N,4], optionally 'p0_pred', 'p0_true', 'q0_pred', 'q0_true'}
    """
    all_preds = {}
    for label, entry in models_dict.items():
        if isinstance(entry, dict):
            model       = entry["model"]
            ptdf_params = entry.get("ptdf_params")
            max_buses   = entry.get("max_buses", 0)
        else:
            model, ptdf_params, max_buses = entry, None, 0

        pred_rows, true_rows = [], []
        p0_pred_rows, p0_true_rows = [], []
        q0_pred_rows, q0_true_rows = [], []
        dt_pred_rows, dt_true_rows = [], []  # [STSI 010626]:delta_theta edge-level

        n_nets = len(networks)
        prog_step = max(1, n_nets // 5)
        for i_net, net in enumerate(networks):
            bus_names = list(net.buses.index)
            try:
                bus_res, pred_line, masks, dt_data = predict_network_results_with_masks(
                    model, net, use_edge_features=True,
                    ptdf_params=ptdf_params, max_buses=max_buses,
                    use_pnom_share=use_pnom_share,
                    collect_delta_theta=True)  # [STSI 010626]:collect edge delta_theta
            except RuntimeError as exc:
                print(f"[WARNING] {label} skipped: {exc}"); continue

            true_bus, true_line, _ = model_results(net, print_results=False, plot_results=False, compute_ptdf=False)

            for t in net.snapshots:
                for bus in bus_names:
                    try:
                        pred_rows.append([bus_res.loc[t,(bus,"V pu")], bus_res.loc[t,(bus,"Angle deg")],
                                          bus_res.loc[t,(bus,"P pu")], bus_res.loc[t,(bus,"Q pu")]])
                        true_rows.append([true_bus.loc[t,(bus,"V pu")], true_bus.loc[t,(bus,"Angle deg")],
                                          true_bus.loc[t,(bus,"P pu")], true_bus.loc[t,(bus,"Q pu")]])
                    except KeyError:
                        continue
                if pred_line is not None and true_line is not None:
                    for line in net.lines.index:
                        try:
                            p0_pred_rows.append(pred_line.loc[t,(line,"P0 pu")]); p0_true_rows.append(true_line.loc[t,(line,"P0 pu")])
                            q0_pred_rows.append(pred_line.loc[t,(line,"Q0 pu")]); q0_true_rows.append(true_line.loc[t,(line,"Q0 pu")])
                        except KeyError:
                            continue

            # [STSI 010626]:accumulate delta_theta from this network
            if "dt_pred" in dt_data:
                dt_pred_rows.append(dt_data["dt_pred"])
            if "dt_true" in dt_data:
                dt_true_rows.append(dt_data["dt_true"])

            if (i_net + 1) % prog_step == 0 or (i_net + 1) == n_nets:
                print(f"  [{label}] {i_net+1}/{n_nets} networks processed")
        entry_dict = {"pred": np.array(pred_rows), "true": np.array(true_rows)}
        if p0_pred_rows:
            entry_dict["p0_pred"] = np.array(p0_pred_rows); entry_dict["p0_true"] = np.array(p0_true_rows)
        if q0_pred_rows:
            entry_dict["q0_pred"] = np.array(q0_pred_rows); entry_dict["q0_true"] = np.array(q0_true_rows)
        if dt_pred_rows:  # [STSI 010626]:store concatenated delta_theta arrays
            entry_dict["dt_pred"] = np.concatenate(dt_pred_rows)
        if dt_true_rows:
            entry_dict["dt_true"] = np.concatenate(dt_true_rows)
        all_preds[label] = entry_dict

    if include_dc:  # [STSI 090926]:add DC (lpf) baseline to full test-set predictions
        _dc_dataset = PowerFlowDataset(networks, use_edge_features=True)
        _dc_index_map = {(ni, ti): k for k, (ni, ti) in enumerate(_dc_dataset._index)}
        pred_rows, true_rows = [], []
        p0_pred_rows, p0_true_rows = [], []
        q0_pred_rows, q0_true_rows = [], []
        dt_pred_rows, dt_true_rows = [], []
        n_nets = len(networks)
        prog_step = max(1, n_nets // 5)
        for i_net, net in enumerate(networks):
            bus_names = list(net.buses.index)
            try:
                dc_bus, dc_line, _ = predict_network_results_dc(net)
            except Exception as exc:
                print(f"[WARNING] DC (lpf) skipped: {exc}"); continue
            true_bus, true_line, _ = model_results(net, print_results=False, plot_results=False, compute_ptdf=False)
            for t_idx, t in enumerate(net.snapshots):
                for bus in bus_names:
                    try:
                        pred_rows.append([dc_bus.loc[t,(bus,"V pu")], dc_bus.loc[t,(bus,"Angle deg")],
                                          dc_bus.loc[t,(bus,"P pu")], dc_bus.loc[t,(bus,"Q pu")]])
                        true_rows.append([true_bus.loc[t,(bus,"V pu")], true_bus.loc[t,(bus,"Angle deg")],
                                          true_bus.loc[t,(bus,"P pu")], true_bus.loc[t,(bus,"Q pu")]])
                    except KeyError:
                        continue
                if dc_line is not None and true_line is not None:
                    for line in net.lines.index:
                        try:
                            p0_pred_rows.append(dc_line.loc[t,(line,"P0 pu")]); p0_true_rows.append(true_line.loc[t,(line,"P0 pu")])
                            q0_pred_rows.append(dc_line.loc[t,(line,"Q0 pu")]); q0_true_rows.append(true_line.loc[t,(line,"Q0 pu")])
                        except KeyError:
                            continue
                ds_idx = _dc_index_map.get((i_net, t_idx))
                if ds_idx is not None:
                    data = _dc_dataset[ds_idx]
                    fwd_mask = data.forward_edge_mask.numpy()
                    ei = data.edge_index[:, fwd_mask].numpy()
                    vang_dc = np.radians([dc_bus.loc[t, (bus, "Angle deg")] for bus in bus_names])
                    theta_true = data.y[:, 1].numpy()
                    dt_true_rows.append(theta_true[ei[0]] - theta_true[ei[1]])
                    dt_pred_rows.append(vang_dc[ei[0]] - vang_dc[ei[1]])
            if (i_net + 1) % prog_step == 0 or (i_net + 1) == n_nets:
                print(f"  [DC (lpf)] {i_net+1}/{n_nets} networks processed")
        dc_entry = {"pred": np.array(pred_rows), "true": np.array(true_rows)}
        if p0_pred_rows:
            dc_entry["p0_pred"] = np.array(p0_pred_rows); dc_entry["p0_true"] = np.array(p0_true_rows)
        if q0_pred_rows:
            dc_entry["q0_pred"] = np.array(q0_pred_rows); dc_entry["q0_true"] = np.array(q0_true_rows)
        if dt_pred_rows:
            dc_entry["dt_pred"] = np.concatenate(dt_pred_rows)
        if dt_true_rows:
            dc_entry["dt_true"] = np.concatenate(dt_true_rows)
        all_preds["DC (lpf)"] = dc_entry

    return all_preds


_METRIC_COLORS = {0: "C0", 1: "C1", 2: "C2", 3: "C3", "p0": "C4", "q0": "C5", "dt": "C6"}  # [STSI 100926]:fixed per-metric colors for by_run/by_metric parity


def _extract_metric_arrays(data, metric_key):
    """Return (true_arr, pred_arr) for a metric_key, or (None, None) if unavailable for this run."""
    RAD2DEG = 180.0 / np.pi
    if metric_key in (0, 1, 2, 3):
        return data["true"][:, metric_key], data["pred"][:, metric_key]
    if metric_key == "p0":
        if "p0_pred" in data and len(data["p0_pred"]) > 0:
            return data["p0_true"], data["p0_pred"]
        return None, None
    if metric_key == "q0":
        if "q0_pred" in data and len(data["q0_pred"]) > 0:
            return data["q0_true"], data["q0_pred"]
        return None, None
    if metric_key == "dt":
        if "dt_pred" in data and len(data["dt_pred"]) > 0:
            return data["dt_true"] * RAD2DEG, data["dt_pred"] * RAD2DEG
        return None, None
    raise ValueError(f"Unknown metric_key: {metric_key!r}")


def _metric_specs_for_run(data):
    """Per-run panel list (matches original has_lf/has_dt gating) -- used by layout='by_run'."""
    specs = [(0, "V_mag (pu)"), (1, "V_ang (deg)"), (2, "P (pu)"), (3, "Q (pu)")]
    if "p0_pred" in data and len(data["p0_pred"]) > 0:
        specs += [("p0", "P0 Line (pu)"), ("q0", "Q0 Line (pu)")]
    if "dt_pred" in data and len(data["dt_pred"]) > 0:
        specs.append(("dt", "Δθ edge (deg)"))
    return specs


def _metric_specs_union(all_preds):
    """Union panel list across all runs (any run has it) -- used by layout='by_metric'."""
    has_lf = any("p0_pred" in d and len(d["p0_pred"]) > 0 for d in all_preds.values())
    has_dt = any("dt_pred" in d and len(d["dt_pred"]) > 0 for d in all_preds.values())
    specs = [(0, "V_mag (pu)"), (1, "V_ang (deg)"), (2, "P (pu)"), (3, "Q (pu)")]
    if has_lf:
        specs += [("p0", "P0 Line (pu)"), ("q0", "Q0 Line (pu)")]
    if has_dt:
        specs.append(("dt", "Δθ edge (deg)"))
    return specs


def _chunk_labels(seq, size):
    """Wrap a flat list into row-chunks of at most `size` (mirrors plot_timing_vs_accuracy's _chunk)."""
    seq = list(seq)
    if not size or size < 1 or size >= len(seq):
        return [seq]
    return [seq[i:i + size] for i in range(0, len(seq), size)]


def _resolve_operating_range(metric_name, operating_ranges):
    """Look up an (lo, hi) axis-clip range for a display metric name, if provided."""
    return (operating_ranges or {}).get(metric_name)


def _warn_unmatched_operating_ranges(operating_ranges, valid_names):
    """Print a loud warning for operating_ranges keys that do not match any known metric name.

    Silent no-op keys (e.g. missing a space, or a name that was never a panel title) are the
    most common cause of 'range clipping did nothing'.
    """
    if not operating_ranges:
        return
    unmatched = [k for k in operating_ranges if k not in valid_names]
    if unmatched:
        print(f"[operating_ranges] WARNING: key(s) {unmatched} do not match any plotted metric name "
              f"and will have NO effect. Valid names for this call: {sorted(valid_names)}")


def _add_run_label_legend(fig, labels, short_labels, base_font_size=10,
                           side="right", bottom_frac=0.10, right_frac=0.75, label_wrap=None):
    """Shared legend mapping short panel titles (e.g. R1) to full run labels.

    side : "right" (default) places a single-column legend right of the axes;
           "bottom" places a wrapped multi-column legend below the figure.
    label_wrap : int, optional -- wrap legend text at this many characters (None/0 = off)
    """
    import textwrap
    from matplotlib.patches import Patch
    def _fmt(l):
        text = f"{short_labels[l]}: {l}"
        return textwrap.fill(text, label_wrap) if label_wrap else text
    handles = [Patch(facecolor="none", edgecolor="none", label=_fmt(l)) for l in labels]
    if side == "right":
        fig.legend(handles=handles, loc="center left", bbox_to_anchor=(right_frac, 0.5),
                   ncol=1, fontsize=base_font_size * 0.75, frameon=True)
        plt.tight_layout(rect=[0, 0, right_frac, 1])  # [STSI 110926]:reserve space on the right, not bottom
    else:
        fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, -0.02),
                   ncol=max(1, min(3, len(handles))), fontsize=base_font_size * 0.75, frameon=True)
        plt.tight_layout(rect=[0, bottom_frac, 1, 0.95])


def _plot_scatter_panel(ax, true_arr, pred_arr, metric_label, color, base_font_size,
                         operating_range=None, panel_title=None):
    lo, hi = min(true_arr.min(), pred_arr.min()), max(true_arr.max(), pred_arr.max())
    ax.scatter(true_arr, pred_arr, s=2, alpha=0.3, color=color)
    ax.plot([lo, hi], [lo, hi], "k--", lw=0.8, label="ideal")
    ax.set_aspect("equal", adjustable="datalim")
    ax.set_xlabel(f"True {metric_label}", fontsize=base_font_size * 1.0)
    ax.set_ylabel(f"Predicted {metric_label}", fontsize=base_font_size * 1.0)
    ax.set_title(panel_title or metric_label, fontsize=base_font_size * 1.1)
    ax.legend(fontsize=base_font_size * 0.75)
    ax.tick_params(axis="both", labelsize=base_font_size * 0.85)
    ax.grid(True, alpha=0.3)
    if operating_range is not None:
        ax.set_xlim(operating_range); ax.set_ylim(operating_range)  # [STSI 100926]:view-only clip; stats unaffected


def _plot_error_hist_panel(ax, true_arr, pred_arr, metric_label, bins, color, base_font_size,
                            panel_title=None):
    err = pred_arr - true_arr
    mae = np.mean(np.abs(err)); mu = np.mean(err); sigma = np.std(err)
    ci_lo, ci_hi = np.percentile(err, [2.5, 97.5])
    ax.hist(err, bins=bins, alpha=0.6, color=color,
            label=f"MAE={mae:.4g}\nμ={mu:.4g}  σ={sigma:.4g}\n95% CI=[{ci_lo:.4g},{ci_hi:.4g}]")
    ax.axvline(mu, color="k", linestyle="-", linewidth=1.2, label="mean")
    ax.axvline(ci_lo, color="k", linestyle="--", linewidth=0.9)
    ax.axvline(ci_hi, color="k", linestyle="--", linewidth=0.9, label="95% CI")
    ax.set_xlabel(f"Error ({metric_label})", fontsize=base_font_size * 1.0)
    ax.set_ylabel("Count", fontsize=base_font_size * 1.0)
    ax.set_title(panel_title or metric_label, fontsize=base_font_size * 1.1)
    ax.legend(fontsize=base_font_size * 0.8)
    ax.tick_params(axis="both", labelsize=base_font_size * 0.85)
    ax.grid(True, alpha=0.3)
    # operating_ranges intentionally NOT applied here -- x-axis is error, not the operating value


def plot_full_test_scatter(all_preds, title_prefix="", base_font_size=10,
                            layout="by_run", operating_ranges=None, max_cols=3,
                            legend_label_wrap=None):  # [STSI 100926]:added layout/operating_ranges/max_cols; [STSI 110926]:legend_label_wrap
    """
    Scatter (predicted vs true): 4 bus + optional 2 line flow + optional 1 Δθ panels.

    layout : "by_run" (default, backward compatible) - one figure per run, one panel per metric.
             "by_metric" - one figure per metric, one panel per run (grid wrapped at max_cols).
    operating_ranges : dict {display_metric_name: (lo, hi)}, optional.
        Clips the true/pred axis view to the given range (e.g. {"V_mag (pu)": (0.9, 1.1)}).
        Stats/points are NOT filtered - this only sets ax.set_xlim/set_ylim.
    max_cols : int, grid width for layout="by_metric" (ignored for "by_run").
    legend_label_wrap : int, optional - wrap the shared run-label legend text at this many characters (None/0 = off).
    """
    _warn_unmatched_operating_ranges(operating_ranges, {c for _, c in _metric_specs_union(all_preds)})
    if layout == "by_run":
        for label, data in all_preds.items():
            metric_specs = _metric_specs_for_run(data)
            n_panels = len(metric_specs)
            fig, axes = plt.subplots(1, n_panels, figsize=(5.5 * n_panels, 5))
            for ax, (metric_key, cname) in zip(np.atleast_1d(axes), metric_specs):
                true_arr, pred_arr = _extract_metric_arrays(data, metric_key)
                _plot_scatter_panel(ax, true_arr, pred_arr, cname, _METRIC_COLORS[metric_key],
                                     base_font_size, _resolve_operating_range(cname, operating_ranges))
            fig.suptitle(f"{title_prefix} Full test-set scatter — {label}", fontsize=base_font_size * 1.4)
            plt.tight_layout(); plt.show()

    elif layout == "by_metric":
        metric_specs = _metric_specs_union(all_preds)
        labels = list(all_preds.keys())
        rows = _chunk_labels(labels, max_cols)
        ncols = max(len(r) for r in rows)
        nrows = len(rows)
        short_labels = {lbl: f"R{i + 1}" for i, lbl in enumerate(labels)}  # [STSI 100926]:short panel titles; full run info moved to shared legend
        for metric_key, cname in metric_specs:
            fig, axes = plt.subplots(nrows, ncols, figsize=(5.0 * ncols, 4.5 * nrows), squeeze=False)
            op_range = _resolve_operating_range(cname, operating_ranges)
            for r_i, row_labels in enumerate(rows):
                for c_i in range(ncols):
                    ax = axes[r_i][c_i]
                    if c_i >= len(row_labels):
                        ax.set_visible(False); continue
                    label = row_labels[c_i]
                    true_arr, pred_arr = _extract_metric_arrays(all_preds[label], metric_key)
                    if true_arr is None:
                        ax.set_title(short_labels[label], fontsize=base_font_size * 1.0); ax.set_visible(False); continue
                    _plot_scatter_panel(ax, true_arr, pred_arr, cname, f"C{(r_i * ncols + c_i) % 10}",
                                         base_font_size, op_range, panel_title=short_labels[label])
            fig.suptitle(f"{title_prefix} Full test-set scatter — {cname}", fontsize=base_font_size * 1.4)
            _add_run_label_legend(fig, labels, short_labels, base_font_size=base_font_size, label_wrap=legend_label_wrap)  # [STSI 100926]:run info moved to shared legend; [STSI 110926]:legend_label_wrap
            plt.show()
    else:
        raise ValueError(f"Unknown layout: {layout!r} (expected 'by_run' or 'by_metric')")


def plot_error_histograms(all_preds, title_prefix="", bins=80, base_font_size=10,
                           layout="by_run", operating_ranges=None, max_cols=3,
                           legend_label_wrap=None):  # [STSI 100926]:added layout/operating_ranges/max_cols; [STSI 110926]:legend_label_wrap
    """
    Error histograms (pred - true): 4 bus + optional 2 line flow + optional 1 Δθ panels.

    layout : "by_run" (default) | "by_metric" - see plot_full_test_scatter for semantics.
    operating_ranges : accepted for signature parity with the scatter/error-vs-range plots but
        has NO effect here - the histogram x-axis is the prediction error, not the operating
        value, so there is no matching axis to clip.
    max_cols : int, grid width for layout="by_metric" (ignored for "by_run").
    legend_label_wrap : int, optional - wrap the shared run-label legend text at this many characters (None/0 = off).
    """
    if layout == "by_run":
        for label, data in all_preds.items():
            metric_specs = _metric_specs_for_run(data)
            n_panels = len(metric_specs)
            fig, axes = plt.subplots(1, n_panels, figsize=(5.5 * n_panels, 5))
            for ax, (metric_key, cname) in zip(np.atleast_1d(axes), metric_specs):
                true_arr, pred_arr = _extract_metric_arrays(data, metric_key)
                _plot_error_hist_panel(ax, true_arr, pred_arr, cname, bins, _METRIC_COLORS[metric_key], base_font_size)
            fig.suptitle(f"{title_prefix} {label}", fontsize=base_font_size * 1.4)
            plt.tight_layout(); plt.show()

    elif layout == "by_metric":
        metric_specs = _metric_specs_union(all_preds)
        labels = list(all_preds.keys())
        rows = _chunk_labels(labels, max_cols)
        ncols = max(len(r) for r in rows)
        nrows = len(rows)
        short_labels = {lbl: f"R{i + 1}" for i, lbl in enumerate(labels)}  # [STSI 100926]:short panel titles; full run info moved to shared legend
        for metric_key, cname in metric_specs:
            fig, axes = plt.subplots(nrows, ncols, figsize=(5.0 * ncols, 4.5 * nrows), squeeze=False)
            for r_i, row_labels in enumerate(rows):
                for c_i in range(ncols):
                    ax = axes[r_i][c_i]
                    if c_i >= len(row_labels):
                        ax.set_visible(False); continue
                    label = row_labels[c_i]
                    true_arr, pred_arr = _extract_metric_arrays(all_preds[label], metric_key)
                    if true_arr is None:
                        ax.set_title(short_labels[label], fontsize=base_font_size * 1.0); ax.set_visible(False); continue
                    _plot_error_hist_panel(ax, true_arr, pred_arr, cname, bins, f"C{(r_i * ncols + c_i) % 10}",
                                            base_font_size, panel_title=short_labels[label])
            fig.suptitle(f"{title_prefix} {cname} — error histograms", fontsize=base_font_size * 1.4)
            _add_run_label_legend(fig, labels, short_labels, base_font_size=base_font_size, label_wrap=legend_label_wrap)  # [STSI 100926]:run info moved to shared legend; [STSI 110926]:legend_label_wrap
            plt.show()
    else:
        raise ValueError(f"Unknown layout: {layout!r} (expected 'by_run' or 'by_metric')")


In [ ]:
def plot_errors_vs_operating_range(
    all_preds,
    n_bins: int = 20,
    title_prefix: str = "",
    smoothing_mode: str = None,   # None | "gaussian" | "rolling_avg"
    smoothing_sigma: float = 1.0, # used for gaussian
    smoothing_window: int = 5,    # used for rolling_avg
    base_font_size=10,
    operating_ranges: dict = None,  # [STSI 100926]:metric-name-keyed axis-clip range (view-only)
):
    """
    For each of the 4 bus quantities: one figure, one subplot per model (stacked).
    x-axis = operating range (bin centres of true values)
    y-axis = signed prediction error with filled IQR, std bands, and smoothed median.
    All subplots share unified y-axis range for comparability.

    smoothing_mode:
        None        — raw median line (no smoothing)
        "gaussian"  — scipy gaussian_filter1d (sigma=smoothing_sigma)
        "rolling_avg" — uniform rolling average (window=smoothing_window)
    """
    from scipy import ndimage

    def _smooth(arr):
        if smoothing_mode == "gaussian":
            return ndimage.gaussian_filter1d(arr, sigma=smoothing_sigma)
        elif smoothing_mode == "rolling_avg":
            kernel = np.ones(smoothing_window) / smoothing_window
            return np.convolve(arr, kernel, mode="same")
        return arr  # None — no smoothing

    col_specs = [
        (0, "V_mag (pu)"),
        (1, "V_ang (deg)"),
        (2, "P (pu)"),
        (3, "Q (pu)"),
        ("dt", "Δθ edge (deg)"),  # [STSI 020626]:added delta-theta as 5th quantity
    ]
    _warn_unmatched_operating_ranges(operating_ranges, {c for _, c in col_specs})
    colors   = plt.cm.tab10.colors
    labels   = list(all_preds.keys())
    n_models = len(labels)

    for col_idx, col_name in col_specs:
        # ── Pre-pass: bin all models, collect bounds for unified y-range ────────
        all_bounds = []
        all_data_per_model = {}

        for lbl, data in all_preds.items():
            if col_idx == "dt":  # [STSI 020626]:handle edge delta-theta separately
                if "dt_pred" not in data or len(data["dt_pred"]) == 0:
                    continue
                RAD2DEG = 180.0 / np.pi
                t_col = data["dt_true"] * RAD2DEG
                p_col = data["dt_pred"] * RAD2DEG
            else:
                t_col = data["true"][:, col_idx]
                p_col = data["pred"][:, col_idx]
            err   = p_col - t_col
            vmin, vmax = t_col.min(), t_col.max()
            if vmin == vmax:
                continue
            bin_edges   = np.linspace(vmin, vmax, n_bins + 1)
            bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
            bin_idx     = np.clip(np.digitize(t_col, bin_edges[:-1]) - 1, 0, n_bins - 1)

            q05, q25, q50, q75, q95, std_vals, valid_centers = [], [], [], [], [], [], []
            for b in range(n_bins):
                mask = bin_idx == b
                if mask.sum() >= 5:
                    vals = err[mask]
                    q05.append(np.percentile(vals, 5))
                    q25.append(np.percentile(vals, 25))
                    q50.append(np.percentile(vals, 50))
                    q75.append(np.percentile(vals, 75))
                    q95.append(np.percentile(vals, 95))
                    std_vals.append(np.std(vals))
                    valid_centers.append(bin_centers[b])

            if valid_centers:
                all_bounds.extend(q05 + q95 + std_vals)
                all_data_per_model[lbl] = {
                    "xc": np.array(valid_centers),
                    "q05": np.array(q05), "q25": np.array(q25),
                    "q50": np.array(q50), "q75": np.array(q75),
                    "q95": np.array(q95), "std": np.array(std_vals),
                }

        if not all_data_per_model:  # [STSI 020626]:skip quantity if no model has data
            continue

        # ── Unified y-range ───────────────────────────────────────────────────
        if all_bounds:
            y_min = np.percentile(all_bounds, 2.5)
            y_max = np.percentile(all_bounds, 97.5)
            margin = (y_max - y_min) * 0.1
            y_min -= margin; y_max += margin
        else:
            y_min, y_max = -1, 1

        # ── Main plot loop ────────────────────────────────────────────────────
        fig, axes = plt.subplots(
            n_models, 1,
            figsize=(10, max(3.5, n_models * 3.5)),
            squeeze=False,
        )
        axes = axes.flatten()
        fig.suptitle(
            f"{title_prefix} -- {col_name}: error spread vs operating range",
            fontsize=base_font_size * 1.3,
        )

        for row, (lbl, data) in enumerate(all_preds.items()):
            ax = axes[row]
            if lbl not in all_data_per_model:
                ax.set_title(lbl, fontsize=base_font_size * 1.1); continue

            pd_ = all_data_per_model[lbl]
            xc  = pd_["xc"]
            q05, q25, q50 = pd_["q05"], pd_["q25"], pd_["q50"]
            q75, q95      = pd_["q75"], pd_["q95"]
            std_vals      = pd_["std"]
            color         = colors[row % 10]

            q50_smooth   = _smooth(q50)
            std_smooth   = _smooth(std_vals)

            ax.fill_between(xc, q05, q95,   alpha=0.15, color=color)
            ax.fill_between(xc, q25, q75,   alpha=0.40, color=color)
            ax.fill_between(xc, q50 - std_smooth, q50 + std_smooth, alpha=0.25, color=color)
            ax.plot(xc, q50_smooth, color=color, linewidth=2.0)
            ax.axhline(0, color="red", linewidth=0.9, linestyle="--", alpha=0.6)
            ax.set_xlabel(f"True {col_name}", fontsize=base_font_size * 1.0); ax.set_ylabel("Prediction error", fontsize=base_font_size * 1.0)
            ax.set_title(lbl, fontsize=base_font_size * 1.1); ax.set_ylim(y_min, y_max)
            op_range = _resolve_operating_range(col_name, operating_ranges)  # [STSI 100926]:optional axis-clip
            if op_range is not None:
                ax.set_xlim(op_range)  # view-only clip; percentile/std bands unaffected
            ax.tick_params(axis="both", labelsize=base_font_size * 0.85)
            ax.grid(True, alpha=0.3)

        from matplotlib.lines import Line2D
        from matplotlib.patches import Patch
        struct_handles = [  # [STSI 100626]:dict-keyed multi-model shared legend — structural legend
            Patch(facecolor='gray', alpha=0.15, label='5-95 percentile'),
            Patch(facecolor='gray', alpha=0.40, label='IQR (25-75%)'),
            Patch(facecolor='gray', alpha=0.25, label='Mean ± 1σ'),
            Line2D([0], [0], color='gray', linewidth=2.0, label='Median'),
            Line2D([0], [0], color='red', linewidth=0.9, linestyle='--', alpha=0.6, label='zero error'),
        ]
        fig.legend(handles=struct_handles, loc="lower center",
                   bbox_to_anchor=(0.5, -0.02), ncol=5, fontsize=base_font_size * 0.85, frameon=True)
        plt.tight_layout(rect=[0, 0.06, 1, 0.95])
        plt.show()


def plot_line_flow_errors(
    runs,
    title_prefix="Line Flow Errors",
    label_mode="auto",
    compact_threshold=RUN_LABEL_COMPACT_THRESHOLD,
    index_threshold=RUN_LABEL_INDEX_THRESHOLD,
):
    """
    Bar chart comparing line_flow_mae across runs.
    Reads test_metrics['line_flow_mae'] from each run dict.
    Runs that pre-date line-flow tracking will show NaN bars.
    """
    x = list(range(len(runs)))
    active_label_mode, label_entries = resolve_run_label_policy(
        runs,
        label_mode=label_mode,
        compact_threshold=compact_threshold,
        index_threshold=index_threshold,
    )
    styles = _get_run_styles(runs)
    colors = [styles[id(r)][0] for r in runs]
    tick_labels = [entry["plot_label"] for entry in label_entries]
    tick_rotation = 0 if active_label_mode == "index" else 30
    tick_ha = "center" if active_label_mode == "index" else "right"
    tick_fontsize = 9 if active_label_mode == "index" else 8

    means = []
    for r in runs:
        tm = r.get("test_metrics", {})
        val = tm.get("line_flow_mae", float("nan"))
        means.append(float(val) if val is not None else float("nan"))

    fig, ax = plt.subplots(figsize=(max(8, len(runs) * 1.5), 6))
    fig.suptitle(f"{title_prefix} — Line Flow MAE [p.u.]", fontsize=13)

    bars = ax.bar(x, means, alpha=0.8, color=colors)
    ax.set_xticks(x)
    ax.set_xticklabels(tick_labels, rotation=tick_rotation, ha=tick_ha, fontsize=tick_fontsize)
    ax.set_ylabel("Line Flow MAE [p.u.]")
    ax.set_title("Line Flow MAE per run")
    ax.grid(True, axis="y", alpha=0.3)

    valid = [m for m in means if m == m]
    top = max(valid) if valid else 1.0
    for bar, mean in zip(bars, means):
        if mean == mean:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + top * 0.02,
                f"{mean:.4f}",
                ha="center", va="bottom", fontsize=7,
            )

    if active_label_mode == "index":
        legend_lines = [
            plt.Line2D([0], [0], color=colors[i], linewidth=6, alpha=0.8,
                       label=label_entries[i]["mapping_label"])
            for i in range(len(runs))
        ]
        fig.legend(
            handles=legend_lines,
            loc="lower center",
            bbox_to_anchor=(0.5, -0.02),
            ncol=max(1, min(3, len(runs))),
            fontsize=8,
            frameon=True,
        )
        plt.tight_layout(rect=[0, 0.12 + 0.03 * (len(runs) // 3), 1, 1])
    else:
        plt.tight_layout()
    plt.show()

In [ ]:

def plot_timing_vs_accuracy(  # [STSI 260526]:merged single + multi into subplot-grid function
    runs,
    y_metrics="vmag_mae",
    y_metrics_row2=None,  # [STSI 010626]:optional second row of RMSE metrics for 2-row grid
    x_metric="training_time",
    title_prefix="Timing vs Accuracy",
    ref_baselines: dict = None,
    x_axis_log: bool = False,
    timing_key: str = "both",
    label_mode: str = "auto",
    compact_threshold: int = RUN_LABEL_COMPACT_THRESHOLD,
    index_threshold: int = RUN_LABEL_INDEX_THRESHOLD,
    annotation_mode: str = "all",  # [STSI 260526]:default to annotate all points for readability,
    figsize_per_subplot=(10, 7),
    max_cols=None,  # None = no wrapping; int wraps each metric row-block into grid rows of at most max_cols
    legend_side="bottom",  # [STSI 040926]:"bottom" (default) or "right" (single-col RHS legend)
    legend_label_wrap=None,  # [STSI 040926]:wrap long legend labels at this many characters
    metrics_key: str = "test_metrics",  # [STSI 300526]:configurable key for external test set
    base_font_size=10,
):
    """
    Scatter plot: x = timing metric, y = one or more accuracy metrics (subplot grid).

    y_metrics : str or list[str]
        Single metric key or list of metric keys.
        e.g. "vmag_mae" or ["vmag_mae", "vang_rmse", "p_mae"]

    ref_baselines : dict {label: metrics_dict}, optional
        Reference points from evaluate_dc_baseline (or any solver).
        Plotted as black stars with bold labels on each subplot.

    x_metric options:
        "training_time"         — seconds per full training run (run['training_time'])
        "inference_time_s"      — seconds for entire test-set inference (test_metrics key)
        "infer_ms_per_snap"     — ms per snapshot (test_metrics key)

    timing_key : "single" | "batch" | "both"
        Controls which timing metric(s) to plot:
        - "single": per-snap timing only (circle markers)
        - "batch":  batched timing only (diamond markers)
        - "both":   both on same subplot (circle + diamond)

    max_cols : int
        Maximum columns in subplot grid (default 3).
    """
    import math

    # --- Normalize y_metrics to list[str] ---
    if isinstance(y_metrics, str):
        y_metrics = [y_metrics]

    x_metric_labels = {
        "training_time":     "Training time [s]",
        "inference_time_s":  "Inference time [s] (full test set)",
        "infer_ms_per_snap": "Inference time [ms / snapshot]",
    }
    y_metric_labels = {  # [STSI 010626]:extended with all RMSE + delta_theta labels
        "vmag_mae":          "Vmag MAE [p.u.]",
        "vmag_rmse":         "Vmag RMSE [p.u.]",
        "vang_mae":          "Vang MAE [rad]",
        "vang_rmse":         "Vang RMSE [rad]",
        "p_mae":             "P MAE [p.u.]",
        "p_rmse":            "P RMSE [p.u.]",
        "q_mae":             "Q MAE [p.u.]",
        "q_rmse":            "Q RMSE [p.u.]",
        "line_flow_mae":     "Line Flow P MAE [p.u.]",
        "line_flow_rmse":    "Line Flow P RMSE [p.u.]",
        "line_flow_q0_mae":  "Line Flow Q MAE [p.u.]",
        "line_flow_q0_rmse": "Line Flow Q RMSE [p.u.]",
        "delta_theta_mae":   "\u0394\u03b8 MAE [rad]",
        "delta_theta_rmse":  "\u0394\u03b8 RMSE [rad]",
    }

    # --- Grid layout ---  # [STSI 040926]:wrap each row-block by max_cols (mirrors DOE coefficient plot)
    def _chunk(seq, size):
        seq = [m for m in seq if m is not None]
        if not size or size < 1 or size >= len(seq):
            return [seq]
        return [seq[i:i + size] for i in range(0, len(seq), size)]

    # MAE block (y_metrics) and optional RMSE block (y_metrics_row2) chunked independently
    _grid_rows = list(_chunk(y_metrics, max_cols))
    _row2_clean = [m for m in (y_metrics_row2 or []) if m is not None]
    if _row2_clean:
        _grid_rows += list(_chunk(_row2_clean, max_cols))

    nrows = len(_grid_rows)
    ncols = max((len(ch) for ch in _grid_rows), default=1)
    fw, fh = figsize_per_subplot
    fig, axes = plt.subplots(nrows, ncols, figsize=(fw * ncols, fh * nrows), squeeze=False)

    # Build flat list: (metric, row, col) positions
    _all_metrics = [(ym, _ri, _ci)
                    for _ri, _chunk_row in enumerate(_grid_rows)
                    for _ci, ym in enumerate(_chunk_row)]

    # --- Shared setup ---
    _alias = {"infer_ms_per_snap": "inference_time_per_snapshot_ms"}
    _timing_modes = {
        "single": [("total_time_ms", "o", "per-snap")],
        "batch":  [("batch_solve_time_ms", "D", "batched")],
        "both":   [("total_time_ms", "o", ""),
                    ("batch_solve_time_ms", "D", "")],
    }
    modes = _timing_modes.get(timing_key, [("total_time_ms", "o", "")])

    styles = _get_run_styles(runs)
    active_label_mode, label_entries = resolve_run_label_policy(
        runs,
        label_mode=label_mode,
        compact_threshold=compact_threshold,
        index_threshold=index_threshold,
    )
    ann_mode = _resolve_annotation_mode(annotation_mode, active_label_mode)

    # --- Per-subplot loop ---
    for ym, row, col in _all_metrics:
        ax = axes[row][col]
        _leg_handles, _leg_labels_seen = [], []
        annotated_runs = set()

        for tkey, default_mk, mode_suffix in modes:
            _alias_runs = {"infer_ms_per_snap": tkey}
            for i, r in enumerate(runs):
                tm = r.get(metrics_key, {})
                if x_metric == "training_time":
                    xval = r.get("training_time", float("nan"))
                else:
                    xval = tm.get(_alias_runs.get(x_metric, x_metric), float("nan"))
                yval = tm.get(ym, float("nan"))
                xval = float(xval) if xval is not None else float("nan")
                yval = float(yval) if yval is not None else float("nan")
                if xval != xval or yval != yval:
                    continue
                color, _, mk = styles[id(r)]
                mk_use = default_mk if len(modes) > 1 else (mk or "o")
                suffix = f" ({mode_suffix})" if mode_suffix else ""
                plot_label = label_entries[i]["plot_label"]
                legend_label = label_entries[i]["legend_label"]
                sc = ax.scatter(xval, yval, color=color, marker=mk_use, s=80, zorder=3)
                if legend_label not in _leg_labels_seen:
                    _leg_handles.append(sc)
                    _leg_labels_seen.append(legend_label)
                should_annotate = (
                    ann_mode == "all"
                    or (ann_mode == "first-per-run" and i not in annotated_runs)
                )
                if should_annotate:
                    ax.annotate(
                        plot_label + suffix,
                        xy=(xval, yval),
                        xytext=(8, 6),  # clears the marker radius so the leader line is visible
                        textcoords="offset points",
                        fontsize=base_font_size * 0.75,
                        color=color,
                        arrowprops=dict(arrowstyle="-", color=color, lw=0.6, alpha=0.6, shrinkA=0, shrinkB=2, zorder=4),
                    )
                    if ann_mode == "first-per-run":
                        annotated_runs.add(i)

        # ── DC / reference baselines ─────────────────────────────────────────
        # DC has no Q model — its q_mae/line_flow_q0_mae are a naive zero-guess,
        # not a real solve. Flag those points with an X marker instead of a star.
        _DC_Q_GUESS_METRICS = {"q_mae", "q_rmse", "line_flow_q0_mae", "line_flow_q0_rmse"}
        if ref_baselines:
            for dc_label, dc_metrics in ref_baselines.items():
                if x_metric == "training_time":
                    continue
                xval_dc = float(dc_metrics.get(_alias.get(x_metric, x_metric), float("nan")))
                yval_dc = float(dc_metrics.get(ym, float("nan")))
                if xval_dc != xval_dc or yval_dc != yval_dc:
                    continue
                is_dc_q_guess = dc_label.strip().lower().startswith("dc") and ym in _DC_Q_GUESS_METRICS
                marker_dc = "X" if is_dc_q_guess else "*"
                size_dc = 200 if is_dc_q_guess else 280
                label_dc = f"{dc_label} (Q=0 guess)" if is_dc_q_guess else dc_label
                sc = ax.scatter(xval_dc, yval_dc, color="black", marker=marker_dc, s=size_dc, zorder=5)
                if label_dc not in _leg_labels_seen:
                    _leg_handles.append(sc)
                    _leg_labels_seen.append(label_dc)
                ax.annotate(
                    label_dc,
                    xy=(xval_dc, yval_dc),
                    xytext=(12, 9),  # clears the larger DC marker radius so the leader line is visible
                    textcoords="offset points",
                    fontsize=base_font_size * 0.8,
                    color="black",
                    fontweight="bold",
                    arrowprops=dict(arrowstyle="-", color="black", lw=0.6, alpha=0.6, shrinkA=0, shrinkB=2, zorder=6),
                )

        # ── Axes formatting ───────────────────────────────────────────────────
        ax.set_xlabel(x_metric_labels.get(x_metric, x_metric), fontsize=base_font_size * 1.0)
        ax.set_ylabel(y_metric_labels.get(ym, ym), fontsize=base_font_size * 1.0)
        ax.tick_params(axis="both", labelsize=base_font_size * 0.85)
        if x_axis_log:
            ax.set_xscale("log")
        ax.grid(True, alpha=0.3)
        ax.set_title(f"{title_prefix} — {y_metric_labels.get(ym, ym)}" if len(_all_metrics) > 1
                     else f"{title_prefix} — {x_metric_labels.get(x_metric, x_metric)} vs {y_metric_labels.get(ym, ym)}",
                     fontsize=base_font_size * 1.1)


    # --- Hide unused subplots ---
    _used = {(r, c) for _, r, c in _all_metrics}
    for r in range(nrows):
        for c in range(ncols):
            if (r, c) not in _used:
                axes[r][c].set_visible(False)

    # Build extra handles for reference baselines  [STSI 100626]:ref solver handles in shared legend
    from matplotlib.lines import Line2D
    extra_handles = []
    if ref_baselines:
        for ref_name in ref_baselines:
            extra_handles.append(
                Line2D([0], [0], color="black", marker="*", markersize=10,
                       linestyle="None", label=ref_name))
        if any(name.strip().lower().startswith("dc") for name in ref_baselines):
            extra_handles.append(
                Line2D([0], [0], color="black", marker="X", markersize=9,
                       linestyle="None", label="DC Q=0 guess (naive)"))
    if len(modes) > 1:
        for _mk, _ml in [("o", "per-snap"), ("D", "batched")]:  
            extra_handles.append(
                Line2D([0], [0], marker=_mk, color="gray", linestyle="None",
                       markersize=7, label=_ml))
    _add_shared_legend(fig, label_entries, styles, bottom_frac=0.12,
                       extra_handles=extra_handles, side=legend_side,
                       label_wrap=legend_label_wrap,
                       base_font_size=base_font_size)  # [STSI 100626]:shared legend via _add_shared_legend
    plt.show()


def plot_inference_time_bar(
    runs,
    title_prefix="Timing",
    label_mode="auto",
    compact_threshold=RUN_LABEL_COMPACT_THRESHOLD,
    index_threshold=RUN_LABEL_INDEX_THRESHOLD,
    metrics_key: str = "test_metrics",  # [STSI 300526]:configurable key for external test set
    base_font_size=10,
):
    """
    Bar chart of inference time per snapshot (ms) across runs.
    Reads test_metrics['inference_time_per_snapshot_ms'].
    """
    x = list(range(len(runs)))
    active_label_mode, label_entries = resolve_run_label_policy(
        runs,
        label_mode=label_mode,
        compact_threshold=compact_threshold,
        index_threshold=index_threshold,
    )
    styles = _get_run_styles(runs)
    colors = [styles[id(r)][0] for r in runs]
    tick_labels = [entry["plot_label"] for entry in label_entries]
    tick_rotation = 0 if active_label_mode == "index" else 30
    tick_ha = "center" if active_label_mode == "index" else "right"
    tick_fontsize = base_font_size * (0.9 if active_label_mode == "index" else 0.8)

    ms_vals = []
    for r in runs:
        tm = r.get(metrics_key, {})
        val = tm.get("inference_time_per_snapshot_ms", float("nan"))
        ms_vals.append(float(val) if val is not None else float("nan"))

    fig, ax = plt.subplots(figsize=(max(8, len(runs) * 1.5), 6))
    fig.suptitle(f"{title_prefix} — Inference time per snapshot [ms]", fontsize=base_font_size * 1.3)

    bars = ax.bar(x, ms_vals, alpha=0.8, color=colors)
    ax.set_xticks(x)
    ax.set_xticklabels(tick_labels, rotation=tick_rotation, ha=tick_ha, fontsize=tick_fontsize)
    ax.set_ylabel("Inference time [ms / snapshot]", fontsize=base_font_size * 1.0)
    ax.set_title("Inference time per snapshot", fontsize=base_font_size * 1.1)
    ax.tick_params(axis="y", labelsize=base_font_size * 0.85)
    ax.grid(True, axis="y", alpha=0.3)

    valid = [v for v in ms_vals if v == v]
    top = max(valid) if valid else 1.0
    for bar, val in zip(bars, ms_vals):
        if val == val:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + top * 0.02,
                f"{val:.2f}",
                ha="center", va="bottom", fontsize=base_font_size * 0.75,
            )

    if active_label_mode == "index":
        legend_lines = [
            plt.Line2D([0], [0], color=colors[i], linewidth=6, alpha=0.8,
                       label=label_entries[i]["mapping_label"])
            for i in range(len(runs))
        ]
        fig.legend(
            handles=legend_lines,
            loc="lower center",
            bbox_to_anchor=(0.5, -0.02),
            ncol=max(1, min(3, len(runs))),
            fontsize=base_font_size * 0.85,
            frameon=True,
        )
        plt.tight_layout(rect=[0, 0.12 + 0.03 * (len(runs) // 3), 1, 1])
    else:
        plt.tight_layout()
    plt.show()



In [ ]:
# =============================================================================
# SECTION 12d: CROSS-SYSTEM GENERALISATION EVALUATION
# =============================================================================

def cross_system_eval(
    runs_by_system: dict,
    networks_by_system: dict,
    metric: str = "vmag_mae",
    seed: int = 42,
):
    """
    Evaluate every run in each train-system against every test-system's networks.

    Args:
        runs_by_system:    {"ieee9": runs_list, "cigre14": runs_list, "ieee30": runs_list, ...}
        networks_by_system: {"ieee9": full_networks_list, ...}
            Uses the stored test split if runs[0]["test_networks"] exists,
            otherwise reconstructs from the full list + seed.
        metric:  which test_metrics key to compare (default "vmag_mae")
        seed:    seed for reconstructing test split when not stored

    Returns:
        results: dict[train_system][run_label][test_system] = float (metric value)
        test_nets: dict[test_system] = list of test networks used
    """
    from collections import defaultdict

    # ── Build test network sets per system ────────────────────────────────────
    test_nets = {}
    for sys_name, nets in networks_by_system.items():
        runs = runs_by_system.get(sys_name, [])
        stored = runs[0].get("test_networks") if runs else None
        if stored is not None:
            test_nets[sys_name] = stored
            print(f"  {sys_name}: using stored test split ({len(stored)} networks)")
        else:
            _, _, split = reconstruct_test_split(nets, seed=seed)
            test_nets[sys_name] = split
            print(f"  {sys_name}: reconstructed test split ({len(split)} networks)")

    # ── Evaluate every run on every test system ───────────────────────────────
    results = defaultdict(lambda: defaultdict(dict))

    for train_sys, runs in runs_by_system.items():
        _, label_entries = resolve_run_label_policy(runs)
        label_entry_by_run = _index_label_entries(label_entries)
        for r in runs:
            lbl = label_entry_by_run[id(r)]["legend_label"]

            # Skip learned_edge PTDF runs — fixed dimension, can't cross-test
            if r.get("_ptdf_params") is not None and r.get("_max_buses", 0) > 0:
                print(f"  SKIP (learned_edge PTDF): {train_sys} / {lbl}")
                continue

            model = r["model"]

            for test_sys, eval_networks in test_nets.items():
                vmag_errs, vang_errs, p_errs, q_errs = [], [], [], []
                for net in eval_networks:
                    try:
                        true_bus, _, _ = model_results(
                            net, print_results=False, plot_results=False, compute_ptdf=False
                        )
                        bus_res, _, _, _ = predict_network_results_with_masks(
                            model, net, use_edge_features=True,
                            ptdf_params=None, max_buses=0,
                        )
                    except RuntimeError:
                        continue

                    for bus in net.buses.index:
                        if (bus, "V pu") in true_bus.columns and (bus, "V pu") in bus_res.columns:
                            vmag_errs.extend(np.abs(bus_res[bus]["V pu"].values - true_bus[bus]["V pu"].values))
                        if (bus, "Angle deg") in true_bus.columns and (bus, "Angle deg") in bus_res.columns:
                            vang_errs.extend(np.abs(bus_res[bus]["Angle deg"].values - true_bus[bus]["Angle deg"].values))
                        if (bus, "P pu") in true_bus.columns and (bus, "P pu") in bus_res.columns:
                            p_errs.extend(np.abs(bus_res[bus]["P pu"].values - true_bus[bus]["P pu"].values))
                        if (bus, "Q pu") in true_bus.columns and (bus, "Q pu") in bus_res.columns:
                            q_errs.extend(np.abs(bus_res[bus]["Q pu"].values - true_bus[bus]["Q pu"].values))

                metric_val = {
                    "vmag_mae": float(np.mean(vmag_errs)) if vmag_errs else float("nan"),
                    "vang_mae": float(np.mean(vang_errs)) if vang_errs else float("nan"),
                    "p_mae": float(np.mean(p_errs)) if p_errs else float("nan"),
                    "q_mae": float(np.mean(q_errs)) if q_errs else float("nan"),
                }.get(metric, float("nan"))
                results[train_sys][lbl][test_sys] = metric_val

    return results, test_nets



In [ ]:
# ─── Edge-Delta Visualization Functions (task 029) ───────────────────────────

def plot_error_vs_hop_distance(  # [STSI 270526]:task 029
    runs,
    test_networks,
    dataset,
    metric="vang",
    max_hops=None,
    figsize=(10, 6),
    label_mode="auto",
    compact_threshold=RUN_LABEL_COMPACT_THRESHOLD,
    index_threshold=RUN_LABEL_INDEX_THRESHOLD,
):
    """
    Bin bus-level error by BFS depth from slack.
    Shows whether edge_delta models improve accuracy at distant buses.

    Parameters:
        runs: list of run_info dicts (must have 'model' key loaded)
        test_networks: list of PyPSA networks
        dataset: PowerFlowDataset for evaluation
        metric: 'vang' for angle error, 'vmag' for voltage magnitude
    """
    fig, ax = plt.subplots(figsize=figsize)
    col_idx = 1 if metric == "vang" else 0  # y[:, 1]=angle, y[:, 0]=vmag
    styles = _get_run_styles(runs)  # [STSI 100626]:shared legend via _add_shared_legend
    _, label_entries = resolve_run_label_policy(runs, label_mode=label_mode,
        compact_threshold=compact_threshold, index_threshold=index_threshold)
    label_entry_by_run = _index_label_entries(label_entries)

    for run in runs:
        model = run["model"]
        angle_mode = getattr(model, "angle_mode", "node")

        hop_errors = {}  # {hop: [errors]}
        model.eval()
        with torch.no_grad():
            for i, data in enumerate(dataset):
                # Get BFS depth
                net_idx, t_idx = dataset._index[i]
                network = test_networks[net_idx]
                slack_idx = int((data.x[:, 0] == 1).nonzero(as_tuple=True)[0][0])
                edge_fwd = data.edge_index[:, ::2].numpy()
                depths = compute_bfs_depth(edge_fwd, slack_idx, data.num_nodes)

                # Get predictions
                out = model(data)
                pred = out[0][:, col_idx].cpu().numpy()
                true = data.y[:, col_idx].cpu().numpy()
                errors = np.abs(pred - true)

                # For edge_delta, reconstruct θ if metric is vang
                if angle_mode == "edge_delta" and metric == "vang":
                    delta_theta_pred = out[2] if len(out) > 2 else None
                    if delta_theta_pred is not None:
                        vang_recon = reconstruct_theta_from_delta(
                            delta_theta_pred, data.bfs_edge_idx,
                            data.bfs_signs, data.bfs_node_order,
                            data.bfs_parent, data.x.size(0)
                        ).cpu().numpy()
                        errors = np.abs(vang_recon - true)

                for node_idx in range(data.num_nodes):
                    d = depths[node_idx]
                    if d >= 0:
                        hop_errors.setdefault(d, []).append(errors[node_idx])

        # Plot mean error per hop
        hops = sorted(hop_errors.keys())
        if max_hops:
            hops = [h for h in hops if h <= max_hops]
        means = [np.mean(hop_errors[h]) for h in hops]
        color, _, mk = styles[id(run)]  # [STSI 100626]:use run-style color for consistency
        ax.plot(hops, means, marker=mk or "o", color=color)

    ax.set_xlabel("BFS Hop Distance from Slack")
    ax.set_ylabel(f"Mean |{metric}| Error")
    ax.set_title(f"{metric.upper()} Error vs Distance from Slack Bus")
    ax.grid(True, alpha=0.3)
    _add_shared_legend(fig, label_entries, styles, bottom_frac=0.12)  # [STSI 100626]:shared legend via _add_shared_legend
    plt.show()
    return fig


def plot_delta_theta_comparison_bar(  # [STSI 270526]:task 029
    runs,
    figsize=(10, 6),
    label_mode="auto",
    compact_threshold=RUN_LABEL_COMPACT_THRESHOLD,
    index_threshold=RUN_LABEL_INDEX_THRESHOLD,
):
    """
    Grouped bar chart comparing angle accuracy metrics across runs:
    - vang_mae (node-level, absolute angle)
    - delta_theta_mae (edge-level, direct Δθ — only for edge_delta runs)

    Highlights the difference between direct edge accuracy and
    accumulated node-level accuracy.
    """
    styles = _get_run_styles(runs)  # [STSI 100626]:shared legend via _add_shared_legend
    _, label_entries = resolve_run_label_policy(runs, label_mode=label_mode,
        compact_threshold=compact_threshold, index_threshold=index_threshold)
    label_entry_by_run = _index_label_entries(label_entries)

    fig, ax = plt.subplots(figsize=figsize)

    tick_labels = []
    vang_vals = []
    dt_vals = []

    for run in runs:
        tm = run.get("test_metrics", {})
        tick_labels.append(label_entry_by_run[id(run)]["short_label"])  # [STSI 100626]:shared legend via _add_shared_legend
        vang_vals.append(tm.get("vang_mae", np.nan))
        dt_vals.append(tm.get("delta_theta_mae", np.nan))

    x = np.arange(len(tick_labels))
    width = 0.35

    ax.bar(x - width/2, vang_vals, width, label="θ MAE (node-level)", color="steelblue")
    ax.bar(x + width/2, dt_vals, width, label="Δθ MAE (edge-level)", color="coral")

    ax.set_xlabel("Run")
    ax.set_ylabel("MAE (radians)")
    ax.set_title("Angle Accuracy: Node θ vs Edge Δθ")
    ax.set_xticks(x)
    ax.set_xticklabels(tick_labels, rotation=0, ha="center", fontsize=9)
    ax.grid(True, alpha=0.3, axis="y")

    # Category patches + mapping  [STSI 100626]:category patches + run mapping in shared legend
    from matplotlib.patches import Patch
    extra_handles = [
        Patch(facecolor="steelblue", alpha=0.8, label="θ MAE (node-level)"),
        Patch(facecolor="coral", alpha=0.8, label="Δθ MAE (edge-level)"),
    ]
    _add_shared_legend(fig, label_entries, styles, extra_handles=extra_handles,
                       legend_key="mapping_label")  # [STSI 100626]:shared legend via _add_shared_legend
    plt.show()
    return fig


def plot_timing_breakdown_stacked(  # [STSI 270526]:task 029
    runs,
    figsize=(10, 6),
    label_mode="auto",
    compact_threshold=RUN_LABEL_COMPACT_THRESHOLD,
    index_threshold=RUN_LABEL_INDEX_THRESHOLD,
):
    """
    Stacked horizontal bar showing timing breakdown per run:
    - solve_ms (GNN forward pass)
    - theta_recon_ms (BFS θ reconstruction, 0 for node mode)
    - postproc_ms (mask application + line flow)
    """
    styles = _get_run_styles(runs)  # [STSI 100626]:shared legend via _add_shared_legend
    _, label_entries = resolve_run_label_policy(runs, label_mode=label_mode,
        compact_threshold=compact_threshold, index_threshold=index_threshold)
    label_entry_by_run = _index_label_entries(label_entries)

    fig, ax = plt.subplots(figsize=figsize)

    tick_labels = []
    solve_times = []
    recon_times = []
    postproc_times = []

    for run in runs:
        tm = run.get("test_metrics", {})
        tick_labels.append(label_entry_by_run[id(run)]["short_label"])  # [STSI 100626]:shared legend via _add_shared_legend

        # These are independently measured timings
        solve = tm.get("solve_time_ms", 0) or 0
        recon = tm.get("theta_reconstruction_time_ms", 0) or 0
        postproc = tm.get("postproc_time_ms", 0) or 0

        solve_times.append(solve)
        recon_times.append(recon)
        postproc_times.append(postproc)

    y = np.arange(len(tick_labels))

    ax.barh(y, solve_times, color="steelblue")
    ax.barh(y, recon_times, left=solve_times, color="coral")
    lefts = np.array(solve_times) + np.array(recon_times)
    ax.barh(y, postproc_times, left=lefts, color="seagreen")

    ax.set_yticks(y)
    ax.set_yticklabels(tick_labels, fontsize=9)
    ax.set_xlabel("Time (ms per snapshot)")
    ax.set_title("Inference Timing Breakdown")
    ax.grid(True, alpha=0.3, axis="x")

    # Category patches + run mapping  [STSI 100626]:category patches + run mapping in shared legend
    from matplotlib.patches import Patch
    extra_handles = [
        Patch(facecolor="steelblue", alpha=0.8, label="GNN Forward"),
        Patch(facecolor="coral", alpha=0.8, label="θ Reconstruction"),
        Patch(facecolor="seagreen", alpha=0.8, label="Post-processing"),
    ]
    _add_shared_legend(fig, label_entries, styles, extra_handles=extra_handles,
                       legend_key="mapping_label")  # [STSI 100626]:shared legend via _add_shared_legend
    plt.show()
    return fig


def plot_line_flow_accuracy_comparison(  # [STSI 270526]:task 029
    runs,
    figsize=(10, 6),
    label_mode="auto",
    compact_threshold=RUN_LABEL_COMPACT_THRESHOLD,
    index_threshold=RUN_LABEL_INDEX_THRESHOLD,
):
    """
    Bar chart comparing line flow P0 MAE across runs.
    For edge_delta runs (task 028): uses direct Δθ line flows.
    For node runs: uses θ-based line flows.

    Requires test_metrics['line_flow_mae'] to be populated (from evaluate_gnn_on_test_set).
    """
    from matplotlib.patches import Patch

    styles = _get_run_styles(runs)  # [STSI 100626]:shared legend via _add_shared_legend
    _, label_entries = resolve_run_label_policy(runs, label_mode=label_mode,
        compact_threshold=compact_threshold, index_threshold=index_threshold)
    label_entry_by_run = _index_label_entries(label_entries)

    fig, ax = plt.subplots(figsize=figsize)

    tick_labels = []
    lf_maes = []
    bar_colors = []

    for run in runs:
        tm = run.get("test_metrics", {})
        am = run.get("angle_mode", "node")
        tick_labels.append(label_entry_by_run[id(run)]["short_label"])  # [STSI 100626]:shared legend via _add_shared_legend
        lf_maes.append(tm.get("line_flow_mae", np.nan))
        bar_colors.append("coral" if am == "edge_delta" else "steelblue")

    x = np.arange(len(tick_labels))
    ax.bar(x, lf_maes, color=bar_colors)

    ax.set_xlabel("Run")
    ax.set_ylabel("Line Flow P0 MAE (p.u.)")
    ax.set_title("Line Flow Accuracy: Direct Δθ vs Node θ")
    ax.set_xticks(x)
    ax.set_xticklabels(tick_labels, rotation=0, ha="center", fontsize=9)
    ax.grid(True, alpha=0.3, axis="y")

    # Category patches + mapping  [STSI 100626]:category patches + run mapping in shared legend
    extra_handles = [
        Patch(facecolor="steelblue", alpha=0.8, label="Node mode (θ→flows)"),
        Patch(facecolor="coral", alpha=0.8, label="Edge-delta (Δθ→flows)"),
    ]
    _add_shared_legend(fig, label_entries, styles, extra_handles=extra_handles,
                       legend_key="mapping_label")  # [STSI 100626]:shared legend via _add_shared_legend
    plt.show()
    return fig


In [ ]:
def print_top_runs(
    runs,
    metrics_key="test_metrics_external",
    top_n=5,
    rank_metrics=None,
    exclude_keys=None,
):
    """
    Print top-N runs per metric and overall best run by rank-sum.

    Parameters
    ----------
    runs         : list of run dicts
    metrics_key  : which metrics dict to read from each run
    top_n        : number of top runs to show per metric
    rank_metrics : list of metric keys for overall rank-sum (lower=better)
    exclude_keys : set of run dict keys to suppress in output
    """
    import math

    if exclude_keys is None:
        exclude_keys = {
            "history", "test_metrics_external", "test_metrics", "model",
            "_ptdf_params", "_source_pkl", "run_key", "physics_mode",
            "weight_physics", "weight_ptdf", "use_ptdf_loss_flag", "max_w_flow",
            "flow_target", "y_matrix_source", "use_pnom_share", "use_global_pool",
            "node_feature_mode", "use_residual", "norm_type", "activation",
            "use_dropout", "drop_rate", "seed", "bus_count_min", "bus_count_max",
            "_max_buses", "loss_weight_mode", "ptdf_alpha", "ptdf_branch_mode",
            "head_mode", "vmag_mode", "conv_type", "hidden_dim", "num_layers",
            "batch_size", "lr", "num_scenarios",
        }

    if rank_metrics is None:
        rank_metrics = [
            "vmag_mae", "vang_mae", "delta_theta_mae",
            "p_mae", "q_mae", "line_flow_mae", "line_flow_q0_mae",
        ]

    def _frac_str(r, frac_key, start_key, stop_key):
        frac = r.get(frac_key)
        if frac is None or frac == 0:
            return None
        start = r.get(start_key, "?")
        stop  = r.get(stop_key, "inf") or "inf"
        return f"{frac}({start}-{stop})"

    def _print_run(rank_label, val_label, val, idx, r):
        fp_s  = _frac_str(r, "fraction_physics",   "physics_activation_start",    "physics_activation_stop")
        ft_s  = _frac_str(r, "fraction_ptdf",       "ptdf_activation_start",       "ptdf_activation_stop")
        dcL_s = _frac_str(r, "fraction_dcf_local",  "dcf_local_activation_start",  "dcf_local_activation_stop")
        dcG_s = _frac_str(r, "fraction_dcf_global", "dcf_global_activation_start", "dcf_global_activation_stop")
        acL_s = _frac_str(r, "fraction_acf_local",  "acf_local_activation_start",  "acf_local_activation_stop")
        acG_s = _frac_str(r, "fraction_acf_global", "acf_global_activation_start", "acf_global_activation_stop")
        fracs = "  ".join(f"{k}={v}" for k, v in [
            ("fp", fp_s), ("ft", ft_s), ("dcL", dcL_s), ("dcG", dcG_s),
            ("acL", acL_s), ("acG", acG_s)
        ] if v is not None)
        print(f"  {rank_label} | Idx {idx:3d} | {val_label} = {val:.6g}")
        print(f"           tag={r.get('tag')}  am={r.get('angle_mode')}  ramp={r.get('activation_ramp_epochs')}")
        if fracs:
            print(f"           {fracs}")
        print(f"           train={r.get('final_train_loss', float('nan')):.4f}  val={r.get('final_val_loss', float('nan')):.4f}")

    # Collect all scalar metrics from the chosen key
    all_metrics = set()
    for r in runs:
        for k, v in (r.get(metrics_key) or {}).items():
            if isinstance(v, (int, float)) and not isinstance(v, bool) and not math.isnan(v):
                all_metrics.add(k)

    # Per-metric top-N
    metric_scores = {}
    for metric in sorted(all_metrics):
        scored = []
        for idx, r in enumerate(runs):
            val = (r.get(metrics_key) or {}).get(metric)
            if val is not None and isinstance(val, (int, float)) and not math.isnan(val):
                scored.append((val, idx, r))
        if not scored:
            continue
        metric_scores[metric] = scored
        scored.sort(key=lambda x: x[0])
        print(f"\n{'='*70}")
        print(f"Metric: {metric}  (top {min(top_n, len(scored))} runs, key='{metrics_key}')")
        print(f"{'='*70}")
        for rank, (val, idx, r) in enumerate(scored[:top_n], 1):
            _print_run(f"Rank {rank:2d}", metric, val, idx, r)

    # Overall best run (rank-sum across rank_metrics)
    rank_sum   = {}
    rank_count = {}
    for metric in rank_metrics:
        if metric not in metric_scores:
            continue
        for rank_pos, (val, idx, r) in enumerate(sorted(metric_scores[metric], key=lambda x: x[0]), 1):
            rank_sum[idx]   = rank_sum.get(idx, 0) + rank_pos
            rank_count[idx] = rank_count.get(idx, 0) + 1

    n_covered = sum(1 for m in rank_metrics if m in metric_scores)
    full_coverage = {idx for idx, cnt in rank_count.items() if cnt == n_covered}

    if full_coverage:
        best_idx = min(full_coverage, key=lambda i: rank_sum[i])
    elif rank_sum:
        best_idx = min(rank_sum, key=lambda i: rank_sum[i] / max(rank_count.get(i, 1), 1))
    else:
        print("\n[No runs with rank metrics available]")
        return

    best_r   = runs[best_idx]
    best_tm  = (best_r.get(metrics_key) or {})
    avg_rank = rank_sum[best_idx] / max(rank_count.get(best_idx, 1), 1)

    print(f"\n{'='*70}")
    print(f"OVERALL BEST RUN  (lowest avg rank across {rank_metrics})")
    print(f"{'='*70}")
    _print_run("Best  ", f"avg_rank={avg_rank:.1f}", avg_rank, best_idx, best_r)
    print(f"           Per-metric values:")
    for m in rank_metrics:
        v = best_tm.get(m)
        if v is not None and not math.isnan(v):
            if m in metric_scores:
                sorted_m = sorted(metric_scores[m], key=lambda x: x[0])
                rank_pos = next((i+1 for i, (_, idx, _) in enumerate(sorted_m) if idx == best_idx), "?")
                print(f"             {m:25s}: {v:.6g}  (rank {rank_pos}/{len(sorted_m)})")



In [ ]:
def get_top_runs(
    runs,
    metrics_key="test_metrics_external",
    top_n=5,
    rank_metrics=None,
    require_full_coverage=True,
    always_include_idx=None,
    include_worst_n=None,
    selection_mode="composite",
):
    """
    Return selected runs ranked by average rank across rank_metrics (lower is better).

    ``average_rank`` is the mean ordinal rank across the available metrics. It is
    not an MAE/RMSE value: 1.0 means first place on every included metric, 2.0
    means second place on average, and so on.

    Returned runs also contain explicit selection metadata:
    ``selection_type`` (``single_metric`` or ``composite``), ``selected_metric``,
    ``selected_metric_value`` (the actual value from ``metrics_key`` for a
    single-metric selection), ``selected_metric_rank``, and
    ``selected_metric_count``. Every returned run also contains ``metric_values``,
    ``metric_ranks``, and ``metric_counts`` dictionaries for all available
    metrics. Composite selections use ``selected_metric`` = ``"composite"``;
    their complete metric details are provided by those dictionaries.
    """
    import math

    runs = [dict(r, _orig_index=r.get("_orig_index", idx + 1)) for idx, r in enumerate(runs)]

    if rank_metrics is None:
        rank_metrics = [
            "vmag_mae", "vang_mae", "delta_theta_mae",
            "p_mae", "q_mae", "line_flow_mae", "line_flow_q0_mae",
        ]

    metric_scores = {}
    for metric in rank_metrics:
        scored = []
        for idx, run in enumerate(runs):
            value = (run.get(metrics_key) or {}).get(metric)
            if value is not None and isinstance(value, (int, float)) and not math.isnan(value):
                scored.append((value, idx))
        if scored:
            metric_scores[metric] = sorted(scored, key=lambda item: item[0])

    n_covered = sum(1 for metric in rank_metrics if metric in metric_scores)
    rank_sum = {}
    rank_count = {}
    for metric in rank_metrics:
        if metric not in metric_scores:
            continue
        for rank_position, (_, idx) in enumerate(metric_scores[metric], 1):
            rank_sum[idx] = rank_sum.get(idx, 0) + rank_position
            rank_count[idx] = rank_count.get(idx, 0) + 1

    if not rank_sum:
        return []

    if require_full_coverage:
        eligible = {idx for idx, count in rank_count.items() if count == n_covered}
    else:
        eligible = set(rank_sum.keys())
    if not eligible:
        eligible = set(rank_sum.keys())

    for idx in rank_sum:
        runs[idx]["average_rank"] = rank_sum[idx] / max(rank_count.get(idx, 1), 1)

    # Keep the actual value and ordinal rank for every ranked metric on each
    # returned run. This makes composite selections inspectable in a DataFrame.
    for idx in rank_sum:
        metric_values = {}
        metric_ranks = {}
        metric_counts = {}
        for metric, scored in metric_scores.items():
            for rank_position, (value, scored_idx) in enumerate(scored, 1):
                if scored_idx == idx:
                    metric_values[metric] = value
                    metric_ranks[metric] = rank_position
                    metric_counts[metric] = len(scored)
                    break
        runs[idx]["metric_values"] = metric_values
        runs[idx]["metric_ranks"] = metric_ranks
        runs[idx]["metric_counts"] = metric_counts

    composite_sorted = sorted(
        eligible,
        key=lambda idx: rank_sum[idx] / max(rank_count.get(idx, 1), 1),
    )

    selected_indices = []
    selection_metadata = {}

    def add_selection(idx, selection_type, metric=None):
        if idx in selected_indices:
            return
        selected_indices.append(idx)
        if selection_type == "single_metric":
            scored = metric_scores[metric]
            value, _ = scored[0]
            rank = next(
                rank_position
                for rank_position, (_, scored_idx) in enumerate(scored, 1)
                if scored_idx == idx
            )
            selection_metadata[idx] = {
                "selection_type": "single_metric",
                "selected_metric": metric,
                "selected_metric_value": value,
                "selected_metric_rank": rank,
                "selected_metric_count": len(scored),
            }
        else:
            selection_metadata[idx] = {
                "selection_type": "composite",
                "selected_metric": "composite",
                "selected_metric_value": runs[idx].get("metric_values", {}),
                "selected_metric_rank": runs[idx].get("metric_ranks", {}),
                "selected_metric_count": runs[idx].get("metric_counts", {}),
            }

    if selection_mode == "composite":
        for idx in composite_sorted[:top_n]:
            add_selection(idx, "composite")

    elif selection_mode == "single":
        for metric in rank_metrics:
            if metric in metric_scores:
                add_selection(metric_scores[metric][0][1], "single_metric", metric)
            if len(selected_indices) >= top_n:
                break

    elif selection_mode == "both":
        # Distribute up to top_n single-metric slots round-robin across the
        # requested metrics, then append the composite winner as one extra row.
        composite_best = composite_sorted[0] if composite_sorted else None
        metric_order = [metric for metric in rank_metrics if metric in metric_scores]
        max_metric_depth = max(
            (len(metric_scores[metric]) for metric in metric_order),
            default=0,
        )
        for rank_offset in range(max_metric_depth):
            for metric in metric_order:
                if len(selected_indices) >= top_n:
                    break
                scored = metric_scores[metric]
                if rank_offset >= len(scored):
                    continue
                candidate_idx = scored[rank_offset][1]
                if candidate_idx != composite_best:
                    add_selection(candidate_idx, "single_metric", metric)
            if len(selected_indices) >= top_n:
                break
        if composite_best is not None:
            add_selection(composite_best, "composite")
    else:
        raise ValueError(
            f"Unknown selection_mode: {selection_mode!r} "
            "(expected 'composite', 'single', or 'both')"
        )

    if always_include_idx:
        for idx in always_include_idx:
            if isinstance(idx, int) and 0 <= idx < len(runs) and idx not in selected_indices:
                add_selection(idx, "composite")

    if include_worst_n:
        worst_indices = sorted(
            eligible,
            key=lambda idx: rank_sum[idx] / max(rank_count.get(idx, 1), 1),
            reverse=True,
        )
        for idx in worst_indices[:include_worst_n]:
            add_selection(idx, "composite")

    returned_runs = []
    for idx in selected_indices:
        run = runs[idx]
        run.update(selection_metadata[idx])
        returned_runs.append(run)
    return returned_runs


In [ ]:

#Plotting utilities

class pdf_output:
    """Context manager: save every plt.show() figure as a separate PDF in pdf_dir.

    Wraps the block with a patched plt.show() that captures each figure and
    writes it to pdf_dir/{prefix}_{N:02d}.pdf before displaying inline.
    No changes to individual plot functions are required.

    Example — standalone:
        with pdf_output(os.path.join(TRAINING_RESULTS_DIR, "plots"), prefix="mixed"):
            fig1 = plot_topology_stats(all_networks)
            plt.show()
            fig2 = plot_electrical_params(all_networks)
            plt.show()
    """

    def __init__(self, pdf_dir: str, prefix: str = ""):
        self._pdf_dir = pdf_dir
        self._prefix = _sanitise_filename(prefix)
        self._counter = 0
        self._orig_show = None
        self._orig_fonttypes = None

    def __enter__(self):
        os.makedirs(self._pdf_dir, exist_ok=True)
        self._orig_show = plt.show
        # [STSI 070926]:embed real (selectable/editable) text in PDF instead of vector-outlined glyphs
        self._orig_fonttypes = (plt.rcParams["pdf.fonttype"], plt.rcParams["ps.fonttype"])
        plt.rcParams["pdf.fonttype"] = 42
        plt.rcParams["ps.fonttype"] = 42
        _self = self

        def _saving_show(*args, **kwargs):
            _fig = plt.gcf()
            if _fig.get_axes():
                _self._counter += 1
                _fname = (
                    f"{_self._prefix}_{_self._counter:02d}.pdf"
                    if _self._prefix
                    else f"plot_{_self._counter:02d}.pdf"
                )
                _fig.savefig(
                    os.path.join(_self._pdf_dir, _fname), bbox_inches="tight"
                )
                print(f"  [PDF] Saved: {_fname}")
            _self._orig_show(*args, **kwargs)

        plt.show = _saving_show
        return self

    def __exit__(self, *_):
        if self._orig_show is not None:
            plt.show = self._orig_show
        if self._orig_fonttypes is not None:
            plt.rcParams["pdf.fonttype"], plt.rcParams["ps.fonttype"] = self._orig_fonttypes


In [ ]:
# ==== UQ SPLIT HELPERS (task 105) — function-level copies of the canonical Training split helpers (cell 15) ====
# [STSI 231026]:new cell (task 105) - split/manifest helpers copied verbatim (make_checkpoint_path / post_curriculum_start not needed)
import hashlib as _hashlib, json as _json
# [COPY of GNN_Powerflow_V2.7_Training::SPLIT_PCTS_4WAY]
SPLIT_PCTS_4WAY   = (70, 10, 10, 10)   # train / val / cal / test
SPLIT_PCTS_LEGACY = (70, 15, 15)       # train / val / test (pre-UQ runs)
_SPLIT_ROLES = {3: ("train", "val", "test"), 4: ("train", "val", "cal", "test")}

# [COPY of GNN_Powerflow_V2.7_Training::_split_counts]
def _split_counts(n, split_pcts, warn=True):
    """Per-role counts from integer-percentage boundaries -> (counts, fallback_applied).

    Small-n fallback (user decision C, 2026-10-23): ONLY when the integer boundaries leave a role empty
    (tiny test datasets), the missing networks are taken one by one from the currently largest role (train),
    so every role gets >= 1 network. Boundaries are unchanged whenever every role is already non-empty.
    """
    split_pcts = tuple(int(p) for p in split_pcts)
    assert sum(split_pcts) == 100 and len(split_pcts) in _SPLIT_ROLES, split_pcts
    roles = _SPLIT_ROLES[len(split_pcts)]
    cuts, acc = [0], 0
    for p in split_pcts[:-1]:
        acc += p
        cuts.append(n * acc // 100)                # integer arithmetic, no float drift
    cuts.append(n)
    if warn and cuts[1] != int(0.7 * n) and split_pcts[0] == 70:
        print(f"[WARN] train boundary {cuts[1]} != legacy int(0.7*n)={int(0.7*n)}")
    counts = [cuts[i + 1] - cuts[i] for i in range(len(roles))]
    if min(counts) > 0:
        return counts, False
    if n < len(roles):
        raise ValueError(f"split {split_pcts} needs at least {len(roles)} networks (one per role {roles}), got n={n}")
    raw = list(counts)
    for i in range(len(counts)):
        if counts[i] == 0:
            j = max(range(len(counts)), key=lambda k: counts[k])   # largest role (train); first on ties
            counts[j] -= 1
            counts[i] = 1
    if warn:
        print(f"[WARN] split fallback: n={n}, split_pcts={split_pcts} leaves an empty role {dict(zip(roles, raw))}; "
              f"using {dict(zip(roles, counts))} (missing networks taken from the largest role, same permutation)")
    return counts, True

# [COPY of GNN_Powerflow_V2.7_Training::split_positions]
def split_positions(n, split_seed, split_pcts=SPLIT_PCTS_4WAY):
    """Role -> positions into the ordered network list. Uses a LOCAL RNG only."""
    split_pcts = tuple(int(p) for p in split_pcts)
    counts, _ = _split_counts(n, split_pcts)
    perm = list(range(n))
    random.Random(split_seed).shuffle(perm)        # same permutation as random.seed(s); random.shuffle(...)
    roles = _SPLIT_ROLES[len(split_pcts)]
    out, start = {}, 0
    for r, c in zip(roles, counts):                # consecutive slices of the same permutation
        out[r] = perm[start:start + c]
        start += c
    return out

# [COPY of GNN_Powerflow_V2.7_Training::networks_fingerprint]
def networks_fingerprint(networks):
    sig = [(len(n.buses), len(n.lines), len(n.snapshots)) for n in networks]
    return _hashlib.sha1(_json.dumps(sig).encode()).hexdigest()[:12]

# [COPY of GNN_Powerflow_V2.7_Training::make_split_manifest]
def make_split_manifest(networks, split_seed, split_pcts=SPLIT_PCTS_4WAY, dataset_source=None):
    positions = split_positions(len(networks), split_seed, split_pcts)
    core = {"n": len(networks), "split_seed": int(split_seed), "split_pcts": [int(p) for p in split_pcts],
            "positions": positions,
            "fingerprint": networks_fingerprint(networks)}
    _, _fallback = _split_counts(len(networks), split_pcts, warn=False)
    return {**core, "version": 1, "dataset_source": dataset_source,
            "split_fallback": _fallback, "role_counts": {r: len(p) for r, p in positions.items()},
            "hash": _hashlib.sha1(_json.dumps(core, sort_keys=True).encode()).hexdigest()[:12]}

# [COPY of GNN_Powerflow_V2.7_Training::networks_for_role]
def networks_for_role(networks, manifest, role):
    assert len(networks) == manifest["n"] and networks_fingerprint(networks) == manifest["fingerprint"], \
        "network list does not match the split manifest (different file or order)"
    return [networks[p] for p in manifest["positions"][role]]

In [ ]:
# ==== UQ COMMON LIBRARY (task 105) — ORIGIN. Trainers copy defs as [COPY of GNN_Powerflow_V2.8_UQ_Analysis::<name>] ====
# [STSI 231026]:new cell (task 105) - P6 flat-row prediction contract, path matrix, streaming forward, scores, timing, [PROP-2] line flows
import json, math, time
UQ_LEVELS = (0.80, 0.90, 0.95)
UQ_PRIMARY_LEVEL = 0.90
STATE_COLS = ("vmag", "theta", "p", "q")                       # contract column order (θ in rad)
BUS_TYPE_CODE = {"PQ": 0, "PV": 1, "slack": 2}
UQ_STRATA = (("vmag", "PQ"), ("theta", "PQ"), ("theta", "PV"), ("q", "PV"), ("p", "slack"), ("q", "slack"))
UQ_UNITS = {"vmag": "pu", "theta": "rad", "p": "pu", "q": "pu", "line": "pu"}   # [STSI 231026]:default metadata["units"]

def path_matrix_from_data(data):
    """[N, E_fwd] path-incidence matrix (θ = P @ Δθ_fwd); reuses data.bfs_path_matrix when present."""
    if hasattr(data, "bfs_path_matrix"):
        return data.bfs_path_matrix
    n, e = data.x.size(0), data.edge_index[:, ::2].size(1)
    P = torch.zeros(n, e)
    for bi in range(len(data.bfs_node_order)):        # BFS order: parents before children (Analysis 1361-1369)
        c, p = int(data.bfs_node_order[bi]), int(data.bfs_parent[bi])
        P[c] = P[p].clone(); P[c, int(data.bfs_edge_idx[bi])] = -float(data.bfs_signs[bi])
    return P

def unknown_mask_from_data(data):
    m = torch.zeros(data.x.size(0), 4, dtype=torch.bool)
    pq, pv, sl = data.pq_mask.bool(), data.pv_mask.bool(), data.slack_mask.bool()
    m[pq, 0] = m[pq, 1] = True; m[pv, 1] = m[pv, 3] = True; m[sl, 2] = m[sl, 3] = True
    return m

def _pad_edge_attr_for_model(model, data):
    """Zero-pad data.edge_attr to the edge width of the model's first conv (fallback factored out of
    predict_network_results_with_masks; the original is unchanged)."""
    if hasattr(model, 'convs') and model.convs and hasattr(model.convs[0], 'lin_edge') and model.convs[0].lin_edge is not None:
        _expected = model.convs[0].lin_edge.in_channels
        if data.edge_attr.size(1) < _expected:
            data.edge_attr = torch.cat([data.edge_attr, torch.zeros(data.edge_attr.size(0), _expected - data.edge_attr.size(1))], dim=1)
    return data

def uq_forward_snapshots(model, networks, net_ids=None, use_pnom_share=None, keep_embeddings=False):
    """Yield one dict per snapshot from the eval-mode embeddings path (PTDF skipped). edge_delta only."""
    assert getattr(model, "angle_mode", "node") == "edge_delta", "UQ path supports angle_mode='edge_delta' only"
    model.eval()
    ds = PowerFlowDataset(networks, use_edge_features=True, ptdf_branch_mode="all", angle_mode="edge_delta",
                          use_pnom_share=(model.node_embedding.in_features >= 8) if use_pnom_share is None else use_pnom_share)
    for i in range(len(ds)):
        net_idx, t_idx = ds._index[i]
        data = ds[i]
        _pad_edge_attr_for_model(model, data)          # same zero-pad fallback as predict_network_results_with_masks
        with torch.no_grad():
            node_pred, h_nodes, h_edges, dtp = model(data, return_embeddings=True)
        P = path_matrix_from_data(data)
        unk = unknown_mask_from_data(data)
        mean = data.y.clone().float()
        pred4 = torch.stack([node_pred[:, 0], P @ dtp, node_pred[:, 1], node_pred[:, 2]], dim=1)
        mean[unk] = pred4[unk]
        bt = torch.full((data.x.size(0),), 0, dtype=torch.int8)
        bt[data.pv_mask.bool()] = 1; bt[data.slack_mask.bool()] = 2
        rec = dict(net=(net_ids[net_idx] if net_ids is not None else net_idx), snap=t_idx, data=data, P=P,
                   mean=mean, y=data.y.float(), unknown=unk, bus_type=bt, node_pred=node_pred,
                   dt_mean=dtp, dt_true=data.y_delta_theta,
                   line_limit=line_limits_for(networks[net_idx], int(data.dc_flow_mask[::2].sum())))   # [PROP-2]
        if keep_embeddings:
            rec.update(h_nodes=h_nodes, h_edges_fwd=h_edges[::2])
        yield rec

def _uq_np(x, dtype=None):
    """torch / array-like -> numpy (optionally cast); None stays None."""
    if x is None:
        return None
    if torch.is_tensor(x):
        x = x.detach().cpu().numpy()
    return np.asarray(x) if dtype is None else np.asarray(x, dtype=dtype)

def _uq_cat(parts, axis=0, what=""):
    """Concatenate optional per-snapshot arrays: None if every part is None, error if only some are."""
    n_none = sum(p is None for p in parts)
    if n_none == len(parts):
        return None
    assert n_none == 0, f"{what}: given for {len(parts) - n_none} of {len(parts)} snapshots (all or none)"
    return np.concatenate(parts, axis=axis)

def _uq_cat_extra(dicts, what="extra"):
    keys = set(dicts[0])
    assert all(set(d) == keys for d in dicts), f"{what}: keys differ between snapshots"
    return {k: np.concatenate([d[k] for d in dicts], axis=0) for k in sorted(keys)}

class UQContractBuilder:
    """Accumulate per-snapshot records into the flat-row P6 contract."""
    def __init__(self): self._r, self._e, self._l = [], [], []

    def add(self, rec, std=None, dt_std=None, samples=None, dt_samples=None, extra=None, line=None):
        """rec: one uq_forward_snapshots record. std [N,4] / samples [M,N,4] / extra {k: [N,...]} in physical units;
        dt_std [E_fwd] / dt_samples [M,E_fwd]; line: dict from line_block(...) ([PROP-2]); rows get net/snap from rec."""
        unk = _uq_np(rec["unknown"], bool)
        y = _uq_np(rec["y"], np.float32)
        n = unk.shape[0]
        std = _uq_np(std, np.float32)
        if std is not None:
            assert std.shape == (n, 4), f"std shape {std.shape} != {(n, 4)}"
            std = np.where(unk, std, 0.0).astype(np.float32)          # known coordinates carry std 0
        samples = _uq_np(samples, np.float32)
        if samples is not None:
            assert samples.ndim == 3 and samples.shape[1:] == (n, 4), f"samples shape {samples.shape}"
            samples = np.where(unk[None], samples, y[None]).astype(np.float32)   # known coordinates hold the truth input
        ext = {k: _uq_np(v) for k, v in (extra or {}).items()}
        for k, v in ext.items():
            assert v.shape[0] == n, f"extra[{k!r}] has {v.shape[0]} rows, expected {n} (row-aligned)"
        net, snap = int(rec["net"]), int(rec["snap"])
        self._r.append(dict(mean=_uq_np(rec["mean"], np.float32), y=y, unknown=unk, std=std,
                            bus_type=_uq_np(rec["bus_type"], np.int8), net=np.full(n, net, np.int32),
                            snap=np.full(n, snap, np.int32), samples=samples, extra=ext))
        dtm = _uq_np(rec["dt_mean"], np.float32)
        ne = dtm.shape[0]
        dts, dtsm = _uq_np(dt_std, np.float32), _uq_np(dt_samples, np.float32)
        assert dts is None or dts.shape == (ne,), f"dt_std shape {dts.shape} != {(ne,)}"
        assert dtsm is None or (dtsm.ndim == 2 and dtsm.shape[1] == ne), f"dt_samples shape {dtsm.shape}"
        self._e.append(dict(dt_mean=dtm, dt_true=_uq_np(rec["dt_true"], np.float32), dt_std=dts, dt_samples=dtsm,
                            net=np.full(ne, net, np.int32), snap=np.full(ne, snap, np.int32)))
        if line is None:                                                 # [PROP-2] line section (optional, all or none)
            self._l.append(None)
            return
        lm = _uq_np(line["mean"], np.float32)
        nl = lm.shape[0]
        assert lm.shape == (nl, len(LINE_QTYS)), f"line mean shape {lm.shape}"
        ls, lsm = _uq_np(line.get("std"), np.float32), _uq_np(line.get("samples"), np.float32)
        assert ls is None or ls.shape == lm.shape, f"line std shape {ls.shape}"
        assert lsm is None or (lsm.ndim == 3 and lsm.shape[1:] == lm.shape), f"line samples shape {lsm.shape}"
        lext = {k: _uq_np(v) for k, v in (line.get("extra") or {}).items()}
        for k, v in lext.items():
            assert v.shape[0] == nl, f"line extra[{k!r}] has {v.shape[0]} rows, expected {nl} (row-aligned)"
        self._l.append(dict(mean=lm, y=_uq_np(line["y"], np.float32), std=ls, samples=lsm, extra=lext,
                            net=np.full(nl, net, np.int32), snap=np.full(nl, snap, np.int32),
                            edge_pos=_uq_np(line["edge_pos"], np.int32), limit=_uq_np(line["limit"], np.float32)))

    def build(self, metadata):
        """Concatenate to numpy float32/bool/int arrays; std/samples=None if never given; line=None if no add passed it."""
        assert self._r, "UQContractBuilder.build: no records added"
        md = {"units": dict(UQ_UNITS), "angle_unit": "rad", "state_cols": list(STATE_COLS), "line_qtys": list(LINE_QTYS),
              **metadata}
        md = json.loads(json.dumps(md))                                 # must be JSON-serialisable (tuples -> lists)
        R, E = self._r, self._e
        cat = lambda rows, k, axis=0: _uq_cat([r[k] for r in rows], axis, k)
        c = dict(mean=cat(R, "mean"), y=cat(R, "y"), unknown=cat(R, "unknown"), std=cat(R, "std"),
                 bus_type=cat(R, "bus_type"), net=cat(R, "net"), snap=cat(R, "snap"),
                 samples=cat(R, "samples", 1), extra=_uq_cat_extra([r["extra"] for r in R]),
                 edge=dict(dt_mean=cat(E, "dt_mean"), dt_true=cat(E, "dt_true"), dt_std=cat(E, "dt_std"),
                           dt_samples=cat(E, "dt_samples", 1), net=cat(E, "net"), snap=cat(E, "snap")),
                 line=None, metadata=md)
        n_line = sum(l is not None for l in self._l)
        if n_line:                                                       # [PROP-2]
            assert n_line == len(self._l), f"line: given for {n_line} of {len(self._l)} snapshots (all or none)"
            L = self._l
            c["line"] = dict(mean=cat(L, "mean"), y=cat(L, "y"), std=cat(L, "std"), net=cat(L, "net"),
                             snap=cat(L, "snap"), edge_pos=cat(L, "edge_pos"), limit=cat(L, "limit"),
                             samples=cat(L, "samples", 1), extra=_uq_cat_extra([l["extra"] for l in L], "line extra"))
        return c

def det_predict_uq(run, networks, net_ids=None, population=""):
    b = UQContractBuilder()
    for rec in uq_forward_snapshots(run["model"], networks, net_ids):
        b.add(rec, line=line_block(rec))                                               # [PROP-2] means only, std None
    return b.build({"method": "deterministic", "uncertainty_source": None, "run_keys": [run["run_key"]],
                    "split_hash": (run.get("split_manifest") or {}).get("hash"), "population": population,
                    "interval_type": None, "levels": UQ_LEVELS, "angle_unit": "rad",
                    "line_scope": "lines", "line_flow_fn": "delta_theta_pi_model", "line_sigma_method": None})

def z_of(level): return float(torch.distributions.Normal(0., 1.).icdf(torch.tensor(0.5 + level / 2)))
def gauss_nll(y, mu, sd):  return 0.5 * np.log(2 * np.pi * sd ** 2) + (y - mu) ** 2 / (2 * sd ** 2)
def gauss_crps(y, mu, sd):
    z = (y - mu) / sd; from scipy.stats import norm
    return sd * (z * (2 * norm.cdf(z) - 1) + 2 * norm.pdf(z) - 1 / np.sqrt(np.pi))
def ens_crps_fair(samples, y):            # samples [M, ...]; fair (unbiased) ensemble CRPS
    M = samples.shape[0]
    t1 = np.abs(samples - y[None]).mean(0)
    t2 = np.abs(samples[:, None] - samples[None, :]).sum((0, 1)) / (2 * M * (M - 1))
    return t1 - t2
def interval_score(y, lo, hi, level):
    a = 1 - level
    return (hi - lo) + (2 / a) * (lo - y) * (y < lo) + (2 / a) * (y - hi) * (y > hi)

def time_fn(fn, n_warmup=3, n_repeat=20):
    for _ in range(n_warmup): fn()
    ts = []
    for _ in range(n_repeat):
        t0 = time.perf_counter(); fn(); ts.append((time.perf_counter() - t0) * 1e3)
    q = np.percentile(ts, [25, 50, 75])
    return {"median_ms": float(q[1]), "iqr_ms": float(q[2] - q[0]), "n": n_repeat}

def assert_same_split(*runs):
    h = {r["split_manifest"]["hash"] for r in runs}
    assert len(h) == 1, f"runs are on different splits: {h}"
    return h.pop()

# ---- [PROP-2] line-flow helpers (task 105) — lines only; Δθ-head π-model (calculate_line_flows_from_delta_theta); pu ----
LINE_QTYS = ("p0", "q0", "p1", "q1", "smax")          # column order of c["line"]["mean"/"y"/"std"]

def line_limits_for(network, n_line_rows):
    """[n_line_rows] float64 lines.s_limit_pu in forward-line-edge order; NaN where missing (unrated)."""
    ok = network.lines.bus0.isin(network.buses.index) & network.lines.bus1.isin(network.buses.index)
    lines = network.lines[ok]                         # same filter and order as the dataset edge loop (Training cell 11)
    assert len(lines) == n_line_rows, f"{len(lines)} lines vs {n_line_rows} line edges"
    if "s_limit_pu" not in lines.columns:
        return torch.full((n_line_rows,), float("nan"), dtype=torch.float64)
    return torch.tensor(lines["s_limit_pu"].astype(float).to_numpy(), dtype=torch.float64)

def line_rows(rec):
    """Line-edge view of one uq_forward_snapshots record (transformers excluded)."""
    d = rec["data"]
    lm = d.dc_flow_mask[::2].bool()                   # forward LINE edges; lines precede transformers
    return dict(mask=lm, edge_pos=lm.nonzero(as_tuple=True)[0], ei=d.edge_index[:, ::2][:, lm],
                ea=d.edge_attr[::2][lm, :4].double(),  # [r, x, b/2, tap]; zero-padded model columns are ignored
                limit=rec["line_limit"])

def _line_flow_core(vf_raw, vt, dt, ea):
    r, x, bh, tap = ea.unbind(-1)
    z2 = r * r + x * x
    g, b = r / z2, -x / z2                            # 1 / (r + jx)
    t = torch.where(tap.abs() > 1e-12, tap, torch.ones_like(tap))
    vf, c, s = vf_raw / t, torch.cos(dt), torch.sin(dt)
    p0 = vf ** 2 * g - vf * vt * (g * c + b * s)
    q0 = -vf ** 2 * (b + bh) - vf * vt * (g * s - b * c)
    p1 = vt ** 2 * g - vt * vf * (g * c - b * s)
    q1 = -vt ** 2 * (b + bh) + vt * vf * (g * s + b * c)
    smax = torch.maximum(torch.hypot(p0, q0), torch.hypot(p1, q1))
    return torch.stack([p0, q0, p1, q1, smax], dim=-1)

def line_flows_batched(V, dt, ei, ea):
    """V [...,N] pu, dt [...,L] rad (Δθ on line rows), ei [2,L], ea [L,4] -> [...,L,5] float64 (LINE_QTYS order)."""
    V, dt = V.double(), dt.double()
    return _line_flow_core(V[..., ei[0]], V[..., ei[1]], dt, ea)

def line_flow_jacobian(V, dt, ei, ea):
    """[L,5,3] partials of LINE_QTYS w.r.t. (V_from, V_to, Δθ) at one point. smax: active end; 0 where |S| = 0."""
    with torch.enable_grad():
        vf = V.double()[ei[0]].detach().clone().requires_grad_(True)
        vt = V.double()[ei[1]].detach().clone().requires_grad_(True)
        d = dt.double().detach().clone().requires_grad_(True)
        out = _line_flow_core(vf, vt, d, ea)
        J = torch.zeros(out.size(0), 5, 3, dtype=torch.float64)
        for k in range(5):                            # each line depends only on its own inputs -> exact per-line partials
            J[:, k] = torch.stack(torch.autograd.grad(out[:, k].sum(), (vf, vt, d), retain_graph=k < 4), dim=-1)
    return torch.nan_to_num(J)                        # hypot gradient at |S| = 0 is NaN -> 0

def line_delta_var(J, var_from, var_to, cov_from_to, var_dt):
    """First-order variance [L,5] from J [L,5,3] and [L] (or scalar) input moments. V–Δθ covariance is taken as 0."""
    vf, vt, cft, vd = (torch.as_tensor(v, dtype=torch.float64).reshape(-1, 1)
                       for v in (var_from, var_to, cov_from_to, var_dt))
    Jf, Jt, Jd = J[..., 0], J[..., 1], J[..., 2]
    return (Jf ** 2 * vf + Jt ** 2 * vt + 2 * Jf * Jt * cft + Jd ** 2 * vd).clamp_min(0.0)

def line_block(rec, std=None, samples=None, extra=None):
    """Contract `line` rows of one snapshot: mean = f(rec mean state), y = f(dataset Vmag, Δθ)."""
    lr = line_rows(rec); m = lr["mask"]
    return dict(mean=line_flows_batched(rec["mean"][:, 0], rec["dt_mean"][m], lr["ei"], lr["ea"]),
                y=line_flows_batched(rec["y"][:, 0], rec["dt_true"][m], lr["ei"], lr["ea"]),
                std=std, samples=samples, extra=(extra or {}), edge_pos=lr["edge_pos"], limit=lr["limit"])

def line_loading(x, limit):
    """x [L] or [L,k] in pu -> loading fraction; NaN where unrated (limit NaN or <= 0)."""
    lim = np.asarray(limit, dtype=float); lim = np.where(lim > 0, lim, np.nan)
    return np.asarray(x, dtype=float) / lim.reshape(-1, *([1] * (np.ndim(x) - 1)))

In [ ]:
# ==== UQ METHOD COPIES (task 109) — verbatim [COPY of …] blocks; edit the trainer origins, then re-copy ====
# [STSI 261026]:new cell (task 109) - C (BLL), A1 (mean-scale heads) and B1 (ensemble) predictors incl. [PROP-2] line parts, copied from their trainer origins (test_uq_copy_parity.py)
import glob, itertools, math

# [COPY of GNN_Powerflow_V2.8_UQ-C_BayesLastLayer::BLL_COLS]
BLL_COLS = {  # col: (head attr, out row, kind, mask attr, contract col)
    "vmag_pq": ("pq_head", 0, "node", "pq_mask", 0), "q_pv": ("pv_head", 0, "node", "pv_mask", 3),
    "p_slack": ("slack_head", 0, "node", "slack_mask", 2), "q_slack": ("slack_head", 1, "node", "slack_mask", 3),
    "dtheta": ("edge_angle_pred", 0, "edge", None, None)}
BLL_SIGMA2_GRID = np.logspace(-1, 1, 21)          # × var(r0) on val
BLL_TAU_GRID = np.logspace(-1, 5, 25)             # C1 likelihood temperature
BLL_KAPPA_GRID = np.logspace(-2, 4, 25)           # C2 posterior scale
BLL_THETA_C_GRID = np.logspace(-2, 2, 25)         # × σ²_dθ (θ path constant)
BLL_CHUNK = 16384                                 # rows per chunk in the val-NLL grids

# [COPY of GNN_Powerflow_V2.8_UQ-C_BayesLastLayer::BLL_FLOW_MC_SAMPLES]
BLL_FLOW_MC_SAMPLES, BLL_FLOW_MC_SEED = 256, 0
LINE_COV_VDELTA_NOTE = "zero (C-7: independent vmag_pq and dtheta posteriors; PHYS-3 deferred)"

# [COPY of GNN_Powerflow_V2.8_UQ-C_BayesLastLayer::bll_sqrt]
def bll_sqrt(S):                                   # R with R Rᵀ = S (eigh; robust to PSD-singular S)
    lam, U = torch.linalg.eigh(S.double()); return U * lam.clamp_min(0.0).sqrt()

# [COPY of GNN_Powerflow_V2.8_UQ-C_BayesLastLayer::bll_line_std]
def bll_line_std(rec, post, roots, n_samples=BLL_FLOW_MC_SAMPLES, seed=BLL_FLOW_MC_SEED):
    """(std_mc [L,5], std_delta [L,5]) for one keep_embeddings record. roots = {col: bll_sqrt(post[col]["S"])}."""
    lr, d = line_rows(rec), rec["data"]; lm, pq = lr["mask"], d.pq_mask.bool()
    pV, pD = post["vmag_pq"], post["dtheta"]
    Hv = torch.cat([rec["h_nodes"][pq], torch.ones(int(pq.sum()), 1)], 1).double()            # [n_pq, d]
    He = torch.cat([rec["h_edges_fwd"][lm], torch.ones(int(lm.sum()), 1)], 1).double()       # [L, d]
    g = torch.Generator().manual_seed((seed * 1_000_003 + int(rec["net"]) * 1_009 + int(rec["snap"])) % 2 ** 63)
    rn = lambda *s: torch.randn(*s, generator=g, dtype=torch.float64)
    V = rec["mean"][:, 0].double().expand(n_samples, -1).clone()                              # PV/slack V fixed (σ = 0)
    V[:, pq] += rn(n_samples, Hv.size(1)) @ roots["vmag_pq"].T @ Hv.T + math.sqrt(pV["sigma2"]) * rn(n_samples, Hv.size(0))
    dt = (rec["dt_mean"][lm].double() + rn(n_samples, He.size(1)) @ roots["dtheta"].T @ He.T
          + math.sqrt(pD["sigma2"]) * rn(n_samples, He.size(0)))                              # independent of V (C-7)
    std_mc = line_flows_batched(V, dt, lr["ei"], lr["ea"]).std(0)                             # [L,5]
    J = line_flow_jacobian(rec["mean"][:, 0], rec["dt_mean"][lm], lr["ei"], lr["ea"])         # diagnostic
    Hn = torch.zeros(d.x.size(0), Hv.size(1), dtype=torch.float64); Hn[pq] = Hv              # 0 rows at PV/slack
    hf, ht = Hn[lr["ei"][0]], Hn[lr["ei"][1]]
    var_f = ((hf @ pV["S"]) * hf).sum(1) + pV["sigma2"] * pq[lr["ei"][0]].double()
    var_t = ((ht @ pV["S"]) * ht).sum(1) + pV["sigma2"] * pq[lr["ei"][1]].double()
    cov_ft = ((hf @ pV["S"]) * ht).sum(1)                                                     # shared weights; noise independent
    var_dt = ((He @ pD["S"]) * He).sum(1) + pD["sigma2"]
    return std_mc, line_delta_var(J, var_f, var_t, cov_ft, var_dt).sqrt()

# [COPY of GNN_Powerflow_V2.8_UQ-C_BayesLastLayer::bll_rec_std]
def bll_rec_std(rec, post, c_th):
    """Node / θ (PROP-1) / Δθ closed-form σ for one keep_embeddings record -> (std [N,4], var_model [N,4], dt_var_model [E])."""
    std = torch.zeros_like(rec["mean"]); var_model = torch.zeros_like(rec["mean"])
    for col, (head, row, kind, mask, ccol) in BLL_COLS.items():
        if kind != "node": continue
        m = getattr(rec["data"], mask).bool()
        if not m.any(): continue
        h_t = torch.cat([rec["h_nodes"][m], torch.ones(int(m.sum()), 1)], 1).double()
        q = ((h_t @ post[col]["S"]) * h_t).sum(1)
        var_model[m, ccol] = q.float(); std[m, ccol] = torch.sqrt(q + post[col]["sigma2"]).float()
    He = torch.cat([rec["h_edges_fwd"], torch.ones(rec["h_edges_fwd"].size(0), 1)], 1).double()
    Sd = post["dtheta"]["S"]
    dt_var_model = ((He @ Sd) * He).sum(1)
    A = rec["P"].double() @ He
    th_var_model = ((A @ Sd) * A).sum(1)
    th_var = th_var_model + c_th * rec["P"].abs().sum(1).double()
    unk = rec["unknown"]
    std[:, 1] = torch.where(unk[:, 1], torch.sqrt(th_var).float(), torch.zeros(()))
    var_model[:, 1] = torch.where(unk[:, 1], th_var_model.float(), torch.zeros(()))
    std[~unk] = 0.0
    return std, var_model, dt_var_model

# [COPY of GNN_Powerflow_V2.8_UQ-C_BayesLastLayer::bll_predict_uq]
def bll_predict_uq(run, networks, net_ids=None, population=""):
    """P6 contract for a C1/C2 run (mean = forward pass, w0)."""
    post, c_th = run["uq_artifacts"]["bll_posterior"], run["uq_artifacts"]["theta_c"]
    b = UQContractBuilder()
    roots = {c: bll_sqrt(post[c]["S"]) for c in ("vmag_pq", "dtheta")}                        # [PROP-2] once per call
    for rec in uq_forward_snapshots(run["model"], networks, net_ids, keep_embeddings=True):
        std, var_model, dt_var_model = bll_rec_std(rec, post, c_th)
        l_std, l_std_delta = bll_line_std(rec, post, roots)                                    # [PROP-2]
        b.add(rec, std=std, dt_std=torch.sqrt(dt_var_model + post["dtheta"]["sigma2"]).float(),
              extra={"var_model": var_model},
              line=line_block(rec, std=l_std.float(), extra={"std_delta": l_std_delta.float()}))
    a = run["uq_artifacts"]
    return b.build({"method": a["method"], "uncertainty_source": "bll_closed_form" if a["method"] == "C1" else "vbll",
                    "mean_source": "w0", "run_keys": [run["run_key"]], "split_hash": run["split_manifest"]["hash"],
                    "population": population, "interval_type": "gaussian", "levels": UQ_LEVELS, "angle_unit": "rad",
                    "line_scope": "lines", "line_flow_fn": "delta_theta_pi_model", "line_sigma_method": "mc_weight_space",
                    "line_mc_samples": BLL_FLOW_MC_SAMPLES, "line_mc_seed": BLL_FLOW_MC_SEED,
                    "line_std_diagnostic": "delta", "line_cov_VDelta": LINE_COV_VDELTA_NOTE})

# [COPY of GNN_Powerflow_V2.8_UQ-A1_MeanScale::A1_FLOW_CHANNELS]
A1_FLOW_CHANNELS = ("flow_p0", "flow_q0", "flow_smax")
A1_FLOW_QTY = {"flow_p0": 0, "flow_q0": 1, "flow_smax": 4}        # column in LINE_QTYS
A1_CFG = dict(family="gaussian", nu=None, sigma_min=1e-3, k_folds=5, max_epochs=100, lr=5e-3,
              batch_rows=8192, fold_seed=0, theta_primary="direct",
              flow_channels=A1_FLOW_CHANNELS)                      # [PROP-2] () -> state channels only; line σ = baseline
A1_FAMILIES = ("gaussian", "student_t")            # NLL registry (a1_nll); switch with A1_CFG["family"] (+ "nu")
A1_THETA_VARIANTS = ("direct", "indep", "comonotone")
A1_NODE_ROWS = {  # ch: (row masks OR-ed, contract col); residual r = y[:, col] − frozen μ[:, col]
    "vmag_pq": (("pq_mask",), 0), "q_pv": (("pv_mask",), 3), "p_slack": (("slack_mask",), 2),
    "q_slack": (("slack_mask",), 3), "theta_direct": (("pq_mask", "pv_mask"), 1)}
A1_QQ_PROBS = np.linspace(0.005, 0.995, 199)       # stored held-out standardized-residual quantiles (QQ plot)

# [COPY of GNN_Powerflow_V2.8_UQ-A1_MeanScale::a1_sigma]
def a1_sigma(H, w, b, s, sigma_min):           # functional σ in physical units; H [n,h]
    return s * (F.softplus(H @ w.reshape(-1) + b.reshape(())) + sigma_min)

# [COPY of GNN_Powerflow_V2.8_UQ-A1_MeanScale::a1_std_from_scale]
def a1_std_from_scale(sig, family="gaussian", nu=None):
    return sig if family == "gaussian" else sig * math.sqrt(nu / (nu - 2))

# [COPY of GNN_Powerflow_V2.8_UQ-A1_MeanScale::_a1_sig]
def _a1_sig(a, ch, H):                         # predictive std of channel ch in physical units
    sd = a["scale_state"][ch]
    return a1_std_from_scale(a1_sigma(H.float(), sd["weight"], sd["bias"], a["normalizers"][ch], a["sigma_min"]),
                             a["family"], a["nu"])

# [COPY of GNN_Powerflow_V2.8_UQ-A1_MeanScale::a1_state_std]
def a1_state_std(rec, a):
    """(std [N,4], dt_std [E_fwd], extra {theta_std_direct/indep/comonotone [N]}) for one keep_embeddings record."""
    std = torch.zeros_like(rec["mean"]); d = rec["data"]
    for ch, (masks, col) in A1_NODE_ROWS.items():
        if ch == "theta_direct": continue
        m = getattr(d, masks[0]).bool()
        if m.any(): std[m, col] = _a1_sig(a, ch, rec["h_nodes"][m])
    s_e = _a1_sig(a, "dtheta", rec["h_edges_fwd"])
    Pabs = rec["P"].abs().double()
    th_m = d.pq_mask.bool() | d.pv_mask.bool()
    th_direct = torch.zeros(std.size(0))
    if th_m.any(): th_direct[th_m] = _a1_sig(a, "theta_direct", rec["h_nodes"][th_m])
    th_indep = torch.sqrt(Pabs @ s_e.double() ** 2).float()                  # (i) independent edges (PROP-1 baseline)
    th_comon = (Pabs @ s_e.double()).float()                                 # (ii) comonotone edges (PROP-1 baseline)
    u1 = rec["unknown"][:, 1]
    th = {v: torch.where(u1, x, torch.zeros(())) for v, x in
          (("direct", th_direct), ("indep", th_indep), ("comonotone", th_comon))}          # 0 at slack (known θ)
    std[:, 1] = th[a["theta_primary"]]
    std[~rec["unknown"]] = 0.0
    return std, s_e, {f"theta_std_{v}": x for v, x in th.items()}

# [COPY of GNN_Powerflow_V2.8_UQ-A1_MeanScale::a1_line_std]
def a1_line_std(rec, a, std, s_e):
    """[PROP-2] (line std [L,5] float64, std_indep [L,5] float64). Baseline: first-order propagation of the vmag_pq and
    dtheta σ with zero V–V and V–Δθ covariance (σ_V = 0 at PV/slack); flow-scale heads replace p0/q0/smax."""
    lr = line_rows(rec); lm = lr["mask"]
    J = line_flow_jacobian(rec["mean"][:, 0], rec["dt_mean"][lm], lr["ei"], lr["ea"])
    sV = std[:, 0].double()                                                              # 0 at PV/slack (known)
    l_ind = line_delta_var(J, sV[lr["ei"][0]] ** 2, sV[lr["ei"][1]] ** 2, 0.0, s_e[lm].double() ** 2).sqrt()
    l_std = l_ind.clone()
    for ch in a.get("flow_channels", ()):                                                # primary: flow-scale heads
        l_std[:, A1_FLOW_QTY[ch]] = _a1_sig(a, ch, rec["h_edges_fwd"][lm]).double()
    return l_std, l_ind

# [COPY of GNN_Powerflow_V2.8_UQ-A1_MeanScale::a1_predict_uq]
def a1_predict_uq(run, networks, net_ids=None, population=""):
    """P6 contract for an A1 run: mean = frozen forward pass (bit-identical to det_predict_uq), σ from the scale heads."""
    a = run["uq_artifacts"]; fam, nu = a["family"], a["nu"]
    b = UQContractBuilder()
    for rec in uq_forward_snapshots(run["model"], networks, net_ids, keep_embeddings=True):
        std, s_e, extra = a1_state_std(rec, a)
        l_std, l_ind = a1_line_std(rec, a, std, s_e)                                     # [PROP-2]
        b.add(rec, std=std, dt_std=s_e, extra=extra,
              line=line_block(rec, std=l_std.float(), extra={"std_indep": l_ind.float()}))
    fc = tuple(a.get("flow_channels", ()))
    return b.build({"method": "A1", "uncertainty_source": "mean_scale_head", "family": fam, "nu": nu,
                    "run_keys": [run["run_key"]], "split_hash": run["split_manifest"]["hash"], "population": population,
                    "interval_type": fam, "levels": UQ_LEVELS, "angle_unit": "rad",
                    "theta_primary": a["theta_primary"],
                    "line_scope": "lines", "line_flow_fn": "delta_theta_pi_model",
                    "line_sigma_method": "flow_scale_head" if fc else "indep_delta",
                    "line_std_source": {q: ("head" if f"flow_{q}" in fc else "indep_delta") for q in LINE_QTYS},
                    "line_baseline": "extra.std_indep: first-order propagation, zero V-V and V-dtheta covariance"})

# [COPY of GNN_Powerflow_V2.8_UQ-B1_Ensemble::SIGMA_FLOOR_Q]
SIGMA_FLOOR_Q = 0.01                          # conformal-score floor: SIGMA_FLOOR_Q-quantile of σ̄ (task 109 uses the same rule)
B1_EXTEND_GAIN = 0.02                         # B-3: EXTEND if the (M−1)→M relative gain (mean over strata) > 2 % and CI low > 0
B1_DIV_SPREAD_MIN, B1_DIV_CORR_MAX, B1_DIV_MIN_STRATA = 0.3, 0.9, 4   # B-5: LOW DIVERSITY if >= 4 of the 7 verdict strata are low
B1_LINE_QTYS_REPORTED = ("p0", "q0", "smax")  # [PROP-2] report-only line rows (never counted in B-3 / B-5)

# [COPY of GNN_Powerflow_V2.8_UQ-B1_Ensemble::load_ensemble_members]
def load_ensemble_members(paths_or_globs):
    """Glob the member pickles and check that they form ONE ensemble (B-4): one split hash, one config (run key without
    `_is<N>`), distinct init_seed, same features. Raises AssertionError naming the offending files; returns runs by init_seed."""
    if isinstance(paths_or_globs, str):
        paths_or_globs = [paths_or_globs]
    runs, files = [], []
    for g in paths_or_globs:
        for p in sorted(glob.glob(g)):
            for r in load_sweep_results(p):
                runs.append(r); files.append(os.path.basename(p))
    assert runs, f"no member pickles found for {paths_or_globs}"
    def _files_by(vals):                       # {value: [files]} -> error messages name the files
        out = {}
        for v, f in zip(vals, files):
            out.setdefault(v, []).append(f)
        return out
    h = _files_by([r["split_manifest"]["hash"] for r in runs])
    assert len(h) == 1, f"members are on different splits (split_manifest hash -> files): {h}"
    assert_same_split(*runs)
    base = _files_by([re.sub(r"_is\d+", "", r["run_key"]) for r in runs])
    assert len(base) == 1, f"members differ in config (run key without _is<N> -> files): {base}"
    seeds = _files_by([r["init_seed"] for r in runs])
    dup = {s: f for s, f in seeds.items() if len(f) > 1}
    assert not dup, f"duplicate init_seed (seed -> files): {dup}"
    feat = _files_by([(r["angle_mode"], r["use_pnom_share"], r["hidden_dim"], r["model"].convs[0].lin_edge.in_channels)
                      for r in runs])
    assert len(feat) == 1, f"feature config mismatch ((angle_mode, use_pnom_share, hidden_dim, edge_dim) -> files): {feat}"
    order = sorted(range(len(runs)), key=lambda i: runs[i]["init_seed"])
    print(f"[load_ensemble_members] M={len(runs)} | init_seeds {[runs[i]['init_seed'] for i in order]} | "
          f"split {next(iter(h))} | files {[files[i] for i in order]}")
    return [runs[i] for i in order]

# [COPY of GNN_Powerflow_V2.8_UQ-B1_Ensemble::resolve_checkpoint_path]
def resolve_checkpoint_path(path, results_dir=None):   # [STSI 261026]:task 111 - cross-machine checkpoint fallback (origin; Analysis copies it)
    """Cross-machine checkpoint fallback (task 111) -> (path | None, how). how: "stored" (stored path exists) |
    "basename_fallback" (results_dir/basename(path) exists; results_dir defaults to TRAINING_RESULTS_DIR at call time;
    ntpath splits on both separators) | "missing". Changes only WHERE a checkpoint file is read, never WHICH one."""
    import ntpath
    if not path:
        return None, "missing"
    if os.path.isfile(path):
        return path, "stored"
    alt = os.path.join(TRAINING_RESULTS_DIR if results_dir is None else results_dir, ntpath.basename(str(path)))
    if os.path.isfile(alt):
        seen = resolve_checkpoint_path.__dict__.setdefault("_logged", set())
        if path not in seen:
            seen.add(path)
            print(f"[ckpt] stored path not on this machine: {path} -> using {alt} (same file name, TRAINING_RESULTS_DIR)")
        return alt, "basename_fallback"
    return None, "missing"

# [COPY of GNN_Powerflow_V2.8_UQ-B1_Ensemble::load_member_variant]
def load_member_variant(run, which="best"):      # "best" | "final" | "post_curriculum"
    if which == "best": return run["model"]
    path, _how = resolve_checkpoint_path((run.get("alt_checkpoints") or {}).get(which))   # [STSI 261026]:task 111 - cross-machine fallback
    assert path, f"{which} checkpoint missing for {run['run_key']}"
    sd = torch.load(path, map_location="cpu")
    sd = {k: v for k, v in sd.items() if not k.startswith("uq_bll.")}   # [STSI 261026]:task 111 - drop C2 hook params (X3: not part of PowerFlowGNN)
    m = copy.deepcopy(run["model"]); m.load_state_dict(sd); return m.eval()   # strict for every remaining key

# [COPY of GNN_Powerflow_V2.8_UQ-B1_Ensemble::uq_forward_snapshots_multi]
def uq_forward_snapshots_multi(models, networks, net_ids=None):
    """One PowerFlowDataset pass; per snapshot one eval forward per model (return_embeddings=True). Yields the
    uq_forward_snapshots record of model 0 plus "member_mean" [M,N,4] (known coords = truth for every member) and
    "member_dt" [M,E_fwd]. Graphs are built once, so member rows are aligned by construction."""
    models = list(models)
    assert models, "uq_forward_snapshots_multi: no models"
    for mdl in models:
        assert getattr(mdl, "angle_mode", "node") == "edge_delta", "UQ path supports angle_mode='edge_delta' only"
        mdl.eval()
    pn = {mdl.node_embedding.in_features >= 8 for mdl in models}
    assert len(pn) == 1, "members disagree on use_pnom_share (node feature width)"
    ds = PowerFlowDataset(networks, use_edge_features=True, ptdf_branch_mode="all", angle_mode="edge_delta",
                          use_pnom_share=pn.pop())
    for i in range(len(ds)):
        net_idx, t_idx = ds._index[i]
        data = ds[i]
        P = path_matrix_from_data(data)
        unk = unknown_mask_from_data(data)
        means, dts, first = [], [], None
        for mdl in models:
            _pad_edge_attr_for_model(mdl, data)    # same zero-pad fallback as uq_forward_snapshots (edge widths are equal)
            with torch.no_grad():
                node_pred, h_nodes, h_edges, dtp = mdl(data, return_embeddings=True)
            mean = data.y.clone().float()
            pred4 = torch.stack([node_pred[:, 0], P @ dtp, node_pred[:, 1], node_pred[:, 2]], dim=1)
            mean[unk] = pred4[unk]
            means.append(mean); dts.append(dtp)
            if first is None:
                first = (node_pred, mean, dtp)
        bt = torch.full((data.x.size(0),), 0, dtype=torch.int8)
        bt[data.pv_mask.bool()] = 1; bt[data.slack_mask.bool()] = 2
        rec = dict(net=(net_ids[net_idx] if net_ids is not None else net_idx), snap=t_idx, data=data, P=P,
                   mean=first[1], y=data.y.float(), unknown=unk, bus_type=bt, node_pred=first[0],
                   dt_mean=first[2], dt_true=data.y_delta_theta,
                   line_limit=line_limits_for(networks[net_idx], int(data.dc_flow_mask[::2].sum())),
                   member_mean=torch.stack(means), member_dt=torch.stack(dts))
        assert "line_limit" in rec                 # [PROP-2] same record as uq_forward_snapshots -> line_block works
        yield rec

# [COPY of GNN_Powerflow_V2.8_UQ-B1_Ensemble::ensemble_predict_uq]
def ensemble_predict_uq(runs, networks, net_ids=None, population="", which="best", keep_samples=True):
    models = [load_member_variant(r, which) for r in runs]
    b = UQContractBuilder()
    for rec in uq_forward_snapshots_multi(models, networks, net_ids):
        mm, md = rec["member_mean"], rec["member_dt"]
        rec["mean"], rec["dt_mean"] = mm.mean(0), md.mean(0)
        std = mm.var(0, unbiased=False).sqrt(); std[~rec["unknown"]] = 0.0
        lr = line_rows(rec)                                                                  # [PROP-2]
        mf = line_flows_batched(mm[:, :, 0], md[:, lr["mask"]], lr["ei"], lr["ea"])         # [M,L,5] member flows
        lb = line_block(rec, std=mf.var(0, unbiased=False).sqrt().float(),                   # centre f(μ̄) (D-P2-6)
                        samples=(mf.float() if keep_samples else None))
        y_l = lb["y"].numpy()
        crps = ens_crps_fair(mf.numpy(), y_l) if mf.size(0) > 1 else np.abs(mf[0].numpy() - y_l)
        lb["extra"] = {"mean_of_flows": mf.mean(0).float(), "crps_fair": torch.from_numpy(crps).float()}
        b.add(rec, std=std, dt_std=md.var(0, unbiased=False).sqrt(),
              samples=(mm if keep_samples else None), dt_samples=(md if keep_samples else None), line=lb)
    return b.build({"method": "B1", "uncertainty_source": "deep_ensemble_disagreement", "M": len(runs),
                    "variance_denominator": "M", "member_run_keys": [r["run_key"] for r in runs],
                    "init_seeds": [r["init_seed"] for r in runs], "checkpoint_criterion": which,
                    "split_hash": runs[0]["split_manifest"]["hash"], "population": population,
                    "interval_type": "conformal_only", "levels": UQ_LEVELS, "angle_unit": "rad",
                    "line_scope": "lines", "line_flow_fn": "delta_theta_pi_model",
                    "line_sigma_method": "ensemble_members", "line_center": "f(ensemble-mean state)",
                    "line_variance_denominator": "M"})

# [STSI 261026]:task 110 - B1 val diagnostics (b1_saturation, b1_diversity, [PROP-2] b1_line_center_gap; B1_LINE_QTYS_REPORTED is already copied inside the SIGMA_FLOOR_Q block above) and C report helpers (uqc_hyper_table, uqc_c1_vs_c2_table, [PROP-2] uqc_line_delta_ratio + its constants) for sections (c) / (d)

# [COPY of GNN_Powerflow_V2.8_UQ-B1_Ensemble::b1_strata]
def b1_strata(contract):
    """[(name, y [n], member values [M,n], net [n])] for the 7 verdict strata: UQ_STRATA (unknown coords) + Δθ (all fwd edges)."""
    S, e = contract["samples"], contract["edge"]
    assert S is not None and e["dt_samples"] is not None, "B1 diagnostics need ensemble_predict_uq(..., keep_samples=True)"
    out = []
    for col, bus in UQ_STRATA:
        j = STATE_COLS.index(col)
        m = (contract["bus_type"] == BUS_TYPE_CODE[bus]) & contract["unknown"][:, j]
        out.append((f"{col}|{bus}", contract["y"][m, j], S[:, m, j], contract["net"][m]))
    out.append(("dtheta|edge", e["dt_true"], e["dt_samples"], e["net"]))
    return out

# [COPY of GNN_Powerflow_V2.8_UQ-B1_Ensemble::_b1_wq]
def _b1_wq(s_sorted, W, cols, level, chunk=32):
    """Split-conformal quantile per bootstrap row: the ceil((n+1)·level)-th smallest score with integer row weights
    W[b, cols] (n = Σw); inf if n is too small, NaN if n = 0. s_sorted [n_rows] ascending, cols [n_rows], W [B,K] -> [B]."""
    W = np.atleast_2d(np.asarray(W, np.float64))
    out = np.full(W.shape[0], np.nan)
    if not s_sorted.size:
        return out
    for c0 in range(0, W.shape[0], chunk):
        cw = np.cumsum(W[c0:c0 + chunk][:, cols], axis=1)      # chunked: never materialise [B, n_rows]
        n = cw[:, -1]
        k = np.ceil((n + 1) * level)
        pos = (cw < k[:, None]).sum(1)                          # first index with cumulative weight >= k
        q = s_sorted[np.minimum(pos, s_sorted.size - 1)]
        out[c0:c0 + chunk] = np.where(n == 0, np.nan, np.where(k > n, np.inf, q))
    return out

# [COPY of GNN_Powerflow_V2.8_UQ-B1_Ensemble::_b1_nanmean0]
def _b1_nanmean0(X):
    """Column mean ignoring NaN; NaN (no warning) where a column has no finite value."""
    X = np.asarray(X, np.float64)
    ok = np.isfinite(X)
    cnt = ok.sum(0)
    return np.where(cnt > 0, np.where(ok, X, 0.0).sum(0) / np.maximum(cnt, 1), np.nan)

# [COPY of GNN_Powerflow_V2.8_UQ-B1_Ensemble::b1_saturation]
def b1_saturation(contract, val_groups_a, val_groups_b, n_boot=500, seed=0, level=UQ_PRIMARY_LEVEL):
    """Saturation over all C(M,m) member subsets, m = 1..M, on the 7 verdict strata (val only):
    RMSE(μ̄) and fair CRPS (m = 1 -> MAE) over all val rows; conformal width@level: q̂ on half A of
    |y − μ̄| / max(σ̄, floor), floor = SIGMA_FLOOR_Q-quantile of σ̄ on half A (σ̄ ≡ 0 -> unscaled scores),
    width = mean 2·q̂·max(σ̄, floor) on half B. Gain (M−1)→M: relative, mean over strata, network-bootstrap CI (halves
    resampled separately; floors fixed at the full-sample value)."""
    ga = np.asarray(sorted({int(g) for g in val_groups_a})); gb = np.asarray(sorted({int(g) for g in val_groups_b}))
    assert ga.size and gb.size and not set(ga.tolist()) & set(gb.tolist()), "halves must be non-empty and disjoint"
    nets = np.concatenate([ga, gb]); K = nets.size; srt = np.argsort(nets)
    rng = np.random.default_rng(seed)
    W = np.concatenate([rng.multinomial(ga.size, np.full(ga.size, 1.0 / ga.size), size=n_boot),
                        rng.multinomial(gb.size, np.full(gb.size, 1.0 / gb.size), size=n_boot)], axis=1).astype(np.float64)
    strata = b1_strata(contract)
    M = strata[0][2].shape[0]
    m_from, m_to = M - 1, M
    rows, boot = [], {}                              # boot[(stratum, m)] = list of per-subset bootstrap metric arrays
    for name, y, S, net in strata:
        y, S, net = np.asarray(y, np.float64), np.asarray(S, np.float64), np.asarray(net)
        keep = np.isin(net, nets)
        y, S, net = y[keep], S[:, keep], net[keep]
        idx = srt[np.searchsorted(nets[srt], net)] if net.size else net.astype(int)
        inA = idx < ga.size
        for m in range(1, M + 1):
            for sub in itertools.combinations(range(M), m):
                Ss = S[list(sub)]
                mu, sd = Ss.mean(0), Ss.std(0)
                err = np.abs(y - mu)
                crps = ens_crps_fair(Ss, y) if m >= 2 else np.abs(Ss[0] - y)
                floor = float(np.quantile(sd[inA], SIGMA_FLOOR_Q)) if inA.any() else np.nan
                scaled = bool(np.isfinite(floor) and floor > 0)
                sig = np.maximum(sd, floor) if scaled else np.ones_like(sd)
                oA = np.argsort(err[inA] / sig[inA], kind="stable")
                sA, iA = (err[inA] / sig[inA])[oA], idx[inA][oA]
                q = float(_b1_wq(sA, np.ones((1, 1)), np.zeros(sA.size, dtype=np.int64), level)[0])
                nB = int((~inA).sum())
                rows.append({"m": m, "subset": tuple(int(s) for s in sub), "stratum": name, "n": int(y.size),
                             "RMSE": float(np.sqrt(np.mean(err ** 2))) if y.size else np.nan,
                             "CRPS": float(np.mean(crps)) if y.size else np.nan,
                             f"width{int(round(level * 100))}": float(np.mean(2 * q * sig[~inA])) if nB else np.nan,
                             "scaled": scaled})
                if m in (m_from, m_to) and n_boot:
                    cn, cs = np.bincount(idx, minlength=K).astype(np.float64), np.bincount(idx, crps, minlength=K)
                    b_crps = (W @ cs) / np.maximum(W @ cn, 1e-300)
                    bq = _b1_wq(sA, W, iA, level)
                    sbn = np.bincount(idx[~inA], sig[~inA], minlength=K); cbn = np.bincount(idx[~inA], minlength=K).astype(np.float64)
                    b_w = 2 * bq * (W @ sbn) / np.maximum(W @ cbn, 1e-300)
                    boot.setdefault((name, m), []).append((b_crps, b_w))
    wcol = f"width{int(round(level * 100))}"
    subsets = pd.DataFrame(rows)
    table = (subsets.groupby(["m", "stratum"], sort=False)
             .agg(n_subsets=("subset", "size"), n=("n", "first"), RMSE=("RMSE", "mean"), CRPS=("CRPS", "mean"),
                  **{wcol: (wcol, "mean")}, scaled=("scaled", "all")))
    gain_rows, per_stratum = {}, {}
    verdict = "n/a (M < 2)"
    if M >= 2:
        names = [s[0] for s in strata]
        with np.errstate(divide="ignore", invalid="ignore"):
            for k, metric in enumerate(("CRPS", wcol)):
                pt = np.array([(table.loc[(m_from, s), metric] - table.loc[(m_to, s), metric]) / table.loc[(m_from, s), metric]
                               for s in names], dtype=np.float64)
                pt[~np.isfinite(pt)] = np.nan
                per_stratum[metric] = dict(zip(names, pt))
                lo = hi = np.nan
                if n_boot:
                    bg = []
                    for s in names:                     # per stratum: mean over the C(M, M−1) subsets, then relative gain
                        a = np.mean([x[k] for x in boot[(s, m_from)]], axis=0); c = boot[(s, m_to)][0][k]
                        g = (a - c) / a; g[~np.isfinite(g)] = np.nan; bg.append(g)
                    bmean = _b1_nanmean0(np.vstack(bg))
                    if np.isfinite(bmean).any():
                        lo, hi = np.percentile(bmean[np.isfinite(bmean)], [2.5, 97.5])
                gain_rows[metric] = {"gain_mean_over_strata": float(_b1_nanmean0(pt[:, None])[0]),
                                     "ci95_lo": float(lo), "ci95_hi": float(hi), "threshold": B1_EXTEND_GAIN,
                                     "n_strata": int(np.isfinite(pt).sum()), "n_boot": int(n_boot)}
        verdict = b1_extend_verdict(pd.DataFrame(gain_rows).T)
    gain = pd.DataFrame(gain_rows).T
    gain.index.name = f"gain {m_from}->{m_to}"
    return {"table": table, "subsets": subsets, "gain": gain, "gain_per_stratum": pd.DataFrame(per_stratum),
            "verdict": verdict, "m_from": m_from, "m_to": m_to, "M": M, "level": level}

# [COPY of GNN_Powerflow_V2.8_UQ-B1_Ensemble::b1_extend_verdict]
def b1_extend_verdict(gain):
    """B-3: EXTEND if, for CRPS or width@90, the mean-over-strata relative gain > B1_EXTEND_GAIN and its CI lower bound > 0."""
    ok = [(float(r["gain_mean_over_strata"]) > B1_EXTEND_GAIN) and (float(r["ci95_lo"]) > 0) for _, r in gain.iterrows()]
    return "EXTEND" if any(ok) else "SATURATED"

# [COPY of GNN_Powerflow_V2.8_UQ-B1_Ensemble::_b1_div_row]
def _b1_div_row(y, S):
    y, S = np.asarray(y, np.float64), np.asarray(S, np.float64)
    M, n = S.shape[0], y.size
    if n == 0:
        return {"n": 0, "mean_sigma": np.nan, "RMSE": np.nan, "spread_over_RMSE": np.nan, "mean_pair_err_corr": np.nan}
    mu, sd = S.mean(0), S.std(0)
    rmse, spread = float(np.sqrt(np.mean((mu - y) ** 2))), float(np.mean(sd))
    corr = np.nan
    if M >= 2 and n >= 2:
        with np.errstate(divide="ignore", invalid="ignore"):
            C = np.corrcoef(S - y[None])
        cu = C[np.triu_indices(M, 1)]
        corr = float(np.nanmean(cu)) if np.isfinite(cu).any() else np.nan
    return {"n": int(n), "mean_sigma": spread, "RMSE": rmse,
            "spread_over_RMSE": (spread / rmse if rmse > 0 else np.nan), "mean_pair_err_corr": corr}

# [COPY of GNN_Powerflow_V2.8_UQ-B1_Ensemble::b1_diversity]
def b1_diversity(contract):
    """Per stratum: mean σ̄ / RMSE(μ̄) and the mean pairwise correlation of member errors. B-5 verdict counts only the 7
    state/Δθ strata; [PROP-2] line rows (B1_LINE_QTYS_REPORTED, from line.samples) are report-only."""
    rows = {}
    for name, y, S, _ in b1_strata(contract):
        rows[name] = {**_b1_div_row(y, S), "counts_for_verdict": True}
    L = contract.get("line")
    if L is not None and L.get("samples") is not None:                                     # [PROP-2] report-only
        for q in B1_LINE_QTYS_REPORTED:
            k = LINE_QTYS.index(q)
            rows[f"line_{q}"] = {**_b1_div_row(L["y"][:, k], L["samples"][:, :, k]), "counts_for_verdict": False}
    t = pd.DataFrame(rows).T
    for c in ("n", "mean_sigma", "RMSE", "spread_over_RMSE", "mean_pair_err_corr"):
        t[c] = pd.to_numeric(t[c])
    t["counts_for_verdict"] = t["counts_for_verdict"].astype(bool)
    t["low"] = [bool((r["spread_over_RMSE"] < B1_DIV_SPREAD_MIN) or (r["mean_pair_err_corr"] > B1_DIV_CORR_MAX))
                for _, r in t.iterrows()]
    vt = t[t["counts_for_verdict"]]
    n_low = int(vt["low"].sum())
    return {"table": t, "verdict": "LOW DIVERSITY" if n_low >= B1_DIV_MIN_STRATA else "DIVERSE", "n_low": n_low,
            "n_strata": int(len(vt)), "rule": f">= {B1_DIV_MIN_STRATA} of {len(vt)} strata with spread/RMSE < "
                                              f"{B1_DIV_SPREAD_MIN} or mean pairwise error corr > {B1_DIV_CORR_MAX}"}

# [COPY of GNN_Powerflow_V2.8_UQ-B1_Ensemble::b1_line_center_gap]
def b1_line_center_gap(contract):
    """[PROP-2] D-P2-6 diagnostic: |f(μ̄) − mean_m F_m| / max(line σ, floor_q), floor_q = SIGMA_FLOOR_Q-quantile of line σ
    (this contract), plus RMSE vs y of both centres. Report-only."""
    L = contract["line"]
    assert L is not None and "mean_of_flows" in L["extra"], "contract has no B1 line section"
    rows = {}
    for q in B1_LINE_QTYS_REPORTED:
        k = LINE_QTYS.index(q)
        mu, mof, sd, y = (np.asarray(a[:, k], np.float64) for a in (L["mean"], L["extra"]["mean_of_flows"], L["std"], L["y"]))
        if not y.size:
            rows[f"line_{q}"] = {"n": 0}; continue
        floor = max(float(np.quantile(sd, SIGMA_FLOOR_Q)), 1e-12)
        g = np.abs(mu - mof) / np.maximum(sd, floor)
        rows[f"line_{q}"] = {"n": int(y.size), "median_gap/sigma": float(np.median(g)), "p90_gap/sigma": float(np.percentile(g, 90)),
                             "max_abs_gap": float(np.max(np.abs(mu - mof))), "sigma_floor": floor,
                             "RMSE_f(mean_state)": float(np.sqrt(np.mean((mu - y) ** 2))),
                             "RMSE_mean_of_flows": float(np.sqrt(np.mean((mof - y) ** 2)))}
    return pd.DataFrame(rows).T

# [COPY of GNN_Powerflow_V2.8_UQ-C_BayesLastLayer::UQ_C_REPORT_POPS]
UQ_C_REPORT_POPS = ("val", "test")
UQ_C_LAT_SNAPSHOTS, UQ_C_LAT_WARMUP, UQ_C_LAT_REPEAT = 20, 3, 20
UQ_C_LINE_REPORT_QTYS = ("p0", "q0", "smax")
UQ_C_SMALL_S = 0.02                                   # pu; |S| below this is where the line-flow σ is non-smooth

# [COPY of GNN_Powerflow_V2.8_UQ-C_BayesLastLayer::uqc_hyper_table]
def uqc_hyper_table(run):
    a = run["uq_artifacts"]; post = a["bll_posterior"]; dg = a["diagnostics"]
    rows = {}
    for c, p in post.items():
        rows[c] = {"sigma2": p["sigma2"], "alpha": p["alpha"], "tau": p["tau"], "kappa": p["kappa"], "n_rows": p["n_rows"],
                   "n_snapshots": p["n_snapshots"], "counts_from": p.get("counts_population", "train"),
                   "edge_of_grid": p["grid_edge_flag"], "w0_val_rmse": dg["w0_val_rmse"].get(c),
                   "m_refit_val_rmse": dg["m_refit_val_rmse"].get(c), "epi_frac_val": dg["epistemic_frac"][c]["val"]}
    rows["theta_c"] = {"sigma2": a["theta_c"], "edge_of_grid": a["theta_grid_edge_flag"]}
    return pd.DataFrame(rows).T

# [COPY of GNN_Powerflow_V2.8_UQ-C_BayesLastLayer::uqc_c1_vs_c2_table]
def uqc_c1_vs_c2_table(run):
    """C1-on-C2 diagnostic: closed-form S on C2's backbone vs C2's (κ-rescaled) learned S."""
    c1p, c2p = run["uq_artifacts"]["diagnostics"]["c1_on_c2"], run["uq_artifacts"]["bll_posterior"]
    rows = {}
    for c in c2p:
        S1, S2 = c1p[c]["S"].double(), c2p[c]["S"].double()
        (s1, l1), (s2, l2) = torch.linalg.slogdet(S1), torch.linalg.slogdet(S2)
        rows[c] = {"trace_ratio_C1/C2": float(torch.trace(S1) / torch.trace(S2)),
                   "logdet_C1-logdet_C2": float(l1 - l2) if s1 > 0 and s2 > 0 else np.nan,
                   "sigma2_C1": c1p[c]["sigma2"], "sigma2_C2": c2p[c]["sigma2"]}
    return pd.DataFrame(rows).T

# [COPY of GNN_Powerflow_V2.8_UQ-C_BayesLastLayer::uqc_line_delta_ratio]
def uqc_line_delta_ratio(c):
    """[PROP-2] std_delta / std_mc per quantity: median, p10, p90, for true smax < UQ_C_SMALL_S and the rest."""
    L = c["line"]; small = L["y"][:, LINE_QTYS.index("smax")] < UQ_C_SMALL_S
    out = []
    for qn in LINE_QTYS:
        k = LINE_QTYS.index(qn)
        with np.errstate(divide="ignore", invalid="ignore"):
            ratio = L["extra"]["std_delta"][:, k].astype(np.float64) / L["std"][:, k].astype(np.float64)
        for lab, m in ((f"smax<{UQ_C_SMALL_S}", small), (f"smax>={UQ_C_SMALL_S}", ~small)):
            rr = ratio[m & np.isfinite(ratio)]
            p10, p50, p90 = np.percentile(rr, [10, 50, 90]) if rr.size else (np.nan,) * 3
            out.append({"qty": qn, "rows": lab, "n": int(rr.size), "median": p50, "p10": p10, "p90": p90})
    return pd.DataFrame(out).set_index(["qty", "rows"])

In [ ]:
# ==== UQ N-1 / DATASET LOADERS (task 109) — verbatim [COPY of …] blocks (Contingency + DataGen origins) ====
# [STSI 261026]:new cell (task 109) - run loading by key, N-1 dataset + base-failed snapshot handling, [PROP-2] line-limit decode / attach; load_dataset_subset / read_n1_sidecar marked with their DataGen origin

# [COPY of GNN_Powerflow_V2.7_Contingency::load_runs_by_key]
def load_runs_by_key(run_keys, sweep_paths):
    """D6: return {run_key: run_dict} for the user-supplied keys; first occurrence in sweep_paths order wins."""
    if not run_keys:
        raise ValueError("run_keys must be a non-empty list (D6: no default model)")
    found, dups = {}, {}
    for p in sweep_paths:
        for r in load_sweep_results(p):
            k = r.get("run_key")
            if k in run_keys:
                if k in found:
                    dups.setdefault(k, []).append(p)
                else:
                    found[k] = r
    missing = [k for k in run_keys if k not in found]
    if missing:
        raise KeyError(f"run_keys not found in {len(sweep_paths)} file(s): {missing}")
    for k, ps in dups.items():
        print(f"[load_runs_by_key] duplicate {k!r} also in {ps}; using {found[k].get('_source_pkl')}")
    return {k: found[k] for k in run_keys}

# [COPY of GNN_Powerflow_V2.7_Contingency::BASE_CHECK_FIELDS]
BASE_CHECK_FIELDS = ("base_nonconverged_snapshots", "base_sanity_failed_snapshots")   # [STSI 151026]:task 093: task 092 sidecar row fields

# [COPY of GNN_Powerflow_V2.7_Contingency::base_failed_snapshots]
def base_failed_snapshots(row):
    """str(snapshot) labels where the pre-outage base failed (inferred non-converged OR N-1 sanity) -> base_nonconverged."""
    return set(row["base_nonconverged_snapshots"]) | set(row["base_sanity_failed_snapshots"])

# [COPY of GNN_Powerflow_V2.7_Contingency::drop_base_failed]
def drop_base_failed(df, row):
    """Drop records of base-failed snapshots (task 093: class base_nonconverged, never scored, never severe)."""
    bad = base_failed_snapshots(row)
    return df[~df["snapshot"].map(str).isin(bad)] if bad and len(df) else df

# [COPY of GNN_Powerflow_V2.7_Contingency::OUTAGE_CHECK_FIELDS]
OUTAGE_CHECK_FIELDS = ("failed_snapshots", "nonconverged_snapshots", "sanity_failed_snapshots")   # [STSI 201026]:task 100 (DataGen task 099)

# [COPY of GNN_Powerflow_V2.7_Contingency::failed_snapshot_reasons]
def failed_snapshot_reasons(row):
    """{str(snapshot): 'non_converged' | 'sanity_rejected'} for post-outage snapshots without an accepted solution."""
    nc = {str(s) for s in row["nonconverged_snapshots"]}
    return {str(s): ("non_converged" if str(s) in nc else "sanity_rejected") for s in row["failed_snapshots"]}

# [COPY of GNN_Powerflow_V2.7_Contingency::load_n1_dataset]
def load_n1_dataset(n1_index_path):
    """Returns (nets, header, rows_df); nets[i] is the .nc at rows_df.file_position == i
    (solved + partial outages; solved snapshots only)."""
    meta_path = n1_index_path[:-len(".json")] + "_meta.json"
    header, rows = read_n1_sidecar(meta_path)
    missing = [f for f in BASE_CHECK_FIELDS if f not in rows.columns]   # [STSI 151026]:task 093: no fallback for old sidecars
    if missing or not all(isinstance(v, list) for f in BASE_CHECK_FIELDS for v in rows[f]):
        raise ValueError(f"{meta_path}: sidecar rows lack base-check field(s) {missing or list(BASE_CHECK_FIELDS)}; "
                         "regenerate the N-1 dataset with DataGen task 092+ (no fallback)")
    granularity = (header.get("outage_check") or {}).get("granularity")   # [STSI 201026]:task 100 D-F3d: no fallback
    missing = [f for f in (*OUTAGE_CHECK_FIELDS, "n_solved_snapshots") if f not in rows.columns]
    if granularity != "per_snapshot" or missing or not all(isinstance(v, list) for f in OUTAGE_CHECK_FIELDS
                                                           for v in rows[f]):
        raise ValueError(f"{meta_path}: sidecar lacks the per-snapshot outage check (outage_check granularity="
                         f"{granularity!r}, missing {missing}); regenerate the N-1 dataset with DataGen task 099+ "
                         "(no fallback)")
    nets = load_dataset_list(n1_index_path)
    stored = rows[rows.file_position.notna()]
    assert list(stored.file_position.astype(int)) == list(range(len(nets))), "sidecar/index misaligned"
    bad = [int(p) for p, n in zip(stored.file_position, stored.n_solved_snapshots)
           if len(nets[int(p)].snapshots) != int(n)]
    assert not bad, f"stored snapshot count != n_solved_snapshots at file positions {bad[:10]}"
    return nets, header, rows

# [COPY of GNN_Powerflow_V2.7_Contingency::load_source_base_nets]
def load_source_base_nets(rows_df):
    """{(source_index, source_position): pre-outage base net} for every base case referenced in the sidecar."""
    out = {}
    for src, grp in rows_df.groupby("source_index"):
        for pos, _, net in load_dataset_subset(src, sorted(set(int(p) for p in grp.source_position))):
            out[(src, pos)] = net
    return out

# [COPY of GNN_Powerflow_V2.7_Contingency::decode_line_limits]
def decode_line_limits(header):
    """Sidecar header line_limits -> (real {system: {name: (v, src)}}, synth {(system, bus0, bus1, circuit): v}).
    [STSI 281026]:task 117 F5 - only branch-keyed sidecars (DataGen tasks 115/124); name-keyed ones raise (no fallback)."""
    lim = header["line_limits"]
    if lim.get("keying") != LINE_LIMIT_KEYING:
        raise ValueError("N-1 sidecar line_limits are name-keyed (pre-task-115, audit F5); regenerate the N-1 dataset "
                         "with DataGen task 124+ (runbook 122 Part C; no fallback)")
    real = {s: {n: (float(v), src) for n, (v, src) in d.items()} for s, d in lim["real"].items()}
    synth = {(s, b0, b1, c): float(v) for s, b0, b1, c, v in lim["synthetic"]}
    return real, synth

# [COPY of GNN_Powerflow_V2.6_DataGen::load_dataset_subset]
def load_dataset_subset(index_path, positions):
    """Load only `positions` of a load_dataset_list-format index; same path logic as load_dataset_list."""
    with open(index_path, "r", encoding="utf-8") as f:
        meta = json.load(f)
    base_dir = os.path.dirname(index_path)
    return [(int(p), meta["files"][p], pypsa.Network(os.path.join(base_dir, meta["files"][p]))) for p in positions]

# [COPY of GNN_Powerflow_V2.6_DataGen::read_n1_sidecar]
def read_n1_sidecar(meta_path):
    """Returns (header_dict_without_rows, pd.DataFrame(rows))."""
    with open(meta_path, "r", encoding="utf-8") as f:
        meta = json.load(f)
    rows = meta.pop("rows")
    return meta, pd.DataFrame(rows)

# [STSI 281026]:task 117 F5 - branch-keyed limit helpers (LINE_LIMIT_KEYING, is_base_bus, branch_key) and attach_line_limits re-copied verbatim from DataGen cell 74 (task 115)
# [COPY of GNN_Powerflow_V2.6_DataGen::LINE_LIMIT_KEYING]
LINE_LIMIT_KEYING = "system_bus0_bus1_circuit"   # [STSI 281026]:task 115 F5 - sidecar marker; consumers refuse other keyings

# [COPY of GNN_Powerflow_V2.6_DataGen::is_base_bus]
def is_base_bus(system, bus):
    """Bus of the unmodified base system (added buses are 'Bus <k>'; IEEE9 base = Bus 1..9).
    [STSI 281026]:task 115 F5"""
    b = str(bus)
    if system == "ieee9":
        return b.startswith("Bus ") and b[4:].strip().isdigit() and int(b[4:]) <= 9
    return not b.startswith("Bus ")


# [COPY of GNN_Powerflow_V2.6_DataGen::branch_key]
def branch_key(system, name, bus0, bus1):
    """(bus0, bus1, circuit) of a physical base line, None for lines touching an added bus (never rated).
    circuit = group 3 of '^L-(\\d+)-(\\d+)-(.*)$' stripped ('L-6-8-1 ' -> '1'), else the line name (IEEE9 'Line k').
    Self-contained (local import) so that consumer [COPY of ...] blocks need no extra globals.
    [STSI 281026]:task 115 F5"""
    import re
    if not (is_base_bus(system, bus0) and is_base_bus(system, bus1)):
        return None
    m = re.match(r"^L-(\d+)-(\d+)-(.*)$", str(name))
    return (str(bus0), str(bus1), m.group(3).strip() if m else str(name))


# [COPY of GNN_Powerflow_V2.6_DataGen::attach_line_limits]
def attach_line_limits(net, system, real_ratings, synthetic_limits):
    """Write lines.s_limit_pu / lines.limit_source in place (pass a copy). Never touches s_nom/v_nom.
    [STSI 281026]:task 115 F5 - synthetic by (system, bus0, bus1, circuit); added-bus lines unrated (NaN);
    real ratings by name (pair-correct on base lines, R-D1 8) for base lines only."""
    s_lim, src = [], []
    for name, row in net.lines.iterrows():
        k = branch_key(system, name, row.bus0, row.bus1)
        if k is None:
            v, s = float("nan"), "unrated"
        elif name in real_ratings:
            v, s = real_ratings[name]
        else:
            v, s = synthetic_limits.get((system, *k), float("nan")), "synthetic"
        s_lim.append(float(v)); src.append(s)
    net.lines["s_limit_pu"] = s_lim
    net.lines["limit_source"] = src
    return net

In [ ]:
# ==== UQ HARNESS (task 109) — ORIGIN; method-agnostic, contract in -> DataFrames out ====
# [STSI 261026]:new cell (task 109) - strata views, point / probabilistic metrics, AUROC, selective risk, σ reliability, over-confidence, group bootstrap, [PROP-2] line strata / loading intervals / conditional coverage
from sklearn.metrics import roc_auc_score
# ---- [PROP-2] line strata (task 109) — pu; one pooled stratum per quantity; two-sided; evaluation-only bands ----
LINE_STRATA = (("smax", "line"), ("p0", "line"), ("q0", "line"))       # D-P2-7; D-P2-2
LINE_SMAX_BANDS_PU = (0.0, 0.02, 0.05, 0.2, 0.5, 1.0, np.inf)          # predicted |S|max bands (evaluation only)
LINE_LOADING_BANDS = (0.0, 0.5, 0.9, 0.95, 1.05, 1.2, np.inf)          # true loading bands, rated rows (evaluation only)
IEEE57_LIMIT_SYSTEM = "ieee57"                                         # system key of the N-1 sidecar line_limits

STRATA_ALL = UQ_STRATA + (("dtheta", "edge"),) + LINE_STRATA          # [PROP-2]
HARNESS_COLS = ("method", "population", "stratum", "metric", "value", "lo", "hi")
CAL_POPULATIONS = ("cal", "cal_clean")  # X4: contracts of these populations may only reach cal2_fit
DRMSE_FLAG_PCT = 2.0                  # Q1: report-only flag when RMSE rises by more than 2 % against the reference
SIGMA_EPS = 1e-12                     # guards σ = 0 in |err|/σ, NLL and floors (unknown coords of σ-methods have σ > 0)
NOTE_DET = "n/a (deterministic: no σ)"
NOTE_FLAT = "flat (deterministic: no σ ranking)"


def stratum_name(stratum):
    return f"{stratum[0]}|{stratum[1]}"


def _state_rows(c, stratum):
    """(column j, row mask) of a state stratum: bus type matches and the coordinate is `unknown`."""
    j = STATE_COLS.index(stratum[0])
    return j, (c["bus_type"] == BUS_TYPE_CODE[stratum[1]]) & c["unknown"][:, j]


def line_view(c, qty):
    """(y, mu, sd|None, samples|None, groups, crps_fair|None) for one LINE_QTYS column; None if c['line'] is None."""
    L = c.get("line")
    if L is None:
        return None
    k = LINE_QTYS.index(qty)
    return (L["y"][:, k], L["mean"][:, k], None if L["std"] is None else L["std"][:, k],
            None if L["samples"] is None else L["samples"][:, :, k], L["net"],
            L["extra"]["crps_fair"][:, k] if "crps_fair" in L["extra"] else None)


def stratum_view(c, stratum):
    """(y, mu, sd|None, samples|None, groups) for unknown coords of one stratum (edge stratum from c['edge']).
    [PROP-2] line strata: first five items of line_view(c, qty); None when c['line'] is None (stratum skipped)."""
    col, kind = stratum
    if kind == "line":
        v = line_view(c, col)
        return None if v is None else v[:5]
    if kind == "edge":
        e = c["edge"]
        return e["dt_true"], e["dt_mean"], e["dt_std"], e["dt_samples"], e["net"]
    j, m = _state_rows(c, stratum)
    return (c["y"][m, j], c["mean"][m, j], None if c["std"] is None else c["std"][m, j],
            None if c["samples"] is None else c["samples"][:, m, j], c["net"][m])


def _stratum_crps_extra(c, stratum):
    """Stored fair ensemble CRPS of a stratum (B1 without samples): state extra['crps_fair'] [R,4], edge
    edge['dt_crps_fair'], [PROP-2] line line_view(...)[5]; None if absent."""
    col, kind = stratum
    if kind == "line":
        v = line_view(c, col)
        return None if v is None else v[5]
    if kind == "edge":
        return c["edge"].get("dt_crps_fair")
    if "crps_fair" not in (c.get("extra") or {}):
        return None
    j, m = _state_rows(c, stratum)
    return c["extra"]["crps_fair"][m, j]


def interval_view(c, ints_lv, stratum):
    """(lo, hi) of one stratum from one level of cal2_apply(...), row-aligned with stratum_view(c, stratum); None if
    the interval set has no rows for it ([PROP-2] line section absent)."""
    col, kind = stratum
    if kind == "line":
        L = ints_lv.get("line")
        if L is None:
            return None
        k = LINE_QTYS.index(col)
        return L[0][:, k], L[1][:, k]
    if kind == "edge":
        return ints_lv["edge"]
    j, m = _state_rows(c, stratum)
    return ints_lv["state"][0][m, j], ints_lv["state"][1][m, j]


def _label(c, method=None, population=None):
    md = c.get("metadata") or {}
    return (method or md.get("analysis_method") or md.get("method"), population or md.get("population"))


def _is_ensemble(c):
    return (c.get("metadata") or {}).get("method") == "B1"


def assert_same_rows(a, b):
    """Two contracts describe identical rows in identical order (state net/snap/bus_type/unknown/y, edge net/snap/dt_true,
    [PROP-2] line net/snap/edge_pos/y when both have a line section). Required before any delta is computed."""
    for k in ("net", "snap", "bus_type", "unknown", "y"):
        assert np.array_equal(a[k], b[k]), f"state rows differ ({k}): contracts are not comparable"
    for k in ("net", "snap", "dt_true"):
        assert np.array_equal(a["edge"][k], b["edge"][k]), f"edge rows differ ({k}): contracts are not comparable"
    la, lb = a.get("line"), b.get("line")
    if la is not None and lb is not None:
        for k in ("net", "snap", "edge_pos", "y"):
            assert np.array_equal(la[k], lb[k]), f"line rows differ ({k}): contracts are not comparable"


def contract_groups(c):
    """Sorted unique group ids (c['net']); edge and [PROP-2] line rows carry the same ids, so they follow their net."""
    parts = [c["net"], c["edge"]["net"]] + ([c["line"]["net"]] if c.get("line") is not None else [])
    return np.unique(np.concatenate([np.asarray(p) for p in parts]))


def _gw_mean(gw, gi, x, G):
    """Group-weighted mean of row values x [n] for every weight row of gw [K,G] -> [K]. +inf rows (unbounded intervals)
    make every resample that contains them +inf; NaN where the total weight is 0."""
    x = np.asarray(x, np.float64)
    K = gw.shape[0]
    if x.size == 0:
        return np.full(K, np.nan)
    n = np.bincount(gi, minlength=G).astype(np.float64)
    fin = np.isfinite(x)
    s = np.bincount(gi, weights=np.where(fin, x, 0.0), minlength=G)
    den = gw @ n
    with np.errstate(invalid="ignore", divide="ignore"):
        out = (gw @ s) / den
    if not fin.all():
        present = (gw > 0).astype(np.float64)
        out = np.where(present @ np.bincount(gi, weights=(x == np.inf).astype(np.float64), minlength=G) > 0, np.inf, out)
        out = np.where(present @ np.bincount(gi, weights=(np.isnan(x) | (x == -np.inf)).astype(np.float64),
                                             minlength=G) > 0, np.nan, out)
    return np.where(den > 0, out, np.nan)


def _gw_joint(gw, gi, ok, G):
    """Group-weighted fraction of groups whose rows are ALL ok (per-network joint coverage) -> [K]."""
    present = np.bincount(gi, minlength=G) > 0
    allok = present & (np.bincount(gi, weights=1.0 - np.asarray(ok, np.float64), minlength=G) == 0)
    den = gw @ present.astype(np.float64)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(den > 0, (gw @ allok.astype(np.float64)) / den, np.nan)


def _gw_quantile(gw, gi, x, q):
    """Group-weighted q-quantile (inverted CDF) per weight row -> [K]; row 0 (all ones) uses np.percentile (linear)."""
    x = np.asarray(x, np.float64)
    out = np.full(gw.shape[0], np.nan)
    if x.size == 0:
        return out
    o = np.argsort(x, kind="stable"); xs, gs = x[o], gi[o]
    for k in range(gw.shape[0]):
        cw = np.cumsum(gw[k, gs])
        if cw[-1] > 0:
            out[k] = xs[min(int(np.searchsorted(cw, q * cw[-1])), xs.size - 1)]
    out[0] = float(np.percentile(x, 100.0 * q))
    return out


def _crps_fair(S, y, chunk=250_000):
    """Fair ensemble CRPS over the leading member axis, chunked over rows; M = 1 -> |member − y|."""
    S, y = np.asarray(S, np.float64), np.asarray(y, np.float64)
    if S.shape[0] < 2:
        return np.abs(S[0] - y)
    if not y.shape[0]:
        return np.zeros(y.shape)
    return np.concatenate([ens_crps_fair(S[:, i:i + chunk], y[i:i + chunk]) for i in range(0, y.shape[0], chunk)])


def group_bootstrap(metric_fn, c, n_boot=500, seed=0, **kw):
    """Resample unique c['net'] groups with replacement (edges follow their net); returns point, lo2.5, hi97.5.
    metric_fn(c, gw=[K,G] group weights, ug=[G] group ids, **kw) -> list of row dicts {stratum, metric, value, ...}; row 0
    of gw is all ones (point estimate), rows 1..n_boot are multinomial group counts. A scalar value has no CI (lo/hi NaN).
    Returns a tidy DataFrame: method, population, stratum, metric, value, lo, hi (+ the extra keys of the rows)."""
    method, population = _label(c, kw.pop("method", None), kw.pop("population", None))
    assert population not in CAL_POPULATIONS, f"cal contracts ({population!r}) are used by cal2_fit only (X4)"
    ug = contract_groups(c)
    G = ug.size
    rng = np.random.default_rng(seed)
    W = (rng.multinomial(G, np.full(G, 1.0 / G), size=int(n_boot)).astype(np.float64) if (n_boot and G)
         else np.zeros((0, G)))
    gw = np.vstack([np.ones((1, G)), W])
    out = []
    for r in metric_fn(c, gw=gw, ug=ug, **kw):
        r = dict(r)
        v = np.atleast_1d(np.asarray(r.pop("value"), np.float64))
        lo = hi = np.nan
        if v.size == gw.shape[0] and v.size > 1:
            b = v[1:][~np.isnan(v[1:])]
            if b.size:
                with np.errstate(invalid="ignore"):
                    lo, hi = (float(x) for x in np.percentile(b, [2.5, 97.5]))
        out.append({"method": method, "population": population, "stratum": r.pop("stratum"), "metric": r.pop("metric"),
                    "value": float(v[0]), "lo": lo, "hi": hi, **r})
    cols = list(HARNESS_COLS) + sorted({k for r in out for k in r} - set(HARNESS_COLS))
    return pd.DataFrame(out, columns=cols)


def _views(c):
    """[(stratum, name, y64, mu64, sd64|None, samples|None, groups)] for every stratum with rows ([PROP-2] None skipped)."""
    out = []
    for st in STRATA_ALL:
        v = stratum_view(c, st)
        if v is None or not len(v[0]):
            continue
        y, mu, sd, S, g = v
        out.append((st, stratum_name(st), np.asarray(y, np.float64), np.asarray(mu, np.float64),
                    None if sd is None else np.asarray(sd, np.float64), S, np.asarray(g)))
    return out


def point_metrics(c, ref=None, n_boot=500, seed=0, method=None, population=None):
    """Rows per stratum: n, RMSE, MAE; if ref (retrained deterministic reference, same population): dRMSE_pct and
    flag_gt2pct on the SAME rows (asserted), paired bootstrap (same group weights for both)."""
    if ref is not None:
        assert_same_rows(c, ref)
    def fn(c, gw, ug):
        G, rows = ug.size, []
        for st, name, y, mu, _, _, g in _views(c):
            gi, e = np.searchsorted(ug, g), mu - y
            rmse = np.sqrt(_gw_mean(gw, gi, e ** 2, G))
            rows += [dict(stratum=name, metric="n", value=float(y.size)),
                     dict(stratum=name, metric="RMSE", value=rmse),
                     dict(stratum=name, metric="MAE", value=_gw_mean(gw, gi, np.abs(e), G))]
            rv = None if ref is None else stratum_view(ref, st)
            if rv is None:
                continue
            rr = np.sqrt(_gw_mean(gw, gi, (np.asarray(rv[1], np.float64) - np.asarray(rv[0], np.float64)) ** 2, G))
            with np.errstate(invalid="ignore", divide="ignore"):
                d = 100.0 * (rmse - rr) / rr
            rows += [dict(stratum=name, metric="dRMSE_pct", value=d),
                     dict(stratum=name, metric="flag_gt2pct", value=float(d[0] > DRMSE_FLAG_PCT))]
        return rows
    return group_bootstrap(fn, c, n_boot=n_boot, seed=seed, method=method, population=population)


def _raw_half_width(c, sd, level):
    """Raw (pre-conformal) half width for interval_type gaussian / student_t; None otherwise (det, B1 conformal_only)."""
    md = c.get("metadata") or {}
    it = md.get("interval_type")
    if sd is None or it not in ("gaussian", "student_t"):
        return None
    if it == "student_t":
        from scipy.stats import t as _t
        nu = float(md["nu"])
        return _t.ppf(0.5 + level / 2, nu) * sd / math.sqrt(nu / (nu - 2))
    return z_of(level) * sd


def prob_metrics(c, intervals=None, levels=None, n_boot=500, seed=0, method=None, population=None):
    """Per stratum and level: coverage / width / interval score (conformal if `intervals` = cal2_apply(...) is given,
    else the raw gaussian / student_t interval; none for det and B1 'conformal_only'); simultaneous intervals also report
    joint_coverage (all rows of a network covered). NLL: gaussian (A1/C) | moment-matched gaussian (B1, labelled) | n/a
    (det). CRPS: gaussian | fair ensemble CRPS (B1, from samples or stored extra) | MAE (det, point mass)."""
    md = c.get("metadata") or {}
    if intervals is not None:
        levels = tuple(sorted(intervals)) if levels is None else tuple(levels)
        itype = "conformal_" + str(next(iter(intervals.values())).get("mode", "marginal"))
    else:
        levels = tuple(md.get("levels") or UQ_LEVELS) if levels is None else tuple(levels)
        itype = md.get("interval_type")
    ens, fam = _is_ensemble(c), md.get("interval_type")
    def fn(c, gw, ug):
        G, rows = ug.size, []
        for st, name, y, mu, sd, S, g in _views(c):
            gi = np.searchsorted(ug, g)
            sd_ = None if sd is None else np.maximum(sd, SIGMA_EPS)
            for lv in levels:
                if intervals is not None:
                    iv = interval_view(c, intervals[lv], st)
                    if iv is None:
                        continue
                    lo, hi = (np.asarray(a, np.float64) for a in iv)
                    if np.isnan(lo).all():
                        continue
                else:
                    h = _raw_half_width(c, sd_, lv)
                    if h is None:
                        continue
                    lo, hi = mu - h, mu + h
                cov = ((lo <= y) & (y <= hi)).astype(np.float64)
                w = hi - lo
                with np.errstate(invalid="ignore", over="ignore"):
                    isc = np.where(np.isfinite(w), interval_score(y, lo, hi, lv), np.inf)
                tag = dict(level=float(lv), interval=itype)
                rows += [dict(stratum=name, metric="coverage", value=_gw_mean(gw, gi, cov, G), **tag),
                         dict(stratum=name, metric="width", value=_gw_mean(gw, gi, w, G), **tag),
                         dict(stratum=name, metric="interval_score", value=_gw_mean(gw, gi, isc, G), **tag)]
                if itype == "conformal_simultaneous":
                    rows.append(dict(stratum=name, metric="joint_coverage", value=_gw_joint(gw, gi, cov, G), **tag))
            if sd_ is None:
                rows.append(dict(stratum=name, metric="NLL", value=np.nan, note=NOTE_DET))
                rows.append(dict(stratum=name, metric="CRPS", value=_gw_mean(gw, gi, np.abs(y - mu), G),
                                 note="MAE (deterministic point mass)"))
                continue
            if fam == "student_t":
                from scipy.stats import t as _t
                nu = float(md["nu"]); scl = sd_ / math.sqrt(nu / (nu - 2))
                rows.append(dict(stratum=name, metric="NLL", value=_gw_mean(gw, gi, -_t.logpdf(y, nu, mu, scl), G),
                                 note="student_t"))
            else:
                rows.append(dict(stratum=name, metric="NLL", value=_gw_mean(gw, gi, gauss_nll(y, mu, sd_), G),
                                 note=("moment-matched gaussian (ensemble mean/std; members have no density)" if ens
                                       else "gaussian")))
            if ens:
                crps = _crps_fair(S, y) if S is not None else _stratum_crps_extra(c, st)
                note = "fair ensemble CRPS (" + ("samples" if S is not None else "stored extra") + ")"
                val = np.full(gw.shape[0], np.nan) if crps is None else _gw_mean(gw, gi, crps, G)
            else:
                val, note = _gw_mean(gw, gi, gauss_crps(y, mu, sd_), G), ("gaussian" if fam != "student_t"
                                                                           else "gaussian (moment-matched student_t)")
            rows.append(dict(stratum=name, metric="CRPS", value=val, note=note))
        return rows
    return group_bootstrap(fn, c, n_boot=n_boot, seed=seed, method=method, population=population)


def _auroc_weighted(gw, gi, lab, score):
    """Weighted Mann-Whitney AUROC (ties count 1/2) of `score` for binary `lab` per weight row -> [K]."""
    o = np.argsort(score, kind="stable")
    s, l, g = score[o], np.asarray(lab, np.float64)[o], gi[o]
    starts = np.flatnonzero(np.r_[True, s[1:] != s[:-1]])
    out = np.full(gw.shape[0], np.nan)
    for k in range(gw.shape[0]):
        w = gw[k, g]
        P, N = np.add.reduceat(w * l, starts), np.add.reduceat(w * (1.0 - l), starts)
        tp, tn = P.sum(), N.sum()
        if tp > 0 and tn > 0:
            out[k] = float((P * (np.cumsum(N) - 0.5 * N)).sum() / (tp * tn))
    return out


def large_error_auroc(c, thresholds, n_boot=500, seed=0, method=None, population=None):
    """AUROC of σ for the large-error label |err| > thresholds[stratum] (ref_thresholds of the deterministic reference on
    the same population); point = sklearn roc_auc_score, CI = weighted AUROC on group resamples; n/a if sd is None."""
    def fn(c, gw, ug):
        rows = []
        for st, name, y, mu, sd, _, g in _views(c):
            if st not in thresholds:
                continue
            tag = dict(stratum=name, metric="AUROC_large_error", threshold=float(thresholds[st]))
            if sd is None:
                rows.append(dict(value=np.nan, note=NOTE_DET, **tag)); continue
            lab = np.abs(mu - y) > thresholds[st]
            if lab.all() or not lab.any():
                rows.append(dict(value=np.nan, note="n/a (single class)", **tag)); continue
            v = _auroc_weighted(gw, np.searchsorted(ug, g), lab, sd)
            v[0] = float(roc_auc_score(lab, sd))
            rows.append(dict(value=v, note="", **tag))
        return rows
    return group_bootstrap(fn, c, n_boot=n_boot, seed=seed, method=method, population=population)


def _selective_weighted(gw, gi, e2, sd, fracs):
    """RMSE of the fraction `fracs` of rows with the smallest σ (weight-based cut) per weight row -> [K, F]."""
    o = np.argsort(sd, kind="stable")
    e, g = e2[o], gi[o]
    f = np.asarray(fracs, np.float64) * (1.0 - 1e-12)
    out = np.full((gw.shape[0], f.size), np.nan)
    for k in range(gw.shape[0]):
        w = gw[k, g]
        cw, ce = np.cumsum(w), np.cumsum(w * e)
        if cw[-1] <= 0:
            continue
        idx = np.minimum(np.searchsorted(cw, f * cw[-1]), cw.size - 1)
        out[k] = np.sqrt(ce[idx] / cw[idx])
    return out


def selective_risk(c, fracs=np.linspace(0.1, 1.0, 10), n_boot=500, seed=0, method=None, population=None):
    """RMSE of the fraction with smallest σ (ties keep row order). Deterministic contracts: flat (overall RMSE)."""
    def fn(c, gw, ug):
        G, rows = ug.size, []
        for st, name, y, mu, sd, _, g in _views(c):
            gi, e2 = np.searchsorted(ug, g), (mu - y) ** 2
            if sd is None:
                flat = np.sqrt(_gw_mean(gw, gi, e2, G))
                rows += [dict(stratum=name, metric="selective_RMSE", frac=float(f), value=flat, note=NOTE_FLAT) for f in fracs]
                continue
            v = _selective_weighted(gw, gi, e2, sd, fracs)
            rows += [dict(stratum=name, metric="selective_RMSE", frac=float(f), value=v[:, i], note="")
                     for i, f in enumerate(fracs)]
        return rows
    return group_bootstrap(fn, c, n_boot=n_boot, seed=seed, method=method, population=population)


def sigma_reliability(c, n_bins=10, n_boot=500, seed=0, method=None, population=None):
    """Per σ-decile (edges from the point sample, fixed across resamples): mean σ, RMSE, mean |err|. n/a for det."""
    def fn(c, gw, ug):
        G, rows = ug.size, []
        for st, name, y, mu, sd, _, g in _views(c):
            if sd is None:
                rows.append(dict(stratum=name, metric="sigma_reliability", value=np.nan, note=NOTE_DET)); continue
            gi, e = np.searchsorted(ug, g), mu - y
            edges = np.quantile(sd, np.linspace(0.0, 1.0, n_bins + 1))
            b = np.clip(np.searchsorted(edges[1:-1], sd, side="right"), 0, n_bins - 1)
            for bi in range(n_bins):
                m = b == bi
                if not m.any():
                    continue
                tag = dict(stratum=name, bin=bi, sigma_lo=float(edges[bi]), sigma_hi=float(edges[bi + 1]), note="")
                rows += [dict(metric="mean_sigma", value=_gw_mean(gw, gi[m], sd[m], G), **tag),
                         dict(metric="RMSE", value=np.sqrt(_gw_mean(gw, gi[m], e[m] ** 2, G)), **tag),
                         dict(metric="mean_abs_err", value=_gw_mean(gw, gi[m], np.abs(e[m]), G), **tag)]
        return rows
    return group_bootstrap(fn, c, n_boot=n_boot, seed=seed, method=method, population=population)


def overconfidence(c, n_boot=500, seed=0, method=None, population=None):
    """mean(|err|/σ) and p90(|err|/σ) per stratum (≈ 0.80 / 1.64 for a calibrated gaussian). n/a for det."""
    def fn(c, gw, ug):
        G, rows = ug.size, []
        for st, name, y, mu, sd, _, g in _views(c):
            if sd is None:
                rows += [dict(stratum=name, metric=m, value=np.nan, note=NOTE_DET) for m in ("mean_abs_z", "p90_abs_z")]
                continue
            gi, z = np.searchsorted(ug, g), np.abs(mu - y) / np.maximum(sd, SIGMA_EPS)
            rows += [dict(stratum=name, metric="mean_abs_z", value=_gw_mean(gw, gi, z, G), note=""),
                     dict(stratum=name, metric="p90_abs_z", value=_gw_quantile(gw, gi, z, 0.90), note="")]
        return rows
    return group_bootstrap(fn, c, n_boot=n_boot, seed=seed, method=method, population=population)


def ref_thresholds(ref_c, q=0.90):
    """{stratum: q-quantile of |err|} of the deterministic reference (large-error label; one threshold for all methods)."""
    return {st: float(np.quantile(np.abs(mu - y), q)) for st, _, y, mu, *_ in _views(ref_c)}


def loading_intervals(c, ints, level=UQ_PRIMARY_LEVEL):
    """Per line row: y, mu, lo, hi as loading fractions = smax pu values / limit (D-P2-1); NaN where unrated."""
    L, k = c["line"], LINE_QTYS.index("smax"); lo, hi = ints[level]["line"]
    f = lambda x: line_loading(x, L["limit"])
    return pd.DataFrame({"net": L["net"], "snap": L["snap"], "edge_pos": L["edge_pos"], "limit": L["limit"],
                         "y": f(L["y"][:, k]), "mu": f(L["mean"][:, k]), "lo": f(lo[:, k]), "hi": f(hi[:, k])})


def _band_labels(edges):
    return [f"[{a:g}, {b:g})" for a, b in zip(edges[:-1], edges[1:])]


def line_conditional_coverage(c, ints, level=UQ_PRIMARY_LEVEL, by="pred_smax", n_boot=500, seed=0, method=None,
                              population=None):
    """[PROP-2] evaluation-only conditional coverage of the LINE_STRATA intervals (never a calibration stratum).
    by="pred_smax": band of line.mean[:, smax] over LINE_SMAX_BANDS_PU (all rows);
    by="true_loading": band of line_loading(line.y[:, smax], limit) over LINE_LOADING_BANDS (rated rows only; empty if none).
    Tidy rows (metric n | coverage | width) with columns band, by, level; CI by group_bootstrap over line.net."""
    assert by in ("pred_smax", "true_loading"), by
    L = c.get("line")
    if L is None or ints[level].get("line") is None:
        return pd.DataFrame(columns=list(HARNESS_COLS) + ["band", "by", "level"])
    ks = LINE_QTYS.index("smax")
    if by == "pred_smax":
        key, edges = np.asarray(L["mean"][:, ks], np.float64), LINE_SMAX_BANDS_PU
        keep = np.ones(key.size, bool)
    else:
        key, edges = line_loading(L["y"][:, ks], L["limit"]), LINE_LOADING_BANDS
        keep = np.isfinite(key)
    band = np.clip(np.searchsorted(np.asarray(edges[1:-1]), key, side="right"), 0, len(edges) - 2)
    labels = _band_labels(edges)
    lo_all, hi_all = ints[level]["line"]
    def fn(c, gw, ug):
        G, rows = ug.size, []
        gi = np.searchsorted(ug, np.asarray(L["net"]))
        for st in LINE_STRATA:
            k = LINE_QTYS.index(st[0])
            y, lo, hi = (np.asarray(a[:, k], np.float64) for a in (L["y"], lo_all, hi_all))
            if np.isnan(lo).all():
                continue
            for b, lab in enumerate(labels):
                m = keep & (band == b)
                tag = dict(stratum=stratum_name(st), band=lab, by=by, level=float(level))
                rows.append(dict(metric="n", value=float(m.sum()), **tag))
                rows.append(dict(metric="coverage", value=_gw_mean(gw, gi[m], ((lo[m] <= y[m]) & (y[m] <= hi[m])), G), **tag))
                rows.append(dict(metric="width", value=_gw_mean(gw, gi[m], hi[m] - lo[m], G), **tag))
        return rows
    return group_bootstrap(fn, c, n_boot=n_boot, seed=seed, method=method, population=population)

In [ ]:
# ==== CAL-2 split conformal (task 109) — ORIGIN; cal used ONLY here ====
# [STSI 261026]:new cell (task 109) - val σ floors, marginal / simultaneous split conformal fit + apply (state, Δθ, [PROP-2] line smax/p0/q0 in pu, two-sided), q̂/z diagnostic


def sigma_floor_from_val(val_c, q=SIGMA_FLOOR_Q):
    """{stratum: floor} = q-quantile of the method's own σ on val in that stratum (state, Δθ, [PROP-2] line strata);
    None for deterministic contracts (std None). Floors never use cal (X4)."""
    if val_c["std"] is None:
        return None
    pop = (val_c.get("metadata") or {}).get("population")
    assert pop in (None, "", "val"), f"σ floors come from val only, got population {pop!r}"
    floors = {}
    for st in STRATA_ALL:
        v = stratum_view(val_c, st)
        if v is None or v[2] is None or not len(v[2]):
            continue
        floors[st] = max(float(np.quantile(np.asarray(v[2], np.float64), q)), SIGMA_EPS)
    return floors


def _k(n, level): return math.ceil((n + 1) * level)


def cal2_fit(cal_c, floors=None, levels=UQ_LEVELS, mode="marginal"):
    """{stratum: {level: q̂}}; mode 'marginal' (coordinate scores) | 'simultaneous' (max per cal network)."""
    assert mode in ("marginal", "simultaneous"), mode
    assert (cal_c["std"] is not None) == (floors is not None), \
        "σ-methods need val floors (sigma_floor_from_val); deterministic contracts take floors=None"
    out, n_scores = {}, {}
    for st in STRATA_ALL:
        v = stratum_view(cal_c, st)
        if v is None:                                     # [PROP-2] contract without a line section
            continue
        y, mu, sd, _, g = v
        y, mu = np.asarray(y, np.float64), np.asarray(mu, np.float64)
        s = np.abs(y - mu) / (np.maximum(np.asarray(sd, np.float64), floors[st]) if sd is not None else 1.0)
        if mode == "simultaneous":
            s = pd.Series(s).groupby(np.asarray(g)).max().to_numpy()
        s = np.sort(s); n = len(s)
        if n == 0:
            continue
        out[st] = {lv: (float(s[_k(n, lv) - 1]) if _k(n, lv) <= n else math.inf) for lv in levels}
        n_scores[st] = n
    md = cal_c.get("metadata") or {}
    return {"mode": mode, "qhat": out, "floors": floors, "n_cal_groups": int(len(np.unique(cal_c["net"]))),
            "n_scores": n_scores, "levels": tuple(levels), "cal_population": md.get("population"),
            "method": md.get("analysis_method") or md.get("method")}


def cal2_apply(c, cal):
    """{level: {"state": (lo [R,4], hi [R,4]), "edge": (lo, hi) [E], "line": (lo [L,5], hi [L,5]) | None, "mode"}};
    NaN on known coords and in line columns p1/q1 (not calibrated). Two-sided: μ ± q̂·max(σ, floor) (det: μ ± q̂), pu."""
    sig, fl, qh = c["std"] is not None, cal["floors"], cal["qhat"]
    assert sig == (fl is not None), "contract and calibration disagree on σ (σ-method vs deterministic)"
    def half(st, sd, lv):
        return qh[st][lv] * (np.maximum(np.asarray(sd, np.float64), fl[st]) if fl is not None else 1.0)
    out = {}
    mu = np.asarray(c["mean"], np.float64)
    e, L, est = c["edge"], c.get("line"), ("dtheta", "edge")
    emu = np.asarray(e["dt_mean"], np.float64)
    for lv in cal["levels"]:
        lo, hi = np.full(mu.shape, np.nan), np.full(mu.shape, np.nan)
        for st in UQ_STRATA:
            if st not in qh:
                continue
            j, m = _state_rows(c, st)
            h = half(st, c["std"][m, j] if sig else None, lv)
            lo[m, j], hi[m, j] = mu[m, j] - h, mu[m, j] + h
        if est in qh:
            h = half(est, e["dt_std"] if sig else None, lv)
            edge = (emu - h, emu + h)
        else:
            edge = (np.full(emu.shape, np.nan), np.full(emu.shape, np.nan))
        line = None
        if L is not None:                                 # [PROP-2] smax / p0 / q0 only; p1 / q1 stay NaN
            lmu = np.asarray(L["mean"], np.float64)
            llo, lhi = np.full(lmu.shape, np.nan), np.full(lmu.shape, np.nan)
            for st in LINE_STRATA:
                if st not in qh:
                    continue
                k = LINE_QTYS.index(st[0])
                h = half(st, L["std"][:, k] if sig else None, lv)
                llo[:, k], lhi[:, k] = lmu[:, k] - h, lmu[:, k] + h
            line = (llo, lhi)
        out[lv] = {"state": (lo, hi), "edge": edge, "line": line, "mode": cal["mode"]}
    return out


def qhat_over_z(cal):
    """σ-methods: q̂/z_level per stratum (1 = the raw gaussian interval was already calibrated). Empty for det."""
    cols = ["mode", "stratum", "level", "qhat", "z", "qhat_over_z"]
    if cal["floors"] is None:
        return pd.DataFrame(columns=cols)
    return pd.DataFrame([{"mode": cal["mode"], "stratum": stratum_name(st), "level": lv, "qhat": q, "z": z_of(lv),
                          "qhat_over_z": q / z_of(lv)} for st, d in cal["qhat"].items() for lv, q in d.items()],
                        columns=cols)

In [ ]:
# ==== UQ ANALYSIS CONFIG (task 109) — user-editable ====
# [STSI 261026]:new cell (task 109) - run config: paths, run keys, toggles, populations, manifest assertion
UQ_SPLIT_SEED = 42
DATASET_PATH = os.path.join(TRAINING_NETWORKS_DIR, "mixed_1500_w.json")
ORIGINAL_RUNS_PATH = os.path.join(TRAINING_RESULTS_DIR, "best_runs_57.pkl")
ORIGINAL_RUN_KEY = ("adap-fp0.1_ft0.3_flow-dcL0.1-dcG0.1-acL0.2-acG0.1_kvl0.3_balDC_bs64_lr1.00e-04_h128_L6_gatv2_"
    "manual_s42_hmwith_encoder_pn0_cb-res_agelu_do0.1_amedge_delta_vmresidual_phys60_ptdf60-85_dcfL20-65_dcfG40-75_"
    "acfL50_acfG80_kvl0_ramp10")
UQ_REF_RUNS_PATH  = os.path.join(TRAINING_RESULTS_DIR, f"uq_ref_ss{UQ_SPLIT_SEED}.pkl")
UQ_A1_PATH        = os.path.join(TRAINING_RESULTS_DIR, f"uq_a1_ss{UQ_SPLIT_SEED}.pkl")
UQ_C_POSTHOC_PATH = os.path.join(TRAINING_RESULTS_DIR, f"uq_c_posthoc_ss{UQ_SPLIT_SEED}.pkl")
UQ_B1_GLOBS       = [UQ_REF_RUNS_PATH, os.path.join(TRAINING_RESULTS_DIR, f"uq_b1_member_ss{UQ_SPLIT_SEED}_is*.pkl")]
UQ_B1_PHYSFREE_GLOB = os.path.join(TRAINING_RESULTS_DIR, f"uq_b1_physfree_ss{UQ_SPLIT_SEED}_is*.pkl")  # optional
IEEE57_PATH = os.path.join(TRAINING_NETWORKS_DIR, "ieee57_1500_w.json")
N1_INDEX_PATH = os.path.join(TRAINING_NETWORKS_DIR, "n1_ieee57_1500_u1_t50.json")   # [STSI 281026]:task 117 F5 - branch-keyed N-1 (= UC_N1 default, task 124); change together with UC_N1
RUN_IEEE118 = False
N_BOOT = 500
UQ_CACHE_DIR = os.path.join(TRAINING_RESULTS_DIR, "uq_analysis_cache")
METHODS = ("original", "reference", "A1", "B1", "C1", "C2")   # a missing pickle -> method skipped with a printed note

# ---- populations (task 109) — group id `net`: manifest position (mixed), dataset position (IEEE), N-1 base-case code ----
IEEE118_PATH = os.path.join(TRAINING_NETWORKS_DIR, "ieee118_1500_w.json")     # only if RUN_IEEE118
IEEE_PERM_SEED, IEEE_N_RANKS = 42, 100       # ranks 0-99 of the random.Random(42) permutation (= Analysis cell 42)
N1_GROUP_CODE_BASE = 1_000_000               # N-1 group code = source-file ordinal * BASE + source_position
SIGMA_METHODS = ("A1", "B1", "C1", "C2")     # σ-methods: floors from val; reference / original are deterministic
CAL_POPULATION = {"original": "cal_clean"}   # S-2: the published Original calibrates on the 75 clean cal nets; others "cal"
B1_DROP_SAMPLES_POPS = ("ieee57_n1",)        # keep_samples=False there; fair CRPS stored in extra / edge / line extra
B1_CHUNK_NETS = 200                          # networks per ensemble pass when samples are dropped
UQ_POPULATIONS = {
    "val": "manifest val (150) — σ floors only",
    "cal": "manifest cal (150) — CAL-2 for reference, A1, B1, C1, C2",
    "cal_clean": "manifest cal ∩ legacy 70/15/15 test (75) — CAL-2 for the published Original",
    "test": "manifest test (150) — internal, preliminary",
    "ieee57": "ieee57_1500_w ranks 0-99 — PRIMARY ordinary; [PROP-2] synthetic N-1 limits by branch (added-bus lines NaN)",   # [STSI 281026]:task 117 F5
    "ieee57_n1": "n1_ieee57_1500_u1_t50, base-failed snapshots dropped — PRIMARY N-1; limits on the nets",   # [STSI 281026]:task 117 F5
    "ieee118": "ieee118_1500_w ranks 0-99 — optional (RUN_IEEE118)",
}


def uq_ieee_positions(n, n_ranks=None, seed=None):
    """Dataset positions of ranks 0..n_ranks-1 of the random.Random(seed) permutation (= random.seed; random.shuffle)."""
    perm = list(range(int(n)))
    random.Random(IEEE_PERM_SEED if seed is None else seed).shuffle(perm)
    return perm[:(IEEE_N_RANKS if n_ranks is None else n_ranks)]


def uq_cal_clean_positions(manifest):
    """S-2: manifest cal ∩ split_positions(n, seed, SPLIT_PCTS_LEGACY)['test'] in cal order (n = 1500: ranks 1275-1349)."""
    legacy_test = set(split_positions(manifest["n"], manifest["split_seed"], SPLIT_PCTS_LEGACY)["test"])
    return [p for p in manifest["positions"]["cal"] if p in legacy_test]


def uq_assert_manifest(manifest, networks):
    """The loaded mixed list must be the manifest's list (n + fingerprint) and the manifest hash must reproduce."""
    assert int(manifest["split_seed"]) == UQ_SPLIT_SEED, f"manifest split_seed {manifest['split_seed']} != {UQ_SPLIT_SEED}"
    assert tuple(manifest["split_pcts"]) == tuple(SPLIT_PCTS_4WAY), f"manifest split_pcts {manifest['split_pcts']}"
    assert len(networks) == manifest["n"] and networks_fingerprint(networks) == manifest["fingerprint"], \
        f"loaded mixed list ({len(networks)} nets) does not match the split manifest fingerprint {manifest['fingerprint']}"
    assert make_split_manifest(networks, manifest["split_seed"], manifest["split_pcts"])["hash"] == manifest["hash"], \
        "split manifest hash does not reproduce from the loaded networks"
    print(f"[uq-config] manifest OK: hash {manifest['hash']} | fingerprint {manifest['fingerprint']} | "
          f"roles {manifest.get('role_counts') or {r: len(p) for r, p in manifest['positions'].items()}}")

In [ ]:
# ==== UQ PREDICT ALL (task 109) — contracts per method × population (cached); val floors + CAL-2 on cal / cal_clean ====
# [STSI 261026]:new cell (task 109) - load runs (missing -> skipped), split-hash + manifest assertion, populations (lazy), contract cache keyed by run keys + split hash + population, UQ_CAL[method]
import hashlib as _uq_hashlib
_UQ_DATA = globals().get("_UQ_DATA") or {}          # memo across re-runs: dataset path -> loaded mixed list
_UQ_POPS = {}                                       # population (name, spec) -> (nets, ids); built only on a cache miss


def _uq_skip(m, why):
    print(f"[uq-predict-all] {m}: SKIPPED — {why}")


def _uq_atomic_pickle(obj, path):
    with open(path + ".tmp", "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(path + ".tmp", path)


def uq_contract_concat(parts):
    """Row-concatenate P6 contracts of one method / population (samples on axis 1); metadata of the first part."""
    def cat(xs, axis=0):
        n_none = sum(x is None for x in xs)
        assert n_none in (0, len(xs)), "contract parts disagree on an optional array"
        return None if n_none else np.concatenate(xs, axis=axis)
    def cat_extra(ds):
        assert all(set(d) == set(ds[0]) for d in ds), "contract parts disagree on extra keys"
        return {k: np.concatenate([d[k] for d in ds], axis=0) for k in ds[0]}
    p0 = parts[0]
    c = {k: cat([p[k] for p in parts], 1 if k == "samples" else 0)
         for k in ("mean", "y", "unknown", "std", "bus_type", "net", "snap", "samples")}
    c["extra"] = cat_extra([p["extra"] for p in parts])
    c["edge"] = {k: cat([p["edge"].get(k) for p in parts], 1 if k == "dt_samples" else 0) for k in p0["edge"]}
    if p0["line"] is None:
        assert all(p["line"] is None for p in parts), "contract parts disagree on the line section"
        c["line"] = None
    else:
        c["line"] = {k: (cat_extra([p["line"]["extra"] for p in parts]) if k == "extra" else
                         cat([p["line"][k] for p in parts], 1 if k == "samples" else 0)) for k in p0["line"]}
    c["metadata"] = dict(p0["metadata"])
    return c


def uq_b1_predict_nosamples(runs, networks, net_ids, population, chunk=None):
    """B1 contract with keep_samples=False semantics: ensemble_predict_uq per chunk of networks, the fair CRPS is computed
    inside the loop from the member samples (state extra['crps_fair'] [R,4], edge['dt_crps_fair']; line extra already
    has it), then all samples are dropped."""
    chunk = int(chunk or B1_CHUNK_NETS)
    parts = []
    for i in range(0, len(networks), chunk):
        c = ensemble_predict_uq(runs, networks[i:i + chunk], net_ids=list(net_ids[i:i + chunk]), population=population,
                                keep_samples=True)
        c["extra"]["crps_fair"] = _crps_fair(c["samples"], c["y"]).astype(np.float32)
        c["edge"]["dt_crps_fair"] = _crps_fair(c["edge"]["dt_samples"], c["edge"]["dt_true"]).astype(np.float32)
        c["samples"], c["edge"]["dt_samples"] = None, None
        if c["line"] is not None:
            c["line"]["samples"] = None
        parts.append(c)
    out = uq_contract_concat(parts)
    out["metadata"].update(keep_samples=False, crps_fair_stored="extra.crps_fair | edge.dt_crps_fair | line.extra.crps_fair")
    return out


# ---- 1. runs (a missing pickle -> method skipped with a note) ----
UQ_RUNS = {}
if "original" in METHODS:
    if not os.path.isfile(ORIGINAL_RUNS_PATH):
        _uq_skip("original", f"pickle not found: {ORIGINAL_RUNS_PATH}")
    else:
        try:
            UQ_RUNS["original"] = load_runs_by_key([ORIGINAL_RUN_KEY], [ORIGINAL_RUNS_PATH])[ORIGINAL_RUN_KEY]
        except KeyError:
            _uq_skip("original", f"ORIGINAL_RUN_KEY not found in {ORIGINAL_RUNS_PATH}")
for _m, _p in (("reference", UQ_REF_RUNS_PATH), ("A1", UQ_A1_PATH)):
    if _m in METHODS:
        if os.path.isfile(_p):
            UQ_RUNS[_m] = load_sweep_results(_p)[0]
        else:
            _uq_skip(_m, f"pickle not found: {_p}")
if {"C1", "C2"} & set(METHODS):
    _by = {}
    if os.path.isfile(UQ_C_POSTHOC_PATH):
        _by = {(r.get("uq_method") or r["uq_artifacts"]["method"]): r for r in load_sweep_results(UQ_C_POSTHOC_PATH)}
    for _m in ("C1", "C2"):
        if _m in METHODS:
            if _m in _by:
                UQ_RUNS[_m] = _by[_m]
            else:
                _uq_skip(_m, f"no {_m} run in {UQ_C_POSTHOC_PATH}" if _by else f"pickle not found: {UQ_C_POSTHOC_PATH}")
if "B1" in METHODS:
    if not (os.path.isfile(UQ_B1_GLOBS[0]) and glob.glob(UQ_B1_GLOBS[1])):
        _uq_skip("B1", f"needs the reference {UQ_B1_GLOBS[0]} + >= 1 member pickle {UQ_B1_GLOBS[1]}")
    else:
        UQ_RUNS["B1"] = load_ensemble_members(UQ_B1_GLOBS)

# ---- 2. one split hash for all new runs; loaded mixed list == manifest ----
_uq_new = [UQ_RUNS[m] for m in ("reference", "A1", "C1", "C2") if m in UQ_RUNS] + list(UQ_RUNS.get("B1", []))
if _uq_new:
    assert_same_split(*_uq_new)
if DATASET_PATH not in _UQ_DATA:
    _UQ_DATA[DATASET_PATH] = load_dataset_list(DATASET_PATH)
_uq_mixed = _UQ_DATA[DATASET_PATH]
UQ_MANIFEST = (_uq_new[0]["split_manifest"] if _uq_new
               else make_split_manifest(_uq_mixed, UQ_SPLIT_SEED, SPLIT_PCTS_4WAY, dataset_source=DATASET_PATH))
uq_assert_manifest(UQ_MANIFEST, _uq_mixed)
UQ_SPLIT_HASH = UQ_MANIFEST["hash"]


# ---- 3. populations (built lazily, only when a contract is not cached) ----
def _uq_pop_spec(pop):
    """Population identity for the cache key (basenames: the cache stays valid when DATA_ROOT moves)."""
    b = os.path.basename
    if pop in ("val", "cal", "test", "cal_clean"):
        return f"{pop}|{b(DATASET_PATH)}|{UQ_MANIFEST['fingerprint']}"
    if pop in ("ieee57", "ieee118"):
        path = IEEE57_PATH if pop == "ieee57" else IEEE118_PATH
        lim = f"|limits:{b(N1_INDEX_PATH)}:{IEEE57_LIMIT_SYSTEM}" if pop == "ieee57" else ""
        return f"{pop}|{b(path)}|perm{IEEE_PERM_SEED}|ranks0-{IEEE_N_RANKS - 1}{lim}"
    if pop == "ieee57_n1":
        return f"{pop}|{b(N1_INDEX_PATH)}|drop_base_failed|code{N1_GROUP_CODE_BASE}"
    raise KeyError(pop)


def _uq_build_population(pop):
    """-> (networks, group ids) of one population (see UQ_POPULATIONS)."""
    man = UQ_MANIFEST
    if pop in ("val", "cal", "test"):
        return networks_for_role(_uq_mixed, man, pop), list(man["positions"][pop])
    if pop == "cal_clean":
        keep = set(uq_cal_clean_positions(man))
        sel = [(n, p) for n, p in zip(networks_for_role(_uq_mixed, man, "cal"), man["positions"]["cal"]) if p in keep]
        print(f"[populations] cal_clean: {len(sel)} of {len(man['positions']['cal'])} cal networks (legacy test ∩ cal)")
        return [n for n, _ in sel], [p for _, p in sel]
    if pop in ("ieee57", "ieee118"):
        path = IEEE57_PATH if pop == "ieee57" else IEEE118_PATH
        with open(path, "r", encoding="utf-8") as f:
            pos = uq_ieee_positions(len(json.load(f)["files"]))
        nets = [n for _, _, n in load_dataset_subset(path, pos)]
        if pop == "ieee57":
            # [PROP-2] synthetic IEEE57 limits for the ordinary IEEE57 population (same source and matching as Contingency)
            n1_header = read_n1_sidecar(N1_INDEX_PATH[:-len(".json")] + "_meta.json")[0]   # = load_n1_dataset header
            _real, _synth = decode_line_limits(n1_header)
            nets = [attach_line_limits(n.copy(), IEEE57_LIMIT_SYSTEM, _real.get(IEEE57_LIMIT_SYSTEM, {}), _synth)
                    for n in nets]
            _n_rated = int(nets[0].lines["s_limit_pu"].notna().sum())
            assert _n_rated > 0, "no IEEE57 line matched the N-1 synthetic limits"
            print(f"[limits] ieee57 ordinary: {_n_rated}/{len(nets[0].lines)} lines rated (synthetic, by branch; added-bus lines NaN)")   # [STSI 281026]:task 117 F5 - branch-keyed limits
        return nets, pos
    if pop == "ieee57_n1":
        global UQ_N1_ROWS
        nets, n1_header, rows = load_n1_dataset(N1_INDEX_PATH)
        stored = rows[rows.file_position.notna()].sort_values("file_position")
        srcs = sorted(stored.source_index.unique())
        out, ids, meta = [], [], []
        for _, row in stored.iterrows():
            net = nets[int(row.file_position)]
            keep = list(drop_base_failed(pd.DataFrame({"snapshot": list(net.snapshots)}), row)["snapshot"])
            n_drop = len(net.snapshots) - len(keep)
            if not keep:
                continue
            if n_drop:
                net = net.copy(snapshots=keep)
            code = srcs.index(row.source_index) * N1_GROUP_CODE_BASE + int(row.source_position)
            out.append(net); ids.append(code)
            meta.append({"group": code, "file_position": int(row.file_position), "source_index": row.source_index,
                         "source_position": int(row.source_position), "outage_name": row.outage_name,
                         "n_snapshots": len(keep), "n_base_failed_dropped": int(n_drop)})
        UQ_N1_ROWS = pd.DataFrame(meta)
        print(f"[populations] ieee57_n1: {len(out)} outage nets, {len(set(ids))} base cases (groups), "
              f"{sum(m['n_snapshots'] for m in meta)} snapshots after dropping base-failed snapshots")
        return out, ids
    raise KeyError(pop)


def uq_population(pop):
    key = (pop, _uq_pop_spec(pop))
    if key not in _UQ_POPS:
        _UQ_POPS[key] = _uq_build_population(pop)
    return _UQ_POPS[key]


# ---- 4. contracts per method × population (cache: run keys + split hash + population) ----
def _uq_run_keys(m):
    return [r["run_key"] for r in UQ_RUNS[m]] if m == "B1" else [UQ_RUNS[m]["run_key"]]


def uq_method_populations(m):
    return ((["val"] if m in SIGMA_METHODS else []) + [CAL_POPULATION.get(m, "cal"), "test", "ieee57", "ieee57_n1"]
            + (["ieee118"] if RUN_IEEE118 else []))


def uq_cache_path(m, pop):
    key = json.dumps({"run_keys": _uq_run_keys(m), "split_hash": UQ_SPLIT_HASH, "population": pop,
                      "population_spec": _uq_pop_spec(pop)}, sort_keys=True)
    return os.path.join(UQ_CACHE_DIR, f"{m}_{pop}_{_uq_hashlib.sha1(key.encode()).hexdigest()[:10]}.pkl")


def _uq_predict(m, nets, ids, pop):
    r = UQ_RUNS[m]
    if m in ("original", "reference"):
        return det_predict_uq(r, nets, net_ids=ids, population=pop)
    if m == "A1":
        return a1_predict_uq(r, nets, net_ids=ids, population=pop)
    if m in ("C1", "C2"):
        return bll_predict_uq(r, nets, net_ids=ids, population=pop)
    if m == "B1":
        if pop in B1_DROP_SAMPLES_POPS:
            return uq_b1_predict_nosamples(r, nets, ids, pop)
        return ensemble_predict_uq(r, nets, net_ids=ids, population=pop)
    raise KeyError(m)


def uq_contract(m, pop):
    """Load one cached contract (task 110 reads arrays through this)."""
    with open(UQ_CONTRACT_PATHS[(m, pop)], "rb") as f:
        return pickle.load(f)


os.makedirs(UQ_CACHE_DIR, exist_ok=True)
UQ_CONTRACT_PATHS, UQ_CACHE_STATS = {}, {"loaded": [], "computed": []}
UQ_METHODS_LOADED = [m for m in METHODS if m in UQ_RUNS]
for _m in UQ_METHODS_LOADED:
    for _pop in uq_method_populations(_m):
        _path = uq_cache_path(_m, _pop)
        if os.path.isfile(_path):
            UQ_CACHE_STATS["loaded"].append((_m, _pop))
        else:
            _nets, _ids = uq_population(_pop)
            _t0 = time.time()
            _c = _uq_predict(_m, _nets, _ids, _pop)
            _c["metadata"].update(analysis_method=_m, cache_file=os.path.basename(_path), split_hash_analysis=UQ_SPLIT_HASH)
            _uq_atomic_pickle(_c, _path)
            print(f"[uq-predict-all] {_m:9s} {_pop:9s}: {len(_c['y'])} state rows, "
                  f"{0 if _c['line'] is None else len(_c['line']['y'])} line rows ({time.time() - _t0:.1f}s) -> {os.path.basename(_path)}")
            del _c
            UQ_CACHE_STATS["computed"].append((_m, _pop))
        UQ_CONTRACT_PATHS[(_m, _pop)] = _path
print(f"[uq-predict-all] contracts: {len(UQ_CACHE_STATS['computed'])} computed, {len(UQ_CACHE_STATS['loaded'])} loaded "
      f"from cache ({UQ_CACHE_DIR}) | methods: {UQ_METHODS_LOADED} | split {UQ_SPLIT_HASH}")

# ---- 5. σ floors on val, CAL-2 (marginal + simultaneous) on cal (Original: cal_clean) ----
UQ_CAL = {}
for _m in UQ_METHODS_LOADED:
    _cal_pop = CAL_POPULATION.get(_m, "cal")
    _floors = sigma_floor_from_val(uq_contract(_m, "val")) if _m in SIGMA_METHODS else None
    _cal_c = uq_contract(_m, _cal_pop)                                     # X4: cal reaches cal2_fit only
    UQ_CAL[_m] = {"cal_population": _cal_pop, "floors": _floors,
                  "marginal": cal2_fit(_cal_c, _floors, mode="marginal"),
                  "simultaneous": cal2_fit(_cal_c, _floors, mode="simultaneous")}
    del _cal_c
if UQ_CAL:
    _q = pd.DataFrame({_m: {stratum_name(st): d[UQ_PRIMARY_LEVEL] for st, d in UQ_CAL[_m]["marginal"]["qhat"].items()}
                       for _m in UQ_CAL})
    print(f"[CAL-2] marginal q̂ at {UQ_PRIMARY_LEVEL:.0%} (σ-methods: in σ units; deterministic: pu / rad):")
    print(_q.to_string())

In [ ]:
# ==== UQ SECTIONS LIBRARY (task 110) — ORIGIN; sections (a)–(e) read cached contracts + UQ_CAL and display tidy tables ====
# [STSI 261026]:new cell (task 110) - section helpers: core / plot / [PROP-2] line tables, with_line_std / with_theta_std (same CAL-2 code path), cross-method tables + styling, generated decision aid + line-flow view, CSV export (UQ_Results/)
from IPython.display import Markdown, display
UQ_POP_ORDER = ("ieee57", "ieee57_n1", "test", "ieee118")         # primary first; the internal test is preliminary
UQ_POP_LABEL = {"ieee57": "IEEE57 ordinary (primary)", "ieee57_n1": "IEEE57 N-1 (primary)",
                "test": "internal test (preliminary)", "ieee118": "IEEE118 (optional)"}
UQ_LOADING_POPS = ("ieee57", "ieee57_n1")                          # D-P2-1: only these populations carry line limits
UQ_METHOD_LABEL = {"original": "Original (published)", "reference": "Reference (retrained)", "A1": "UQ-A1 mean-scale",
                   "B1": "UQ-B1 ensemble", "C1": "BLL C1", "C2": "BLL C2", "B1_physfree": "UQ-B1 physics-free"}
UQ_CROSS_METHODS = ("original", "reference", "A1", "B1", "C1", "C2")   # row order of every cross-method table
UQ_DECISION_METHODS = ("A1", "B1", "C1", "C2")                       # judged against the reference's constant-width CAL-2
UQ_EXPORT_DIR = os.path.join(TRAINING_RESULTS_DIR, "UQ_Results")    # created on run (outside the repo), never committed
STATE_STRATA = tuple(st for st in STRATA_ALL if st[1] != "line")    # UQ_STRATA + Δθ
STATE_STRATA_NAMES = tuple(stratum_name(st) for st in STATE_STRATA)
LINE_STRATA_NAMES = tuple(stratum_name(st) for st in LINE_STRATA)
LOADING_NEAR_LIMIT_BAND = "[0.95, 1.05)"                            # LINE_LOADING_BANDS label of the near-limit band
DECISION_COV_TOL = 0.02        # coverage counts as "matched" when its CI contains the level (or, without a CI, |cov − level| <= tol)
DECISION_MIN_WIN_FRAC = 0.5    # check (1) passes when the method is narrower than the reference in MORE than this share of matched strata
UQ_SECTIONS = {}               # section -> {table name: tidy DataFrame | dict | str}; filled by the section cells, exported as CSV
_UQ_SEC_MEMO = {}              # (method, population, tag) -> harness tables (computed once per session)


def uq_report_pops(method=None):
    """Evaluation populations in report order (ieee57, ieee57_n1, test, [ieee118]); with method: only those with a contract."""
    pops = [p for p in UQ_POP_ORDER if p != "ieee118" or RUN_IEEE118]
    return [p for p in pops if method is None or (method, p) in UQ_CONTRACT_PATHS]


def uq_available(method):
    return method in UQ_CAL and bool(uq_report_pops(method))


def pop_label(pop):
    return UQ_POP_LABEL.get(pop, pop)


def _pop_rank(pop):
    return UQ_POP_ORDER.index(pop) if pop in UQ_POP_ORDER else len(UQ_POP_ORDER)


def _empty():
    return pd.DataFrame(columns=list(HARNESS_COLS))


def _cat(dfs):
    dfs = [d for d in dfs if d is not None and len(d)]
    return pd.concat(dfs, ignore_index=True) if dfs else _empty()


def _strata_rows(df, names):
    return df[df["stratum"].isin(names)].reset_index(drop=True) if len(df) else df


def _note_row(method, pop, stratum, note, metric="note"):
    return pd.DataFrame([{"method": method, "population": pop, "stratum": stratum, "metric": metric, "value": np.nan,
                          "lo": np.nan, "hi": np.nan, "note": note}])


def loading_table(c, ints, levels=UQ_LEVELS, n_boot=500, seed=0, method=None, population=None):
    """[PROP-2] Loading-% intervals on RATED line rows only (D-P2-1: smax pu interval / limit): n_rated, coverage, median
    and p90 width [% of limit] per level; group-bootstrap CI over line.net."""
    L = c.get("line")
    if L is None or ints[levels[0]].get("line") is None:
        return _empty()
    lim = np.asarray(L["limit"], np.float64)
    rated = np.isfinite(lim) & (lim > 0)
    li = {lv: loading_intervals(c, ints, lv) for lv in levels}
    def fn(c, gw, ug):
        G, rows = ug.size, []
        gi = np.searchsorted(ug, np.asarray(L["net"])[rated])
        for lv, d in li.items():
            y, lo, hi = (d[k].to_numpy(np.float64)[rated] for k in ("y", "lo", "hi"))
            w = 100.0 * (hi - lo)
            tag = dict(stratum="loading|line", level=float(lv), note="rated rows; smax pu interval / limit")
            rows += [dict(metric="n_rated", value=float(rated.sum()), **tag),
                     dict(metric="coverage", value=_gw_mean(gw, gi, ((lo <= y) & (y <= hi)).astype(np.float64), G), **tag),
                     dict(metric="median_width_pct", value=_gw_quantile(gw, gi, w, 0.5), **tag),
                     dict(metric="p90_width_pct", value=_gw_quantile(gw, gi, w, 0.9), **tag)]
        return rows
    return group_bootstrap(fn, c, n_boot=n_boot, seed=seed, method=method, population=population)


def _harness_tables(c, ref, cal, method, pop):
    """Every harness table of one contract (state, Δθ and [PROP-2] line strata in one pass)."""
    kw = dict(n_boot=N_BOOT, seed=0, method=method, population=pop)
    ints_m, ints_s = cal2_apply(c, cal["marginal"]), cal2_apply(c, cal["simultaneous"])
    t = {"point": point_metrics(c, ref=ref, **kw), "raw": prob_metrics(c, **kw),
         "cal2_marginal": prob_metrics(c, intervals=ints_m, **kw),
         "cal2_simultaneous": prob_metrics(c, intervals=ints_s, **kw),
         "auroc": large_error_auroc(c, ref_thresholds(ref), **kw) if ref is not None else _empty(),
         "selective": selective_risk(c, **kw), "reliability": sigma_reliability(c, **kw),
         "overconfidence": overconfidence(c, **kw)}
    if pop in UQ_LOADING_POPS:                                    # [PROP-2] loading only where limits exist
        t["loading"] = loading_table(c, ints_m, **kw)
        t["cond_true_loading"] = line_conditional_coverage(c, ints_m, by="true_loading", **kw)
    else:
        t["loading"] = _note_row(method, pop, "loading|line", "pu only (no limits)")
        t["cond_true_loading"] = _empty()
    t["cond_pred_smax"] = line_conditional_coverage(c, ints_m, by="pred_smax", **kw)
    L = c.get("line")
    t["line_rows"] = {"all": 0 if L is None else int(len(L["y"])),
                      "rated": 0 if L is None else int(np.sum(np.isfinite(L["limit"]) & (np.asarray(L["limit"]) > 0)))}
    return t


def _ref_contract(method, pop, c):
    if method == "reference":
        return c
    return uq_contract("reference", pop) if ("reference", pop) in UQ_CONTRACT_PATHS else None


def sec_tables(method, pop):
    """Memoised harness tables of one method × population (contracts are loaded from the cache, never kept in memory)."""
    key = (method, pop, "primary")
    if key not in _UQ_SEC_MEMO:
        c = uq_contract(method, pop)
        _UQ_SEC_MEMO[key] = _harness_tables(c, _ref_contract(method, pop, c), UQ_CAL[method], method, pop)
        del c
    return _UQ_SEC_MEMO[key]


CORE_KEYS = ("point", "raw", "cal2_marginal", "cal2_simultaneous", "auroc", "selective", "reliability", "overconfidence")


def section_core_tables(method, populations=None):
    """Standard (state + Δθ strata) table set of one method over the report populations; tidy with CI columns lo / hi."""
    pops = populations or uq_report_pops(method)
    T = [sec_tables(method, p) for p in pops]
    out = {k: _strata_rows(_cat([t[k] for t in T]), STATE_STRATA_NAMES) for k in CORE_KEYS}
    out["qhat_over_z"] = pd.concat([qhat_over_z(UQ_CAL[method][m]) for m in ("marginal", "simultaneous")],
                                   ignore_index=True).assign(method=method)
    return out


def section_line_tables(method, populations=None):
    """[PROP-2] line block of one method: line point Δ, raw / CAL-2 line coverage + width + interval score (smax, p0, q0),
    loading-% intervals (rated rows; ieee57 / ieee57_n1 only, else 'pu only (no limits)'), conditional coverage by
    predicted |S| band (all populations) and by true-loading band (ieee57_n1; ieee57 with synthetic limits)."""
    pops = populations or uq_report_pops(method)
    T = [sec_tables(method, p) for p in pops]
    out = {k: _strata_rows(_cat([t[k] for t in T]), LINE_STRATA_NAMES)
           for k in ("point", "raw", "cal2_marginal", "cal2_simultaneous", "auroc")}
    for k in ("loading", "cond_pred_smax", "cond_true_loading"):
        out[k] = _cat([t[k] for t in T])
    out["line_rows"] = {p: t["line_rows"] for p, t in zip(pops, T)}
    return out


# ---- re-calibration with a substituted σ (same code path: sigma_floor_from_val -> cal2_fit -> cal2_apply) ----
def with_line_std(c, key):
    """Shallow copy of a contract with line['std'] = line['extra'][key] and metadata['line_sigma_method'] updated."""
    L = c.get("line")
    assert L is not None and key in L["extra"], f"contract has no line extra {key!r}"
    out = dict(c)
    out["line"] = {**L, "std": L["extra"][key]}
    out["metadata"] = {**(c.get("metadata") or {}), "line_sigma_method": f"extra.{key}"}
    return out


def with_theta_std(c, variant):
    """Shallow copy with std[:, θ] = extra['theta_std_<variant>'] (A1 θ-variant panel; 0 on known θ)."""
    k = f"theta_std_{variant}"
    assert c["std"] is not None and k in c["extra"], f"contract has no extra {k!r}"
    std = np.array(c["std"], copy=True)
    std[:, 1] = np.where(c["unknown"][:, 1], np.asarray(c["extra"][k], np.float32), 0.0)
    out = dict(c)
    out["std"] = std
    out["metadata"] = {**(c.get("metadata") or {}), "theta_primary": variant}
    return out


def recalibrate(method, transform):
    """CAL-2 for a σ-substituted contract: floors on val, marginal + simultaneous q̂ on the method's cal population.
    The cal contract reaches cal2_fit only (X4)."""
    cal_pop = CAL_POPULATION.get(method, "cal")
    floors = sigma_floor_from_val(transform(uq_contract(method, "val")))
    cal_c = transform(uq_contract(method, cal_pop))
    return {"cal_population": cal_pop, "floors": floors,
            "marginal": cal2_fit(cal_c, floors, mode="marginal"),
            "simultaneous": cal2_fit(cal_c, floors, mode="simultaneous")}


def sigma_panel(method, variants, transform_of, strata_names, populations=None, level=UQ_PRIMARY_LEVEL):
    """One CAL-2 fit per σ variant (transform_of(v) -> contract transform; None = primary UQ_CAL fit). Rows: variant ×
    population × stratum: CAL-2 coverage / width / interval score at `level` and AUROC (reference p90 large-error label).
    Returns (tidy table, {variant: cal})."""
    pops = populations or uq_report_pops(method)
    cals, rows = {}, []
    for v in variants:
        tf = transform_of(v)
        cals[v] = UQ_CAL[method] if tf is None else recalibrate(method, tf)
        for p in pops:
            c = uq_contract(method, p)
            ct = c if tf is None else tf(c)
            ref = _ref_contract(method, p, c)
            kw = dict(n_boot=N_BOOT, seed=0, method=method, population=p)
            pm = prob_metrics(ct, intervals=cal2_apply(ct, cals[v]["marginal"]), levels=(level,), **kw)
            pm = pm[pm.metric.isin(["coverage", "width", "interval_score"])]
            au = large_error_auroc(ct, ref_thresholds(ref), **kw) if ref is not None else _empty()
            rows.append(_strata_rows(_cat([pm, au]), strata_names).assign(variant=v))
            del c, ct
    return _cat(rows), cals


# ---- display / export ----
def _isnan(v):
    return v is None or (isinstance(v, (float, np.floating)) and np.isnan(v))


def _fmt(v, lo=np.nan, hi=np.nan, d=4):
    if _isnan(v):
        return "n/a"
    if np.isfinite(v) and float(v).is_integer() and 1 <= abs(v) < 1e12 and _isnan(lo) and _isnan(hi):
        return f"{int(v)}"                                             # counts (n, n_rated) without CI
    s = f"{v:.{d}g}"
    if not _isnan(lo) and not _isnan(hi) and np.isfinite(lo) and np.isfinite(hi):
        s += f" [{lo:.{d}g}, {hi:.{d}g}]"
    return s


def wide_ci(df, index_extra=(), col_extra=(), d=4):
    """Tidy -> display table: rows (population, method, stratum, *index_extra), columns metric[@level][, col_extra];
    cells 'value [lo, hi]' (2.5 / 97.5 % group bootstrap) or the row note when there is no value. IEEE57 first."""
    if df is None or not len(df):
        return pd.DataFrame()
    x = df.copy()
    note = x["note"].fillna("").astype(str) if "note" in x.columns else pd.Series("", index=x.index)
    x["cell"] = [(n if (_isnan(v) and n) else _fmt(v, lo, hi, d)) for v, lo, hi, n in zip(x.value, x.lo, x.hi, note)]
    lev = x["level"].astype(float) if "level" in x.columns else pd.Series(np.nan, index=x.index)
    col = [m if _isnan(l) else f"{m}@{int(round(100 * l))}" for m, l in zip(x.metric, lev)]
    for ce in col_extra:
        col = [f"{c} {ce}={v}" for c, v in zip(col, x[ce])]
    x["column"] = col
    idx = ["population", "method", "stratum"] + list(index_extra)
    w = x.pivot_table(index=idx, columns="column", values="cell", aggfunc="first", sort=False)
    w = w.reindex(sorted(w.index, key=lambda k: (_pop_rank(k[0]), UQ_CROSS_METHODS.index(k[1]) if k[1] in UQ_CROSS_METHODS
                                                 else 99, str(k[1]), _stratum_rank(k[2]), *[str(e) for e in k[3:]])))
    return w.rename(index=UQ_POP_LABEL, level=0)


def _stratum_rank(name):
    names = [stratum_name(st) for st in STRATA_ALL]
    return (names.index(name), "") if name in names else (len(names), str(name))


def uq_keep(section, name, df, export=True, index=False):
    """Store a section table in UQ_SECTIONS[section][name] and export it (one CSV per population, split hash in the name)."""
    UQ_SECTIONS.setdefault(section, {})[name] = df
    if export and isinstance(df, pd.DataFrame) and len(df):
        uq_export(section, name, df, index=index)
    return df


def uq_export(section, name, df, index=False):
    """CSV per section × table × population in UQ_EXPORT_DIR: {section}_{name}_{population}_{split hash}.csv (atomic)."""
    os.makedirs(UQ_EXPORT_DIR, exist_ok=True)
    groups = df.groupby("population", sort=False) if "population" in df.columns else [("all", df)]
    paths = []
    for pop, g in groups:
        p = os.path.join(UQ_EXPORT_DIR, f"{section}_{name}_{pop}_{UQ_SPLIT_HASH}.csv")
        g.to_csv(p + ".tmp", index=index)
        os.replace(p + ".tmp", p)
        paths.append(p)
    return paths


def show(title, df):
    print(f"\n--- {title} ---")
    if df is None or not len(df):
        print("(no rows)")
    else:
        display(df)


CORE_VIEWS = (
    ("point", "Point metrics: n, RMSE, MAE; ΔRMSE% vs the retrained reference (flag_gt2pct = 1 -> > 2 %)", None),
    ("raw", "Raw (pre-conformal) intervals, NLL, CRPS (deterministic: no σ -> n/a; CRPS = MAE)", None),
    ("cal2_marginal", "CAL-2 marginal (PRIMARY): coverage / width / interval score at 80 / 90 / 95 %", None),
    ("cal2_simultaneous", "CAL-2 simultaneous (secondary): joint coverage per network (N-1: base case), coverage, width",
     ("joint_coverage", "coverage", "width")),
    ("auroc", "Large-error AUROC of σ (label |err| > reference p90 |err| on the same population)", None),
    ("overconfidence", "Over-confidence |err|/σ: mean and p90 (calibrated gaussian ~ 0.80 / 1.64)", None),
)


def show_section_core(section, methods):
    """Display + store the standard state / Δθ tables of the given methods side by side (IEEE57 first)."""
    methods = [m for m in methods if uq_available(m)]
    if not methods:
        print(f"[section {section}] no method available — skipped")
        return {}
    core = {m: section_core_tables(m) for m in methods}
    for key, title, metrics in CORE_VIEWS:
        df = uq_keep(section, key, _cat([core[m][key] for m in methods]))
        if metrics is not None and len(df):
            df = df[df.metric.isin(metrics)]
        if key == "raw" and all(m in ("original", "reference", "B1") for m in methods):
            title += " — raw intervals: n/a (" + ("deterministic members" if "B1" in methods else "deterministic") + ")"
        show(title, wide_ci(df))
    for key in ("selective", "reliability"):                       # plotted below; stored + exported as tidy tables
        uq_keep(section, key, _cat([core[m][key] for m in methods]))
    qz = uq_keep(section, "qhat_over_z", _cat([core[m]["qhat_over_z"] for m in methods]))
    if len(qz):
        show("q̂ / z (pre-conformal calibration diagnostic; 1 = raw gaussian already calibrated)",
             qz.pivot_table(index=["method", "mode", "stratum"], columns="level", values="qhat_over_z", sort=False))
    for m in methods:
        section_plots(m, core=core[m])
    return core


LINE_VIEWS = (
    ("point", "[PROP-2] line point metrics (pu): RMSE / MAE; ΔRMSE% vs reference (line centre f(μ))"),
    ("raw", "[PROP-2] raw line intervals (σ-methods only), NLL, CRPS"),
    ("cal2_marginal", "[PROP-2] CAL-2 marginal line intervals (pu): coverage / width / interval score at 80 / 90 / 95 %"),
    ("cal2_simultaneous", "[PROP-2] CAL-2 simultaneous line intervals: joint coverage, coverage, width"),
)


def show_line_block(section, methods):
    """[PROP-2] display + store the line block of the given methods: line Δ, raw / CAL-2 line tables, loading-% intervals
    (rated rows, IEEE57 / N-1), conditional coverage@90 by predicted |S| band and by true-loading band (evaluation only)."""
    methods = [m for m in methods if uq_available(m)]
    if not methods:
        return {}
    lines = {m: section_line_tables(m) for m in methods}
    for key, title in LINE_VIEWS:
        df = uq_keep(section, f"line_{key}", _cat([lines[m][key] for m in methods]))
        show(title, wide_ci(df))
    ld = uq_keep(section, "line_loading", _cat([lines[m]["loading"] for m in methods]))
    show("[PROP-2] loading-% intervals (rated rows only; % of the line limit; IEEE57 ordinary limits are synthetic)",
         wide_ci(ld))
    cp = uq_keep(section, "line_cond_pred_smax", _cat([lines[m]["cond_pred_smax"] for m in methods]))
    show(f"[PROP-2] conditional coverage@{UQ_PRIMARY_LEVEL:.0%} by predicted |S|max band [pu] (evaluation only, D-P2-2)",
         wide_ci(cp, index_extra=("band",)))
    ct = uq_keep(section, "line_cond_true_loading", _cat([lines[m]["cond_true_loading"] for m in methods]))
    show(f"[PROP-2] conditional coverage@{UQ_PRIMARY_LEVEL:.0%} by TRUE loading band (rated rows; IEEE57 N-1 primary, "
         "IEEE57 ordinary on synthetic limits; evaluation only)", wide_ci(ct, index_extra=("band",)))
    UQ_SECTIONS.setdefault(section, {}).setdefault("line_rows", {}).update(
        {(m, p): v for m in methods for p, v in lines[m]["line_rows"].items()})
    return lines


# ---- standard plots ----
def _yerr(g):
    v, lo, hi = (np.asarray(g[k], np.float64) for k in ("value", "lo", "hi"))
    return np.nan_to_num(np.vstack([np.maximum(v - lo, 0.0), np.maximum(hi - v, 0.0)]), nan=0.0, posinf=0.0)


def _finish(fig, title):
    fig.suptitle(title, fontsize=10)
    fig.tight_layout()
    plt.show()
    plt.close(fig)
    return fig


def section_plots(method, populations=None, core=None):
    """Standard plots of one method: CAL-2 coverage vs nominal, width@90 per stratum (CI bars), selective-risk curves
    (RMSE of the most confident fraction / overall RMSE), σ-decile reliability (σ-methods). Returns the figures."""
    core = core or section_core_tables(method, populations)
    lab, figs = UQ_METHOD_LABEL.get(method, method), []
    cm = core["cal2_marginal"]
    pops = sorted(set(cm.population), key=_pop_rank) if len(cm) else []
    if not pops:
        return figs
    cov = cm[cm.metric == "coverage"]
    fig, axes = plt.subplots(1, len(pops), figsize=(4.0 * len(pops), 3.4), squeeze=False)
    for ax, p in zip(axes[0], pops):
        for st, g in cov[cov.population == p].groupby("stratum", sort=False):
            g = g.sort_values("level")
            ax.errorbar(g.level.astype(float), g.value, yerr=_yerr(g), marker="o", ms=3, capsize=2, lw=1, label=st)
        ax.plot([0.78, 0.97], [0.78, 0.97], "k--", lw=0.8)
        ax.set_title(pop_label(p), fontsize=9); ax.set_xlabel("nominal level"); ax.set_ylabel("CAL-2 coverage")
    axes[0][-1].legend(fontsize=6)
    figs.append(_finish(fig, f"{lab}: CAL-2 marginal coverage vs nominal"))
    wd = cm[(cm.metric == "width") & np.isclose(cm.level.astype(float), UQ_PRIMARY_LEVEL)]
    fig, axes = plt.subplots(1, len(pops), figsize=(4.0 * len(pops), 3.4), squeeze=False)
    for ax, p in zip(axes[0], pops):
        g = wd[wd.population == p]
        ax.bar(np.arange(len(g)), g.value.replace(np.inf, np.nan), yerr=_yerr(g), capsize=2)
        ax.set_xticks(np.arange(len(g))); ax.set_xticklabels(g.stratum, rotation=60, fontsize=7)
        ax.set_yscale("log"); ax.set_title(pop_label(p), fontsize=9); ax.set_ylabel(f"width@{UQ_PRIMARY_LEVEL:.0%} (pu / rad)")
    figs.append(_finish(fig, f"{lab}: CAL-2 width@{UQ_PRIMARY_LEVEL:.0%} per stratum"))
    sr = core["selective"]
    if len(sr) and not (sr["note"] == NOTE_FLAT).all():
        fig, axes = plt.subplots(1, len(pops), figsize=(4.0 * len(pops), 3.4), squeeze=False)
        for ax, p in zip(axes[0], pops):
            for st, g in sr[sr.population == p].groupby("stratum", sort=False):
                g = g.sort_values("frac")
                full = float(g.value.iloc[-1]) if len(g) else np.nan
                ax.plot(g.frac, g.value / full if full else g.value, marker=".", lw=1, label=st)
            ax.axhline(1.0, color="k", lw=0.6)
            ax.set_title(pop_label(p), fontsize=9); ax.set_xlabel("retained fraction (smallest σ)"); ax.set_ylabel("RMSE / RMSE(all)")
        axes[0][-1].legend(fontsize=6)
        figs.append(_finish(fig, f"{lab}: selective risk"))
    rl = core["reliability"]
    if len(rl) and (rl.metric == "mean_sigma").any():
        fig, axes = plt.subplots(1, len(pops), figsize=(4.0 * len(pops), 3.4), squeeze=False)
        for ax, p in zip(axes[0], pops):
            g = rl[rl.population == p]
            for st, gs in g.groupby("stratum", sort=False):
                s = gs[gs.metric == "mean_sigma"].sort_values("bin").value.to_numpy()
                r = gs[gs.metric == "RMSE"].sort_values("bin").value.to_numpy()
                if s.size and r.size:
                    ax.loglog(s, r, marker="o", ms=3, lw=1, label=st)
            lim = ax.get_xlim(); ax.plot(lim, lim, "k--", lw=0.8)
            ax.set_title(pop_label(p), fontsize=9); ax.set_xlabel("mean σ per σ-decile"); ax.set_ylabel("RMSE per σ-decile")
        axes[0][-1].legend(fontsize=6)
        figs.append(_finish(fig, f"{lab}: σ-decile reliability (raw σ)"))
    return figs


# ---- (e) cross-method tables ----
def _pick(df, stratum, metric, level=None):
    if df is None or not len(df):
        return None
    q = df[(df.stratum == stratum) & (df.metric == metric)]
    if level is not None and "level" in q.columns:
        q = q[np.isclose(q.level.astype(float), level)]
    return q.iloc[0] if len(q) else None


def _v(r, part="value"):
    return np.nan if r is None else float(r[part])


def _diff(a, b):
    """a − b for two INDEPENDENT populations; CI ≈ ±sqrt(h_a² + h_b²) from the percentile half-widths (normal approx.)."""
    if a is None or b is None:
        return np.nan, np.nan, np.nan
    v = float(a.value) - float(b.value)
    h = math.sqrt(((float(a.hi) - float(a.lo)) / 2) ** 2 + ((float(b.hi) - float(b.lo)) / 2) ** 2)
    return (v, v - h, v + h) if np.isfinite(h) else (v, np.nan, np.nan)


def _qz(method, stratum, level=UQ_PRIMARY_LEVEL):
    q = qhat_over_z(UQ_CAL[method]["marginal"])
    q = q[(q.stratum == stratum) & np.isclose(q.level.astype(float), level)]
    return float(q.qhat_over_z.iloc[0]) if len(q) else np.nan


def _cross_methods():
    return [m for m in UQ_CROSS_METHODS if uq_available(m)]


def cross_table(pop, level=UQ_PRIMARY_LEVEL):
    """Numeric cross-method state table of one population: index (stratum, method); value / _lo / _hi per metric."""
    rows = []
    for st in STATE_STRATA_NAMES:
        for m in _cross_methods():
            if (m, pop) not in UQ_CONTRACT_PATHS:
                continue
            T = sec_tables(m, pop)
            pt, cm, cs, raw, au = (T[k] for k in ("point", "cal2_marginal", "cal2_simultaneous", "raw", "auroc"))
            r = {"stratum": st, "method": m}
            for name, rr in (("dRMSE_pct", _pick(pt, st, "dRMSE_pct")), ("cov", _pick(cm, st, "coverage", level)),
                             ("width", _pick(cm, st, "width", level)), ("IS", _pick(cm, st, "interval_score", level)),
                             ("CRPS", _pick(raw, st, "CRPS")), ("NLL", _pick(raw, st, "NLL")),
                             ("AUROC", _pick(au, st, "AUROC_large_error")),
                             ("sim_joint_cov", _pick(cs, st, "joint_coverage", level))):
                r.update({name: _v(rr), f"{name}_lo": _v(rr, "lo"), f"{name}_hi": _v(rr, "hi")})
            r["flag_gt2pct"] = _v(_pick(pt, st, "flag_gt2pct"))
            r["qhat_over_z"] = _qz(m, st, level) if m in SIGMA_METHODS else np.nan
            cov_test = (_pick(sec_tables(m, "test")["cal2_marginal"], st, "coverage", level)
                        if (m, "test") in UQ_CONTRACT_PATHS else None)
            for P in ("ieee57", "ieee57_n1"):
                cp = _pick(sec_tables(m, P)["cal2_marginal"], st, "coverage", level) if (m, P) in UQ_CONTRACT_PATHS else None
                r.update(dict(zip((f"ret_{P}", f"ret_{P}_lo", f"ret_{P}_hi"), _diff(cp, cov_test))))
            # [PROP-2] headline line number: smax CAL-2 coverage retention (same for every stratum row of a method)
            ls = {P: (_pick(sec_tables(m, P)["cal2_marginal"], "smax|line", "coverage", level)
                      if (m, P) in UQ_CONTRACT_PATHS else None) for P in ("ieee57", "ieee57_n1", "test")}
            for P in ("ieee57_n1", "ieee57"):
                r.update(dict(zip((f"line_ret_{P}", f"line_ret_{P}_lo", f"line_ret_{P}_hi"), _diff(ls[P], ls["test"]))))
            rows.append(r)
    return pd.DataFrame(rows).set_index(["stratum", "method"]) if rows else pd.DataFrame()


def line_cross_table(pop, level=UQ_PRIMARY_LEVEL):
    """[PROP-2] Numeric cross-method LINE table of one population: index method; smax / p0 / q0 ΔRMSE%, CAL-2
    coverage + width (pu); loading coverage + median loading width [% of limit] (ieee57 / ieee57_n1 only); smax joint
    coverage, interval score, CRPS, AUROC; smax coverage in the true-loading band [0.95, 1.05) (ieee57_n1)."""
    rows = []
    for m in _cross_methods():
        if (m, pop) not in UQ_CONTRACT_PATHS:
            continue
        T = sec_tables(m, pop)
        r = {"method": m}
        for q in ("smax", "p0", "q0"):
            st = f"{q}|line"
            for name, rr in ((f"dRMSE_pct_{q}", _pick(T["point"], st, "dRMSE_pct")),
                             (f"cov_{q}", _pick(T["cal2_marginal"], st, "coverage", level)),
                             (f"width_{q}", _pick(T["cal2_marginal"], st, "width", level))):
                r.update({name: _v(rr), f"{name}_lo": _v(rr, "lo"), f"{name}_hi": _v(rr, "hi")})
            r[f"flag_gt2pct_{q}"] = _v(_pick(T["point"], st, "flag_gt2pct"))
        lt = T["loading"] if pop in UQ_LOADING_POPS else None
        for name, metric in (("loading_cov", "coverage"), ("loading_medw_pct", "median_width_pct")):
            rr = _pick(lt, "loading|line", metric, level)
            r.update({name: _v(rr), f"{name}_lo": _v(rr, "lo"), f"{name}_hi": _v(rr, "hi")})
        for name, key, metric in (("smax_sim_joint_cov", "cal2_simultaneous", "joint_coverage"),
                                  ("smax_IS", "cal2_marginal", "interval_score"), ("smax_CRPS", "raw", "CRPS"),
                                  ("smax_AUROC", "auroc", "AUROC_large_error")):
            rr = _pick(T[key], "smax|line", metric, level if metric in ("joint_coverage", "interval_score") else None)
            r.update({name: _v(rr), f"{name}_lo": _v(rr, "lo"), f"{name}_hi": _v(rr, "hi")})
        rr = None
        if pop == "ieee57_n1" and len(T["cond_true_loading"]):
            ct = T["cond_true_loading"]
            q = ct[(ct.stratum == "smax|line") & (ct.metric == "coverage") & (ct.band == LOADING_NEAR_LIMIT_BAND)]
            rr = q.iloc[0] if len(q) else None
        r.update({"smax_cov_nearlimit": _v(rr), "smax_cov_nearlimit_lo": _v(rr, "lo"), "smax_cov_nearlimit_hi": _v(rr, "hi")})
        rows.append(r)
    return pd.DataFrame(rows).set_index("method") if rows else pd.DataFrame()


def _ci_contains(v, lo, hi, level=UQ_PRIMARY_LEVEL, tol=DECISION_COV_TOL):
    if _isnan(v):
        return False
    if not _isnan(lo) and not _isnan(hi):
        return bool(lo <= level <= hi)
    return abs(v - level) <= tol


def _display_frame(num, groups):
    """Numeric value / _lo / _hi columns -> display strings 'v [lo, hi]' for the given metric names (others dropped)."""
    out = pd.DataFrame(index=num.index)
    nan = [np.nan] * len(num)
    for g, label in groups:
        if g in num.columns:
            out[label] = [_fmt(v, a, b) for v, a, b in zip(num[g], num[f"{g}_lo"] if f"{g}_lo" in num.columns else nan,
                                                            num[f"{g}_hi"] if f"{g}_hi" in num.columns else nan)]
    return out


CROSS_COLS = (("dRMSE_pct", "ΔRMSE%"), ("cov", "CAL-2 cov@90"), ("width", "width@90"), ("IS", "IS@90"),
              ("CRPS", "CRPS"), ("NLL", "NLL"), ("AUROC", "AUROC"), ("qhat_over_z", "q̂/z"),
              ("ret_ieee57", "retention ieee57−test"), ("ret_ieee57_n1", "retention N-1−test"),
              ("sim_joint_cov", "simultaneous cov@90"),
              ("line_ret_ieee57_n1", "[PROP-2] smax retention N-1−test"),
              ("line_ret_ieee57", "[PROP-2] smax retention ieee57−test"))
LINE_CROSS_COLS = tuple((f"{k}_{q}", f"{lab} {q}") for q in ("smax", "p0", "q0")
                        for k, lab in (("dRMSE_pct", "ΔRMSE%"), ("cov", "cov@90"), ("width", "width@90 [pu]"))) + (
    ("loading_cov", "loading cov@90"), ("loading_medw_pct", "median loading width@90 [% limit]"),
    ("smax_sim_joint_cov", "smax simultaneous cov@90"), ("smax_IS", "smax IS@90"), ("smax_CRPS", "smax CRPS"),
    ("smax_AUROC", "smax AUROC"), ("smax_cov_nearlimit", "smax cov@90, true loading [0.95, 1.05)"))


def style_cross(num, cols, flag_cols, width_col, cov_col, group_level=None, pop=None):
    """Display strings + Styler: red background for a ΔRMSE flag; bold for the best width among methods whose coverage@90
    CI contains 0.90 (per stratum block when group_level is given). Deterministic rows show AUROC / NLL / q̂/z as n/a."""
    disp = _display_frame(num, cols)
    meth = num.index.get_level_values("method") if isinstance(num.index, pd.MultiIndex) else num.index
    for c, lab in cols:
        if c in ("NLL", "AUROC", "qhat_over_z", "smax_AUROC") and lab in disp.columns:
            disp[lab] = [("n/a (deterministic)" if m in ("original", "reference") else s) for s, m in zip(disp[lab], meth)]
    if pop is not None and pop not in UQ_LOADING_POPS:
        for lab in ("loading cov@90", "median loading width@90 [% limit]"):
            if lab in disp.columns:
                disp[lab] = "pu only (no limits)"
    styles = pd.DataFrame("", index=disp.index, columns=disp.columns)
    lab_of = dict(cols)
    for fc, dc in flag_cols:
        if fc in num.columns and lab_of.get(dc) in styles.columns:
            styles.loc[(num[fc] == 1.0).to_numpy(), lab_of[dc]] = "background-color: #f4b6b6"
    if width_col in num.columns and cov_col in num.columns:
        blocks = num.groupby(level=group_level, sort=False) if group_level is not None else [(None, num)]
        for _, b in blocks:
            ok = [i for i, r in b.iterrows() if np.isfinite(r[width_col])
                  and _ci_contains(r[cov_col], r.get(f"{cov_col}_lo"), r.get(f"{cov_col}_hi"))]
            if ok:
                best = min(ok, key=lambda i: b.loc[i, width_col])
                styles.loc[[best], lab_of[width_col]] = "font-weight: bold"
    return disp, disp.style.apply(lambda _: styles, axis=None)


# ---- generated decision aid (R-S §6) ----
def _cov_tab(pop, strata, level=UQ_PRIMARY_LEVEL):
    """{(method, stratum): (cov, lo, hi, width)} from the CAL-2 marginal tables of one population."""
    out = {}
    for m in _cross_methods():
        if (m, pop) not in UQ_CONTRACT_PATHS:
            continue
        cm = sec_tables(m, pop)["cal2_marginal"]
        for st in strata:
            c, w = _pick(cm, st, "coverage", level), _pick(cm, st, "width", level)
            if c is not None:
                out[(m, st)] = (_v(c), _v(c, "lo"), _v(c, "hi"), _v(w))
    return out


def decision_checks(strata, level=UQ_PRIMARY_LEVEL):
    """Per UQ method vs the reference (constant-width CAL-2), computed only from the CAL-2 marginal tables:
    (1) width at matched coverage on the internal test (where CAL-2 matches coverage by construction): share of matched
        strata (both coverage CIs contain the level) in which the method is narrower; pass if > DECISION_MIN_WIN_FRAC;
    (2) conditional coverage across strata on IEEE57 and N-1: mean |cov − level| over strata; pass if <= reference on both;
    (3) coverage retention on IEEE57 and N-1: mean (cov_P − cov_test) over strata; pass if >= reference on both."""
    tabs = {p: _cov_tab(p, strata, level) for p in ("test", "ieee57", "ieee57_n1")}
    rows = []
    for m in [m for m in UQ_DECISION_METHODS if uq_available(m)]:
        r = {"method": m}
        t = tabs["test"]
        matched = [st for st in strata if (m, st) in t and ("reference", st) in t
                   and _ci_contains(*t[(m, st)][:3], level) and _ci_contains(*t[("reference", st)][:3], level)]
        wins = [st for st in matched if t[(m, st)][3] < t[("reference", st)][3]]
        r.update(n_matched=len(matched), n_narrower=len(wins),
                 check1_width=(None if not matched else len(wins) / len(matched) > DECISION_MIN_WIN_FRAC))
        dev, ret = {}, {}
        for P in ("ieee57", "ieee57_n1"):
            tp = tabs[P]
            for who in (m, "reference"):
                d = [abs(tp[(who, st)][0] - level) for st in strata if (who, st) in tp]
                dev[(who, P)] = float(np.mean(d)) if d else np.nan
                rr = [tp[(who, st)][0] - t[(who, st)][0] for st in strata if (who, st) in tp and (who, st) in t]
                ret[(who, P)] = float(np.mean(rr)) if rr else np.nan
            r[f"mean_abs_cov_dev_{P}"], r[f"ref_mean_abs_cov_dev_{P}"] = dev[(m, P)], dev[("reference", P)]
            r[f"mean_retention_{P}"], r[f"ref_mean_retention_{P}"] = ret[(m, P)], ret[("reference", P)]
        ok2 = [dev[(m, P)] <= dev[("reference", P)] for P in ("ieee57", "ieee57_n1")
               if np.isfinite(dev[(m, P)]) and np.isfinite(dev[("reference", P)])]
        ok3 = [ret[(m, P)] >= ret[("reference", P)] for P in ("ieee57", "ieee57_n1")
               if np.isfinite(ret[(m, P)]) and np.isfinite(ret[("reference", P)])]
        r["check2_conditional"] = all(ok2) if len(ok2) == 2 else None
        r["check3_retention"] = all(ok3) if len(ok3) == 2 else None
        cks = [r["check1_width"], r["check2_conditional"], r["check3_retention"]]
        r["verdict"] = ("does not earn its keep" if any(c is False for c in cks) else
                        "undetermined (a check has no data)" if any(c is None for c in cks) else "earns its keep")
        rows.append(r)
    return pd.DataFrame(rows).set_index("method") if rows else pd.DataFrame()


def loading_checks(level=UQ_PRIMARY_LEVEL, pop="ieee57_n1"):
    """[PROP-2] loading on IEEE57 N-1 (rated rows) vs the reference: |loading cov − level| <= reference's, and median
    loading width narrower at matched coverage (both loading coverage CIs contain the level)."""
    if not uq_available("reference") or ("reference", pop) not in UQ_CONTRACT_PATHS:
        return pd.DataFrame()
    def get(m):
        lt = sec_tables(m, pop)["loading"]
        c, w = _pick(lt, "loading|line", "coverage", level), _pick(lt, "loading|line", "median_width_pct", level)
        return (_v(c), _v(c, "lo"), _v(c, "hi"), _v(w))
    ref, rows = get("reference"), []
    for m in [m for m in UQ_DECISION_METHODS if uq_available(m) and (m, pop) in UQ_CONTRACT_PATHS]:
        g = get(m)
        matched = _ci_contains(*g[:3], level) and _ci_contains(*ref[:3], level)
        rows.append({"method": m, "loading_cov": g[0], "ref_loading_cov": ref[0], "loading_medw_pct": g[3],
                     "ref_loading_medw_pct": ref[3],
                     "loading_cov_closer_or_equal": (None if _isnan(g[0]) or _isnan(ref[0]) else
                                                     abs(g[0] - level) <= abs(ref[0] - level)),
                     "narrower_at_matched_coverage": (g[3] < ref[3]) if matched else None})
    return pd.DataFrame(rows).set_index("method") if rows else pd.DataFrame()


def _yn(x):
    return "n/a" if _isnan(x) else ("yes" if bool(x) else "no")


def decision_markdown(state, line=None, loading=None, level=UQ_PRIMARY_LEVEL):
    """(state verdict text, [PROP-2] separate line-flow view text | None). The line view never changes the state verdict."""
    head = (f"**Decision aid (generated from the CAL-2 tables at {level:.0%}; R-S §6).** A UQ method earns its keep only "
            "if it beats the reference's constant-width CAL-2 on (1) width at matched coverage (internal test, where CAL-2 "
            f"matches coverage by construction; pass if narrower in > {DECISION_MIN_WIN_FRAC:.0%} of the matched strata), "
            "(2) conditional coverage across strata (mean |coverage − level| over strata on IEEE57 and IEEE57 N-1, <= the "
            "reference's on both), (3) coverage retention (mean coverage_P − coverage_test over strata, >= the reference's "
            "on IEEE57 and N-1).\n")
    def para(df, strata_txt):
        if df is None or not len(df):
            return "_No UQ method or no reference available — nothing to compare._\n"
        out = []
        for m, r in df.iterrows():
            out.append(f"- **{UQ_METHOD_LABEL.get(m, m)}: {r['verdict']}** — (1) narrower in {int(r['n_narrower'])} of "
                       f"{int(r['n_matched'])} matched {strata_txt} strata: {_yn(r['check1_width'])}; (2) mean |cov − "
                       f"{level:.2f}| IEEE57 {r['mean_abs_cov_dev_ieee57']:.3f} (ref {r['ref_mean_abs_cov_dev_ieee57']:.3f}), "
                       f"N-1 {r['mean_abs_cov_dev_ieee57_n1']:.3f} (ref {r['ref_mean_abs_cov_dev_ieee57_n1']:.3f}): "
                       f"{_yn(r['check2_conditional'])}; (3) mean retention IEEE57 {r['mean_retention_ieee57']:+.3f} (ref "
                       f"{r['ref_mean_retention_ieee57']:+.3f}), N-1 {r['mean_retention_ieee57_n1']:+.3f} (ref "
                       f"{r['ref_mean_retention_ieee57_n1']:+.3f}): {_yn(r['check3_retention'])}.")
        return "\n".join(out) + "\n"
    txt = head + "\n**State verdict** (state + Δθ strata):\n\n" + para(state, "state")
    line_txt = None
    if line is not None:
        line_txt = ("**[PROP-2] Line-flow view** (separate; applies the same three checks to the line strata smax / p0 / "
                    "q0 and to loading on IEEE57 N-1; it does NOT change the state verdict above):\n\n" + para(line, "line"))
        if loading is not None and len(loading):
            for m, r in loading.iterrows():
                line_txt += (f"- {UQ_METHOD_LABEL.get(m, m)} loading on IEEE57 N-1 (rated rows): coverage "
                             f"{r['loading_cov']:.3f} vs reference {r['ref_loading_cov']:.3f} (closer to {level:.2f} or "
                             f"equal: {_yn(r['loading_cov_closer_or_equal'])}); median width {r['loading_medw_pct']:.3g} % "
                             f"vs {r['ref_loading_medw_pct']:.3g} % of limit (narrower at matched coverage: "
                             f"{_yn(r['narrower_at_matched_coverage'])}).\n")
    return txt, line_txt

## (a) ORIGINAL — deterministic baseline (not a UQ method)
<!-- [STSI 261026]:new cell (task 110) - section (a) banner -->

**What is compared.**
- **(a1) Published Original** — the selected run `ORIGINAL_RUN_KEY` from `best_runs_57.pkl`. It is shown first in the **unchanged** `check_hparam_results` view (external IEEE57, dangling-bus post-processing, solver baselines), which reproduces the numbers already reported. It is then evaluated in the UQ harness on **raw** outputs, with CAL-2 calibrated on the **75 clean cal networks** (`cal_clean`, S-2).
- **(a2) Retrained reference** — `uq_ref_ss42.pkl`, the comparison baseline of every ΔRMSE% in this notebook, CAL-2 on the 150 cal networks.
- **Reproducibility a1 vs a2** — per-stratum RMSE on IEEE57 with Δ%, report-only.

**Populations.** IEEE57 ordinary (**primary**) → IEEE57 N-1 (**primary**) → internal test (**preliminary**) → IEEE118 (optional).

**How to read the tables.** Every cell is `point [2.5 %, 97.5 %]` from a group bootstrap over networks (N-1: base cases). Deterministic models have no σ: raw intervals, NLL and AUROC are "n/a", CRPS = MAE, and CAL-2 widths are constant per stratum. **[PROP-2] line block:** line flows P0 / Q0 / |S|max in pu (lines only); loading = |S|max interval / line limit, only on IEEE57 and IEEE57 N-1 (rated lines; IEEE57 ordinary limits are synthetic); conditional-coverage bands are **evaluation-only** (D-P2-2), never a calibration stratum.

**S6 selection bias.** The published run was **selected** among many runs on its IEEE57 results, so its IEEE57 numbers are optimistically biased. The retrained reference was not selected on IEEE57; use it as the baseline.

In [ ]:
# ==== SECTION (a) ORIGINAL (task 110) — deterministic baseline: a1 published run, a2 retrained reference ====
# [STSI 261026]:new cell (task 110) - (a1) published check_hparam_results view (unchanged V2.7 view, external IEEE57) + harness/CAL-2 (cal_clean), (a2) retrained reference, reproducibility a1 vs a2, [PROP-2] line block for both
RUN_PUBLISHED_VIEW = True                       # the unchanged V2.7 view re-evaluates on 100 IEEE57 nets + solver baselines
PUBLISHED_VIEW_KW = dict(external_test_set=True, force_external_reeval=False,   # = Analysis cells 42 / 47 (reported numbers)
                         run_factorial_analysis=False)                          # one run: no factorial analysis
UQ_SECTIONS["a"] = {}
if "original" in UQ_RUNS and RUN_PUBLISHED_VIEW:
    print("(a1) PUBLISHED view — check_hparam_results, unchanged (dangling-bus post-processing; external IEEE57 ranks "
          f"0-{IEEE_N_RANKS - 1}). These numbers use the V2.7 evaluation path, NOT the UQ harness below.")
    with open(IEEE57_PATH, "r", encoding="utf-8") as _f:
        _pos57 = uq_ieee_positions(len(json.load(_f)["files"]))
    ieee57_test_networks = [n for _, _, n in load_dataset_subset(IEEE57_PATH, _pos57)]   # = Analysis cell 42 selection
    ref_single_ieee57, ref_batch_ieee57 = build_ref_baselines(None, test_networks=ieee57_test_networks)
    check_hparam_results([UQ_RUNS["original"]], networks=ieee57_test_networks,
                         title_prefix="Published Original — external IEEE57",
                         ref_baselines=ref_single_ieee57, ref_baselines_batch=ref_batch_ieee57, use_pnom_share=False,
                         plot_accuracy_metrics=True, plot_inference_time=True, **PUBLISHED_VIEW_KW)
    UQ_SECTIONS["a"]["published_view"] = "check_hparam_results(external IEEE57) shown"
elif "original" not in UQ_RUNS:
    print("(a1) published Original not loaded (see uq-predict-all) — published view skipped")

print("\n(a1) PUBLISHED Original in the UQ harness (raw outputs, CAL-2 on the 75 clean cal networks)  |  "
      "(a2) RETRAINED reference (CAL-2 on the 150 cal networks)")
for _m in ("original", "reference"):
    if _m in UQ_CAL:
        print(f"  {UQ_METHOD_LABEL[_m]}: CAL-2 on '{UQ_CAL[_m]['cal_population']}' "
              f"({UQ_CAL[_m]['marginal']['n_cal_groups']} cal networks)")
show_section_core("a", ["original", "reference"])

# reproducibility a1 vs a2 (report-only): per-stratum RMSE on ieee57 with Δ% (point_metrics of the Original vs the reference)
if uq_available("original") and uq_available("reference"):
    _pt = UQ_SECTIONS["a"]["point"]
    _rp = _pt[(_pt.population == "ieee57") & _pt.metric.isin(["RMSE", "dRMSE_pct"])]
    _rr = _pt[(_pt.population == "ieee57") & (_pt.method == "reference") & (_pt.metric == "RMSE")]
    _repro = (_rp[_rp.method == "original"].pivot_table(index="stratum", columns="metric", values="value", sort=False)
              .rename(columns={"RMSE": "RMSE_published", "dRMSE_pct": "dRMSE_pct_published_vs_reference"}))
    _repro.insert(1, "RMSE_reference", _rr.set_index("stratum")["value"].reindex(_repro.index))
    uq_keep("a", "reproducibility_ieee57", _repro.reset_index().assign(population="ieee57"))
    show("Reproducibility a1 (published) vs a2 (retrained reference) on IEEE57 ordinary — report-only", _repro)

display(Markdown("#### (a) [PROP-2] line block — published Original and retrained reference (no extra line diagnostics)"))
show_line_block("a", ["original", "reference"])

## (b) UQ-A1 — two-stage mean–scale heads
<!-- [STSI 261026]:new cell (task 110) - section (b) banner -->

**What is compared.** A1 against the retrained reference (same frozen backbone, so the point ΔRMSE is **0 by construction**). Raw Gaussian intervals (z·σ) against CAL-2 (marginal primary, simultaneous secondary) at 80 / 90 / 95 %; NLL, CRPS; large-error AUROC (label: |err| > the reference's p90 |err|); selective risk; σ-decile reliability; mean and p90 of |err|/σ per population (over-confidence shows as a rise from test → IEEE57 → N-1, R-A §7). **θ-variant panel:** direct / independent / comonotone θ σ, each calibrated separately with the same CAL-2 code. Selected epochs and held-out NLL curves come from `uq_artifacts` (val only).

**Populations.** IEEE57 ordinary (**primary**) → IEEE57 N-1 (**primary**) → internal test (**preliminary**) → IEEE118 (optional).

**How to read the tables.** `point [2.5 %, 97.5 %]` (group bootstrap). q̂/z = 1 means the raw Gaussian interval was already calibrated; > 1 means σ is too small. **[PROP-2] line block:** line centre f(μ) (ΔRMSE = 0); line σ = flow-scale heads, with the first-order `std_indep` baseline in a separate **line-σ panel** (each calibrated separately). Loading only on IEEE57 / N-1 (rated lines); bands are evaluation-only (D-P2-2).

In [ ]:
# ==== SECTION (b) UQ-A1 two-stage mean–scale (task 110) ====
# [STSI 261026]:new cell (task 110) - core tables (Δ expected 0, raw gaussian vs CAL-2 at 80/90/95, NLL/CRPS, AUROC, selective risk, σ reliability, |err|/σ shift), θ-variant panel, selected epochs + held-out NLL curves, [PROP-2] line block + line-σ panel (flow head vs std_indep)
UQ_SECTIONS["b"] = {}
if not uq_available("A1"):
    print("[section b] A1 not loaded — section skipped")
else:
    _a1 = UQ_RUNS["A1"]["uq_artifacts"]
    show_section_core("b", ["A1"])
    _oc = UQ_SECTIONS["b"]["overconfidence"]
    _trend = _oc[_oc.metric.isin(["mean_abs_z", "p90_abs_z"])].pivot_table(
        index=["stratum", "metric"], columns="population", values="value", sort=False)
    _trend = _trend[[p for p in ("test", "ieee57", "ieee57_n1") if p in _trend.columns]]
    show("Over-confidence shift test -> ieee57 -> ieee57_n1 (R-A §7: rising |err|/σ = σ too small under topology shift)",
         _trend)

    # θ-variant panel: direct / independent / comonotone θ σ (extra), each through the same CAL-2 code (own floors + q̂)
    _thp, _thcal = sigma_panel("A1", A1_THETA_VARIANTS, lambda v: (lambda c, _v=v: with_theta_std(c, _v)),
                               ("theta|PQ", "theta|PV"))
    uq_keep("b", "theta_variant_panel", _thp)
    UQ_SECTIONS["b"]["theta_variant_cal"] = _thcal
    show(f"θ-variant panel (primary: {_a1['theta_primary']}): CAL-2 coverage / width / IS @90 and AUROC on θ strata — "
         "one CAL-2 fit per variant", wide_ci(_thp, index_extra=("variant",)))

    # selected epochs and held-out NLL curves (stage 2, val only)
    _ep = pd.DataFrame([{"channel": ch, "selected_epoch": int(E),
                         "heldout_NLL@0": float(np.nanmean(np.asarray(_a1["history"][ch]["fold_heldout_nll"], np.float64)[:, 0])),
                         "heldout_NLL@E*": float(np.nanmean(np.asarray(_a1["history"][ch]["fold_heldout_nll"], np.float64)[:, int(E)]))}
                        for ch, E in (_a1.get("selected_epochs") or {}).items()
                        if np.asarray((_a1.get("history") or {}).get(ch, {}).get("fold_heldout_nll", []), np.float64).ndim == 2])
    uq_keep("b", "selected_epochs", _ep)
    show("A1 stage 2: selected epochs E* and held-out NLL (grouped folds on val, standardized units)", _ep)
    if len(_ep):
        _nc = 3; _nr = int(math.ceil(len(_ep) / _nc))
        _fig, _axes = plt.subplots(_nr, _nc, figsize=(4.0 * _nc, 2.6 * _nr), squeeze=False)
        for _ax, _ch in zip(_axes.ravel(), _ep.channel):
            _C = np.asarray(_a1["history"][_ch]["fold_heldout_nll"], np.float64)
            _x = np.arange(_C.shape[1])
            _ax.plot(_x, np.nanmean(_C, 0), lw=1); _ax.fill_between(_x, np.nanmin(_C, 0), np.nanmax(_C, 0), alpha=0.25)
            _ax.axvline(_a1["selected_epochs"][_ch], color="k", ls="--", lw=0.8)
            _ax.set_title(f"{_ch} (E*={_a1['selected_epochs'][_ch]})", fontsize=8); _ax.set_xlabel("epoch", fontsize=8)
        for _ax in _axes.ravel()[len(_ep):]:
            _ax.axis("off")
        _finish(_fig, "A1 stage 2: held-out NLL (fold mean, min–max band)")

    display(Markdown("#### (b) [PROP-2] line block — A1 (line centre f(μ): ΔRMSE = 0 exactly)"))
    show_line_block("b", ["A1"])
    # [PROP-2] line-σ panel: flow-scale heads (primary, UQ_CAL fit) vs std_indep baseline (own floors + own CAL-2 fit)
    _fc = tuple(_a1.get("flow_channels", ()))
    if not _fc:
        print("[PROP-2] A1 has no flow-scale heads: the primary line σ IS std_indep (panel rows coincide)")
    _lsp, _lscal = sigma_panel("A1", ("flow_head", "std_indep"),
                               lambda v: None if v == "flow_head" else (lambda c: with_line_std(c, "std_indep")),
                               LINE_STRATA_NAMES)
    uq_keep("b", "line_sigma_panel", _lsp)
    UQ_SECTIONS["b"]["line_sigma_panel_cal"] = _lscal
    show("[PROP-2] A1 line-σ panel: flow head (primary) vs std_indep (first-order baseline) — CAL-2 width / coverage / IS "
         "@90 and AUROC on smax / p0 / q0; each σ calibrated separately", wide_ci(_lsp, index_extra=("variant",)))

## (c) UQ-B1 — deep ensemble
<!-- [STSI 261026]:new cell (task 110) - section (c) banner -->

**What is compared.** The ensemble (member 0 = the retrained reference, init_seed 42) against the reference: point ΔRMSE of the ensemble mean, CAL-2 coverage / width, NLL (moment-matched Gaussian, labelled), fair ensemble CRPS, AUROC, selective risk, σ reliability. **Raw intervals: n/a (deterministic members)** — B1 intervals exist only after CAL-2. The **member-RMSE spread** is the noise yardstick for the 2 % ΔRMSE flag (Q1). `b1_diversity` and `b1_saturation` are re-run on val (copies of the B1 notebook functions, same numbers). A physics-free ensemble row is shown when `UQ_B1_PHYSFREE_GLOB` matches ≥ 2 pickles.

**Populations.** IEEE57 ordinary (**primary**) → IEEE57 N-1 (**primary**; member samples dropped there, fair CRPS stored) → internal test (**preliminary**) → IEEE118 (optional).

**How to read the tables.** `point [2.5 %, 97.5 %]` (group bootstrap). **[PROP-2] line block:** line centre f(μ̄) (D-P2-6), line σ = member flow spread; line spread/RMSE (report-only rows of `b1_diversity`) and the centre gap |f(μ̄) − mean of member flows| / σ. Loading only on IEEE57 / N-1 (rated lines); bands are evaluation-only (D-P2-2).

In [ ]:
# ==== SECTION (c) UQ-B1 deep ensemble (task 110) ====
# [STSI 261026]:new cell (task 110) - core tables (raw intervals n/a: deterministic members), member RMSE spread (Q1 noise yardstick), b1_diversity / b1_saturation on val, optional physics-free ensemble row, [PROP-2] line block + line spread/RMSE + centre gap
UQ_SECTIONS["c"] = {}


def b1_member_rmse_table(c, population):
    """Per stratum (state, Δθ, [PROP-2] line smax/p0/q0): RMSE of the ensemble mean and the spread of the member RMSEs
    (member 0 = init_seed 42 = the retrained reference). None if the contract carries no member samples."""
    if c["samples"] is None:
        return None
    rows = []
    for st in STATE_STRATA + LINE_STRATA:
        v = stratum_view(c, st)
        if v is None or v[3] is None or not len(v[0]):
            continue
        y, mu, S = np.asarray(v[0], np.float64), np.asarray(v[1], np.float64), np.asarray(v[3], np.float64)
        mr = np.sqrt(np.mean((S - y[None]) ** 2, axis=1))
        rows.append({"population": population, "stratum": stratum_name(st), "M": int(S.shape[0]),
                     "RMSE_ensemble_mean": float(np.sqrt(np.mean((mu - y) ** 2))), "RMSE_member_min": float(mr.min()),
                     "RMSE_member_median": float(np.median(mr)), "RMSE_member_max": float(mr.max()),
                     "member_range_pct_of_member0": float(100.0 * (mr.max() - mr.min()) / mr[0]) if mr[0] > 0 else np.nan,
                     "flag_threshold_pct": DRMSE_FLAG_PCT})
    return pd.DataFrame(rows)


def _b1pf_cache_path(runs, pop):
    key = json.dumps({"run_keys": [r["run_key"] for r in runs], "split_hash": UQ_SPLIT_HASH, "population": pop,
                      "population_spec": _uq_pop_spec(pop)}, sort_keys=True)
    return os.path.join(UQ_CACHE_DIR, f"B1_physfree_{pop}_{_uq_hashlib.sha1(key.encode()).hexdigest()[:10]}.pkl")


if not uq_available("B1"):
    print("[section c] B1 not loaded — section skipped")
else:
    print(f"B1 ensemble: M = {len(UQ_RUNS['B1'])} members (init_seeds {[r['init_seed'] for r in UQ_RUNS['B1']]}); "
          "centre = ensemble mean; σ = member spread (denominator M); raw intervals n/a (deterministic members)")
    show_section_core("c", ["B1"])

    # member RMSE distribution vs the ensemble mean: the noise yardstick for the Q1 ΔRMSE flag
    _mr = []
    for _p in uq_report_pops("B1"):
        _t = b1_member_rmse_table(uq_contract("B1", _p), _p)
        _mr.append(_t if _t is not None else _note_row("B1", _p, "all", "n/a (member samples dropped on this population)"))
    _mr = uq_keep("c", "member_rmse", _cat(_mr))
    show("Member RMSE spread vs the ensemble mean (Q1 noise yardstick: compare member_range_pct with the 2 % flag)", _mr)

    # val diagnostics (copies of the B1 notebook functions -> same numbers as the trainer report)
    _val_b1 = uq_contract("B1", "val")
    _div = b1_diversity(_val_b1)
    uq_keep("c", "diversity_val", _div["table"].reset_index().rename(columns={"index": "stratum"}).assign(population="val"))
    show(f"b1_diversity on val: spread/RMSE and mean pairwise member-error correlation — verdict {_div['verdict']} "
         f"({_div['n_low']} of {_div['n_strata']} strata low; {_div['rule']}; line rows report-only)", _div["table"])
    if len(UQ_RUNS["B1"]) >= 2:
        _halves = (UQ_MANIFEST["positions"]["val"][0::2], UQ_MANIFEST["positions"]["val"][1::2])
        _sat = b1_saturation(_val_b1, *_halves, n_boot=N_BOOT)
        uq_keep("c", "saturation_val", _sat["table"].reset_index().assign(population="val"))
        uq_keep("c", "saturation_gain_val", _sat["gain"].reset_index().assign(population="val"))
        show(f"b1_saturation on val (all member subsets; gain {_sat['m_from']}->{_sat['m_to']}) — verdict {_sat['verdict']}",
             _sat["table"])
        show("saturation gain (mean over strata, network-bootstrap CI)", _sat["gain"])
    del _val_b1

    # optional physics-free ensemble (separate glob; never mixed with the physics members)
    _pf_files = sorted(glob.glob(UQ_B1_PHYSFREE_GLOB))
    if len(_pf_files) >= 2:
        _pf = load_ensemble_members(_pf_files)
        assert_same_split(UQ_RUNS["B1"][0], _pf[0])
        for _p in ["val", "cal"] + uq_report_pops():
            _path = _b1pf_cache_path(_pf, _p)
            if not os.path.isfile(_path):
                _nets, _ids = uq_population(_p)
                _c = (uq_b1_predict_nosamples(_pf, _nets, _ids, _p) if _p in B1_DROP_SAMPLES_POPS
                      else ensemble_predict_uq(_pf, _nets, net_ids=_ids, population=_p))
                _c["metadata"].update(analysis_method="B1_physfree", cache_file=os.path.basename(_path),
                                      split_hash_analysis=UQ_SPLIT_HASH)
                _uq_atomic_pickle(_c, _path)
                del _c
            UQ_CONTRACT_PATHS[("B1_physfree", _p)] = _path
        _fl = sigma_floor_from_val(uq_contract("B1_physfree", "val"))
        _cal_c = uq_contract("B1_physfree", "cal")                           # X4: cal reaches cal2_fit only
        UQ_CAL["B1_physfree"] = {"cal_population": "cal", "floors": _fl,
                                 "marginal": cal2_fit(_cal_c, _fl, mode="marginal"),
                                 "simultaneous": cal2_fit(_cal_c, _fl, mode="simultaneous")}
        del _cal_c
        _rows = []
        for _m in ("B1", "B1_physfree"):
            _ct = section_core_tables(_m)
            _rows += [_ct["point"][_ct["point"].metric.isin(["RMSE", "dRMSE_pct"])],
                      _ct["cal2_marginal"][_ct["cal2_marginal"].metric.isin(["coverage", "width"])
                                           & np.isclose(_ct["cal2_marginal"].level.astype(float), UQ_PRIMARY_LEVEL)],
                      _ct["raw"][_ct["raw"].metric == "CRPS"]]
        _pfr = uq_keep("c", "physics_free_vs_physics", _cat(_rows))
        show(f"Physics-free ensemble (M={len(_pf)}) next to the physics ensemble (M={len(UQ_RUNS['B1'])}): RMSE, ΔRMSE%, "
             "CAL-2 coverage / width @90, CRPS", wide_ci(_pfr))
        _pfd = b1_diversity(uq_contract("B1_physfree", "val"))
        show("b1_diversity on val: physics vs physics-free", pd.concat(
            {"physics": _div["table"][["spread_over_RMSE", "mean_pair_err_corr", "low"]],
             "physics_free": _pfd["table"][["spread_over_RMSE", "mean_pair_err_corr", "low"]]}, axis=1))
    else:
        print(f"[B-5] no physics-free ensemble ({len(_pf_files)} pickle(s) match {UQ_B1_PHYSFREE_GLOB}; >= 2 needed)")

    display(Markdown("#### (c) [PROP-2] line block — B1 (line centre f(μ̄), D-P2-6; σ = member flow spread)"))
    show_line_block("c", ["B1"])
    _ls = _div["table"][~_div["table"]["counts_for_verdict"]][["n", "mean_sigma", "RMSE", "spread_over_RMSE",
                                                               "mean_pair_err_corr"]]
    uq_keep("c", "line_spread_val", _ls.reset_index().rename(columns={"index": "stratum"}).assign(population="val"))
    show("[PROP-2] line spread / RMSE and member-error correlation on val (report-only rows of b1_diversity)", _ls)
    _cg = []
    for _p in uq_report_pops("B1"):
        _cg.append(b1_line_center_gap(uq_contract("B1", _p)).reset_index().rename(columns={"index": "stratum"})
                   .assign(population=_p))
    _cg = uq_keep("c", "line_center_gap", _cat(_cg))
    show("[PROP-2] B1 line centre gap |f(μ̄) − mean_m F_m| / σ_line and RMSE of both centres (D-P2-6 diagnostic, report-only)",
         _cg.set_index(["population", "stratum"]))

## (d) BLL C1 / C2 — Bayesian last layer
<!-- [STSI 261026]:new cell (task 110) - section (d) banner -->

**What is compared.** C1 (closed-form posterior on the reference backbone, **point ΔRMSE = 0 exactly**) and C2 (variational BLL on its own backbone, ΔRMSE flagged above 2 %) side by side with the reference: raw Gaussian vs CAL-2 coverage / width, NLL, CRPS, AUROC, selective risk, σ reliability. Epistemic fraction of the predictive variance per population; chosen α / σ² / τ / κ / c with grid-edge flags; C1-on-C2 against C2's learned S.

**Populations.** IEEE57 ordinary (**primary**) → IEEE57 N-1 (**primary**) → internal test (**preliminary**) → IEEE118 (optional).

**How to read the tables.** `point [2.5 %, 97.5 %]` (group bootstrap). **[PROP-2] line block:** line σ = 256-sample weight-space MC; the delta method is a diagnostic (`std_delta / std`, split at true |S|max < 0.02 pu, where the flow function is non-smooth). The V–Δθ covariance is zero by C-7 (caveat printed below). Loading only on IEEE57 / N-1 (rated lines); bands are evaluation-only (D-P2-2).

In [ ]:
# ==== SECTION (d) BLL C1 / C2 (task 110) ====
# [STSI 261026]:new cell (task 110) - C1 and C2 core tables side by side, epistemic fraction per population, chosen α/σ²/τ/κ/c with edge flags, C1-on-C2 vs C2 S, C2 point Δ flag (C1 = 0 exactly), [PROP-2] line block + std_delta/std ratio + C-7 caveat
UQ_SECTIONS["d"] = {}


def bll_epistemic_fraction(c, sigma2_dtheta, n_boot=500, seed=0, method=None, population=None):
    """Mean epistemic share per stratum: var_model / σ² on the unknown state coordinates (θ: path quadratic form over
    the θ variance incl. the c·|path| term), 1 − σ²_noise / σ² on Δθ; group-bootstrap CI."""
    vm = c["extra"]["var_model"]
    def fn(c, gw, ug):
        G, rows = ug.size, []
        for st in UQ_STRATA:
            j, m = _state_rows(c, st)
            if not m.any():
                continue
            sd = np.asarray(c["std"][m, j], np.float64)
            f = np.asarray(vm[m, j], np.float64) / np.maximum(sd ** 2, 1e-300)
            rows.append(dict(stratum=stratum_name(st), metric="epistemic_frac",
                             value=_gw_mean(gw, np.searchsorted(ug, c["net"][m]), f, G)))
        e = c["edge"]
        sd = np.asarray(e["dt_std"], np.float64)
        f = np.clip(1.0 - float(sigma2_dtheta) / np.maximum(sd ** 2, 1e-300), 0.0, 1.0)
        rows.append(dict(stratum="dtheta|edge", metric="epistemic_frac", value=_gw_mean(gw, np.searchsorted(ug, e["net"]), f, G)))
        return rows
    return group_bootstrap(fn, c, n_boot=n_boot, seed=seed, method=method, population=population)


_c_methods = [m for m in ("C1", "C2") if uq_available(m)]
if not _c_methods:
    print("[section d] neither C1 nor C2 loaded — section skipped")
else:
    show_section_core("d", _c_methods)
    _pt = UQ_SECTIONS["d"]["point"]
    for _m in _c_methods:
        _d = _pt[(_pt.method == _m) & (_pt.metric == "dRMSE_pct")]
        if _m == "C1":
            print(f"C1 point ΔRMSE vs reference: max |Δ| = {np.abs(_d.value).max():.3g} % (expected exactly 0: same backbone, mean = w0)")
        else:
            _fl = _pt[(_pt.method == "C2") & (_pt.metric == "flag_gt2pct") & (_pt.value == 1.0)]
            print(f"C2 point ΔRMSE vs reference (own backbone): {len(_fl)} flagged stratum × population cells (> {DRMSE_FLAG_PCT} %)"
                  + ("" if not len(_fl) else ": " + ", ".join(f"{r.population}/{r.stratum}" for r in _fl.itertuples())))
    _ef = []
    for _m in _c_methods:
        _s2 = UQ_RUNS[_m]["uq_artifacts"]["bll_posterior"]["dtheta"]["sigma2"]
        for _p in uq_report_pops(_m):
            _ef.append(bll_epistemic_fraction(uq_contract(_m, _p), _s2, n_boot=N_BOOT, method=_m, population=_p))
    _ef = uq_keep("d", "epistemic_fraction", _cat(_ef))
    show("Epistemic fraction of the predictive variance per population (C1 / C2)", wide_ci(_ef))
    for _m in _c_methods:
        _run, _a = UQ_RUNS[_m], UQ_RUNS[_m]["uq_artifacts"]
        _h = uqc_hyper_table(_run)
        uq_keep("d", f"hyperparameters_{_m}", _h.reset_index().rename(columns={"index": "column"}))
        show(f"{_m}: chosen α / σ² / τ / κ / c (edge_of_grid = chosen value on the boundary of its grid)", _h)
        for _cn, _p in list(_a["bll_posterior"].items()) + [("theta_c", {"grid_edge_flag": _a["theta_grid_edge_flag"]})]:
            if _p["grid_edge_flag"]:
                print(f"[WARN] {_m} {_cn}: chosen hyperparameter on the EDGE of its grid — widen the grid before trusting it")
        if "c1_on_c2" in (_a.get("diagnostics") or {}):
            _cc = uqc_c1_vs_c2_table(_run)
            uq_keep("d", "c1_on_c2", _cc.reset_index().rename(columns={"index": "column"}))
            show("C1-on-C2 (closed-form S on C2's backbone) vs C2's learned S", _cc)

    display(Markdown("#### (d) [PROP-2] line block — C1 / C2 (line σ = weight-space MC; delta method = diagnostic)"))
    show_line_block("d", _c_methods)
    _dr = []
    for _m in _c_methods:
        for _p in uq_report_pops(_m):
            _dr.append(uqc_line_delta_ratio(uq_contract(_m, _p)).reset_index().assign(method=_m, population=_p))
    _dr = uq_keep("d", "line_delta_ratio", _cat(_dr))
    show(f"[PROP-2] std_delta / std (delta method over weight-space MC): median, p10, p90, split at true smax < "
         f"{UQ_C_SMALL_S} pu", _dr.set_index(["population", "method", "qty", "rows"]) if len(_dr) else _dr)
    _cov_note = uq_contract(_c_methods[0], uq_report_pops(_c_methods[0])[0])["metadata"]["line_cov_VDelta"]
    UQ_SECTIONS["d"]["c7_caveat"] = (
        f"**[PROP-2] Caveat (C-7).** The V–Δθ covariance used for the C line σ is {_cov_note}. The vmag_pq and dtheta "
        "posteriors are independent, so line-flow σ ignores any correlation between voltage and angle errors; where that "
        "correlation is material the C line intervals can be too narrow or too wide. CAL-2 corrects the average scale, "
        "not this shape error.")
    display(Markdown(UQ_SECTIONS["d"]["c7_caveat"]))

## (e) CROSS-METHOD — harness + CAL-2, IEEE57 first
<!-- [STSI 261026]:new cell (task 110) - section (e) banner -->

**What is compared.** Original (published), reference, A1, B1, C1, C2 in **one table per population**: ΔRMSE% (red = flag > 2 %), CAL-2 marginal coverage@90, width@90 (**bold** = narrowest among methods whose coverage CI contains 0.90), interval score@90, CRPS, NLL (n/a for deterministic), AUROC (n/a for deterministic), q̂/z, **coverage retention** (coverage on IEEE57 / N-1 minus internal test), simultaneous coverage@90 and **[PROP-2] line-loading coverage retention** (smax CAL-2 coverage@90). **[PROP-2]** A separate **line table per population** covers smax / p0 / q0 (ΔRMSE%, coverage, width in pu), loading coverage and median loading width (% of limit, IEEE57 / N-1 only), smax simultaneous coverage, interval score, CRPS, AUROC, and smax coverage in the true-loading band [0.95, 1.05) on N-1. Same-process **latency** follows.

**Populations.** IEEE57 ordinary (**primary**) → IEEE57 N-1 (**primary**) → internal test (**preliminary**) → IEEE118 (optional).

**How to read.** After CAL-2 every method reaches about nominal coverage on the internal test by construction, so the ranking rests on (1) width at matched coverage, (2) conditional coverage across strata, (3) coverage retention on IEEE57 / N-1 (R-S §6). The **decision aid** at the end is generated from these tables (not hand-written). The **[PROP-2] line-flow view** is a separate generated paragraph and does not change the state verdict. All tables are exported as CSV to `UQ_Results/` (under `TRAINING_RESULTS_DIR`; population and split hash in the file name).

In [ ]:
# ==== SECTION (e) CROSS-METHOD (task 110) — harness + CAL-2, IEEE57 first ====
# [STSI 261026]:new cell (task 110) - per-population cross-method state table (+ [PROP-2] smax loading coverage retention) and line table with styling, coverage-vs-nominal / width / selective-risk plots, generated decision aid + separate [PROP-2] line-flow view, CSV export
UQ_SECTIONS["e"] = {}
_cm = _cross_methods()
print(f"[section e] methods: {[UQ_METHOD_LABEL[m] for m in _cm]} | populations: {[pop_label(p) for p in uq_report_pops()]}")
for _p in uq_report_pops():
    _num = cross_table(_p)
    if not len(_num):
        continue
    uq_export("e", "cross_state", _num.reset_index().assign(population=_p))
    UQ_SECTIONS["e"].setdefault("cross_state", {})[_p] = _num
    _disp, _sty = style_cross(_num, CROSS_COLS, (("flag_gt2pct", "dRMSE_pct"),), "width", "cov", group_level="stratum", pop=_p)
    print(f"\n=== {pop_label(_p)} — state + Δθ strata (rows: stratum × method) ===")
    print("red = ΔRMSE > 2 % vs reference; bold = narrowest width@90 among methods whose coverage@90 CI contains 0.90; "
          "retention = coverage@90 on IEEE57 / N-1 minus internal test (≈CI, independent populations); "
          "[PROP-2] smax retention = line-loading coverage retention (smax CAL-2 coverage@90)")
    display(_sty)
    _lnum = line_cross_table(_p)
    if len(_lnum):
        uq_export("e", "cross_line", _lnum.reset_index().assign(population=_p))
        UQ_SECTIONS["e"].setdefault("cross_line", {})[_p] = _lnum
        _flags = tuple((f"flag_gt2pct_{q}", f"dRMSE_pct_{q}") for q in ("smax", "p0", "q0"))
        _ld, _ls = style_cross(_lnum, LINE_CROSS_COLS, _flags, "width_smax", "cov_smax", pop=_p)
        print(f"=== {pop_label(_p)} — [PROP-2] line table (smax / p0 / q0 in pu; loading on rated rows) ===")
        display(_ls)

# ---- plots: coverage vs nominal, width@90 per stratum, selective risk ----
_pops = uq_report_pops()
if _cm and _pops:
    _cov = _cat([sec_tables(m, p)["cal2_marginal"] for m in _cm for p in _pops if (m, p) in UQ_CONTRACT_PATHS])
    _cov = _cov[_cov.stratum.isin(STATE_STRATA_NAMES + ("smax|line",))]
    _strata = list(STATE_STRATA_NAMES) + ["smax|line"]
    _fig, _axes = plt.subplots(len(_pops), len(_strata), figsize=(2.3 * len(_strata), 2.2 * len(_pops)), squeeze=False)
    for _i, _p in enumerate(_pops):
        for _j, _st in enumerate(_strata):
            _ax = _axes[_i][_j]
            for _m in _cm:
                _g = _cov[(_cov.population == _p) & (_cov.stratum == _st) & (_cov.method == _m) & (_cov.metric == "coverage")]
                if len(_g):
                    _g = _g.sort_values("level")
                    _ax.errorbar(_g.level.astype(float), _g.value, yerr=_yerr(_g), marker="o", ms=2, lw=0.8, capsize=1.5, label=_m)
            _ax.plot([0.78, 0.97], [0.78, 0.97], "k--", lw=0.6)
            _ax.set_title(f"{_p} | {_st}", fontsize=7); _ax.tick_params(labelsize=6)
    _axes[0][-1].legend(fontsize=6)
    _finish(_fig, "CAL-2 marginal coverage vs nominal level (rows: populations; columns: strata incl. [PROP-2] smax)")
    _fig, _axes = plt.subplots(1, len(_pops), figsize=(6.0 * len(_pops), 3.6), squeeze=False)
    _w = _cov[(_cov.metric == "width") & np.isclose(_cov.level.astype(float), UQ_PRIMARY_LEVEL)]
    for _ax, _p in zip(_axes[0], _pops):
        _x = np.arange(len(_strata)); _bw = 0.8 / max(len(_cm), 1)
        for _k, _m in enumerate(_cm):
            _g = _w[(_w.population == _p) & (_w.method == _m)].set_index("stratum").reindex(_strata)
            _ax.bar(_x + _k * _bw, _g.value.replace(np.inf, np.nan), _bw, yerr=_yerr(_g.fillna(0.0)), capsize=1.5, label=_m)
        _ax.set_xticks(_x + 0.4 - _bw / 2); _ax.set_xticklabels(_strata, rotation=60, fontsize=7); _ax.set_yscale("log")
        _ax.set_title(pop_label(_p), fontsize=9); _ax.set_ylabel(f"CAL-2 width@{UQ_PRIMARY_LEVEL:.0%}")
    _axes[0][-1].legend(fontsize=6)
    _finish(_fig, "CAL-2 width@90 per stratum (bars: point; whiskers: group-bootstrap CI)")
    _sel = _cat([sec_tables(m, p)["selective"] for m in _cm if m in SIGMA_METHODS for p in _pops if (m, p) in UQ_CONTRACT_PATHS])
    if len(_sel):
        _fig, _axes = plt.subplots(len(_pops), len(_strata), figsize=(2.3 * len(_strata), 2.2 * len(_pops)), squeeze=False)
        for _i, _p in enumerate(_pops):
            for _j, _st in enumerate(_strata):
                _ax = _axes[_i][_j]
                for _m in _cm:
                    _g = _sel[(_sel.population == _p) & (_sel.stratum == _st) & (_sel.method == _m)].sort_values("frac")
                    if len(_g):
                        _ax.plot(_g.frac, _g.value / float(_g.value.iloc[-1]), lw=0.8, label=_m)
                _ax.axhline(1.0, color="k", lw=0.5); _ax.set_title(f"{_p} | {_st}", fontsize=7); _ax.tick_params(labelsize=6)
        _axes[0][-1].legend(fontsize=6)
        _finish(_fig, "Selective risk: RMSE of the most confident fraction / overall RMSE (σ-methods)")

# ---- generated decision aid: state verdict, then the separate [PROP-2] line-flow view ----
_dec_state = decision_checks(STATE_STRATA_NAMES)
_dec_line = decision_checks(LINE_STRATA_NAMES)
_dec_load = loading_checks()
_txt_state, _txt_line = decision_markdown(_dec_state, _dec_line, _dec_load)
uq_keep("e", "decision_state", _dec_state.reset_index().assign(population="all") if len(_dec_state) else _dec_state)
uq_keep("e", "decision_line", _dec_line.reset_index().assign(population="all") if len(_dec_line) else _dec_line)
uq_keep("e", "decision_loading", _dec_load.reset_index().assign(population="ieee57_n1") if len(_dec_load) else _dec_load)
UQ_SECTIONS["e"]["decision_text_state"], UQ_SECTIONS["e"]["decision_text_line"] = _txt_state, _txt_line
display(Markdown(_txt_state))
display(Markdown(_txt_line))
print(f"[section e] CSV export: {UQ_EXPORT_DIR}")

In [ ]:
# ==== UQ LATENCY (task 110) — ORIGIN; same process, same snapshots, median/IQR ====
# [STSI 261026]:new cell (task 110) - fair same-process latency on the same IEEE57 snapshots: batch-1 and batch-N per method, vs the reference embeddings path (=1.0) and vs the deployed model(data) path; [PROP-2] + flows + flow σ vs reference + flows (=1.0)
UQ_LAT_SNAPSHOTS, UQ_LAT_BATCH, UQ_LAT_WARMUP, UQ_LAT_REPEAT = 64, 64, 3, 20
UQ_LAT_POPULATION = "ieee57"
UQ_LAT_EXCLUDE_KEYS = ['y_ptdf', 'ptdf_line_index', 'y_line_p',
                       'bfs_edge_idx', 'bfs_signs', 'bfs_node_order',
                       'bfs_parent', 'y_delta_theta', 'bfs_path_matrix']   # batched_gnn_forward keys + per-graph P (block P below)
UQ_LAT_METHODS = ("reference", "A1", "C1", "C2", "B1")


def uq_latency_records(model, networks, n=None):
    """The first n snapshots of `networks` as uq_forward_snapshots records (data, P, unknown, y, line limit). Built once,
    shared by every method: all methods time the SAME snapshots."""
    return list(itertools.islice(uq_forward_snapshots(model, networks), int(n or UQ_LAT_SNAPSHOTS)))


def _lat_block_P(Ps):
    """Block-diagonal path matrix (sparse COO) of per-graph P [N_i, E_fwd_i]; θ = P_block @ Δθ_concat."""
    idx, val, r0, c0 = [], [], 0, 0
    for P in Ps:
        P = P.to_dense() if P.is_sparse else P
        nz = P.nonzero(as_tuple=False)
        idx.append(nz.T + torch.tensor([[r0], [c0]]))
        val.append(P[nz[:, 0], nz[:, 1]].float())
        r0 += P.size(0); c0 += P.size(1)
    return torch.sparse_coo_tensor(torch.cat(idx, 1), torch.cat(val), (r0, c0)).coalesce()


def uq_latency_batch_record(recs):
    """One batched record: Batch.from_data_list (batched_gnn_forward exclude keys), block P built once per batch, stacked
    masks; Batch offsets edge indices per graph, so line rows of the 64 graphs are concatenated correctly."""
    batch = Batch.from_data_list([r["data"] for r in recs], exclude_keys=UQ_LAT_EXCLUDE_KEYS)
    return dict(data=batch, P=_lat_block_P([r["P"] for r in recs]), unknown=unknown_mask_from_data(batch),
                y=batch.y.float(), line_limit=torch.cat([torch.as_tensor(r["line_limit"]) for r in recs]), net=0, snap=0)


def _lat_mean(npred, P, dtp, y, unk):
    pred4 = torch.stack([npred[:, 0], P @ dtp, npred[:, 1], npred[:, 2]], dim=1)
    mean = y.clone()
    mean[unk] = pred4[unk]
    return mean


def _lat_call(method, run_or_runs, rec, with_lines, roots=None):
    """Zero-argument callable for one call path of `method` on one (single or batched) record."""
    data, P, unk, y = rec["data"], rec["P"], rec["unknown"], rec["y"]
    if method == "deployed":                                       # model(data): node heads + bilinear PTDF
        model = run_or_runs["model"]
        return lambda: model(data)
    if method == "B1":
        models = [r["model"] for r in run_or_runs]
        def fn():
            outs = [mdl(data, return_embeddings=True) for mdl in models]
            mm = torch.stack([_lat_mean(o[0], P, o[3], y, unk) for o in outs])
            md = torch.stack([o[3] for o in outs])
            mu, dmu = mm.mean(0), md.mean(0)
            std = mm.var(0, unbiased=False).sqrt(); std[~unk] = 0.0
            dstd = md.var(0, unbiased=False).sqrt()
            if not with_lines:
                return mu, std, dmu, dstd
            lr = line_rows(rec); lm = lr["mask"]
            mf = line_flows_batched(mm[:, :, 0], md[:, lm], lr["ei"], lr["ea"])            # [M, L, 5] member flows
            return line_flows_batched(mu[:, 0], dmu[lm], lr["ei"], lr["ea"]), mf.var(0, unbiased=False).sqrt(), mf.mean(0)
        return fn
    model = run_or_runs["model"]
    def fwd():
        npred, hn, he, dtp = model(data, return_embeddings=True)
        return dtp, hn, he, _lat_mean(npred, P, dtp, y, unk)
    def flows(mean, dtp):
        lr = line_rows(rec)
        return line_flows_batched(mean[:, 0], dtp[lr["mask"]], lr["ei"], lr["ea"])
    if method in ("reference", "original"):
        def fn():
            dtp, _, _, mean = fwd()
            return flows(mean, dtp) if with_lines else mean
        return fn
    if method == "A1":
        a = run_or_runs["uq_artifacts"]
        def fn():
            dtp, hn, he, mean = fwd()
            r = {**rec, "mean": mean, "dt_mean": dtp, "h_nodes": hn, "h_edges_fwd": he[::2]}
            std, s_e, extra = a1_state_std(r, a)                    # 6 linear σ heads + θ variants
            if not with_lines:
                return std, s_e, extra
            return flows(mean, dtp), a1_line_std(r, a, std, s_e)    # + flows + 3 flow heads + std_indep
        return fn
    if method in ("C1", "C2"):
        a = run_or_runs["uq_artifacts"]
        post, c_th = a["bll_posterior"], a["theta_c"]
        def fn():
            dtp, hn, he, mean = fwd()
            r = {**rec, "mean": mean, "dt_mean": dtp, "h_nodes": hn, "h_edges_fwd": he[::2]}
            std, var_model, dt_var_model = bll_rec_std(r, post, c_th)   # quadratic forms (node, Δθ, θ)
            dt_std = torch.sqrt(dt_var_model + post["dtheta"]["sigma2"])
            if not with_lines:
                return std, dt_std
            return flows(mean, dtp), bll_line_std(r, post, roots)    # + flows + 256-sample weight-space MC (+ delta)
        return fn
    raise KeyError(method)


def uq_latency_batched(method, run_or_runs, data_list, batch=True, with_lines=False):
    """time_fn dict for one call path: batch=False -> batch-1 (one call processes every record one at a time);
    batch=True -> one Batch of all records. Adds per-snapshot medians (ms) and n_snapshots."""
    models = [r["model"] for r in run_or_runs] if method == "B1" else [run_or_runs["model"]]
    for mdl in models:
        mdl.eval()
    recs = [uq_latency_batch_record(data_list)] if batch else list(data_list)
    for mdl in models:
        for r in recs:
            _pad_edge_attr_for_model(mdl, r["data"])
    roots = None
    if method in ("C1", "C2"):                                     # once per run, outside the timed call (= bll_predict_uq)
        post = run_or_runs["uq_artifacts"]["bll_posterior"]
        roots = {c: bll_sqrt(post[c]["S"]) for c in ("vmag_pq", "dtheta")}
    calls = [_lat_call(method, run_or_runs, r, with_lines, roots) for r in recs]
    def fn():
        for call in calls:
            call()
    with torch.no_grad():
        t = time_fn(fn, n_warmup=UQ_LAT_WARMUP, n_repeat=UQ_LAT_REPEAT)
    n = len(data_list)
    return {**t, "n_snapshots": n, "batch": n if batch else 1,
            "median_ms_per_snapshot": t["median_ms"] / n, "iqr_ms_per_snapshot": t["iqr_ms"] / n}


def latency_table(methods, data_list, deployed_ref_model):
    """Rows: method × {b1, b<N>}. Columns: per-snapshot median / IQR [ms]; x_vs_ref_embed (reference embeddings + P@Δθ
    = 1.0); x_vs_deployed (reference model(data) incl. PTDF = 1.0; ratios < 1 come from the PTDF skip); [PROP-2]
    median_lines / IQR_lines [ms] and x_vs_ref_embed_lines (reference + flows at μ = 1.0)."""
    assert "reference" in methods, "latency baseline needs the reference"
    n, res = len(data_list), {}
    dep = {b: uq_latency_batched("deployed", {"model": deployed_ref_model}, data_list, batch=b) for b in (False, True)}
    for m, r in methods.items():
        for b in (False, True):
            res[(m, b)] = (uq_latency_batched(m, r, data_list, batch=b),
                           uq_latency_batched(m, r, data_list, batch=b, with_lines=True))
    rows = []
    for b in (False, True):
        bl = f"b{n if b else 1}"
        ref, refl = res[("reference", b)]
        for m in methods:
            t, tl = res[(m, b)]
            rows.append({"method": m, "batch": bl, "median_ms": t["median_ms_per_snapshot"], "IQR_ms": t["iqr_ms_per_snapshot"],
                         "x_vs_ref_embed": t["median_ms"] / ref["median_ms"], "x_vs_deployed": t["median_ms"] / dep[b]["median_ms"],
                         "median_lines_ms": tl["median_ms_per_snapshot"], "IQR_lines_ms": tl["iqr_ms_per_snapshot"],
                         "x_vs_ref_embed_lines": tl["median_ms"] / refl["median_ms"]})
        rows.append({"method": "deployed reference model(data)", "batch": bl, "median_ms": dep[b]["median_ms_per_snapshot"],
                     "IQR_ms": dep[b]["iqr_ms_per_snapshot"], "x_vs_ref_embed": dep[b]["median_ms"] / ref["median_ms"],
                     "x_vs_deployed": 1.0, "median_lines_ms": np.nan, "IQR_lines_ms": np.nan, "x_vs_ref_embed_lines": np.nan})
    df = pd.DataFrame(rows).set_index(["method", "batch"])
    df.attrs.update(n_snapshots=n, warmup=UQ_LAT_WARMUP, repeat=UQ_LAT_REPEAT, population=UQ_LAT_POPULATION)
    return df


UQ_LATENCY = None
_lat_methods = {m: UQ_RUNS[m] for m in UQ_LAT_METHODS if m in UQ_RUNS}
if "reference" not in _lat_methods:
    print("[uq-latency] reference not loaded — latency skipped (the baseline is the reference)")
else:
    _lat_nets, _lat_n = [], 0
    for _net in uq_population(UQ_LAT_POPULATION)[0]:                # first networks of the IEEE57 population (rank order)
        _lat_nets.append(_net); _lat_n += len(_net.snapshots)
        if _lat_n >= UQ_LAT_SNAPSHOTS:
            break
    _lat_recs = uq_latency_records(UQ_RUNS["reference"]["model"], _lat_nets, min(UQ_LAT_SNAPSHOTS, _lat_n))
    _lat_recs = _lat_recs[:UQ_LAT_BATCH] if len(_lat_recs) >= UQ_LAT_BATCH else _lat_recs
    print(f"[uq-latency] {len(_lat_recs)} {UQ_LAT_POPULATION} snapshots, same process / device (cpu), {UQ_LAT_WARMUP} warm-up "
          f"+ {UQ_LAT_REPEAT} repeats per path; methods {list(_lat_methods)}; B1 M = {len(UQ_RUNS.get('B1', []))}")
    UQ_LATENCY = latency_table(_lat_methods, _lat_recs, UQ_RUNS["reference"]["model"])
    uq_keep("e", "latency", UQ_LATENCY.reset_index().assign(population=UQ_LAT_POPULATION))
    show("Latency per snapshot (median / IQR over repeats) and multiples. Baseline 1.0 = reference embeddings path + P@Δθ; "
         "x_vs_deployed: vs the deployed model(data) incl. PTDF (ratios < 1 come from the PTDF skip); [PROP-2] *_lines: "
         "+ flows + flow σ vs reference + flows at μ", UQ_LATENCY.round(4))
    _fig, _axes = plt.subplots(1, 2, figsize=(11, 3.6))
    for _ax, _col, _t in ((_axes[0], "x_vs_ref_embed", "state path (×reference embeddings)"),
                          (_axes[1], "x_vs_ref_embed_lines", "[PROP-2] + flows + flow σ (×reference + flows)")):
        _d = UQ_LATENCY[_col].unstack("batch").dropna(how="all")
        _d.plot.bar(ax=_ax, rot=30, fontsize=8); _ax.axhline(1.0, color="k", lw=0.8); _ax.set_title(_t, fontsize=9)
        _ax.set_ylabel("latency multiple")
    _finish(_fig, "Same-process latency multiples (IEEE57 snapshots)")

## (f) Checkpoint-variant sensitivity on IEEE57 (report-only)
<!-- [STSI 261026]:new cell (task 111) - checkpoint-variant banner -->

**What is compared.** Three checkpoints of the SAME training run on the unseen IEEE57 network, for the retrained reference, every other B1 member (is42 is the reference and is not repeated) and C2 (point mean only):
- **best** — minimum of the full `val_total` (X4 / B-2). Every UQ method, contract and CAL-2 fit uses this checkpoint, unchanged.
- **final** — the last epoch.
- **post_curriculum** — minimum of `val_mse + val_dθ` at epoch ≥ `post_curriculum_start` (absent when the curriculum never ended).

**Hypothesis under test.** The reference, is45, is46 and C2 select their best epoch (38–42) before the AC-physics term starts. Selection on the validation loss may favour early epochs, while the physics terms may help generalisation. The unseen IEEE57 network decides.

**Phases.** *pre-physics* = epoch < `physics_activation_start` (60). KVL (from epoch 0) and the DC / AC-local flow terms (from 20 / 40 / 50) may already be active then. *post-curriculum* = epoch ≥ `post_curriculum_start`. Anything in between is *in-curriculum (physics on)*.

**How to read.** RMSE / MAE per stratum (6 state strata, Δθ per edge, [PROP-2] line smax / p0 / q0) per checkpoint. ΔRMSE% is **paired** against the SAME run's best checkpoint (same rows, same group-bootstrap weights; the CI is in the tidy table and the CSV). C2 rows are the point mean only: its posterior was fitted for best only, so no σ is reported. The pooled summary covers the deterministic runs (reference + B1 members; C2 excluded), grouped by the phase of the best epoch. The readout is generated counts only, with no recommendation. `ieee57_n1` is opt-in (add it to `CKPT_VARIANT_POPS`; ≈ 17 extra full N-1 passes).

**Report-only.** Nothing here changes `UQ_RUNS`, `UQ_CONTRACT_PATHS` or `UQ_CAL`. Variant contracts are cached as `ckptvar_*.pkl` in `UQ_CACHE_DIR` and tracked in `UQ_CKPT_VARIANT_PATHS` only. A checkpoint path stored on another machine is read from `TRAINING_RESULTS_DIR` under the same file name (logged once per path).

In [ ]:
# ==== UQ CHECKPOINT VARIANTS (task 111) — ORIGIN; report-only: every UQ method, contract and CAL-2 fit keeps the best (full val_total) checkpoint ====
# [STSI 261026]:new cell (task 111) - IEEE57 checkpoint-variant view: best / final / post_curriculum per run (reference, B1 members, C2 point mean), cached ckptvar_ contracts (det_predict_uq = section (a2) code path), paired ΔRMSE% vs the run's own best, epoch/phase table, pooled summary, generated readout
import hashlib, ntpath
CKPT_VARIANTS = ("best", "final", "post_curriculum")
CKPT_VARIANT_POPS = ("ieee57",)    # opt-in "ieee57_n1": ≈ (reference ieee57_n1 time in uq-predict-all) × up to 17 models
CKPT_VARIANT_C2 = True             # C2 point mean only (head = posterior mean w0); uq_bll.* keys dropped on load
UQ_CKPT_VARIANT_PATHS = {}         # (label, variant, pop) -> cache file; NEVER merged into UQ_CONTRACT_PATHS
CKPT_PHASES = ("pre-physics", "in-curriculum (physics on)", "post-curriculum", "n/a")
CKPT_SUMMARY_COLS = ("population", "variant", "stratum", "best_phase", "n_runs", "median_dRMSE_pct", "n_better",
                     "n_sig_better", "n_sig_worse")


def ckpt_variant_runs():
    """[(label, run)]: reference, B1 members (is42 deduplicated by run_key), C2 — only those loaded in UQ_RUNS."""
    cand = ([("reference", UQ_RUNS["reference"])] if "reference" in UQ_RUNS else []) + \
           [(f"B1_is{r['init_seed']}", r) for r in UQ_RUNS.get("B1", [])] + \
           ([("C2", UQ_RUNS["C2"])] if CKPT_VARIANT_C2 and "C2" in UQ_RUNS else [])
    out, seen = [], set()
    for lab, r in cand:
        if r["run_key"] not in seen:
            seen.add(r["run_key"]); out.append((lab, r))
    return out


def ckpt_variant_file(run, which):
    """best -> (None, "pickle") (the pickled model); final / post_curriculum -> resolve_checkpoint_path(stored path)."""
    if which == "best":
        return None, "pickle"
    return resolve_checkpoint_path((run.get("alt_checkpoints") or {}).get(which))


def _ckpt_stored(run, which):
    """Stored (training-machine) path of a checkpoint, or None."""
    return run.get("checkpoint_path") if which == "best" else (run.get("alt_checkpoints") or {}).get(which)


def ckpt_epoch(run, which):
    """0-based epoch: best_epoch | len(history["val_total"]) - 1 | post_curriculum_best_epoch; None if unknown or -1."""
    if which == "best":
        e = run.get("best_epoch")
    elif which == "final":
        vt = (run.get("history") or {}).get("val_total")
        e = (len(vt) - 1) if vt is not None and len(vt) else None
    else:
        e = run.get("post_curriculum_best_epoch")
    return None if e is None or int(e) < 0 else int(e)


def ckpt_phase(run, epoch):
    """"n/a" | "pre-physics" (epoch < physics_activation_start) | "post-curriculum" (epoch >= post_curriculum_start) |
    "in-curriculum (physics on)"."""
    if epoch is None:
        return "n/a"
    pa, pc = run.get("physics_activation_start"), run.get("post_curriculum_start")
    if pa is not None and epoch < int(pa):
        return "pre-physics"
    if pc is not None and epoch >= int(pc):
        return "post-curriculum"
    return "in-curriculum (physics on)"


def ckpt_epoch_table(runs):
    """Rows per label × variant: init_seed, variant, epoch (0-based), phase, val_total@epoch,
    val_mse+val_delta_theta_mse@epoch (NaN if the history key is absent), checkpoint (pickle|stored|basename_fallback|missing),
    file (basename), physics_activation_start, post_curriculum_start."""
    rows = []
    for lab, r in runs:
        h = r.get("history") or {}
        for w in CKPT_VARIANTS:
            e = ckpt_epoch(r, w)
            at = {k: (float(h[k][e]) if (h.get(k) is not None and e is not None and e < len(h[k])) else np.nan)
                  for k in ("val_total", "val_mse", "val_delta_theta_mse")}
            stored = _ckpt_stored(r, w)
            rows.append({"method": lab, "init_seed": r.get("init_seed"), "variant": w, "epoch": e, "phase": ckpt_phase(r, e),
                         "val_total": at["val_total"], "val_mse+val_delta_theta_mse": at["val_mse"] + at["val_delta_theta_mse"],
                         "checkpoint": ckpt_variant_file(r, w)[1], "file": ntpath.basename(str(stored)) if stored else "",
                         "physics_activation_start": r.get("physics_activation_start"),
                         "post_curriculum_start": r.get("post_curriculum_start")})
    return pd.DataFrame(rows)


def ckpt_variant_cache_path(label, run, which, pop, file):
    """UQ_CACHE_DIR/ckptvar_{label}_{which}_{pop}_{hash}.pkl. Key: run_key, variant, checkpoint file basename + size
    ("pickle" for best), split hash, population + spec. The ckptvar_ prefix never collides with the method caches."""
    key = json.dumps({"run_key": run["run_key"], "variant": which,
                      "file": ntpath.basename(str(file)) if file else "pickle",
                      "file_size": os.path.getsize(file) if file else None, "split_hash": UQ_SPLIT_HASH,
                      "population": pop, "population_spec": _uq_pop_spec(pop)}, sort_keys=True)
    return os.path.join(UQ_CACHE_DIR, f"ckptvar_{label}_{which}_{pop}_{hashlib.sha1(key.encode()).hexdigest()[:10]}.pkl")


def ckpt_variant_contract(label, run, which, pop):
    """(contract | None, how). Cached det_predict_uq({**run, "model": load_member_variant(run, which)}, *uq_population(pop))
    (population built only on a cache miss); metadata += analysis_method=f"ckptvar:{label}", checkpoint_variant,
    checkpoint_resolution, checkpoint_file, report_only=True; written with _uq_atomic_pickle; path stored in
    UQ_CKPT_VARIANT_PATHS. None if a non-best file is missing."""
    file, how = ckpt_variant_file(run, which)
    if which != "best" and file is None:
        return None, how
    path = ckpt_variant_cache_path(label, run, which, pop, file)
    if os.path.isfile(path):
        with open(path, "rb") as f:
            c = pickle.load(f)
    else:
        nets, ids = uq_population(pop)
        c = det_predict_uq({**run, "model": load_member_variant(run, which)}, nets, net_ids=ids, population=pop)
        c["metadata"].update(analysis_method=f"ckptvar:{label}", checkpoint_variant=which, checkpoint_resolution=how,
                             checkpoint_file=(ntpath.basename(str(file)) if file else "pickle"), report_only=True,
                             cache_file=os.path.basename(path), split_hash_analysis=UQ_SPLIT_HASH)
        os.makedirs(UQ_CACHE_DIR, exist_ok=True)
        _uq_atomic_pickle(c, path)
    UQ_CKPT_VARIANT_PATHS[(label, which, pop)] = path
    return c, how


def ckpt_reference_sanity(best, pop):
    """The reference best-variant contract must equal the section (a2) reference contract bit for bit (state mean,
    edge dt_mean, line mean) — same det_predict_uq code path, same model, same rows."""
    a = uq_contract("reference", pop)
    la, lb = a.get("line"), best.get("line")
    same = (np.array_equal(best["mean"], a["mean"], equal_nan=True)
            and np.array_equal(best["edge"]["dt_mean"], a["edge"]["dt_mean"], equal_nan=True)
            and (la is None) == (lb is None)
            and (lb is None or np.array_equal(lb["mean"], la["mean"], equal_nan=True)))
    assert same, (f"[ckpt-variants] reference best contract on {pop} differs from the section (a2) reference contract "
                  "(stale ckptvar_ cache or a different reference pickle) — delete the ckptvar_reference_best_* cache file")


def ckpt_variant_point_table(pops=None, n_boot=None):
    """Tidy point_metrics rows + columns variant, checkpoint for every run × variant × pop. best: ref=None;
    final / post_curriculum: ref=<same run's best contract> (dRMSE_pct, paired CI, flag_gt2pct). Missing variant ->
    _note_row(label, pop, "all", f"{which}: checkpoint file not found (stored path and TRAINING_RESULTS_DIR)").
    Sanity: if ("reference", pop) in UQ_CONTRACT_PATHS, the reference best contract's mean, edge dt_mean and line mean
    must np.array_equal uq_contract("reference", pop) — same code path as section (a2), else AssertionError."""
    pops = list(CKPT_VARIANT_POPS if pops is None else pops)
    n_boot = N_BOOT if n_boot is None else int(n_boot)
    out = []
    for pop in pops:
        for lab, r in ckpt_variant_runs():
            best, _ = ckpt_variant_contract(lab, r, "best", pop)
            if lab == "reference" and ("reference", pop) in UQ_CONTRACT_PATHS:
                ckpt_reference_sanity(best, pop)
            for w in CKPT_VARIANTS:
                c, how = (best, "pickle") if w == "best" else ckpt_variant_contract(lab, r, w, pop)
                if c is None:
                    out.append(_note_row(lab, pop, "all", f"{w}: checkpoint file not found (stored path and "
                                                          "TRAINING_RESULTS_DIR)").assign(variant=w, checkpoint=how))
                    continue
                out.append(point_metrics(c, ref=(None if w == "best" else best), n_boot=n_boot, method=lab,
                                         population=pop).assign(variant=w, checkpoint=how))
                del c
            del best
    return _cat(out)


def _ckpt_deterministic(label):
    return label == "reference" or str(label).startswith("B1_")


def ckpt_variant_summary(point, epochs):
    """Deterministic runs only (reference + B1 members; C2 excluded): per population × variant (final, post_curriculum)
    × stratum × best_phase (each phase + "all"): n_runs, median dRMSE_pct, n_better (value < 0), n_sig_better (hi < 0),
    n_sig_worse (lo > 0)."""
    cols = list(CKPT_SUMMARY_COLS)
    if point is None or not len(point) or "variant" not in point.columns or epochs is None or not len(epochs):
        return pd.DataFrame(columns=cols)
    phase = epochs[epochs.variant == "best"].set_index("method")["phase"].to_dict()
    p = point[(point.metric == "dRMSE_pct") & point.variant.isin(["final", "post_curriculum"])
              & point.method.map(_ckpt_deterministic).astype(bool)]
    p = p.assign(best_phase=p.method.map(phase).fillna("n/a"))
    rows = []
    for (pop, var, st), g in p.groupby(["population", "variant", "stratum"], sort=False):
        for ph in [ph for ph in CKPT_PHASES if ph in set(g.best_phase)] + ["all"]:
            gg = g if ph == "all" else g[g.best_phase == ph]
            v = gg.value.to_numpy(np.float64)
            rows.append({"population": pop, "variant": var, "stratum": st, "best_phase": ph,
                         "n_runs": int(gg.method.nunique()),
                         "median_dRMSE_pct": float(np.nanmedian(v)) if np.isfinite(v).any() else np.nan,
                         "n_better": int((v < 0).sum()), "n_sig_better": int((gg.hi.to_numpy(np.float64) < 0).sum()),
                         "n_sig_worse": int((gg.lo.to_numpy(np.float64) > 0).sum())})
    return pd.DataFrame(rows, columns=cols)


def ckpt_variant_readout(summary):
    """Markdown: generated counts per population × variant × best_phase; no recommendation."""
    head = ("**(f) Checkpoint-variant readout** (generated counts only; report-only, no recommendation). ΔRMSE% of each "
            "checkpoint vs the SAME run's best (full val_total) checkpoint, pooled over the reference and the B1 members "
            "(C2 excluded). A cell is one run × stratum; significant = the paired 95 % CI excludes 0.\n\n")
    if summary is None or not len(summary):
        return head + "_No deterministic run has a final / post_curriculum checkpoint on this machine — nothing to count._\n"
    lines = []
    for (pop, var, ph), g in summary.groupby(["population", "variant", "best_phase"], sort=False):
        lines.append(f"- **{pop} · {var}** vs best, runs whose best epoch is **{ph}** ({int(g.n_runs.max())} runs, "
                     f"{len(g)} strata): lower RMSE in {int(g.n_better.sum())} of {int(g.n_runs.sum())} run × stratum cells "
                     f"({int(g.n_sig_better.sum())} significantly lower, {int(g.n_sig_worse.sum())} significantly higher); "
                     f"median ΔRMSE% < 0 in {int((g.median_dRMSE_pct < 0).sum())} of {len(g)} strata.")
    return head + "\n".join(lines) + "\n"

In [ ]:
# ==== (f) CHECKPOINT-VARIANT VIEW (task 111) — report-only; UQ methods keep the best (full val_total) checkpoint ====
# [STSI 261026]:new cell (task 111) - run the IEEE57 checkpoint-variant view: epoch/phase/source table, RMSE/MAE + paired ΔRMSE% vs the run's own best, pooled summary by best-epoch phase, generated readout; CSVs ckpt_* in UQ_Results/
UQ_SECTIONS["ckpt"] = {}
_cv_runs = ckpt_variant_runs()
if not _cv_runs:
    print("[ckpt-variants] no reference / B1 / C2 run loaded — view skipped")
else:
    _cv_ep = uq_keep("ckpt", "epochs", ckpt_epoch_table(_cv_runs))
    show("Selected epochs per run and checkpoint (0-based) — phase, val_total, where each file was read", _cv_ep)
    _cv_pt = uq_keep("ckpt", "point", ckpt_variant_point_table())
    _w = _cv_pt[_cv_pt.metric.isin(["RMSE", "MAE", "dRMSE_pct"])].pivot_table(
        index=["population", "method", "variant"], columns=["metric", "stratum"], values="value", sort=False)
    show("IEEE57 point accuracy per checkpoint: RMSE / MAE per stratum; dRMSE_pct vs the SAME run's best "
         "(paired CI in the CSV / tidy table)", _w)
    _cv_sm = uq_keep("ckpt", "summary", ckpt_variant_summary(_cv_pt, _cv_ep))
    show("Pooled over reference + B1 members, by phase of the best epoch", _cv_sm)
    display(Markdown(ckpt_variant_readout(_cv_sm)))